# Agent 2 — Notebook 05

## Final hierarchical hybrid assessment retrieval — metadata-rescue safe

This notebook implements the final Agent 1 → Agent 2 retrieval architecture.

The design principle is:

```text
Metadata decides WHERE Agent 2 is allowed to search.
Semantic + lexical relevance decide WHICH questions are best inside that pool.
```

The implementation is deliberately general. It does not contain topic-name,
question-ID or answer-specific retrieval rules.

Key behaviour:

```text
Agent 1 detailed topic + official reference + lesson evidence
        ↓
Resolve only to exact reference / approved dotted parent
        ↓
Hard metadata filters
        ↓
Qdrant exact-reference pool
        +
PostgreSQL exact-reference pool
        ↓
deduplicate
        ↓
classify retrieval pool

metadata-trusted specific reference
        → quality checks
        → hybrid ranking only
        → semantic score cannot discard an otherwise valid exact match

broad/shared canonical reference
        → lesson-evidence MiniLM
        + direct-topic MiniLM
        + generic BM25 lexical relevance
        + metadata-strength signal
        → one stable hybrid relevance floor
        → quality checks
        ↓
near-duplicate removal
        ↓
count / primary / supporting / coverage selection
        ↓
PostgreSQL full question + mark scheme
        ↓
student and teacher outputs
```

Sibling / same-section fallback remains disabled.


### Final ownership refinement

Candidate identity is preserved per Agent 1 topic until detailed-topic
relevance has been calculated.

```text
same DB question may temporarily exist in multiple topic pools
        ↓
score independently for each detailed Agent 1 topic
        ↓
resolve ownership using detailed-topic relevance
        ↓
keep question once under its strongest topic
        ↓
other topics retain their next-best candidates
```

Short but legitimate exam prompts are also protected. A complete command-style
question is not rejected merely because it contains fewer than eight words.


### Final metadata-mismatch recovery

Some older assessment rows can contain a valid question whose stored official
reference is broader/different from the detailed Agent 1 topic. This notebook
does **not** blindly use sibling-topic fallback.

Instead, only when a topic has **zero exact/canonical candidates**, Agent 2 may
scan the same hard-filtered PostgreSQL assessment bank and perform a
metadata-mismatch rescue:

```text
approved detailed Agent 1 topic
        ↓
exact/canonical Qdrant + PostgreSQL pool
        ↓
if pool > 0 → normal hybrid path
        ↓
if pool = 0
        ↓
scan PostgreSQL under the SAME user hard filters
        ↓
rank using:
  detailed-topic MiniLM
  + lesson-evidence MiniLM
  + generic lexical overlap
        ↓
retain only strong rescue candidates
        ↓
force them through the broad hybrid relevance gate
        ↓
quality gate → ownership → selection
```

Paper, language, marks, code and visual filters remain hard constraints.

**Update:** deterministic AQA syllabus paper routing is applied before assessment retrieval.


### Paper 1 storage-code correction (v2.3.1)

Agent 2 PostgreSQL/Qdrant stores the paper family as:

```text
Paper 1 → paper_code = "1"
Paper 2 → paper_code = "2"
```

The Paper 1 programming variant is stored separately in
`programming_language`. PMT source paths may contain `1B`, but `1B` is not
used as the database/vector-store `paper_code`.

An earlier resolver incorrectly changed a frontend Paper 1 request from `1`
to `1B`, causing exact PostgreSQL/Qdrant filters to return an empty pool.
This version normalises every Paper 1 label/variant back to stored code `1`
while retaining the programming-language filter separately.

No semantic ranking, metadata-rescue, ownership, quality, selection or
cross-paper safety rule was changed.


### Notebook 07 visual-rendering integration (v2.4)

Question retrieval and assessment selection remain in Notebook 05.

After the final selected PostgreSQL question bundles are available, Notebook 05
now writes a rendering request and executes:

`07_question_visual_cropping_and_multipage_rendering.ipynb`

Notebook 07 returns only source-layout artifacts:

- verified question crop(s);
- multi-page continuation crop(s);
- safe source-page fallback(s);
- visual/dependency status;
- a render manifest.

Notebook 05 then merges the returned manifest into `final_df` and continues with
the existing grouping, package generation, student PDF, teacher PDF and audit PDF.

This isolates retrieval bugs from visual-rendering bugs.


### Notebook 07 v1.1 dependency-aware rendering (v2.5 integration)

For a visual question, final student-image order is now:

```text
verified dependency crop(s)
→ verified selected-question crop(s)
```

This prevents Figure/Table references located before the selected subquestion
from disappearing. Retrieval, ranking, topic ownership and selection are
unchanged.


### Question-focus semantic refinement

The main Notebook 04 Qdrant vector remains the enriched retrieval representation.
For direct detailed-topic similarity, near-duplicate comparison and topic ownership,
Notebook 05 now uses a second **question-text-only vector persisted in Qdrant**.
This prevents a subquestion from inheriting topic relevance only because its parent
stem/context is relevant. No hard question-focus rejection threshold is added; the
selector keeps the requested question count and prefers the stronger question-level fit.


## 2026-08-12 precision + frontend-enforcement patch

This version additionally enforces: (1) MCQ option/distractor text cannot create topic relevance; linked mark-scheme evidence is used to validate the correct option, (2) frontend minimum primary/supporting question counts are strict targets when feasible; if the quality-safe pool is short, the best safe partial assessment is returned with an explicit warning, (3) student PDF headings always show generated question number, Agent 1 topic and role plus original AQA question number, and (4) rendered visual images are globally de-duplicated in the student paper.


### Generic post-render visual sanitisation

Before PDF assembly, Notebook 05 now clamps single-page top-level questions to
their database page assignment, rejects unanchored parent-context fallbacks, and
removes tiny continuation boundary slivers. These are structural rules only; no
question/topic-specific exceptions are used.


### Resilient partial-output behaviour (v2.3)

- Detailed Agent 1 references are resolved to the nearest approved canonical Agent 2 parent without collapsing separate topics.
- A topic with no mapping or too few quality-safe questions produces a structured warning and is skipped/partially represented while other valid topics continue.
- Weak/rejected questions are not reintroduced merely to fill a quota.
- Fatal errors are reserved for cases where no trustworthy assessment can be generated (for example, database/Qdrant unavailable or zero candidates across every mapped topic).


## Retrieval design

The final retrieval policy is **hierarchical and metadata-first**.

### Stage A — authoritative pool construction

Hard metadata determines which questions may be considered:

```text
official/canonical reference
paper
minimum/maximum marks
programming language where applicable
code-question permission
visual-question permission
review status
retrieval_enabled
is_active
is_legacy
record_type
mark-scheme link
```

Qdrant and PostgreSQL exact-reference results are combined. PostgreSQL is a
complementary exact source rather than a fallback that runs only when Qdrant
returns zero results.

### Stage B — specific vs broad reference policy

A conservative structural test decides whether the canonical reference can be
trusted as sufficiently specific for the Agent 1 topic.

```text
Specific / metadata-trusted:
    exact Agent 1 reference
    no parent resolution
    canonical reference not shared by multiple approved detailed topics
    detailed topic closely aligns with the canonical concept name

Broad / shared:
    parent canonicalisation was required
    OR multiple Agent 1 topics share the canonical reference
    OR detailed topic is materially narrower than the canonical concept name
```

If uncertain, the system chooses the **broad/shared** path. This is safer than
incorrectly treating a broad syllabus area as a precise topic.

### Stage C — relevance

Specific pools use MiniLM/BM25 for **ranking**, not for rejecting an
authoritatively matched exact-reference question.

Broad/shared pools use one generalized relevance score:

```text
40% lesson-evidence semantic similarity
35% direct detailed-topic semantic similarity
20% BM25 lexical relevance
 5% metadata strength
```

Only broad/shared pools use the single hybrid relevance floor. There are no
stacked adaptive semantic and direct-concept rejection thresholds.

This avoids both failure modes:

```text
over-retrieval:  broad 3.5 pool returning a neighbouring network concept
over-filtering:  exact 3.6.1 "Define cyber security" being discarded by MiniLM
```


## Development record — issue discovered after the first retrieval test

The original Notebook 05 pipeline successfully:

```text
validated Agent 1 topics
filtered Qdrant by official AQA references
ranked candidates with MiniLM
balanced the requested marks
fetched complete PostgreSQL question and mark-scheme records
generated JSON, CSV, Markdown and TXT outputs
```

The original logic is intentionally retained in this notebook so the development
history remains visible.

### Issue observed during output evaluation

Some retrieved questions referred to:

```text
figures
flowcharts
trace tables
diagrams
tables
```

The text output correctly stated that a visual existed, but the original image was
not attached. For example, a question could say `Figure 4 shows...` while the
student could not see Figure 4.

### Impact

```text
The question may be technically relevant but not independently solvable.
The student cannot complete a trace table or interpret a diagram without the source visual.
Manual evaluation of the retrieved assessment becomes less reliable.
```

### Options considered

```text
Option A — exclude visual questions
Option B — attach the original question-page image
Option C — use only visuals completely reconstructed from extracted text
```

### Decision

**Option B was selected.**

For every selected question marked as visual, this phase renders the relevant
original Question Paper PDF page as a PNG image and attaches its path to the
assessment package.

### Why full-page rendering is used in Phase 1

Full-page rendering is more reliable than automatic cropping because it avoids
accidentally cutting off:

```text
figure labels
table headings
continuation text
question instructions
page-spanning visual context
```

### Current scope

```text
render selected visual-question pages only
use the original Question Paper PDF
save PNG images under Agent2/OUTPUT
add image paths to JSON, Markdown, TXT and CSV outputs
display rendered images inside the notebook for evaluation
do not regenerate or modify MiniLM vectors
do not change the current retrieval/ranking logic
```

### Planned future refinement

A later phase may detect the exact question bounding box and crop the page around
the question and visual. That refinement is deliberately not hidden inside this
phase, so the supervisor can see the progression from the identified issue to the
first reliable solution.


## Phase 2 development record — concept relevance and question-quality filtering

Phase 1 solved the missing-visual problem. Manual evaluation of the resulting
assessment then identified a second set of issues.

### Baseline behaviour retained

The existing pipeline already:

```text
filters by official AQA reference
ranks with MiniLM
removes duplicates
tracks selected total marks for reporting
fetches full PostgreSQL mark schemes
```

This baseline logic remains in the notebook as evidence of the original approach.

### Issues observed after Phase 1

```text
1. An official AQA subsection may still contain several different concepts.
2. A weak concept match may be selected when too many competing selection constraints are optimised together.
3. Incomplete/parser-noisy question text may enter the final assessment.
4. "minimum primary questions" does not guarantee supporting-topic coverage.
```

Examples observed during evaluation included:

```text
a structured-programming question for a tracing-focused lesson
question text ending in "Do not"
parser fragments such as "outsid" and "bo"
five primary questions despite approved supporting topics
```

### Options considered

```text
Option A — increase the MiniLM score only
Option B — use an LLM to approve every retrieved question
Option C — deterministic hybrid gate:
           enriched lesson evidence
           + semantic threshold
           + text-quality rules
           + explicit topic-distribution constraints
```

### Decision

**Option C was selected.**

It is deterministic, auditable and suitable for later controlled evaluation.

### Phase 2 implementation

```text
1. Use actual Agent 1 source chunk text when available.
2. Use lesson summary only as a clearly recorded fallback.
3. Apply a provisional semantic relevance threshold.
4. Reject incomplete or parser-noisy question text.
5. Apply the gate before question-count and coverage selection.
6. Require supporting-topic and distinct-reference coverage.
7. Export every gate decision and rejection reason.
```

### Important limitation

The semantic threshold is provisional. Notebook 06 must compare thresholds and
human relevance ratings before any production promotion.


## Phase 2 threshold-development record — fixed to adaptive

### Initial threshold approach

The first Phase 2 implementation used two fixed MiniLM thresholds:

```text
strict threshold  = 0.60
relaxed threshold = 0.55
```

The logic was:

```text
try 0.60
    ↓ insufficient candidate pool
try 0.55
    ↓ insufficient candidate pool
use a documented quality-safe rescue or stop
```

### Issue found during testing

This approach assumed that all official topics would produce similar MiniLM score
distributions. The test showed that this assumption was not reliable.

For example:

```text
tracing questions may naturally score around 0.60–0.68
array questions may have a lower but still meaningful score range
iteration questions may produce a different score distribution again
```

A fixed `0.60` threshold could therefore retain several tracing questions while
rejecting the strongest available array questions. Lowering the global threshold to
`0.55` affected every topic, including topics that already had strong candidates.

### Options considered

```text
Option A — keep lowering one global threshold
Option B — configure a manually selected threshold for each syllabus topic
Option C — calculate a threshold from each topic's own candidate-score distribution
```

### Final decision

**Option C was adopted: per-topic adaptive thresholding.**

The fixed `0.60` and `0.55` values remain in this notebook only as part of the
documented development history. They are no longer the active Phase 2 gate.

### Adaptive threshold logic

For each approved Agent 1 topic, the notebook calculates:

```text
candidate count
quality-safe candidate count
minimum score
maximum score
median score
selected percentile score
best score minus configured margin
required candidate quota for that topic
final adaptive threshold
```

The initial topic threshold is based on:

```text
max(
    selected score percentile,
    best topic score - score margin
)
```

It is then constrained by:

```text
absolute minimum floor
maximum allowed threshold
minimum candidate quota required for topic coverage
```

### Safety rules

```text
the text-quality gate remains non-adaptive
near-duplicate removal remains non-adaptive
the adaptive threshold cannot fall below the absolute floor
a lower quality-safe rescue remains separately labelled
any selected rescue candidate forces human review
```

### Why this is less overfitted

The threshold is not hard-coded for tracing, arrays or iteration. It is derived from
the current approved topic's own retrieved candidates and the current assessment
requirements.


## Final Phase 2 refinement and Phase 3 development record

### Phase 2 issue still found after adaptive thresholding

Adaptive thresholding solved the insufficient candidate-pool problem and successfully
covered all approved topics. However, the evaluation still selected a question
containing:

```text
Complete the decomposition ... boxes and.
Figure 7
```

The previous detector inspected only the final non-empty line. Because `Figure 7`
appeared after the incomplete sentence, the problem was not detected.

### Final Phase 2 decision

The completeness detector now:

```text
normalises the complete question text
splits it into sentence-like segments
ignores standalone visual labels and syllabus headings
checks every meaningful instructional segment
flags a segment ending in a dangling connector such as "and." or "or."
keeps the rule general rather than matching one observed phrase
```

The release state is also refined:

```text
ready_for_release
    only after actual Agent 1 chunk evidence has been used

evaluation_ready
    retrieval is technically valid, but the current notebook used lesson-summary fallback
    or Phase 3 structured fields still require review

needs_user_decision
    marks are outside tolerance or a semantic rescue was selected
```

### Phase 3 issue

The complete raw `marking_guidance` was readable, but the older structured fields
sometimes mixed:

```text
worked code examples into marking points
example commentary into acceptable answers
valid code into rejected answers
examiner notes into the wrong category
```

### Phase 3 decision

The raw marking guidance remains the source of truth. Phase 3 adds a new deterministic
structured view without overwriting the original database values.

It separates:

```text
marking points
acceptable answers
rejected answers
additional guidance
worked examples
assessment objectives
```

Every selected mark scheme receives:

```text
cleanup status
rule confidence
review reasons
legacy fields for comparison
cleaned Phase 3 fields
```

Low-confidence cleanup is marked for human review rather than silently promoted.


## Final retrieval and PDF evaluation refinements

A later end-to-end PDF evaluation exposed eight general issues:

```text
1. an impossible target-marks request was only detected after selection;
2. an exact official-reference match could still assess a neighbouring concept;
3. context belonging to the next subquestion could leak into the current item;
4. weak candidates could enter when the requested count exhausted the pool;
5. selected subquestions sharing one parent were displayed independently;
6. complete source pages exposed unrelated neighbouring questions;
7. student and teacher material were combined into one long audit PDF;
8. extracted mathematical notation and duplicated punctuation were noisy.
```

### Options considered

```text
Option A — add exclusions for the specific questions found during evaluation
Option B — globally raise one fixed semantic threshold
Option C — add bounded, distribution-aware and source-structure-aware rules
```

### Adopted approach

**Option C was adopted.** The implementation below uses:

```text
per-topic direct concept-fit distributions
mark-sensitive confidence margins
quality-first selection with documented shortfalls
semantic context segmentation
source-document question-boundary anchors
parent-question grouping
question-region cropping with full-page fallback
separate student, teacher and combined audit PDFs
display-only notation normalization while raw text stays unchanged
```

A weak question is never rejected because of its ID or because it contains one
hard-coded phrase. Thresholds remain configurable and must later be evaluated in
Notebook 06 across multiple topics.


## Phase 2 refinement record — issues found after the first Phase 2 run

The initial Phase 2 implementation successfully reduced the candidate pool and
introduced supporting-topic coverage. The first evaluated output showed:

```text
60 baseline unique candidates
10 Phase 2 eligible candidates
50 candidates rejected
strict semantic threshold 0.60 used without relaxation
3 primary and 2 supporting questions selected
```

This confirmed that the gate was active. However, the evaluation also revealed
four remaining problems.

### Issue 1 — near-duplicate questions

The same trace-table question appeared under two PMT topical packs. The records had
different IDs and a small extra topical heading, so exact hash/text deduplication did
not recognise them as duplicates.

### Issue 2 — generic incomplete question endings

One selected question ended with:

```text
boxes and.
```

The original quality gate detected known fragments such as `Do not`, `outsid` and
`bo`, but did not yet detect a generic sentence ending in a conjunction or
preposition.

### Issue 3 — not every approved topic was represented

Agent 1 approved three references:

```text
3.1.1 — Representing algorithms
3.2.6 — Data structures / arrays
3.2.2 — Programming concepts / iteration
```

The first Phase 2 run covered only two references because the request required a
minimum of two distinct references.

### Issue 4 — requested marks were not satisfied

The request asked for 20 marks, but the strongest set satisfying the quality and
topic constraints totalled 13 marks. This was not a retrieval crash, but the output
needed an explicit release status rather than silently appearing fully satisfied.

### Options considered

```text
Near duplicates:
A — rely only on content hashes
B — lexical comparison only
C — transparent hybrid comparison using cleaned text, token overlap and MiniLM

Topic coverage:
A — keep only a distinct-reference count
B — require every approved Agent 1 reference when the assessment size permits

Marks:
A — lower quality requirements automatically until the target is reached
B — return the best safe assessment and clearly require a user decision
```

### Decisions

```text
Near duplicates → Option C
Topic coverage  → cover every approved reference by default
Marks handling  → return best safe set and flag "needs_user_decision"
```

### Refinement added in this notebook

```text
1. Remove PMT headings and common formatting noise before duplicate comparison.
2. Detect exact cleaned-text matches.
3. Detect near duplicates using:
   - lexical sequence similarity
   - token Jaccard overlap
   - MiniLM question-to-question similarity
   - same-mark safeguard
4. Detect generic incomplete endings such as "and.", "or.", "to." and "for.".
5. Require all approved official references when the question count permits.
6. Separate technical notebook completion from assessment release readiness.
7. Export duplicate and release-readiness evidence for audit.
```

### Remaining limitation

The current test still uses lesson-summary fallback because actual Agent 1 chunk text
was not supplied. The retrieval code supports real chunk text, but this must be tested
during Streamlit integration.


### Final generalisation review

The first refinement used the observed `boxes and.` output as an example. The final
implementation does **not** keep a dedicated production rule for that phrase.

Instead, it applies a general sentence-completeness rule only when:

```text
the final non-empty line is short
the line does not end with a question mark
the final meaningful token is a clearly incomplete connector:
and / or / the / a / an
```

This reduces overfitting and lowers the risk of rejecting valid questions that
naturally contain words such as `for`, `with`, `by`, `to` or `of`.


## 1. Install dependencies


In [ ]:
# PERFORMANCE: dependencies are intentionally NOT installed during every notebook run.
# Install them once in the project virtual environment, for example:
# pip install "sentence-transformers>=3.0,<6" "qdrant-client>=1.12,<2" "sqlalchemy>=2.0" \
#     "psycopg[binary]>=3.1" "pymupdf>=1.24,<2" pandas numpy torch python-dotenv \
#     "nbconvert>=7,<8" "reportlab>=4.0"
print("Runtime dependency installation skipped; using the active virtual environment.")


## 2. Configuration and Agent 1 sample input

Replace `AGENT1_TOPIC_OUTPUT` with the actual list returned by Agent 1.

The sample below follows the same structure shown in the Agent 1 interface.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import subprocess
import sys
import time
import uuid

import fitz
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import Image as IPythonImage, display
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from sqlalchemy import (
    MetaData,
    Table,
    create_engine,
    select,
    text,
)
from sqlalchemy.engine import Engine
from sqlalchemy.orm import Session


cwd = Path.cwd().resolve()

PROJECT_ROOT = (
    cwd.parent
    if cwd.name.lower() in {"notebooks", "notebook"}
    else cwd
)

OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
MODEL_CACHE_DIR = PROJECT_ROOT / "cache" / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")


DATABASE_URL = os.getenv(
    "AGENT2_DATABASE_URL",
    "",
).strip()

if not DATABASE_URL:
    raise RuntimeError(
        "AGENT2_DATABASE_URL is missing from Agent2/.env"
    )


QDRANT_URL = os.getenv(
    "QDRANT_URL",
    "http://localhost:6333",
).strip()

QDRANT_API_KEY = (
    os.getenv("QDRANT_API_KEY", "").strip()
    or None
)

AGENT1_COLLECTION = os.getenv(
    "QDRANT_COLLECTION",
    "aqa_gcse_computer_science_8525",
).strip()

AGENT2_COLLECTION = (
    os.getenv(
        "AGENT2_QDRANT_COLLECTION",
        "",
    ).strip()
    or f"{AGENT1_COLLECTION}_questions"
)

# Dedicated Qdrant semantic view for the actual assessed question text.
# The main Agent 2 collection keeps the richer Notebook 04 retrieval vector
# (topic/subtopic/question/context). This second collection stores only the
# question-text MiniLM vector so parent context cannot dominate detailed-topic
# ownership or final selection. It is persistent Qdrant storage, not a file cache.
AGENT2_QUESTION_FOCUS_COLLECTION = (
    os.getenv(
        "AGENT2_QUESTION_FOCUS_QDRANT_COLLECTION",
        "",
    ).strip()
    or f"{AGENT2_COLLECTION}_question_focus"
)

# Dedicated Qdrant semantic view for the assessed instruction/skill text.
# This is derived deterministically from each subquestion and stored by
# content hash so the embedding is persistent and reusable rather than a
# local/file cache.
AGENT2_ASSESSED_SKILL_COLLECTION = (
    os.getenv(
        "AGENT2_ASSESSED_SKILL_QDRANT_COLLECTION",
        "",
    ).strip()
    or f"{AGENT2_COLLECTION}_assessed_skill"
)

QDRANT_TIMEOUT_SECONDS = int(
    os.getenv(
        "QDRANT_TIMEOUT_SECONDS",
        "30",
    )
)

raw_threshold = os.getenv(
    "QDRANT_SCORE_THRESHOLD",
    "",
).strip()

QDRANT_SCORE_THRESHOLD = (
    float(raw_threshold)
    if raw_threshold
    else None
)


MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

EXPECTED_VECTOR_SIZE = 384
EXPECTED_QDRANT_POINTS = 820

SPECIFICATION_CODE = "8525"
SPECIFICATION_VERSION = (
    "first_teaching_2020_last_exams_2026"
)

SYLLABUS_PAPER_ROUTING_VERSION = (
    "aqa-8525-direct-paper-routing-v1.0.0"
)

# Agent 2 stores the paper FAMILY in paper_code:
#   Paper 1 -> "1"
#   Paper 2 -> "2"
#
# The Paper 1 language/variant is stored separately in
# programming_language (for the current KB, Python).
#
# A PMT source URL/file may contain "1B", but that is NOT the value stored in
# assessment_topical_topics.paper_code or in the Qdrant paper_code payload.
KB_PAPER_CODE_STORAGE_VERSION = (
    "paper-family-1-2-language-separate-v1.0.0"
)

RETRIEVAL_VERSION = (
    "agent2-official-topic-retrieval-v2.10.0-phase5-evaluation-lock"
)

CANDIDATES_PER_TOPIC = 20
FALLBACK_CANDIDATES_PER_TOPIC = 10

# Exact-reference PostgreSQL is now used as a complementary source,
# not only as a fallback after an empty Qdrant response. This prevents
# valid PMT questions from being hidden by a small Qdrant top-k window.
POSTGRES_EXACT_SCAN_ROWS = 500
POSTGRES_EXACT_CANDIDATES_PER_TOPIC = 120

TOPIC_HANDOFF_CONSOLIDATION_VERSION = (
    "agent1-agent2-canonical-parent-per-topic-v2.9.0-syllabus-paper-routing"
)

# FINAL RETRIEVAL POLICY:
# Do not borrow questions from sibling syllabus references in the same section.
# Detailed Agent 1 -> canonical parent resolution is still allowed earlier in the pipeline.
ALLOW_SECTION_FALLBACK = False
STORE_RETRIEVAL_LOGS = True



# ---------------------------------------------------------
# GENERIC METADATA-MISMATCH RESCUE
# ---------------------------------------------------------
# This is NOT sibling-topic fallback. It runs only when the topic has zero
# exact/canonical candidates and keeps every user-selected hard filter intact.
ENABLE_METADATA_MISMATCH_RESCUE = True
METADATA_MISMATCH_RESCUE_VERSION = (
    "postgres-hard-filtered-detailed-topic-rescue-v1.0.0"
)

# Current AQA bank is small enough for a complete hard-filtered PostgreSQL scan.
POSTGRES_METADATA_RESCUE_SCAN_ROWS = 2000
METADATA_RESCUE_CANDIDATES_PER_TOPIC = 30

# Generic rescue ranking:
#   direct detailed-topic MiniLM + lesson-evidence MiniLM + lexical coverage.
METADATA_RESCUE_DIRECT_TOPIC_WEIGHT = 0.55
METADATA_RESCUE_EVIDENCE_WEIGHT = 0.30
METADATA_RESCUE_LEXICAL_WEIGHT = 0.15

# Conservative pre-filter. Final acceptance still happens later through the
# normal broad/shared hybrid relevance + question-quality gates.
METADATA_RESCUE_MIN_SCORE = 0.38
METADATA_RESCUE_MIN_DIRECT_TOPIC_SCORE = 0.32

# Rescue metadata is deliberately weak because the stored official reference
# did not match the requested detailed topic.
METADATA_RESCUE_METADATA_STRENGTH = 0.35

# ---------------------------------------------------------
# FINAL HIERARCHICAL HYBRID RETRIEVAL POLICY
# ---------------------------------------------------------
HYBRID_RETRIEVAL_POLICY_VERSION = (
    "metadata-specific-vs-broad-enriched-bm25-child-direct-evidence-v1.5.0"
)

# Conservative classification. When the detailed Agent 1 label is not
# strongly aligned with the canonical concept, the safer broad/shared path
# is used.
SPECIFIC_CANONICAL_NAME_ALIGNMENT_MIN = 0.75

# One stable relevance floor is used ONLY for broad/shared canonical pools.
BROAD_HYBRID_RELEVANCE_FLOOR = 0.30

# Relevance weights. These are generic and sum to 1.0.
HYBRID_EVIDENCE_SEMANTIC_WEIGHT = 0.40
HYBRID_DIRECT_TOPIC_WEIGHT = 0.35
HYBRID_BM25_WEIGHT = 0.20
HYBRID_METADATA_WEIGHT = 0.05

BM25_K1 = 1.5
BM25_B = 0.75
BM25_QUERY_VERSION = "agent1-topic-concepts-evidence-v1.0.0"
QUESTION_FOCUS_VECTOR_VERSION = "question-text-minilm-qdrant-v1.0.0"
ASSESSED_SKILL_VECTOR_VERSION = "assessed-instruction-minilm-qdrant-v1.0.0"
CONTEXT_GAP_VERSION = "same-topic-query-enriched-vs-question-only-v1.0.0"
GENERAL_QUALITY_GATE_VERSION = "general-parser-noise-v2.0.0"
FINAL_RELEVANCE_VERIFIER_VERSION = "assessed-skill-child-direct-evidence-v1.2.0"
CHILD_METADATA_SCOPE_VERSION = "parent-inherited-child-direct-evidence-v1.1.0"


# ---------------------------------------------------------
# FINAL CROSS-TOPIC QUESTION OWNERSHIP POLICY
# ---------------------------------------------------------
QUESTION_OWNERSHIP_VERSION = (
    "detailed-topic-ownership-child-direct-evidence-v2.1.0"
)

# Ownership is intentionally driven mostly by the detailed Agent 1 topic,
# not by the broader lesson evidence.
OWNERSHIP_DIRECT_TOPIC_WEIGHT = 0.55
OWNERSHIP_BM25_WEIGHT = 0.25
OWNERSHIP_EVIDENCE_WEIGHT = 0.15
OWNERSHIP_METADATA_WEIGHT = 0.05

# ---------------------------------------------------------
# GENERIC FINAL-PRECISION POLICIES
# ---------------------------------------------------------
# These rules are topic-agnostic. They use the current Agent 1 handoff,
# canonical syllabus metadata and candidate content only.

EXACT_DETECTED_CONCEPT_GATE_VERSION = (
    "agent2-detailed-topic-over-broad-reference-v1.0.0"
)
ENABLE_EXACT_DETECTED_CONCEPT_GATE = True
EXACT_CONCEPT_CANONICAL_TOLERANCE = 0.05

SUBCONCEPT_DIVERSITY_VERSION = (
    "agent2-assessed-skill-semantic-novelty-v1.0.0"
)
SUBCONCEPT_DIVERSITY_WEIGHT = 0.08

ROLE_ALLOCATION_VERSION = (
    "agent2-primary-supporting-question-mark-allocation-v1.0.0"
)

PDF_QUESTION_ANNOTATION_VERSION = (
    "agent2-generated-question-topic-role-source-label-v1.0.0"
)

# ---------------------------------------------------------
# FINAL GENERIC ROLE / ASSESSED-SKILL REFINEMENT
# ---------------------------------------------------------
# These are global role/content policies. They never inspect a syllabus topic
# name, question ID, array/search/iteration keyword list, or transcript-specific
# exception.

# When both Primary and Supporting roles are present, aim for half of the
# selected questions/marks to come from Primary material. This is a preference
# after relevance, hard filters and coverage; it never rescues a weak candidate.
DEFAULT_PRIMARY_PREFERRED_SHARE = 0.50

# If one physical question is a credible match for both a Primary and a
# Supporting Agent 1 topic, Primary may win only when its ownership score is
# genuinely close to the best credible ownership score.
PRIMARY_OWNERSHIP_TIE_MARGIN = 0.04

# Final selection explicitly rewards the student's actual answer-demand fit and
# penalises context-dependent matches. These are global weights, not
# topic-specific thresholds.
ASSESSED_TASK_DIRECTNESS_SELECTION_WEIGHT = 0.25
CONTEXT_DEPENDENCY_SELECTION_PENALTY = 0.08

FINAL_ASSESSED_TASK_OWNERSHIP_VERSION = (
    "agent2-answer-demand-directness-v2.0.0"
)

OWNERSHIP_LEXICAL_NOISE_FIX_VERSION = (
    "agent2-ownership-lexical-noise-v1.0.0"
)

VISUAL_POST_RENDER_SANITIZATION_VERSION = (
    "agent2-visual-post-render-sanitization-v1.1.0-self-contained-no-dependency"
)

# A continuation sliver smaller than this fraction of the source-page height is
# not useful student content and is removed before PDF assembly.
VISUAL_MIN_CONTINUATION_HEIGHT_RATIO = 0.05
FINAL_PRIMARY_BALANCE_VERSION = (
    "agent2-primary-balance-role-aware-v2.0.0"
)

# ---------------------------------------------------------
# FINAL LOCK-CANDIDATE REFINEMENTS
# ---------------------------------------------------------
# All rules below are generic. They depend only on the current Agent 1
# handoff, the request and the candidate content.

SHARED_ROLE_ALLOCATION_VERSION = (
    "agent2-shared-role-allocation-v1.0.0"
)

SEMANTIC_RECALL_QUERY_VERSION = (
    "agent2-agent1-evidence-semantic-recall-v1.0.0"
)

FINAL_ASSESSED_TASK_GATE_VERSION = (
    "agent2-final-assessed-task-adaptive-anchor-v1.1.0"
)

# No high fixed direct-topic threshold is used here. A semantic-only question
# is compared with explicit direct-task anchors from the SAME approved topic.
# This single tolerance is global and topic-agnostic.
FINAL_ASSESSED_TASK_ANCHOR_TOLERANCE = 0.05

# A short question can still be a complete AQA exam prompt, for example:
# "Define the term cyber security."
ABSOLUTE_MIN_QUESTION_WORD_COUNT = 3

# ---------------------------------------------------------
# Resilient partial-output policy
# ---------------------------------------------------------
# Expected data limitations (an unmapped topic, too few quality-safe
# questions for one topic, infeasible coverage/marks) should not abort
# the whole notebook when other valid topics/questions are available.
ALLOW_PARTIAL_TOPIC_OUTPUT = True

# Structured messages are added throughout the notebook and included in
# the exported JSON/release-readiness payload for the frontend to show.
AGENT2_USER_MESSAGES: list[dict[str, Any]] = []
AGENT2_RUN_STATUS = "success"

AGENT2_NO_SAFE_CANDIDATES = False
AGENT2_NO_RETRIEVAL_CANDIDATES = False
AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = False

# Deterministic AQA specification paper-routing state.
AGENT2_SYLLABUS_PAPER_BLOCKED = False
AGENT2_SYLLABUS_PAPER_CHECK: dict[str, Any] = {}
SYLLABUS_UNCLASSIFIED_TOPIC_INDEXES: set[int] = set()

# Topics that were intentionally skipped because the requested paper
# does not contain that topic in the current assessment bank.
PAPER_MISMATCH_TOPIC_INDEXES: set[int] = set()

# User-selected paper/language filters are hard constraints. They are
# never relaxed to fill a quota; incompatible topics are warned and skipped.
USER_HARD_FILTERS_STRICT = True


def paper_family(value: Any) -> str | None:
    text_value = str(value or "").strip().upper()
    if not text_value:
        return None
    if text_value in {"1", "1A", "1B", "1C", "PAPER 1", "PAPER1"}:
        return "1"
    if text_value in {"2", "PAPER 2", "PAPER2"}:
        return "2"
    return text_value


def paper_label(value: Any) -> str:
    family = paper_family(value)
    if family == "1":
        return "Paper 1"
    if family == "2":
        return "Paper 2"
    return str(value or "the requested paper")


# Populated once from PostgreSQL through SyllabusStore after the shared
# Agent 2 PostgreSQL engine is created. All later paper routing is in-memory.
SYLLABUS_PAPER_FAMILY_BY_REFERENCE: dict[str, str] = {}


def syllabus_paper_family_for_reference(
    reference: Any,
) -> str | None:
    """
    Return the AQA 8525 paper that directly assesses an official reference.

    Paper ownership is loaded once per notebook run from PostgreSQL through
    SyllabusStore, then resolved from the run-local lookup below. No database
    query is performed per topic/question.
    """
    cleaned = str(reference or "").strip()
    if not cleaned:
        return None

    return SYLLABUS_PAPER_FAMILY_BY_REFERENCE.get(
        cleaned
    )


def syllabus_paper_label_for_reference(
    reference: Any,
) -> str:
    family = syllabus_paper_family_for_reference(reference)
    if family == "1":
        return "Paper 1"
    if family == "2":
        return "Paper 2"
    return "Unclassified"


def paper_code_matches(candidate_code: Any, requested_code: Any) -> bool:
    if requested_code is None or str(requested_code).strip() == "":
        return True
    candidate_text = str(candidate_code or "").strip().upper()
    requested_text = str(requested_code or "").strip().upper()
    if requested_text in {"1", "PAPER 1", "PAPER1"}:
        return paper_family(candidate_text) == "1"
    if requested_text in {"2", "PAPER 2", "PAPER2"}:
        return paper_family(candidate_text) == "2"
    return candidate_text == requested_text


def knowledge_base_paper_code(
    requested_code: Any = None,
    programming_language: Any = None,
) -> str | None:
    """
    Resolve the UI paper selection to the exact ``paper_code`` value stored
    in Agent 2 PostgreSQL/Qdrant.

    Agent 2 stores paper families, not PMT filename variants:

        Paper 1 / 1 / 1A / 1B / 1C  -> "1"
        Paper 2 / 2                  -> "2"

    Paper 1 programming language is stored independently in the
    ``programming_language`` field. For the current knowledge base the
    supported Paper 1 language is Python.

    Therefore a source PDF/URL containing ``1B`` must NOT cause the database
    paper filter itself to use ``1B``.
    """
    if requested_code is None and "request" in globals():
        requested_code = request.get("paper_code")

    if programming_language is None and "request" in globals():
        programming_language = request.get(
            "programming_language"
        )

    raw = str(
        requested_code or ""
    ).strip().upper()

    if not raw:
        return None

    family = paper_family(raw)

    if family == "2":
        return "2"

    if family == "1":
        language = str(
            programming_language or ""
        ).strip().casefold()

        # Paper 1 variant/language is a SEPARATE metadata field.
        # Automatic/blank and Python both use stored paper_code "1".
        if not language or language == "python":
            return "1"

        # Do not silently substitute another language variant.
        return None

    # Preserve unknown explicit values so downstream diagnostics remain
    # transparent rather than silently widening the request.
    return raw


def requested_paper_debug_label() -> str:
    raw = (
        request.get("paper_code")
        if "request" in globals()
        else None
    )
    resolved = knowledge_base_paper_code()
    language = (
        request.get("programming_language")
        if "request" in globals()
        else None
    )

    return (
        f"{paper_label(raw)} -> stored KB paper_code "
        f"{resolved or 'unsupported/unresolved'}"
        + (
            f" | language={language}"
            if language
            else ""
        )
    )


def should_apply_programming_language_filter() -> bool:
    language = str(request.get("programming_language") or "").strip() if "request" in globals() else ""
    if not language:
        return False
    requested_paper_family = paper_family(request.get("paper_code")) if "request" in globals() else None
    # AQA Paper 2 is not programming-language variant specific.
    # When no paper is selected, do not accidentally remove Paper 2.
    return requested_paper_family == "1"


def add_agent2_user_message(
    *,
    level: str,
    code: str,
    message: str,
    topic: str | None = None,
    requested: int | None = None,
    available: int | None = None,
    details: dict[str, Any] | None = None,
) -> None:
    global AGENT2_RUN_STATUS

    record = {
        "level": str(level),
        "code": str(code),
        "message": str(message),
        "topic": topic,
        "requested": requested,
        "available": available,
        "details": details or {},
    }
    AGENT2_USER_MESSAGES.append(record)

    if str(level).lower() in {"warning", "error"}:
        AGENT2_RUN_STATUS = "partial_success"


def print_agent2_user_messages(title: str = "AGENT 2 NOTICE") -> None:
    if not AGENT2_USER_MESSAGES:
        return

    print("\n" + title)
    print("-" * len(title))
    for item in AGENT2_USER_MESSAGES:
        topic_text = (
            f" [{item['topic']}]"
            if item.get("topic")
            else ""
        )
        print(
            f"{str(item['level']).upper()}{topic_text}: "
            f"{item['message']}"
        )


# ---------------------------------------------------------
# JSON-safe serialization helper
# ---------------------------------------------------------
# Defined early because selection summaries and user-friendly partial-run
# messages need it before the final export cells execute.
def json_safe(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(value, (bool, int, str)):
        return value

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, uuid.UUID):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]

    if hasattr(value, "item"):
        try:
            return json_safe(value.item())
        except (TypeError, ValueError):
            pass

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return str(value)


PRIMARY_ROLE_WEIGHT = 1.00
SUPPORTING_ROLE_WEIGHT = 0.65

EXACT_REFERENCE_BONUS = 0.15
SECTION_FALLBACK_PENALTY = 0.12
HUMAN_CORRECTED_BONUS = 0.02

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ---------------------------------------------------------
# Phase 2 — concept relevance and question-quality gate
# ---------------------------------------------------------
ENABLE_PHASE2_GATE = True
PHASE2_VERSION = "agent2-hierarchical-hybrid-relevance-v2.3.0-enriched-bm25"

# Historical values used in the initial Phase 2 test.
# They are retained for documentation only and are not used
# by the final adaptive gate.
LEGACY_FIXED_STRICT_THRESHOLD = 0.60
LEGACY_FIXED_RELAXED_THRESHOLD = 0.55
LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED = False

# Final Phase 2 approach: calculate a separate threshold for
# each approved Agent 1 topic.
ENABLE_ADAPTIVE_SEMANTIC_THRESHOLD = False
ADAPTIVE_THRESHOLD_VERSION = (
    "agent2-per-topic-adaptive-threshold-v1.0.0"
)

# Topic score distribution controls
ADAPTIVE_SCORE_PERCENTILE = 60.0
ADAPTIVE_TOP_SCORE_MARGIN = 0.10

# Safety bounds
ADAPTIVE_ABSOLUTE_MINIMUM_SCORE = 0.45
ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD = 0.70

# Small numerical tolerance when matching a quota boundary.
ADAPTIVE_THRESHOLD_EPSILON = 1e-9

ENABLE_QUESTION_TEXT_QUALITY_GATE = True
MIN_QUESTION_WORD_COUNT = 8

# Streamlit can pass actual Module 2 chunk text here.
AGENT1_SOURCE_CHUNK_TEXTS: dict[int, str] = {}
MAX_QUERY_EVIDENCE_CHARACTERS = 4000

# ---------------------------------------------------------
# Final Phase 2 and Phase 3 release controls
# ---------------------------------------------------------
REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE = True

PHASE3_VERSION = (
    "agent2-mark-scheme-structured-cleanup-v1.1.0-block-aware"
)
ENABLE_PHASE3_MARK_SCHEME_CLEANUP = True
PHASE3_MIN_RULE_CONFIDENCE = 0.70
PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH = True
PHASE3_STORE_AUDIT_RECORDS = True
PHASE3_BLOCK_PARSER_VERSION = "agent2-mark-scheme-block-parser-v1.0.0"
PHASE3_MERGE_WRAPPED_LINES = True
PHASE3_SPLIT_INLINE_MARKERS = True
PHASE3_REVIEW_ON_AMBIGUOUS_BLOCKS = True


# ---------------------------------------------------------
# Phase 2 refinement — duplicate, coverage and release rules
# ---------------------------------------------------------
PHASE2_REFINEMENT_VERSION = (
    "agent2-concept-quality-gate-v1.1.0"
)

ENABLE_NEAR_DUPLICATE_GATE = True

# A duplicate must normally have the same mark allocation.
REQUIRE_SAME_MARKS_FOR_NEAR_DUPLICATES = True

NEAR_DUPLICATE_LEXICAL_THRESHOLD = 0.92
NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD = 0.80
NEAR_DUPLICATE_SEMANTIC_THRESHOLD = 0.985

# When the number of requested questions allows it, require at
# least one selected question from every approved Agent 1 reference.
COVER_ALL_APPROVED_TOPICS = True

# A set outside this tolerance is returned for evaluation but is
# not marked ready for student release without a user decision.
ALLOW_BEST_AVAILABLE_ASSESSMENT = True

# ---------------------------------------------------------
# Phase 2 resilience — quality-safe topic coverage rescue
# ---------------------------------------------------------
ENABLE_QUALITY_SAFE_TOPIC_RESCUE = True

# Rescue is only considered after both the strict and relaxed
# thresholds fail to produce a sufficient topic-balanced pool.
MIN_TOPIC_RESCUE_SEMANTIC_SCORE = 0.40

# Rescued candidates must still pass every text-quality rule.
# They are marked for human review and prevent automatic release.

# ---------------------------------------------------------
# Phase 1 — original question-page image rendering
# ---------------------------------------------------------
ENABLE_VISUAL_PAGE_RENDERING = True

# Full pages remain available for audit, while the student-facing
# output uses a general question-region crop when a reliable boundary
# can be found. The crop falls back safely to the complete page.
RENDER_FULL_QUESTION_PAGES = True
ENABLE_QUESTION_REGION_CROPPING = True

VISUAL_RENDER_DPI = 180
DISPLAY_RENDERED_IMAGES_IN_NOTEBOOK = False

# Notebook 02 stores PDF page numbers as human-readable,
# one-based page numbers.
DATABASE_PAGE_NUMBERS_ARE_ONE_BASED = True

VISUAL_RENDERING_VERSION = (
    "agent2-notebook7-visual-rendering-v3.0.0"
)



# Notebook 07 is a rendering-only worker. Retrieval/ranking remain here.
NOTEBOOK7_RENDERER_FILE = (
    "07_question_visual_cropping_and_multipage_rendering.ipynb"
)
NOTEBOOK7_RENDERER_INTEGRATION_VERSION = (
    "agent2-notebook05-to-notebook07-render-contract-v1.1.0"
)
NOTEBOOK7_EXECUTION_TIMEOUT_SECONDS = 900

# PERFORMANCE: expensive diagnostics/full-PDF fallback are opt-in.
# These flags do not change retrieval/ranking scores; they only avoid
# development-only scans and unnecessary renderer page searches.
ENABLE_QDRANT_DIAGNOSTIC = (
    os.getenv("AGENT2_ENABLE_QDRANT_DIAGNOSTIC", "0")
    .strip()
    .lower()
    in {"1", "true", "yes", "on"}
)
ALLOW_NOTEBOOK7_FULL_PDF_FALLBACK = (
    os.getenv("AGENT2_ALLOW_FULL_PDF_RENDER_SEARCH", "0")
    .strip()
    .lower()
    in {"1", "true", "yes", "on"}
)

# ---------------------------------------------------------
# Generalized post-Phase-2 concept-fit refinement
# ---------------------------------------------------------
ENABLE_DIRECT_CONCEPT_FIT_GATE = False
CONCEPT_FIT_VERSION = (
    "agent2-direct-concept-fit-v1.0.0-adaptive"
)

# A new threshold is calculated independently for every approved
# topic. These are evaluation parameters, not topic-specific rules.
CONCEPT_FIT_SCORE_PERCENTILE = 35.0
CONCEPT_FIT_TOP_SCORE_MARGIN = 0.18
CONCEPT_FIT_ABSOLUTE_FLOOR = 0.30
CONCEPT_FIT_MAXIMUM_THRESHOLD = 0.68
CONCEPT_FIT_EPSILON = 1e-9

# Larger questions need stronger evidence because one weak extended
# item can otherwise dominate the assessment marks.
CONCEPT_FIT_MARGIN_START_MARKS = 4
CONCEPT_FIT_MARGIN_PER_EXTRA_MARK = 0.01
CONCEPT_FIT_MAXIMUM_MARK_MARGIN = 0.06

# The existing evidence-aware MiniLM score remains dominant. The new
# direct score asks whether the question itself assesses the concise
# approved concept, rather than only sharing the same PMT subsection.
CONCEPT_FIT_EVIDENCE_WEIGHT = 0.60
CONCEPT_FIT_DIRECT_WEIGHT = 0.40

# A rescue is permitted only to retain an otherwise missing approved
# reference. It is never used merely to fill the requested count and
# always blocks automatic release.
ENABLE_CONCEPT_FIT_REFERENCE_RESCUE = False
MIN_CONCEPT_FIT_RESCUE_SCORE = 0.28

# ---------------------------------------------------------
# Generalized context-boundary refinement
# ---------------------------------------------------------
ENABLE_CONTEXT_REFINEMENT = True
CONTEXT_REFINEMENT_VERSION = (
    "agent2-context-segmentation-v1.0.0"
)
CONTEXT_RELEVANCE_ABSOLUTE_FLOOR = 0.34
SELF_CONTAINED_CONTEXT_MINIMUM_SCORE = 0.52
CONTEXT_REFERENCE_MISMATCH_OVERRIDE_SCORE = 0.72

# ---------------------------------------------------------
# Source-region rendering and grouping
# ---------------------------------------------------------
QUESTION_REGION_CROP_VERSION = (
    "agent2-notebook7-question-region-crop-v3.0.0"
)

SOURCE_PAGE_MATCH_VERSION = (
    "agent2-question-to-source-page-match-v1.0.0"
)

# Search only a bounded local window around parser-assigned pages.
# This corrects small page drift without scanning unrelated papers.
SOURCE_PAGE_SEARCH_RADIUS = 4

# A selected source page must contain the current question itself.
# These checks are independent from topic scoring and retrieval.
SOURCE_PAGE_MINIMUM_TOKEN_COVERAGE = 0.52
SOURCE_PAGE_STRONG_TOKEN_COVERAGE = 0.68
SOURCE_PAGE_MINIMUM_ANCHOR_SCORE = 0.72
SOURCE_PAGE_MINIMUM_MEANINGFUL_TOKENS = 5

# Context may extend the crop upward only when it shares an explicit
# Figure/Table/Diagram reference with the selected question.
SOURCE_CONTEXT_MAXIMUM_LOOKBACK_POINTS = 360.0

REQUIRED_DEPENDENCY_VERSION = (
    "agent2-required-visual-context-dependency-v1.0.0"
)

# Search only near the already verified current-question page.
DEPENDENCY_PAGE_SEARCH_RADIUS = 6

# A Figure/Table/Diagram label must appear as a short standalone
# source line. An inline phrase such as "shown in Figure 2" does
# not count as the dependency itself.
DEPENDENCY_STANDALONE_LABEL_MAXIMUM_WORDS = 5

# Structural verification works with vector drawings, embedded
# images, tables, answer boxes and structured pseudocode text.
DEPENDENCY_MINIMUM_DRAWINGS = 2
DEPENDENCY_MINIMUM_STRUCTURED_LINES = 2
DEPENDENCY_REGION_MAXIMUM_HEIGHT_POINTS = 520.0

SPATIAL_DEPENDENCY_BINDING_VERSION = (
    "agent2-label-to-structure-spatial-binding-v1.0.0"
)

# Structure must occur inside the label's own bounded region.
# The region is never artificially extended through the next
# question or visual label merely to meet a minimum crop height.
DEPENDENCY_REGION_TOP_GAP_POINTS = 2.0
DEPENDENCY_REGION_BOTTOM_GAP_POINTS = 4.0
DEPENDENCY_MINIMUM_SPATIAL_HEIGHT_POINTS = 18.0

# Optional support for documents that place a figure caption below
# the figure. The above-label region is also bounded by the previous
# question/label/section boundary, so neighbouring content cannot
# satisfy the dependency.
ALLOW_STRUCTURE_IMMEDIATELY_ABOVE_LABEL = True
DEPENDENCY_ABOVE_REGION_MAXIMUM_HEIGHT_POINTS = 300.0

# General structured-response wording. This is evaluated against the
# current question text and is independent of stale PostgreSQL flags.
STRUCTURED_RESPONSE_DEPENDENCY_VERSION = (
    "agent2-structured-answer-layout-v1.0.0"
)

CROP_ANCHOR_MINIMUM_SIMILARITY = 0.72
CROP_TOP_PADDING_POINTS = 24.0
CROP_BOTTOM_PADDING_POINTS = 14.0
CROP_HORIZONTAL_PADDING_POINTS = 18.0
CROP_MINIMUM_HEIGHT_POINTS = 72.0

GROUP_SHARED_PARENT_SUBQUESTIONS = True
QUESTION_GROUPING_VERSION = (
    "agent2-source-parent-grouping-v1.0.0"
)

# ---------------------------------------------------------
# PDF outputs
# ---------------------------------------------------------
GENERATE_SEPARATE_STUDENT_AND_TEACHER_PDFS = True
PDF_EXPORT_VERSION = (
    "agent2-student-teacher-pdf-v3.0.0-annotated"
)


AGENT1_TOPIC_OUTPUT = [
    {
        "topic": (
            "Algorithm tracing and program execution"
        ),
        "role": "primary",
        "official_reference": "3.1.1",
        "confidence": 0.8172,
        "ranking_score": 0.6502,
        "source_chunks": [3, 4, 5, 6, 7, 8],
    },
    {
        "topic": (
            "One- and two-dimensional arrays"
        ),
        "role": "supporting",
        "official_reference": "3.2.6",
        "confidence": 0.8477,
        "ranking_score": 0.6109,
        "source_chunks": [2, 3, 5, 6, 10],
    },
    {
        "topic": "Iteration",
        "role": "supporting",
        "official_reference": "3.2.2",
        "confidence": 0.5943,
        "ranking_score": 0.3768,
        "source_chunks": [3, 8],
    },
]


LESSON_SUMMARY = (
    "The lesson focused on tracing algorithms and following "
    "program execution. It also covered one- and "
    "two-dimensional arrays and iteration."
)


ASSESSMENT_REQUEST = {
    "number_of_questions": 5,
    "minimum_question_marks": 1,
    "maximum_question_marks": 12,
    # Optional frontend target. Retrieval quality/coverage still take priority.
    "target_total_marks": 20,
    "minimum_primary_questions": 3,

    # Phase 2 topic-distribution requirements
    "minimum_supporting_questions": 1,
    "minimum_distinct_official_references": 2,

    # Frontend checkbox. When True, selection first tries to place one
    # quality-safe question from every approved Agent 1 topic.
    "cover_all_approved_topics": True,

    "include_code_questions": True,
    "include_visual_questions": True,

    # Set to "1B" for Python Paper 1 only,
    # "2" for Paper 2 only, or None for both.
    "paper_code": None,

    # Set to "Python" when required,
    # otherwise keep None.
    "programming_language": None,
}


print(f"Project root:       {PROJECT_ROOT}")
print(f"Qdrant collection:  {AGENT2_COLLECTION}")
print(f"Embedding model:    {MODEL_NAME}")
print(f"Device:             {DEVICE}")
print(
    f"Visual page rendering: "
    f"{ENABLE_VISUAL_PAGE_RENDERING}"
)
print(f"Visual render DPI:   {VISUAL_RENDER_DPI}")


print(f"Phase 2 enabled:     {ENABLE_PHASE2_GATE}")
print(
    "Legacy fixed thresholds: "
    f"{LEGACY_FIXED_STRICT_THRESHOLD} / "
    f"{LEGACY_FIXED_RELAXED_THRESHOLD} "
    "(documentation only)"
)
print(
    f"Adaptive thresholding: "
    f"{ENABLE_ADAPTIVE_SEMANTIC_THRESHOLD}"
)
print(
    f"Adaptive percentile: "
    f"{ADAPTIVE_SCORE_PERCENTILE}"
)
print(
    f"Adaptive score margin: "
    f"{ADAPTIVE_TOP_SCORE_MARGIN}"
)
print(
    "Adaptive safety range: "
    f"{ADAPTIVE_ABSOLUTE_MINIMUM_SCORE}–"
    f"{ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD}"
)

print(
    f"Near-duplicate gate: "
    f"{ENABLE_NEAR_DUPLICATE_GATE}"
)
print(
    f"Cover all approved topics: "
    f"{COVER_ALL_APPROVED_TOPICS}"
)
print(
    f"Quality-safe topic rescue: "
    f"{ENABLE_QUALITY_SAFE_TOPIC_RESCUE}"
)
print(
    f"Minimum rescue semantic score: "
    f"{MIN_TOPIC_RESCUE_SEMANTIC_SCORE}"
)

print(
    f"Require actual Agent 1 chunk evidence for release: "
    f"{REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE}"
)
print(f"Phase 3 cleanup enabled: {ENABLE_PHASE3_MARK_SCHEME_CLEANUP}")
print(f"Phase 3 minimum confidence: {PHASE3_MIN_RULE_CONFIDENCE}")
print(f"Phase 3 block parser: {PHASE3_BLOCK_PARSER_VERSION}")
print(f"Merge wrapped MS lines: {PHASE3_MERGE_WRAPPED_LINES}")
print(f"Split inline MS markers: {PHASE3_SPLIT_INLINE_MARKERS}")

display(pd.DataFrame(AGENT1_TOPIC_OUTPUT))
display(pd.DataFrame([ASSESSMENT_REQUEST]))


## 3. Connect to PostgreSQL and Qdrant

PostgreSQL is the source of truth for complete questions and mark schemes.
Qdrant is used for filtered semantic retrieval.


In [ ]:
engine: Engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    future=True,
)


# ----------------------------------------------------------------
# AQA SYLLABUS PAPER OWNERSHIP — ONE POSTGRESQL READ PER RUN
# ----------------------------------------------------------------
# PostgreSQL remains authoritative. Agent 2 reuses Agent 1's SyllabusStore
# but passes this notebook's already-created PostgreSQL engine, so there is
# no separate per-topic database connection/query.
AGENT1_CODE_ROOT = (
    PROJECT_ROOT.parent
    / "Agent_1"
).resolve()

SYLLABUS_STORE_PATH = (
    AGENT1_CODE_ROOT
    / "app"
    / "services"
    / "syllabus_store.py"
)

if not SYLLABUS_STORE_PATH.is_file():
    raise RuntimeError(
        "Could not locate Agent 1 SyllabusStore at "
        f"{SYLLABUS_STORE_PATH}"
    )

if str(AGENT1_CODE_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(AGENT1_CODE_ROOT),
    )

from app.services.syllabus_store import SyllabusStore

syllabus_store = SyllabusStore(
    engine=engine
)

syllabus_concepts = tuple(
    syllabus_store.get_all_concepts()
)

if not syllabus_concepts:
    raise RuntimeError(
        "PostgreSQL returned zero active AQA syllabus concepts."
    )

paper_lookup: dict[str, str] = {}

for concept in syllabus_concepts:
    reference = str(
        concept.official_reference
        or ""
    ).strip()
    family = paper_family(
        concept.paper
    )

    if not reference or family not in {"1", "2"}:
        continue

    existing_family = paper_lookup.get(
        reference
    )

    if (
        existing_family is not None
        and existing_family != family
    ):
        raise RuntimeError(
            "Conflicting syllabus paper ownership in PostgreSQL for "
            f"{reference}: {existing_family} vs {family}"
        )

    paper_lookup[reference] = family

if not paper_lookup:
    raise RuntimeError(
        "No usable AQA syllabus paper mappings were loaded from PostgreSQL."
    )

SYLLABUS_PAPER_FAMILY_BY_REFERENCE.clear()
SYLLABUS_PAPER_FAMILY_BY_REFERENCE.update(
    paper_lookup
)

print(
    "Syllabus paper routing loaded once from PostgreSQL via SyllabusStore: "
    f"{len(SYLLABUS_PAPER_FAMILY_BY_REFERENCE)} official reference(s)."
)


metadata = MetaData()

topics = Table(
    "assessment_topical_topics",
    metadata,
    autoload_with=engine,
)

questions = Table(
    "assessment_topical_questions",
    metadata,
    autoload_with=engine,
)

mark_schemes = Table(
    "assessment_topical_mark_scheme_entries",
    metadata,
    autoload_with=engine,
)

question_ms_links = Table(
    "assessment_topical_question_mark_scheme_links",
    metadata,
    autoload_with=engine,
)

official_mappings = Table(
    "assessment_topic_official_mappings",
    metadata,
    autoload_with=engine,
)


documents = Table(
    "assessment_topical_documents",
    metadata,
    autoload_with=engine,
)


def require_table_column(
    table: Table,
    candidates: list[str],
    purpose: str,
):
    for candidate in candidates:
        if candidate in table.c.keys():
            return table.c[candidate]

    raise RuntimeError(
        f"Could not find a column for {purpose}. "
        f"Checked {candidates}; available columns are "
        f"{list(table.c.keys())}."
    )


def optional_table_column(
    table: Table,
    candidates: list[str],
):
    for candidate in candidates:
        if candidate in table.c.keys():
            return table.c[candidate]

    return None


QUESTION_DOCUMENT_FK_COLUMN = require_table_column(
    questions,
    [
        "question_document_id",
        "document_id",
        "source_document_id",
    ],
    "the question-paper document foreign key",
)

DOCUMENT_PATH_COLUMN = require_table_column(
    documents,
    [
        "local_cache_path",
        "cache_path",
        "local_path",
        "file_path",
    ],
    "the locally cached PDF path",
)

DOCUMENT_FILE_NAME_COLUMN = optional_table_column(
    documents,
    [
        "file_name",
        "filename",
        "document_name",
    ],
)

DOCUMENT_SOURCE_URL_COLUMN = optional_table_column(
    documents,
    [
        "source_url",
        "url",
    ],
)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection successful.")

print(
    "Question-document FK column: "
    f"{QUESTION_DOCUMENT_FK_COLUMN.name}"
)
print(
    "Document PDF-path column: "
    f"{DOCUMENT_PATH_COLUMN.name}"
)



qdrant_kwargs: dict[str, Any] = {
    "url": QDRANT_URL,
    "timeout": QDRANT_TIMEOUT_SECONDS,
}

if QDRANT_API_KEY is not None:
    qdrant_kwargs["api_key"] = QDRANT_API_KEY

qdrant_client = QdrantClient(
    **qdrant_kwargs
)

collection_names = {
    collection.name
    for collection
    in qdrant_client
    .get_collections()
    .collections
}

if AGENT2_COLLECTION not in collection_names:
    raise RuntimeError(
        f"Qdrant collection not found: {AGENT2_COLLECTION}"
    )

qdrant_point_count = int(
    qdrant_client.count(
        collection_name=AGENT2_COLLECTION,
        exact=True,
    ).count
)

print("Qdrant connection successful.")
print(f"Qdrant points: {qdrant_point_count}")

if qdrant_point_count != EXPECTED_QDRANT_POINTS:
    print(
        "Warning: Qdrant point count differs from "
        f"the expected {EXPECTED_QDRANT_POINTS}."
    )

# The question-focus collection is intentionally separate from the main
# enriched retrieval collection. It is populated lazily by content hash, so
# the first run may add vectors and later runs reuse them directly from Qdrant.
if AGENT2_QUESTION_FOCUS_COLLECTION not in collection_names:
    qdrant_client.create_collection(
        collection_name=AGENT2_QUESTION_FOCUS_COLLECTION,
        vectors_config=models.VectorParams(
            size=EXPECTED_VECTOR_SIZE,
            distance=models.Distance.COSINE,
        ),
    )
    collection_names.add(AGENT2_QUESTION_FOCUS_COLLECTION)
    print(
        "Created Qdrant question-focus collection: "
        f"{AGENT2_QUESTION_FOCUS_COLLECTION}"
    )
else:
    print(
        "Qdrant question-focus collection available: "
        f"{AGENT2_QUESTION_FOCUS_COLLECTION}"
    )

question_focus_point_count = int(
    qdrant_client.count(
        collection_name=AGENT2_QUESTION_FOCUS_COLLECTION,
        exact=True,
    ).count
)
print(f"Qdrant question-focus points: {question_focus_point_count}")

# The assessed-skill collection stores only the deterministic instruction/skill
# view extracted from each subquestion. It is intentionally separate from both
# the enriched retrieval vector and the whole-question focus vector.
if AGENT2_ASSESSED_SKILL_COLLECTION not in collection_names:
    qdrant_client.create_collection(
        collection_name=AGENT2_ASSESSED_SKILL_COLLECTION,
        vectors_config=models.VectorParams(
            size=EXPECTED_VECTOR_SIZE,
            distance=models.Distance.COSINE,
        ),
    )
    collection_names.add(AGENT2_ASSESSED_SKILL_COLLECTION)
    print(
        "Created Qdrant assessed-skill collection: "
        f"{AGENT2_ASSESSED_SKILL_COLLECTION}"
    )
else:
    print(
        "Qdrant assessed-skill collection available: "
        f"{AGENT2_ASSESSED_SKILL_COLLECTION}"
    )

assessed_skill_point_count = int(
    qdrant_client.count(
        collection_name=AGENT2_ASSESSED_SKILL_COLLECTION,
        exact=True,
    ).count
)
print(f"Qdrant assessed-skill points: {assessed_skill_point_count}")


## 4. Load the same MiniLM model used in Notebook 04


In [ ]:
model_started = time.perf_counter()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_CACHE_DIR),
)

VECTOR_SIZE = int(
    model.get_sentence_embedding_dimension()
)

if VECTOR_SIZE != EXPECTED_VECTOR_SIZE:
    raise RuntimeError(
        f"Expected {EXPECTED_VECTOR_SIZE} dimensions, "
        f"but received {VECTOR_SIZE}."
    )

print(
    f"MiniLM loaded in "
    f"{time.perf_counter() - model_started:.2f}s"
)
print(f"Vector size: {VECTOR_SIZE}")


## 5. Validate Agent 1 topics and assessment request

The official AQA reference is the canonical connection key.

This cell verifies that every reference returned by Agent 1 exists in the
approved Notebook 04A mapping table.

### AQA syllabus paper-ownership pre-check

Before assessment retrieval, Notebook 05 now routes each approved Agent 1 topic by its official AQA specification reference:

- **Paper 1 — Computational thinking and programming skills:** direct content from **3.1 Fundamentals of algorithms** and **3.2 Programming**.
- **Paper 2 — Computing concepts:** direct content from **3.3 to 3.8**.

If a specific paper is requested, topics assigned to the other paper are reported to the user and excluded from retrieval. If the approved topics are mixed, only the topics belonging to the requested paper continue through retrieval/ranking. If every approved topic belongs to the opposite paper, the notebook produces a no-assessment result without substituting questions from the wrong paper.

AQA also states that both papers may contain synoptic questions requiring knowledge from across the specification. This routing rule therefore represents the **direct assessment ownership** defined in the specification and is used only to honour the user's explicit paper filter.



## 5A. Agent 1 → Agent 2 canonical topic handoff

### Issue found during integration testing

Agent 1 correctly detected detailed lesson concepts, but concepts sharing the
same approved AQA reference were being passed to Agent 2 as separate retrieval
topics.

Example:

```text
Bubble sort  → 3.1.4
Merge sort   → 3.1.4
```

This duplicated one official syllabus area and could make Agent 2 treat a
sub-concept as the complete assessment topic.

### Adopted approach

Detailed Agent 1 concepts remain available for audit and query evidence, while
the Agent 2 handoff is consolidated by canonical `official_reference`.

```text
Detailed Agent 1 concepts
        ↓
Validate every official reference
        ↓
Group concepts sharing the same official reference
        ↓
Use the approved official concept name as the Agent 2 topic
        ↓
Retain all detailed concepts, roles, confidence and source chunks
```

General aggregation rules:

```text
Canonical name    = approved official concept name
Canonical role    = primary when any grouped concept is primary
Confidence        = strongest grouped confidence
Ranking score     = strongest grouped ranking score
Source chunks     = union without duplicates
Detected concepts = retained as an ordered list
```

This step does not alter retrieval thresholds, ranking weights, candidate
selection, question-quality rules, visual rendering or PDF generation.


In [ ]:
def safe_float(
    value: Any,
    default: float = 0.0,
) -> float:
    try:
        number = float(value)
    except (TypeError, ValueError):
        return default

    return (
        number
        if math.isfinite(number)
        else default
    )


def safe_list(value: Any) -> list[Any]:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    return [value]


def normalise_agent1_topics(
    raw_topics: list[dict[str, Any]],
) -> pd.DataFrame:
    rows = []

    for index, item in enumerate(
        raw_topics,
        start=1,
    ):
        topic_name = (
            item.get("topic")
            or item.get("topic_name")
            or item.get("name")
        )

        reference = (
            item.get("official_reference")
            or item.get("reference")
        )

        role = str(
            item.get("role", "supporting")
        ).strip().lower()

        if role not in {
            "primary",
            "supporting",
        }:
            raise ValueError(
                f"Invalid role for topic {index}: {role}"
            )

        if not topic_name or not reference:
            raise ValueError(
                f"Topic {index} is missing a name or "
                "official reference."
            )

        confidence = safe_float(
            item.get("confidence"),
            0.0,
        )

        ranking_score = safe_float(
            item.get("ranking_score"),
            confidence,
        )

        if not 0.0 <= confidence <= 1.0:
            raise ValueError(
                f"Confidence outside 0–1 for {topic_name}."
            )

        rows.append(
            {
                "agent1_topic_index": index,
                "detected_topic": str(
                    topic_name
                ).strip(),
                "role": role,
                "official_reference": str(
                    reference
                ).strip(),
                "confidence": confidence,
                "ranking_score": ranking_score,
                "source_chunks": safe_list(
                    item.get("source_chunks")
                ),
                "source_chunk_texts": safe_list(
                    item.get("source_chunk_texts")
                ),
            }
        )

    frame = pd.DataFrame(rows)

    if frame.empty:
        raise RuntimeError(
            "Agent 1 returned no topics."
        )

    if not frame["role"].eq("primary").any():
        raise RuntimeError(
            "At least one primary topic is required."
        )

    return frame


agent1_topics_df = normalise_agent1_topics(
    AGENT1_TOPIC_OUTPUT
)


mapping_query = (
    select(
        official_mappings.c.topic_id,
        official_mappings.c.official_reference,
        official_mappings.c.official_concept_name,
        official_mappings.c.official_section_reference,
        official_mappings.c.official_section_name,
        official_mappings.c.pmt_subtopic_code,
        official_mappings.c.pmt_subtopic_name,
    )
    .where(
        official_mappings.c.specification_code
        == SPECIFICATION_CODE,
        official_mappings.c.specification_version
        == SPECIFICATION_VERSION,
        official_mappings.c.mapping_status
        == "approved",
        official_mappings.c.human_approved
        .is_(True),
    )
)

with engine.connect() as connection:
    mapping_df = pd.read_sql(
        mapping_query,
        connection,
    )

mapping_df["topic_id"] = (
    mapping_df["topic_id"].astype(str)
)


# ----------------------------------------------------------------
# Resolve detailed Agent 1 references to Agent 2 canonical retrieval
# references WITHOUT collapsing different detected topics.
#
# Example:
#   Social engineering ... 3.6.2.1 -> retrieval reference 3.6.2
#   Malware ...            3.6.2.2 -> retrieval reference 3.6.2
#
# Both remain separate topics. Each gets its own query embedding,
# ranked candidate pool, adaptive semantic threshold and concept-fit
# profile. If two topics share the same canonical retrieval reference,
# they start from the same broad source set but are ranked separately.
# ----------------------------------------------------------------

if mapping_df["official_reference"].duplicated().any():
    duplicate_mapping_refs = sorted(
        mapping_df.loc[
            mapping_df["official_reference"].duplicated(keep=False),
            "official_reference",
        ].astype(str).unique().tolist()
    )
    raise RuntimeError(
        "Approved Agent 2 mapping contains duplicate official "
        f"references: {duplicate_mapping_refs}"
    )

mapping_df = mapping_df.copy()
mapping_df["official_reference"] = (
    mapping_df["official_reference"].astype(str).str.strip()
)

mapping_lookup = mapping_df.set_index(
    "official_reference",
    drop=False,
)
known_mapping_references = set(
    mapping_lookup.index.astype(str).tolist()
)


def canonical_reference_candidates(reference: Any) -> list[str]:
    """Exact reference followed by progressively broader dotted parents."""
    cleaned = str(reference or "").strip()
    if not cleaned:
        return []

    parts = [
        part.strip()
        for part in cleaned.split(".")
        if part.strip()
    ]

    return [
        ".".join(parts[:length])
        for length in range(len(parts), 0, -1)
    ]


def resolve_agent1_reference(
    reference: Any,
) -> tuple[str | None, str, int]:
    """
    Prefer an exact approved Agent 2 reference. Otherwise use the
    nearest approved dotted parent. Never guess an unrelated section.
    """
    for depth, candidate in enumerate(
        canonical_reference_candidates(reference)
    ):
        if candidate in known_mapping_references:
            strategy = (
                "exact_reference"
                if depth == 0
                else "nearest_approved_parent_reference"
            )
            return candidate, strategy, depth

    return None, "unresolved_reference", -1


resolved_topic_rows: list[dict[str, Any]] = []
unresolved_topic_rows: list[dict[str, Any]] = []

for _, source_row in agent1_topics_df.iterrows():
    original_reference = str(
        source_row["official_reference"]
    ).strip()

    (
        canonical_reference,
        resolution_strategy,
        resolution_depth,
    ) = resolve_agent1_reference(original_reference)

    if canonical_reference is None:
        unresolved_record = source_row.to_dict()
        unresolved_record["agent1_official_reference"] = (
            original_reference
        )
        unresolved_record["reference_resolution_strategy"] = (
            resolution_strategy
        )
        unresolved_topic_rows.append(unresolved_record)
        continue

    mapping_record = mapping_lookup.loc[
        canonical_reference
    ].to_dict()

    detected_topic = str(
        source_row["detected_topic"]
    ).strip()

    resolved_record = source_row.to_dict()

    # Keep the fine-grained Agent 1 reference for audit/query evidence.
    resolved_record["agent1_official_reference"] = (
        original_reference
    )

    # Keep `official_reference` as the canonical retrieval reference so
    # existing Qdrant/PostgreSQL filters continue to work unchanged.
    resolved_record["official_reference"] = canonical_reference
    resolved_record["retrieval_official_reference"] = (
        canonical_reference
    )
    resolved_record["reference_resolution_strategy"] = (
        resolution_strategy
    )
    resolved_record["reference_resolution_depth"] = int(
        resolution_depth
    )
    resolved_record["canonical_parent_applied"] = bool(
        resolution_depth > 0
    )

    for column in [
        "topic_id",
        "official_concept_name",
        "official_section_reference",
        "official_section_name",
        "pmt_subtopic_code",
        "pmt_subtopic_name",
    ]:
        resolved_record[column] = mapping_record[column]

    # One row per approved Agent 1 topic = one independent ranking pool.
    resolved_record["canonical_topic"] = str(
        mapping_record["official_concept_name"]
    ).strip()
    resolved_record["detected_concepts"] = [detected_topic]
    resolved_record["detected_concept_records"] = [
        {
            "detected_topic": detected_topic,
            "agent1_official_reference": original_reference,
            "canonical_official_reference": canonical_reference,
            "role": str(source_row["role"]),
            "confidence": float(source_row["confidence"]),
            "ranking_score": float(source_row["ranking_score"]),
            "source_chunks": safe_list(
                source_row.get("source_chunks")
            ),
        }
    ]
    resolved_record["source_agent1_topic_indices"] = [
        int(source_row["agent1_topic_index"])
    ]
    resolved_record["source_detected_topic_count"] = 1
    resolved_record["source_roles"] = [
        str(source_row["role"])
    ]

    resolved_topic_rows.append(resolved_record)


unknown_topics_df = pd.DataFrame(unresolved_topic_rows)

if not unknown_topics_df.empty:
    print(
        "\nTOPIC HANDOFF WARNING: some approved Agent 1 topics "
        "could not be mapped to an Agent 2 canonical reference. "
        "Those topics will be skipped; mapped topics will continue."
    )
    display(unknown_topics_df)

    for _, unresolved_row in unknown_topics_df.iterrows():
        unresolved_topic_name = str(
            unresolved_row.get("detected_topic") or "Unknown topic"
        ).strip()
        unresolved_reference = str(
            unresolved_row.get("agent1_official_reference")
            or unresolved_row.get("official_reference")
            or ""
        ).strip()

        add_agent2_user_message(
            level="warning",
            code="unresolved_agent1_official_reference",
            topic=unresolved_topic_name,
            message=(
                "No approved Agent 2 canonical mapping was found for "
                f"official reference {unresolved_reference or 'unknown'}. "
                "This topic was skipped; other approved topics will still "
                "be processed."
            ),
            details={
                "agent1_official_reference": unresolved_reference,
                "resolution_strategy": "unresolved_reference",
            },
        )

validated_topics_df = (
    pd.DataFrame(resolved_topic_rows)
    .sort_values("agent1_topic_index")
    .reset_index(drop=True)
)

# If at least one topic resolved, continue with a partial run instead of
# aborting because another topic was skipped. If nothing resolves at all,
# there is no trustworthy retrieval pool to continue with.
if validated_topics_df.empty:
    raise RuntimeError(
        "None of the approved Agent 1 topics could be mapped to Agent 2. "
        "Check the official mapping table before generating an assessment."
    )

if len(validated_topics_df) != len(agent1_topics_df):
    print(
        "Partial handoff accepted: "
        f"{len(validated_topics_df)} of {len(agent1_topics_df)} "
        "Agent 1 topics are retrievable."
    )


# ----------------------------------------------------------------
# AQA 8525 DIRECT PAPER-OWNERSHIP PRE-CHECK
#
# Official specification:
#   Paper 1 directly assesses sections 3.1 and 3.2.
#   Paper 2 directly assesses sections 3.3 to 3.8.
#
# AQA also allows synoptic questions across the specification, but the
# deterministic rule below honours the user's explicit paper selection
# using the direct assessment ownership stated in the specification.
# ----------------------------------------------------------------
all_mapped_topics_df = validated_topics_df.copy()

all_mapped_topics_df["syllabus_paper_family"] = (
    all_mapped_topics_df[
        "agent1_official_reference"
    ].map(
        syllabus_paper_family_for_reference
    )
)
all_mapped_topics_df["syllabus_paper_label"] = (
    all_mapped_topics_df[
        "agent1_official_reference"
    ].map(
        syllabus_paper_label_for_reference
    )
)

requested_syllabus_paper_family = paper_family(
    ASSESSMENT_REQUEST.get("paper_code")
)

if requested_syllabus_paper_family in {"1", "2"}:
    all_mapped_topics_df[
        "syllabus_paper_matches_request"
    ] = (
        all_mapped_topics_df[
            "syllabus_paper_family"
        ]
        == requested_syllabus_paper_family
    )
else:
    all_mapped_topics_df[
        "syllabus_paper_matches_request"
    ] = True

syllabus_unclassified_topics_df = (
    all_mapped_topics_df[
        all_mapped_topics_df[
            "syllabus_paper_family"
        ].isna()
    ].copy()
)

if requested_syllabus_paper_family in {"1", "2"}:
    syllabus_paper_excluded_topics_df = (
        all_mapped_topics_df[
            all_mapped_topics_df[
                "syllabus_paper_family"
            ].notna()
            & (
                all_mapped_topics_df[
                    "syllabus_paper_family"
                ]
                != requested_syllabus_paper_family
            )
        ].copy()
    )

    syllabus_paper_eligible_topics_df = (
        all_mapped_topics_df[
            all_mapped_topics_df[
                "syllabus_paper_family"
            ]
            == requested_syllabus_paper_family
        ].copy()
    )
else:
    syllabus_paper_excluded_topics_df = (
        all_mapped_topics_df.head(0).copy()
    )
    syllabus_paper_eligible_topics_df = (
        all_mapped_topics_df.copy()
    )


def _paper_check_topic_record(
    row: pd.Series,
) -> dict[str, Any]:
    return {
        "agent1_topic_index": int(
            row["agent1_topic_index"]
        ),
        "topic": str(
            row["detected_topic"]
        ),
        "role": str(
            row["role"]
        ),
        "official_reference": str(
            row[
                "agent1_official_reference"
            ]
        ),
        "syllabus_paper_family": (
            row.get(
                "syllabus_paper_family"
            )
        ),
        "syllabus_paper_label": str(
            row.get(
                "syllabus_paper_label"
            )
            or "Unclassified"
        ),
    }


for _, excluded_row in (
    syllabus_paper_excluded_topics_df
    .iterrows()
):
    topic_index = int(
        excluded_row[
            "agent1_topic_index"
        ]
    )
    PAPER_MISMATCH_TOPIC_INDEXES.add(
        topic_index
    )

    topic_name = str(
        excluded_row[
            "detected_topic"
        ]
    )
    official_reference = str(
        excluded_row[
            "agent1_official_reference"
        ]
    )
    actual_paper = str(
        excluded_row[
            "syllabus_paper_label"
        ]
    )
    requested_paper_name = paper_label(
        requested_syllabus_paper_family
    )

    add_agent2_user_message(
        level="warning",
        code="topic_belongs_to_other_aqa_paper",
        topic=topic_name,
        requested=1,
        available=0,
        message=(
            f"AQA reference {official_reference} is directly assessed on "
            f"{actual_paper}, but {requested_paper_name} was requested. "
            "This topic was reported and excluded before retrieval; no "
            "question from the wrong paper will be substituted."
        ),
        details={
            "routing_source": (
                "AQA GCSE Computer Science 8525 specification"
            ),
            "routing_version": (
                SYLLABUS_PAPER_ROUTING_VERSION
            ),
            "official_reference": (
                official_reference
            ),
            "topic_paper": actual_paper,
            "requested_paper": (
                requested_paper_name
            ),
        },
    )


if requested_syllabus_paper_family in {"1", "2"}:
    for _, unclassified_row in (
        syllabus_unclassified_topics_df
        .iterrows()
    ):
        topic_index = int(
            unclassified_row[
                "agent1_topic_index"
            ]
        )
        SYLLABUS_UNCLASSIFIED_TOPIC_INDEXES.add(
            topic_index
        )

        add_agent2_user_message(
            level="warning",
            code="topic_paper_could_not_be_classified",
            topic=str(
                unclassified_row[
                    "detected_topic"
                ]
            ),
            requested=1,
            available=0,
            message=(
                "The topic's official reference could not be assigned to "
                "AQA Paper 1 (3.1-3.2) or Paper 2 (3.3-3.8). It is not "
                "used for a paper-specific retrieval request."
            ),
            details={
                "official_reference": str(
                    unclassified_row[
                        "agent1_official_reference"
                    ]
                ),
                "routing_version": (
                    SYLLABUS_PAPER_ROUTING_VERSION
                ),
            },
        )


AGENT2_SYLLABUS_PAPER_CHECK = {
    "routing_version": (
        SYLLABUS_PAPER_ROUTING_VERSION
    ),
    "routing_source": (
        "AQA GCSE Computer Science 8525 specification"
    ),
    "direct_assessment_rule": {
        "paper_1_sections": ["3.1", "3.2"],
        "paper_2_sections": [
            "3.3",
            "3.4",
            "3.5",
            "3.6",
            "3.7",
            "3.8",
        ],
        "synoptic_note": (
            "Both assessments may contain synoptic questions, but explicit "
            "paper filtering uses direct assessment ownership."
        ),
    },
    "requested_paper_family": (
        requested_syllabus_paper_family
    ),
    "requested_paper_label": (
        paper_label(
            requested_syllabus_paper_family
        )
        if requested_syllabus_paper_family
        else "Both papers"
    ),
    "all_mapped_topics": [
        _paper_check_topic_record(row)
        for _, row in all_mapped_topics_df.iterrows()
    ],
    "eligible_topics": [
        _paper_check_topic_record(row)
        for _, row in (
            syllabus_paper_eligible_topics_df
            .iterrows()
        )
    ],
    "excluded_other_paper_topics": [
        _paper_check_topic_record(row)
        for _, row in (
            syllabus_paper_excluded_topics_df
            .iterrows()
        )
    ],
    "unclassified_topics": [
        _paper_check_topic_record(row)
        for _, row in (
            syllabus_unclassified_topics_df
            .iterrows()
        )
    ],
}

if requested_syllabus_paper_family in {"1", "2"}:
    if not syllabus_paper_eligible_topics_df.empty:
        # Mixed-paper handoff: only requested-paper topics continue.
        validated_topics_df = (
            syllabus_paper_eligible_topics_df
            .sort_values(
                "agent1_topic_index"
            )
            .reset_index(drop=True)
        )

        if not syllabus_paper_excluded_topics_df.empty:
            print(
                "\nAQA PAPER ROUTING: mixed-paper topic handoff detected."
            )
            print(
                f"Requested {paper_label(requested_syllabus_paper_family)}: "
                f"{len(validated_topics_df)} topic(s) will continue; "
                f"{len(syllabus_paper_excluded_topics_df)} topic(s) from "
                "the other paper were excluded before retrieval."
            )
    else:
        # Keep the mapped rows only for the downstream no-assessment summary.
        # The retrieval cell sees this flag and performs zero retrieval calls.
        AGENT2_SYLLABUS_PAPER_BLOCKED = True
        validated_topics_df = (
            all_mapped_topics_df
            .sort_values(
                "agent1_topic_index"
            )
            .reset_index(drop=True)
        )

        add_agent2_user_message(
            level="warning",
            code="all_topics_belong_to_other_aqa_paper",
            message=(
                "None of the approved topics belong to "
                f"{paper_label(requested_syllabus_paper_family)} under the "
                "AQA direct assessment split. Retrieval will be skipped and "
                "no assessment will be generated for this request."
            ),
            requested=len(
                all_mapped_topics_df
            ),
            available=0,
            details={
                "paper_check": (
                    AGENT2_SYLLABUS_PAPER_CHECK
                ),
            },
        )

print(
    "\nAQA syllabus paper routing check:"
)
display(
    all_mapped_topics_df[
        [
            "detected_topic",
            "role",
            "agent1_official_reference",
            "syllabus_paper_label",
            "syllabus_paper_matches_request",
        ]
    ]
)

agent1_detected_concepts_df = validated_topics_df.copy()

# Duplicate canonical references are expected and valid here. Different
# Agent 1 concepts under the same canonical reference remain separate.

validated_topics_df["role_weight"] = (
    validated_topics_df["role"].map(
        {
            "primary": PRIMARY_ROLE_WEIGHT,
            "supporting": SUPPORTING_ROLE_WEIGHT,
        }
    )
)


topic_handoff_manifest_df = (
    validated_topics_df[
        [
            "detected_topic",
            "role",
            "agent1_official_reference",
            "official_reference",
            "reference_resolution_strategy",
            "detected_concepts",
            "source_detected_topic_count",
            "source_roles",
            "confidence",
            "ranking_score",
            "source_chunks",
        ]
    ].copy()
)


print(
    "Agent 1 detailed concepts: "
    f"{len(agent1_detected_concepts_df)}"
)

print(
    "Per-topic Agent 2 retrieval units: "
    f"{len(validated_topics_df)}"
)

print(
    "Topic handoff reference-resolution version: "
    f"{TOPIC_HANDOFF_CONSOLIDATION_VERSION}"
)

display(
    topic_handoff_manifest_df
)

if not unknown_topics_df.empty:
    print_agent2_user_messages(
        "AGENT 2 HANDOFF PARTIAL-SUCCESS NOTICE"
    )


request = dict(ASSESSMENT_REQUEST)
request["syllabus_paper_check"] = json_safe(
    AGENT2_SYLLABUS_PAPER_CHECK
)
request["syllabus_paper_routing_version"] = (
    SYLLABUS_PAPER_ROUTING_VERSION
)
request["cover_all_approved_topics"] = bool(
    request.get(
        "cover_all_approved_topics",
        COVER_ALL_APPROVED_TOPICS,
    )
)

integer_fields = [
    "number_of_questions",
    "minimum_question_marks",
    "maximum_question_marks",
    "minimum_primary_questions",
    "minimum_supporting_questions",
    "minimum_distinct_official_references",
]

for field in integer_fields:
    request[field] = int(request[field])

target_total_marks_raw = request.get("target_total_marks")
if target_total_marks_raw not in (None, ""):
    request["target_total_marks"] = int(target_total_marks_raw)
    if request["target_total_marks"] <= 0:
        raise ValueError("target_total_marks must be positive when supplied.")
else:
    request["target_total_marks"] = None

if request["number_of_questions"] <= 0:
    raise ValueError(
        "number_of_questions must be positive."
    )

requested_minimum_primary_questions = int(
    request["minimum_primary_questions"]
)
requested_minimum_supporting_questions = int(
    request["minimum_supporting_questions"]
)

if request["minimum_primary_questions"] < 0:
    raise ValueError("minimum_primary_questions cannot be negative.")

if request["minimum_supporting_questions"] < 0:
    raise ValueError("minimum_supporting_questions cannot be negative.")

if request["minimum_primary_questions"] > request["number_of_questions"]:
    raise ValueError(
        "minimum_primary_questions cannot exceed number_of_questions. "
        "The frontend value is a hard requirement and will not be reduced."
    )

if request["minimum_supporting_questions"] > request["number_of_questions"]:
    raise ValueError(
        "minimum_supporting_questions cannot exceed number_of_questions. "
        "The frontend value is a hard requirement and will not be reduced."
    )

if (
    request["minimum_primary_questions"]
    + request["minimum_supporting_questions"]
    > request["number_of_questions"]
):
    raise ValueError(
        "minimum_primary_questions + minimum_supporting_questions cannot "
        "exceed number_of_questions. The requested role minima are hard "
        "constraints and will not be silently adapted."
    )


approved_agent1_reference_count = int(
    validated_topics_df["agent1_official_reference"]
    .dropna()
    .astype(str)
    .nunique()
)

canonical_agent2_reference_count = int(
    validated_topics_df["official_reference"]
    .dropna()
    .astype(str)
    .nunique()
)

requested_minimum_distinct_references = int(
    request["minimum_distinct_official_references"]
)

# The frontend may request diversity based on the number of approved
# detailed topics. After canonical-parent resolution, the number of
# distinct detailed Agent 1 references can be smaller. This is not an
# error: per-topic retrieval units stay separate and the assessment
# selector should use the largest feasible reference-diversity target.
request["minimum_distinct_official_references"] = min(
    requested_minimum_distinct_references,
    approved_agent1_reference_count,
    int(request["number_of_questions"]),
)

request["requested_minimum_distinct_official_references"] = (
    requested_minimum_distinct_references
)
request["reference_diversity_adapted"] = bool(
    request["minimum_distinct_official_references"]
    != requested_minimum_distinct_references
)

# Role minima are strict targets when they are feasible, but role absence in the
# approved Agent 1 handoff is not a fatal retrieval error. Notebook 05 continues
# with the best quality-safe partial assessment and reports the shortage.
request["primary_role_available_in_handoff"] = bool(
    validated_topics_df["role"].eq("primary").any()
)
request["supporting_role_available_in_handoff"] = bool(
    validated_topics_df["role"].eq("supporting").any()
)

if (
    request["minimum_supporting_questions"] > 0
    and not request["supporting_role_available_in_handoff"]
):
    add_agent2_user_message(
        level="warning",
        code="requested_supporting_role_not_available_in_handoff",
        message=(
            "Supporting questions were requested, but the approved Agent 1 "
            "handoff contains no supporting topic. Retrieval will continue "
            "with the best quality-safe assessment and will report the "
            "supporting shortfall."
        ),
        requested=int(request["minimum_supporting_questions"]),
        available=0,
    )

if (
    request["minimum_primary_questions"] > 0
    and not request["primary_role_available_in_handoff"]
):
    add_agent2_user_message(
        level="warning",
        code="requested_primary_role_not_available_in_handoff",
        message=(
            "Primary questions were requested, but the approved Agent 1 "
            "handoff contains no primary topic. Retrieval will continue "
            "with the best quality-safe assessment and will report the "
            "primary shortfall."
        ),
        requested=int(request["minimum_primary_questions"]),
        available=0,
    )


# Coverage is tracked using detailed Agent 1 topic indexes, not only
# official references. This matters when several detailed concepts share
# the same broader AQA reference (for example multiple 3.5 topics).
approved_topic_rows = (
    validated_topics_df
    .drop_duplicates(subset=["agent1_topic_index"])
    .copy()
)
approved_topic_rows["agent1_topic_index"] = (
    approved_topic_rows["agent1_topic_index"].astype(int)
)

# Primary topic(s) always come first, then supporting topics in the
# Agent 1 list order. This is the same order used by the selector.
approved_topic_rows["_role_order"] = (
    approved_topic_rows["role"].map({"primary": 0, "supporting": 1}).fillna(2)
)
approved_topic_rows = approved_topic_rows.sort_values(
    ["_role_order", "agent1_topic_index"],
    ascending=[True, True],
).reset_index(drop=True)

approved_agent1_topic_indexes = [
    int(value) for value in approved_topic_rows["agent1_topic_index"].tolist()
]
approved_agent1_topic_names = {
    int(row["agent1_topic_index"]): str(row["detected_topic"])
    for _, row in approved_topic_rows.iterrows()
}

approved_official_references = sorted(
    validated_topics_df["agent1_official_reference"]
    .dropna().astype(str).unique().tolist()
)

request["requested_agent1_topic_indexes"] = approved_agent1_topic_indexes
request["requested_agent1_topic_names"] = approved_agent1_topic_names

if request["cover_all_approved_topics"]:
    # If there are more topics than requested question slots, preserve the
    # requested count and cover as many as possible in primary-first order.
    effective_topic_indexes = approved_agent1_topic_indexes[
        : int(request["number_of_questions"])
    ]
    request["required_agent1_topic_indexes"] = effective_topic_indexes
    request["topic_coverage_mode"] = "all_approved_topics_primary_first"

    if len(approved_agent1_topic_indexes) > int(request["number_of_questions"]):
        add_agent2_user_message(
            level="warning",
            code="approved_topic_count_exceeds_question_count",
            message=(
                "Cover all approved topics is enabled, but there are more "
                "approved topics than requested question slots. The primary "
                "topic is prioritised first, followed by supporting topics in "
                "Agent 1 order; remaining topics are reported instead of "
                "forcing extra questions."
            ),
            requested=len(approved_agent1_topic_indexes),
            available=int(request["number_of_questions"]),
        )
else:
    # Even when full topic coverage is off, keep the first primary topic as
    # the preferred starting topic whenever a primary minimum was requested.
    primary_indexes = [
        int(value)
        for value in approved_topic_rows.loc[
            approved_topic_rows["role"].eq("primary"),
            "agent1_topic_index",
        ].tolist()
    ]
    request["required_agent1_topic_indexes"] = (
        primary_indexes[:1]
        if int(request["minimum_primary_questions"]) > 0
        else []
    )
    request["topic_coverage_mode"] = "role_minima_primary_first"

# Keep reference-level fields for backward-compatible reports, but do not use
# them as the only coverage identity.
required_topic_index_set = set(request["required_agent1_topic_indexes"])
request["required_official_references"] = sorted(
    {
        str(row["agent1_official_reference"])
        for _, row in approved_topic_rows.iterrows()
        if int(row["agent1_topic_index"]) in required_topic_index_set
    }
)

if request["cover_all_approved_topics"]:
    request["minimum_distinct_official_references"] = min(
        int(request["minimum_distinct_official_references"]),
        len(approved_official_references),
        int(request["number_of_questions"]),
    )

print(
    "Detailed Agent 1 official references: "
    f"{approved_agent1_reference_count}"
)

print(
    "Canonical Agent 2 retrieval references: "
    f"{canonical_agent2_reference_count}"
)

print(
    "Effective topic coverage mode: "
    f"{request['topic_coverage_mode']}"
)

print(
    "Cover all approved topics: "
    f"{request['cover_all_approved_topics']}"
)

print(
    "Required Agent 1 topic indexes: "
    f"{request['required_agent1_topic_indexes']}"
)

print(
    "Required official references: "
    f"{request['required_official_references']}"
)

print(
    "Requested minimum distinct references: "
    f"{request['requested_minimum_distinct_official_references']}"
)

print(
    "Effective minimum distinct references: "
    f"{request['minimum_distinct_official_references']}"
)

print(
    "Reference-diversity adaptation applied: "
    f"{request['reference_diversity_adapted']}"
)


display(
    validated_topics_df[
        [
            "detected_topic",
            "role",
            "agent1_official_reference",
            "official_reference",
            "reference_resolution_strategy",
            "official_concept_name",
            "detected_concepts",
            "source_detected_topic_count",
            "official_section_reference",
            "confidence",
            "ranking_score",
            "source_chunks",
        ]
    ]
)


### Canonical-parent compatibility with separate per-topic ranking pools

Agent 1 keeps its detailed topic and reference. When Agent 2 only has a broader approved mapping, the nearest dotted parent is used only as the retrieval filter. Every approved Agent 1 topic still receives its own query embedding, candidate ranking, adaptive threshold and concept-fit profile. Topics sharing a canonical reference may start from the same broad question set, but they are ranked independently; duplicate questions are removed before final assessment selection.


## 6. Build MiniLM query vectors

One query is built for each Agent 1 topic. The query includes the official
section, official subsection, detected concept and lesson summary.


In [ ]:

def clean_evidence_text(value: Any) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(value or ""),
    ).strip()


def collect_topic_evidence(
    row: pd.Series,
) -> tuple[str, str, int]:
    direct_texts = [
        clean_evidence_text(value)
        for value in (
            row.get("source_chunk_texts")
            if isinstance(
                row.get("source_chunk_texts"),
                list,
            )
            else []
        )
        if clean_evidence_text(value)
    ]

    mapped_texts = []

    for chunk_id in (
        row.get("source_chunks")
        if isinstance(row.get("source_chunks"), list)
        else []
    ):
        try:
            key = int(chunk_id)
        except (TypeError, ValueError):
            continue

        text_value = clean_evidence_text(
            AGENT1_SOURCE_CHUNK_TEXTS.get(key)
        )

        if text_value:
            mapped_texts.append(text_value)

    if direct_texts:
        evidence_parts = direct_texts
        evidence_source = "agent1_topic_source_chunk_texts"
    elif mapped_texts:
        evidence_parts = mapped_texts
        evidence_source = "agent1_source_chunk_text_map"
    else:
        evidence_parts = [
            clean_evidence_text(LESSON_SUMMARY)
        ]
        evidence_source = "lesson_summary_fallback"

    evidence = "\n".join(
        value
        for value in evidence_parts
        if value
    )[:MAX_QUERY_EVIDENCE_CHARACTERS]

    return evidence, evidence_source, len(evidence_parts)


evidence_results = validated_topics_df.apply(
    collect_topic_evidence,
    axis=1,
)

validated_topics_df["query_evidence"] = [
    value[0]
    for value in evidence_results
]
validated_topics_df["query_evidence_source"] = [
    value[1]
    for value in evidence_results
]
validated_topics_df["query_evidence_item_count"] = [
    value[2]
    for value in evidence_results
]


def build_query_text(row: pd.Series) -> str:
    detected_concepts = (
        row.get(
            "detected_concepts"
        )
        if isinstance(
            row.get(
                "detected_concepts"
            ),
            list,
        )
        else [
            str(
                row[
                    "detected_topic"
                ]
            )
        ]
    )

    return "\n".join(
        [
            "AQA GCSE Computer Science assessment question.",
            (
                f"Official section: {row['official_section_name']} "
                f"({row['official_section_reference']})."
            ),
            (
                "Detailed Agent 1 topic: "
                f"{row['detected_topic']} "
                f"({row.get('agent1_official_reference') or row['official_reference']})."
            ),
            (
                "Canonical Agent 2 retrieval topic: "
                f"{row['official_concept_name']} "
                f"({row['official_reference']})."
            ),
            (
                "Detected lesson concepts: "
                + ", ".join(
                    str(value)
                    for value in detected_concepts
                )
                + "."
            ),
            f"Canonical Agent 1 role: {row['role']}.",
            "Lesson evidence:",
            str(row["query_evidence"]),
        ]
    )


validated_topics_df["query_text"] = (
    validated_topics_df.apply(
        build_query_text,
        axis=1,
    )
)

query_vectors = model.encode(
    validated_topics_df["query_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
).astype(np.float32)

if query_vectors.shape != (
    len(validated_topics_df),
    VECTOR_SIZE,
):
    raise RuntimeError(
        "Unexpected query-vector shape."
    )

phase2_query_evidence_df = validated_topics_df[
    [
        "detected_topic",
        "detected_concepts",
        "role",
        "official_reference",
        "source_chunks",
        "query_evidence_source",
        "query_evidence_item_count",
        "query_evidence",
        "query_text",
    ]
].copy()

display(phase2_query_evidence_df)

fallback_evidence_topic_count = int(
    (
        validated_topics_df["query_evidence_source"]
        == "lesson_summary_fallback"
    ).sum()
)

print(
    "Topics using actual chunk evidence: "
    f"{len(validated_topics_df) - fallback_evidence_topic_count}"
)
print(
    "Topics using lesson-summary fallback: "
    f"{fallback_evidence_topic_count}"
)


In [ ]:
if not ENABLE_QDRANT_DIAGNOSTIC:
    print(
        "Qdrant full-collection diagnostic skipped for normal runtime. "
        "Set AGENT2_ENABLE_QDRANT_DIAGNOSTIC=1 to run it."
    )
else:
    # ================================================================
    # NOTEBOOK 05 DIAGNOSTIC — QDRANT OFFICIAL-REFERENCE COVERAGE
    #
    # Place after:
    #   - qdrant_client is connected
    #   - validated_topics_df is created
    #
    # Run before:
    #   - exact-reference candidate retrieval
    # ================================================================

    from collections import Counter

    from qdrant_client import models


    approved_references = sorted(
        {
            str(reference).strip()
            for reference in (
                validated_topics_df[
                    "official_reference"
                ].dropna().tolist()
            )
            if str(reference).strip()
        }
    )


    print("=" * 72)
    print("AGENT 2 QDRANT DIAGNOSTIC")
    print("=" * 72)

    print("Qdrant URL:")
    print(QDRANT_URL)

    print("\nCollection:")
    print(AGENT2_COLLECTION)


    collection_point_count = int(
        qdrant_client.count(
            collection_name=AGENT2_COLLECTION,
            exact=True,
        ).count
    )

    print("\nTotal collection points:")
    print(collection_point_count)

    print("\nApproved Agent 1 official references:")
    print(approved_references)


    # ---------------------------------------------------------------
    # Read every payload from the current Agent 2 collection
    # ---------------------------------------------------------------

    all_points = []
    next_offset = None

    while True:
        points, next_offset = qdrant_client.scroll(
            collection_name=AGENT2_COLLECTION,
            limit=256,
            offset=next_offset,
            with_payload=True,
            with_vectors=False,
        )

        all_points.extend(points)

        if next_offset is None:
            break


    payload_reference_counts = Counter()

    payload_keys = Counter()


    for point in all_points:
        payload = point.payload or {}

        payload_keys.update(
            payload.keys()
        )

        official_reference = payload.get(
            "official_reference"
        )

        if official_reference is not None:
            payload_reference_counts[
                str(official_reference).strip()
            ] += 1


    print("\nMost common Qdrant payload keys:")

    for key, count in payload_keys.most_common(30):
        print(f"  {key}: {count}")


    print("\nOfficial references available in Qdrant:")

    if payload_reference_counts:
        for reference, count in sorted(
            payload_reference_counts.items()
        ):
            print(
                f"  {reference}: {count} question(s)"
            )
    else:
        print(
            "  No `official_reference` values were found "
            "in the Qdrant payloads."
        )


    # ---------------------------------------------------------------
    # Count exact matches for each approved Agent 1 reference
    # ---------------------------------------------------------------

    diagnostic_rows = []


    for reference in approved_references:
        reference_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="official_reference",
                    match=models.MatchValue(
                        value=reference
                    ),
                )
            ]
        )

        exact_count = int(
            qdrant_client.count(
                collection_name=AGENT2_COLLECTION,
                count_filter=reference_filter,
                exact=True,
            ).count
        )

        diagnostic_rows.append(
            {
                "official_reference": reference,
                "qdrant_exact_count": exact_count,
                "reference_exists_in_payload_scan": (
                    reference
                    in payload_reference_counts
                ),
            }
        )


    diagnostic_df = pd.DataFrame(
        diagnostic_rows
    )

    display(diagnostic_df)


    # ---------------------------------------------------------------
    # Final diagnosis
    # ---------------------------------------------------------------

    print("\nDIAGNOSIS")


    if collection_point_count == 0:
        print(
            "FAIL: The Agent 2 collection exists but contains zero points."
        )

        print(
            "Start the correct Docker/Qdrant volume or rerun "
            "Notebook 04 indexing."
        )

    elif collection_point_count != 820:
        print(
            f"WARNING: Expected 820 points but found "
            f"{collection_point_count}."
        )

    elif not payload_reference_counts:
        print(
            "FAIL: Qdrant points exist, but their payloads do not "
            "contain `official_reference`."
        )

        print(
            "Rerun Notebook 04A in APPLY mode to update the "
            "official-reference payload metadata."
        )

    elif diagnostic_df[
        "qdrant_exact_count"
    ].sum() == 0:
        print(
            "FAIL: None of the approved Agent 1 references exist "
            "in the current Qdrant payloads."
        )

        print(
            "Check the approved references, the collection name, "
            "and whether Notebook 04A was applied to this collection."
        )

    elif (
        diagnostic_df[
            "qdrant_exact_count"
        ] == 0
    ).any():
        print(
            "PARTIAL FAIL: At least one approved topic has no "
            "exact Qdrant questions."
        )

        print(
            "Correct that official reference or do not approve it "
            "for Agent 2 until the assessment bank supports it."
        )

    else:
        print(
            "PASS: Every approved official reference has exact "
            "Qdrant candidates."
        )

## 7. Retrieve exact-reference candidates

Qdrant remains the preferred vector-retrieval backend. The official AQA reference
is always kept as the canonical topic constraint.

### Issue found during Streamlit integration

A valid assessment run could stop with:

```text
No exact-reference candidates were found.
```

The cause was not always an empty assessment bank. Exact Qdrant retrieval could
also return zero when:

- older Qdrant payloads did not contain the newer specification metadata fields;
- an optional paper or programming-language filter had no matching records;
- Qdrant payload metadata and PostgreSQL metadata were temporarily out of sync.

### Final controlled recovery strategy

For each approved Agent 1 topic, this notebook now attempts:

```text
1. Qdrant exact official reference + specification + requested optional filters
2. Qdrant exact official reference with legacy-payload compatibility
3. Qdrant exact official reference after documented optional-filter relaxation
4. PostgreSQL exact-reference fallback with local MiniLM scoring
```

The official reference and marks range are never removed. Optional filter
relaxation is recorded and forces human review before release. PostgreSQL fallback
uses the same MiniLM model and only includes active, retrieval-enabled,
human-approved or human-corrected scored questions when those fields exist.

This is a general recovery mechanism. It does not hardcode any topic, question ID,
paper, or answer.


### Qdrant vector reuse — enriched retrieval + question focus + assessed skill

Notebook 05 now keeps **three intentional Qdrant semantic representations** without changing the existing candidate-construction, request, coverage, rendering or mark-scheme logic:

```text
Main Agent 2 Qdrant collection
    Notebook 04 enriched vector
    topic + subtopic + question + context
    → candidate discovery / lesson-evidence relevance

Question-focus Qdrant collection
    whole subquestion text only
    → direct detailed-topic relevance
    → same-query context-gap analysis

Assessed-skill Qdrant collection
    deterministic instruction/answer-demand text
    → what the student is actually being asked to do
    → NO_OWNER abstention / final relevance verification
```

The richer BM25 query is unchanged: it uses the Agent 1 detected topic, detected concepts and topic-specific lesson evidence.

The additional precision layers are intentionally downstream of candidate retrieval:

1. **Context-gap analysis** compares the same detailed-topic query against the enriched Notebook 04 vector and the whole-question-only vector.
2. **Assessed-skill checking** focuses on the instruction/answer-demand portion of the subquestion instead of allowing a relevant parent stem to carry an unrelated subquestion.
3. **Ownership can abstain with `NO_OWNER`** when no approved Agent 1 topic credibly owns the assessed skill.
4. **The generalized quality gate** uses parser-fragment structure rather than the old hard-coded `outsid` / `bo` checks.
5. **The final relevance verifier** removes context-carried or assessed-skill-mismatched rows before the existing selection algorithm runs. The existing selection, coverage, hard-filter, question-count and downstream rendering logic is otherwise unchanged.

Question-focus and assessed-skill vectors are both content-hash keyed and persisted in Qdrant. Missing vectors are encoded once, upserted to Qdrant, then re-read from Qdrant before use. There is no filesystem/local persistent embedding cache.


In [ ]:
from sqlalchemy import literal


def exact_match(
    key: str,
    value: Any,
) -> models.FieldCondition:
    return models.FieldCondition(
        key=key,
        match=models.MatchValue(
            value=value
        ),
    )


def build_filter(
    *,
    official_reference: str | None = None,
    section_reference: str | None = None,
    include_specification_metadata: bool = True,
    include_optional_filters: bool = True,
) -> models.Filter:
    conditions = []

    if include_specification_metadata:
        conditions.extend(
            [
                exact_match(
                    "official_specification_code",
                    SPECIFICATION_CODE,
                ),
                exact_match(
                    "official_specification_version",
                    SPECIFICATION_VERSION,
                ),
            ]
        )

    conditions.append(
        models.FieldCondition(
            key="marks",
            range=models.Range(
                gte=request[
                    "minimum_question_marks"
                ],
                lte=request[
                    "maximum_question_marks"
                ],
            ),
        )
    )

    if official_reference is not None:
        conditions.append(
            exact_match(
                "official_reference",
                str(official_reference),
            )
        )

    if section_reference is not None:
        conditions.append(
            exact_match(
                "official_section_reference",
                str(section_reference),
            )
        )

    if include_optional_filters:
        if request.get("paper_code"):
            resolved_paper_code = knowledge_base_paper_code()
            if resolved_paper_code is not None:
                conditions.append(
                    exact_match(
                        "paper_code",
                        resolved_paper_code,
                    )
                )
            else:
                # Deliberately impossible value for an unsupported Paper 1
                # language; never widen to another paper.
                conditions.append(
                    exact_match(
                        "paper_code",
                        "__UNSUPPORTED_REQUESTED_PAPER__",
                    )
                )

        if should_apply_programming_language_filter():
            conditions.append(
                exact_match(
                    "programming_language",
                    request[
                        "programming_language"
                    ],
                )
            )

    if not request[
        "include_code_questions"
    ]:
        conditions.append(
            exact_match(
                "has_code",
                False,
            )
        )

    if not request[
        "include_visual_questions"
    ]:
        conditions.append(
            exact_match(
                "has_visual",
                False,
            )
        )

    return models.Filter(
        must=conditions
    )


# ----------------------------------------------------------------
# QDRANT QUESTION-VECTOR REUSE
# ----------------------------------------------------------------
# Qdrant is the persistent source of truth for stable question embeddings.
# This dictionary is only a run-local lookup of vectors already read from
# Qdrant; it is not a second persistent embedding cache.
QDRANT_QUESTION_VECTOR_LOOKUP: dict[str, np.ndarray] = {}
QDRANT_QUESTION_VECTOR_HASH_LOOKUP: dict[str, str | None] = {}
QDRANT_VECTOR_REUSE_STATS = {
    "registered_from_search": 0,
    "registered_from_retrieve": 0,
    "retrieve_calls": 0,
    "lookup_hits": 0,
}


def _clean_optional_hash(value: Any) -> str | None:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    cleaned = str(value).strip()
    return cleaned or None


def _qdrant_vector_array(raw_vector: Any) -> np.ndarray:
    if raw_vector is None:
        raise RuntimeError(
            "Qdrant returned a question without its stored vector."
        )

    if isinstance(raw_vector, dict):
        if len(raw_vector) != 1:
            raise RuntimeError(
                "Notebook 05 expected one Agent 2 question vector per point, "
                "but Qdrant returned multiple named vectors."
            )
        raw_vector = next(iter(raw_vector.values()))

    vector = np.asarray(
        raw_vector,
        dtype=np.float32,
    )

    if vector.ndim != 1 or vector.shape[0] != VECTOR_SIZE:
        raise RuntimeError(
            "Unexpected Qdrant question-vector shape: "
            f"{vector.shape}; expected ({VECTOR_SIZE},)."
        )

    if not np.isfinite(vector).all():
        raise RuntimeError(
            "Qdrant returned a non-finite question vector."
        )

    return vector


def _register_qdrant_question_point(
    point: Any,
    *,
    source: str,
) -> None:
    payload = point.payload or {}
    question_id = str(
        payload.get(
            "question_id",
            point.id,
        )
    )

    vector = _qdrant_vector_array(
        getattr(point, "vector", None)
    )
    stored_hash = _clean_optional_hash(
        payload.get("embedding_text_hash")
    )

    previous_hash = QDRANT_QUESTION_VECTOR_HASH_LOOKUP.get(
        question_id
    )

    if (
        previous_hash is not None
        and stored_hash is not None
        and previous_hash != stored_hash
    ):
        raise RuntimeError(
            "Qdrant returned conflicting embedding_text_hash values for "
            f"question_id={question_id}. Rerun Notebook 04 indexing."
        )

    QDRANT_QUESTION_VECTOR_LOOKUP[question_id] = vector
    QDRANT_QUESTION_VECTOR_HASH_LOOKUP[question_id] = (
        stored_hash or previous_hash
    )

    if source == "search":
        QDRANT_VECTOR_REUSE_STATS[
            "registered_from_search"
        ] += 1
    elif source == "retrieve":
        QDRANT_VECTOR_REUSE_STATS[
            "registered_from_retrieve"
        ] += 1


def _register_qdrant_points(
    points: list[Any],
    *,
    source: str,
) -> list[Any]:
    for point in points:
        _register_qdrant_question_point(
            point,
            source=source,
        )
    return points


def ensure_qdrant_question_vectors(
    question_ids: list[str],
) -> None:
    ordered_unique_ids = list(
        dict.fromkeys(
            str(question_id)
            for question_id in question_ids
        )
    )

    missing_ids = [
        question_id
        for question_id in ordered_unique_ids
        if question_id not in QDRANT_QUESTION_VECTOR_LOOKUP
    ]

    if not missing_ids:
        QDRANT_VECTOR_REUSE_STATS[
            "lookup_hits"
        ] += len(ordered_unique_ids)
        return

    QDRANT_VECTOR_REUSE_STATS[
        "lookup_hits"
    ] += len(ordered_unique_ids) - len(missing_ids)

    batch_size = 256

    for start in range(0, len(missing_ids), batch_size):
        batch_ids = missing_ids[
            start:start + batch_size
        ]

        QDRANT_VECTOR_REUSE_STATS[
            "retrieve_calls"
        ] += 1

        points = list(
            qdrant_client.retrieve(
                collection_name=AGENT2_COLLECTION,
                ids=batch_ids,
                with_payload=True,
                with_vectors=True,
            )
        )

        _register_qdrant_points(
            points,
            source="retrieve",
        )

    still_missing = [
        question_id
        for question_id in missing_ids
        if question_id not in QDRANT_QUESTION_VECTOR_LOOKUP
    ]

    if still_missing:
        preview = ", ".join(still_missing[:8])
        suffix = (
            " ..."
            if len(still_missing) > 8
            else ""
        )
        raise RuntimeError(
            "Stable question embeddings are missing from the Agent 2 "
            "Qdrant collection for question_id(s): "
            f"{preview}{suffix}. Rerun Notebook 04 question embedding/indexing "
            "before Notebook 05 instead of generating replacement question "
            "embeddings here."
        )


def qdrant_vectors_for_rows(
    rows_df: pd.DataFrame,
) -> np.ndarray:
    if rows_df.empty:
        return np.empty(
            (0, VECTOR_SIZE),
            dtype=np.float32,
        )

    question_ids = rows_df[
        "question_id"
    ].astype(str).tolist()

    ensure_qdrant_question_vectors(
        question_ids
    )

    vectors = []

    for _, row in rows_df.iterrows():
        question_id = str(row["question_id"])
        expected_hash = _clean_optional_hash(
            row.get("embedding_text_hash")
        )
        stored_hash = QDRANT_QUESTION_VECTOR_HASH_LOOKUP.get(
            question_id
        )

        if (
            expected_hash is not None
            and stored_hash is not None
            and expected_hash != stored_hash
        ):
            raise RuntimeError(
                "Question embedding hash mismatch between the current "
                "candidate row and Qdrant for "
                f"question_id={question_id}. Expected {expected_hash}, "
                f"Qdrant has {stored_hash}. Rerun Notebook 04 indexing."
            )

        vectors.append(
            QDRANT_QUESTION_VECTOR_LOOKUP[
                question_id
            ]
        )

    return np.stack(
        vectors,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )



# ----------------------------------------------------------------
# QDRANT QUESTION-FOCUS VECTOR VIEW
# ----------------------------------------------------------------
# The main Agent 2 collection stores an enriched retrieval embedding containing
# topic/subtopic/question/context. For detailed-topic ownership we need a
# representation of what the subquestion itself assesses, so this dedicated
# collection stores only normalized question text.
QUESTION_FOCUS_VECTOR_LOOKUP: dict[str, np.ndarray] = {}
QUESTION_FOCUS_VECTOR_STATS = {
    "qdrant_hits": 0,
    "qdrant_misses": 0,
    "new_vectors_encoded": 0,
    "upserted_points": 0,
    "retrieve_calls": 0,
}


def normalize_question_focus_text(value: Any) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(value or "").strip(),
    )


def question_focus_text_hash(value: Any) -> str:
    normalized = normalize_question_focus_text(value)
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


def question_focus_point_id(
    question_id: Any,
    text_hash: str,
) -> str:
    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                "agent2-question-focus|"
                f"{MODEL_NAME}|{question_id}|{text_hash}"
            ),
        )
    )


def _register_question_focus_point(point: Any) -> None:
    payload = point.payload or {}
    point_id = str(point.id)
    vector = _qdrant_vector_array(
        getattr(point, "vector", None)
    )

    if payload.get("embedding_model") not in {None, MODEL_NAME}:
        raise RuntimeError(
            "Question-focus Qdrant point uses a different embedding model: "
            f"{payload.get('embedding_model')}"
        )

    QUESTION_FOCUS_VECTOR_LOOKUP[point_id] = vector


def qdrant_question_focus_vectors_for_rows(
    rows_df: pd.DataFrame,
) -> np.ndarray:
    """
    Return stable question-text-only MiniLM vectors from Qdrant.

    Missing content-hash points are encoded once, persisted to the dedicated
    Qdrant collection, and then retrieved back from Qdrant before use.
    """
    if rows_df.empty:
        return np.empty(
            (0, VECTOR_SIZE),
            dtype=np.float32,
        )

    records = []
    for _, row in rows_df.iterrows():
        question_id = str(row["question_id"])
        question_text = normalize_question_focus_text(
            row.get("question_text")
        )
        if not question_text:
            raise RuntimeError(
                "Cannot build a question-focus vector for an empty question "
                f"text (question_id={question_id})."
            )
        text_hash = question_focus_text_hash(question_text)
        point_id = question_focus_point_id(
            question_id,
            text_hash,
        )
        records.append(
            {
                "question_id": question_id,
                "question_text": question_text,
                "question_text_hash": text_hash,
                "point_id": point_id,
            }
        )

    ordered_unique = {}
    for record in records:
        ordered_unique.setdefault(
            record["point_id"],
            record,
        )

    missing_point_ids = [
        point_id
        for point_id in ordered_unique
        if point_id not in QUESTION_FOCUS_VECTOR_LOOKUP
    ]

    batch_size = 256
    for start in range(0, len(missing_point_ids), batch_size):
        batch_ids = missing_point_ids[
            start:start + batch_size
        ]
        QUESTION_FOCUS_VECTOR_STATS["retrieve_calls"] += 1
        points = list(
            qdrant_client.retrieve(
                collection_name=AGENT2_QUESTION_FOCUS_COLLECTION,
                ids=batch_ids,
                with_payload=True,
                with_vectors=True,
            )
        )
        for point in points:
            _register_question_focus_point(point)

    found_ids = set(QUESTION_FOCUS_VECTOR_LOOKUP)
    still_missing_records = [
        record
        for point_id, record in ordered_unique.items()
        if point_id not in found_ids
    ]

    QUESTION_FOCUS_VECTOR_STATS["qdrant_hits"] += (
        len(ordered_unique) - len(still_missing_records)
    )
    QUESTION_FOCUS_VECTOR_STATS["qdrant_misses"] += len(
        still_missing_records
    )

    if still_missing_records:
        missing_texts = [
            record["question_text"]
            for record in still_missing_records
        ]
        missing_vectors = model.encode(
            missing_texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        QUESTION_FOCUS_VECTOR_STATS["new_vectors_encoded"] += len(
            still_missing_records
        )

        points_to_upsert = []
        for record, vector in zip(
            still_missing_records,
            missing_vectors,
        ):
            points_to_upsert.append(
                models.PointStruct(
                    id=record["point_id"],
                    vector=vector.tolist(),
                    payload={
                        "question_id": record["question_id"],
                        "question_text_hash": record[
                            "question_text_hash"
                        ],
                        "embedding_model": MODEL_NAME,
                        "vector_version": QUESTION_FOCUS_VECTOR_VERSION,
                    },
                )
            )

        qdrant_client.upsert(
            collection_name=AGENT2_QUESTION_FOCUS_COLLECTION,
            wait=True,
            points=points_to_upsert,
        )
        QUESTION_FOCUS_VECTOR_STATS["upserted_points"] += len(
            points_to_upsert
        )

        # Re-read the newly persisted vectors from Qdrant so downstream code
        # always consumes the persisted representation, even on the first run.
        new_ids = [
            record["point_id"]
            for record in still_missing_records
        ]
        for start in range(0, len(new_ids), batch_size):
            batch_ids = new_ids[start:start + batch_size]
            QUESTION_FOCUS_VECTOR_STATS["retrieve_calls"] += 1
            points = list(
                qdrant_client.retrieve(
                    collection_name=AGENT2_QUESTION_FOCUS_COLLECTION,
                    ids=batch_ids,
                    with_payload=True,
                    with_vectors=True,
                )
            )
            for point in points:
                _register_question_focus_point(point)

    unresolved = [
        record["question_id"]
        for record in records
        if record["point_id"] not in QUESTION_FOCUS_VECTOR_LOOKUP
    ]
    if unresolved:
        raise RuntimeError(
            "Question-focus embeddings could not be persisted/retrieved from "
            "Qdrant for question_id(s): "
            + ", ".join(unresolved[:8])
        )

    return np.stack(
        [
            QUESTION_FOCUS_VECTOR_LOOKUP[
                record["point_id"]
            ]
            for record in records
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


# ----------------------------------------------------------------
# QDRANT ASSESSED-SKILL VECTOR VIEW
# ----------------------------------------------------------------
# The caller provides a deterministic assessed_skill_text column. This helper
# only handles stable content-hash persistence/reuse in Qdrant.
ASSESSED_SKILL_VECTOR_LOOKUP: dict[str, np.ndarray] = {}
ASSESSED_SKILL_VECTOR_STATS = {
    "qdrant_hits": 0,
    "qdrant_misses": 0,
    "new_vectors_encoded": 0,
    "upserted_points": 0,
    "retrieve_calls": 0,
}


def normalize_assessed_skill_text(value: Any) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(value or "").strip(),
    )


def assessed_skill_text_hash(value: Any) -> str:
    normalized = normalize_assessed_skill_text(value)
    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


def assessed_skill_point_id(
    question_id: Any,
    text_hash: str,
) -> str:
    return str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            (
                "agent2-assessed-skill|"
                f"{MODEL_NAME}|{question_id}|{text_hash}"
            ),
        )
    )


def _register_assessed_skill_point(point: Any) -> None:
    payload = point.payload or {}
    point_id = str(point.id)
    vector = _qdrant_vector_array(
        getattr(point, "vector", None)
    )

    if payload.get("embedding_model") not in {None, MODEL_NAME}:
        raise RuntimeError(
            "Assessed-skill Qdrant point uses a different embedding model: "
            f"{payload.get('embedding_model')}"
        )

    ASSESSED_SKILL_VECTOR_LOOKUP[point_id] = vector


def qdrant_assessed_skill_vectors_for_rows(
    rows_df: pd.DataFrame,
) -> np.ndarray:
    """Return stable assessed-skill MiniLM vectors from Qdrant."""
    if rows_df.empty:
        return np.empty(
            (0, VECTOR_SIZE),
            dtype=np.float32,
        )

    if "assessed_skill_text" not in rows_df.columns:
        raise RuntimeError(
            "assessed_skill_text must be created before assessed-skill "
            "Qdrant vectors are requested."
        )

    records = []
    for _, row in rows_df.iterrows():
        question_id = str(row["question_id"])
        assessed_text = normalize_assessed_skill_text(
            row.get("assessed_skill_text")
        )
        if not assessed_text:
            assessed_text = normalize_assessed_skill_text(
                row.get("question_text")
            )
        if not assessed_text:
            raise RuntimeError(
                "Cannot build an assessed-skill vector for empty text "
                f"(question_id={question_id})."
            )
        text_hash = assessed_skill_text_hash(assessed_text)
        point_id = assessed_skill_point_id(
            question_id,
            text_hash,
        )
        records.append(
            {
                "question_id": question_id,
                "assessed_skill_text": assessed_text,
                "assessed_skill_text_hash": text_hash,
                "point_id": point_id,
            }
        )

    ordered_unique = {}
    for record in records:
        ordered_unique.setdefault(record["point_id"], record)

    missing_point_ids = [
        point_id
        for point_id in ordered_unique
        if point_id not in ASSESSED_SKILL_VECTOR_LOOKUP
    ]

    batch_size = 256
    for start in range(0, len(missing_point_ids), batch_size):
        batch_ids = missing_point_ids[start:start + batch_size]
        ASSESSED_SKILL_VECTOR_STATS["retrieve_calls"] += 1
        points = list(
            qdrant_client.retrieve(
                collection_name=AGENT2_ASSESSED_SKILL_COLLECTION,
                ids=batch_ids,
                with_payload=True,
                with_vectors=True,
            )
        )
        for point in points:
            _register_assessed_skill_point(point)

    found_ids = set(ASSESSED_SKILL_VECTOR_LOOKUP)
    still_missing_records = [
        record
        for point_id, record in ordered_unique.items()
        if point_id not in found_ids
    ]

    ASSESSED_SKILL_VECTOR_STATS["qdrant_hits"] += (
        len(ordered_unique) - len(still_missing_records)
    )
    ASSESSED_SKILL_VECTOR_STATS["qdrant_misses"] += len(
        still_missing_records
    )

    if still_missing_records:
        missing_texts = [
            record["assessed_skill_text"]
            for record in still_missing_records
        ]
        missing_vectors = model.encode(
            missing_texts,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        ASSESSED_SKILL_VECTOR_STATS["new_vectors_encoded"] += len(
            still_missing_records
        )

        points_to_upsert = []
        for record, vector in zip(
            still_missing_records,
            missing_vectors,
        ):
            points_to_upsert.append(
                models.PointStruct(
                    id=record["point_id"],
                    vector=vector.tolist(),
                    payload={
                        "question_id": record["question_id"],
                        "assessed_skill_text_hash": record[
                            "assessed_skill_text_hash"
                        ],
                        "embedding_model": MODEL_NAME,
                        "vector_version": ASSESSED_SKILL_VECTOR_VERSION,
                    },
                )
            )

        qdrant_client.upsert(
            collection_name=AGENT2_ASSESSED_SKILL_COLLECTION,
            wait=True,
            points=points_to_upsert,
        )
        ASSESSED_SKILL_VECTOR_STATS["upserted_points"] += len(
            points_to_upsert
        )

        # Consume the persisted representation on the first run as well.
        new_ids = [
            record["point_id"]
            for record in still_missing_records
        ]
        for start in range(0, len(new_ids), batch_size):
            batch_ids = new_ids[start:start + batch_size]
            ASSESSED_SKILL_VECTOR_STATS["retrieve_calls"] += 1
            points = list(
                qdrant_client.retrieve(
                    collection_name=AGENT2_ASSESSED_SKILL_COLLECTION,
                    ids=batch_ids,
                    with_payload=True,
                    with_vectors=True,
                )
            )
            for point in points:
                _register_assessed_skill_point(point)

    unresolved = [
        record["question_id"]
        for record in records
        if record["point_id"] not in ASSESSED_SKILL_VECTOR_LOOKUP
    ]
    if unresolved:
        raise RuntimeError(
            "Assessed-skill embeddings could not be persisted/retrieved from "
            "Qdrant for question_id(s): "
            + ", ".join(unresolved[:8])
        )

    return np.stack(
        [
            ASSESSED_SKILL_VECTOR_LOOKUP[record["point_id"]]
            for record in records
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


def search_points(
    vector: np.ndarray,
    query_filter: models.Filter,
    limit: int,
) -> list[Any]:
    try:
        result = qdrant_client.query_points(
            collection_name=AGENT2_COLLECTION,
            query=vector.tolist(),
            query_filter=query_filter,
            limit=limit,
            score_threshold=QDRANT_SCORE_THRESHOLD,
            with_payload=True,
            with_vectors=True,
        )

        points = (
            result.points
            if hasattr(result, "points")
            else list(result)
        )

        return _register_qdrant_points(
            list(points),
            source="search",
        )

    except (
        AttributeError,
        TypeError,
    ):
        points = list(
            qdrant_client.search(
                collection_name=AGENT2_COLLECTION,
                query_vector=vector.tolist(),
                query_filter=query_filter,
                limit=limit,
                score_threshold=QDRANT_SCORE_THRESHOLD,
                with_payload=True,
                with_vectors=True,
            )
        )

        return _register_qdrant_points(
            points,
            source="search",
        )


def point_to_candidate(
    *,
    point: Any,
    topic_row: pd.Series,
    stage: str,
    retrieval_backend: str = "qdrant",
    retrieval_relaxation: str | None = None,
) -> dict[str, Any]:
    payload = point.payload or {}

    return {
        "question_id": str(
            payload.get(
                "question_id",
                point.id,
            )
        ),
        "agent1_topic_index": int(
            topic_row["agent1_topic_index"]
        ),
        "detected_topic": (
            topic_row["detected_topic"]
        ),
        "detected_concepts": (
            topic_row[
                "detected_concepts"
            ]
        ),
        "agent1_role": topic_row["role"],
        "agent1_confidence": float(
            topic_row["confidence"]
        ),
        "agent1_ranking_score": float(
            topic_row["ranking_score"]
        ),
        "role_weight": float(
            topic_row["role_weight"]
        ),
        "source_chunks": (
            topic_row["source_chunks"]
        ),
        "query_evidence_source": (
            topic_row["query_evidence_source"]
        ),
        "query_evidence": (
            topic_row["query_evidence"]
        ),
        "agent1_official_reference": (
            topic_row.get(
                "agent1_official_reference",
                topic_row["official_reference"],
            )
        ),
        "reference_resolution_strategy": (
            topic_row.get(
                "reference_resolution_strategy",
                "exact_reference",
            )
        ),
        "requested_official_reference": (
            topic_row[
                "official_reference"
            ]
        ),
        "requested_section_reference": (
            topic_row[
                "official_section_reference"
            ]
        ),
        "retrieval_stage": stage,
        "retrieval_backend": retrieval_backend,
        "retrieval_relaxation": retrieval_relaxation,
        "semantic_score": float(
            point.score
        ),
        "question_content_hash": (
            payload.get(
                "question_content_hash"
            )
        ),
        "embedding_text_hash": (
            payload.get(
                "embedding_text_hash"
            )
        ),
        "official_reference": (
            payload.get(
                "official_reference"
            )
        ),
        "official_concept_name": (
            payload.get(
                "official_concept_name"
            )
        ),
        "official_section_reference": (
            payload.get(
                "official_section_reference"
            )
        ),
        "question_number": (
            payload.get("question_number")
        ),
        "question_text": (
            payload.get("question_text")
        ),
        "marks": int(
            payload.get("marks", 0)
        ),
        "has_code": bool(
            payload.get("has_code", False)
        ),
        "has_visual": bool(
            payload.get(
                "has_visual",
                False,
            )
        ),
        "paper_code": (
            payload.get("paper_code")
        ),
        "programming_language": (
            payload.get(
                "programming_language"
            )
        ),
        "review_status": (
            payload.get("review_status")
        ),
    }


def optional_question_column(
    candidates: list[str],
):
    return optional_table_column(
        questions,
        candidates,
    )


QUESTION_CONTENT_HASH_COLUMN = (
    optional_question_column(
        [
            "question_content_hash",
            "content_hash",
        ]
    )
)

QUESTION_EMBEDDING_HASH_COLUMN = (
    optional_question_column(
        [
            "embedding_text_hash",
            "embedding_hash",
        ]
    )
)

QUESTION_RETRIEVAL_ENABLED_COLUMN = (
    optional_question_column(
        ["retrieval_enabled"]
    )
)

QUESTION_IS_ACTIVE_COLUMN = (
    optional_question_column(
        ["is_active"]
    )
)

QUESTION_IS_LEGACY_COLUMN = (
    optional_question_column(
        ["is_legacy"]
    )
)

QUESTION_RECORD_TYPE_COLUMN = (
    optional_question_column(
        ["record_type"]
    )
)

QUESTION_CONTEXT_COLUMN = (
    optional_question_column(
        ["context_text"]
    )
)


def selected_or_null(
    column,
    label: str,
):
    return (
        column.label(label)
        if column is not None
        else literal(None).label(label)
    )


def postgres_exact_rows(
    *,
    topic_row: pd.Series,
    include_optional_filters: bool,
    maximum_rows: int = 250,
) -> pd.DataFrame:
    conditions = [
        questions.c.official_reference
        == str(
            topic_row[
                "official_reference"
            ]
        ),
        questions.c.marks
        >= request[
            "minimum_question_marks"
        ],
        questions.c.marks
        <= request[
            "maximum_question_marks"
        ],
    ]

    if (
        QUESTION_RETRIEVAL_ENABLED_COLUMN
        is not None
    ):
        conditions.append(
            QUESTION_RETRIEVAL_ENABLED_COLUMN
            .is_(True)
        )

    if QUESTION_IS_ACTIVE_COLUMN is not None:
        conditions.append(
            QUESTION_IS_ACTIVE_COLUMN
            .is_(True)
        )

    if QUESTION_IS_LEGACY_COLUMN is not None:
        conditions.append(
            QUESTION_IS_LEGACY_COLUMN
            .is_(False)
        )

    if QUESTION_RECORD_TYPE_COLUMN is not None:
        conditions.append(
            QUESTION_RECORD_TYPE_COLUMN
            == "scored_item"
        )

    if "review_status" in questions.c:
        conditions.append(
            questions.c.review_status.in_(
                [
                    "human_approved",
                    "human_corrected",
                ]
            )
        )

    if not request[
        "include_code_questions"
    ]:
        conditions.append(
            questions.c.has_code.is_(False)
        )

    if not request[
        "include_visual_questions"
    ]:
        conditions.append(
            questions.c.has_visual.is_(False)
        )

    if include_optional_filters:
        if request.get("paper_code"):
            resolved_paper_code = knowledge_base_paper_code()
            if resolved_paper_code is not None:
                conditions.append(
                    topics.c.paper_code
                    == resolved_paper_code
                )
            else:
                conditions.append(
                    topics.c.paper_code
                    == "__UNSUPPORTED_REQUESTED_PAPER__"
                )

        if should_apply_programming_language_filter():
            conditions.append(
                topics.c.programming_language
                == request[
                    "programming_language"
                ]
            )

    candidate_query = (
        select(
            questions.c.id.label(
                "question_id"
            ),
            questions.c.question_number,
            questions.c.question_text,
            selected_or_null(
                QUESTION_CONTEXT_COLUMN,
                "context_text",
            ),
            questions.c.marks,
            questions.c.has_code,
            questions.c.has_visual,
            questions.c.review_status,
            questions.c.official_reference,
            questions.c.official_concept_name,
            questions.c.official_section_reference,
            topics.c.paper_code,
            topics.c.programming_language,
            selected_or_null(
                QUESTION_CONTENT_HASH_COLUMN,
                "question_content_hash",
            ),
            selected_or_null(
                QUESTION_EMBEDDING_HASH_COLUMN,
                "embedding_text_hash",
            ),
        )
        .select_from(
            questions
            .join(
                topics,
                questions.c.topic_id
                == topics.c.id,
            )
            .join(
                question_ms_links,
                question_ms_links.c.question_id
                == questions.c.id,
            )
        )
        .where(*conditions)
        .distinct()
        .limit(maximum_rows)
    )

    with engine.connect() as connection:
        return pd.read_sql(
            candidate_query,
            connection,
        )


def postgres_hard_filtered_bank_rows(
    *,
    maximum_rows: int = POSTGRES_METADATA_RESCUE_SCAN_ROWS,
) -> pd.DataFrame:
    """
    Return the assessment bank under all CURRENT user hard filters but
    without constraining the stored official reference.

    This function is used only for metadata-mismatch recovery after a topic's
    exact/canonical pool is empty. It never relaxes:
      - requested paper,
      - programming language where applicable,
      - marks range,
      - include/exclude code,
      - include/exclude visual,
      - active/retrieval/review status.
    """

    conditions = [
        questions.c.marks
        >= request["minimum_question_marks"],
        questions.c.marks
        <= request["maximum_question_marks"],
    ]

    if QUESTION_RETRIEVAL_ENABLED_COLUMN is not None:
        conditions.append(
            QUESTION_RETRIEVAL_ENABLED_COLUMN.is_(True)
        )

    if QUESTION_IS_ACTIVE_COLUMN is not None:
        conditions.append(
            QUESTION_IS_ACTIVE_COLUMN.is_(True)
        )

    if QUESTION_IS_LEGACY_COLUMN is not None:
        conditions.append(
            QUESTION_IS_LEGACY_COLUMN.is_(False)
        )

    if QUESTION_RECORD_TYPE_COLUMN is not None:
        conditions.append(
            QUESTION_RECORD_TYPE_COLUMN
            == "scored_item"
        )

    if "review_status" in questions.c:
        conditions.append(
            questions.c.review_status.in_(
                [
                    "human_approved",
                    "human_corrected",
                ]
            )
        )

    if not request["include_code_questions"]:
        conditions.append(
            questions.c.has_code.is_(False)
        )

    if not request["include_visual_questions"]:
        conditions.append(
            questions.c.has_visual.is_(False)
        )

    if request.get("paper_code"):
        resolved_paper_code = knowledge_base_paper_code()

        if resolved_paper_code is not None:
            conditions.append(
                topics.c.paper_code
                == resolved_paper_code
            )
        else:
            conditions.append(
                topics.c.paper_code
                == "__UNSUPPORTED_REQUESTED_PAPER__"
            )

    if should_apply_programming_language_filter():
        conditions.append(
            topics.c.programming_language
            == request["programming_language"]
        )

    rescue_query = (
        select(
            questions.c.id.label("question_id"),
            questions.c.question_number,
            questions.c.question_text,
            selected_or_null(
                QUESTION_CONTEXT_COLUMN,
                "context_text",
            ),
            questions.c.marks,
            questions.c.has_code,
            questions.c.has_visual,
            questions.c.review_status,
            questions.c.official_reference,
            questions.c.official_concept_name,
            questions.c.official_section_reference,
            topics.c.paper_code,
            topics.c.programming_language,
            selected_or_null(
                QUESTION_CONTENT_HASH_COLUMN,
                "question_content_hash",
            ),
            selected_or_null(
                QUESTION_EMBEDDING_HASH_COLUMN,
                "embedding_text_hash",
            ),
        )
        .select_from(
            questions
            .join(
                topics,
                questions.c.topic_id
                == topics.c.id,
            )
            .join(
                question_ms_links,
                question_ms_links.c.question_id
                == questions.c.id,
            )
        )
        .where(*conditions)
        .distinct()
        .limit(maximum_rows)
    )

    with engine.connect() as connection:
        return pd.read_sql(
            rescue_query,
            connection,
        )


METADATA_RESCUE_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by",
    "describe", "explain", "for", "from", "give", "how",
    "in", "is", "it", "of", "on", "or", "state", "that",
    "the", "their", "these", "this", "to", "two", "what",
    "when", "which", "with",
}


def metadata_rescue_tokens(
    value: Any,
) -> set[str]:
    return {
        token
        for token in re.findall(
            r"[a-z0-9]+",
            str(value or "").casefold(),
        )
        if len(token) > 1
        and token not in METADATA_RESCUE_STOPWORDS
    }


def metadata_rescue_lexical_coverage(
    *,
    topic_text: Any,
    question_text: Any,
) -> float:
    """
    Generic lexical query coverage.

    This deliberately contains no syllabus-specific keyword list.
    """
    topic_tokens = metadata_rescue_tokens(
        topic_text
    )
    question_tokens = metadata_rescue_tokens(
        question_text
    )

    if not topic_tokens:
        return 0.0

    return float(
        len(topic_tokens & question_tokens)
        / len(topic_tokens)
    )


def postgres_metadata_mismatch_rescue_candidates(
    *,
    topic_row: pd.Series,
    query_vector: np.ndarray,
) -> tuple[
    list[dict[str, Any]],
    dict[str, Any],
]:
    """
    Recover questions whose stored official-reference metadata is inconsistent
    with the detailed Agent 1 topic.

    IMPORTANT:
    - executed only after the normal exact/canonical pool is empty,
    - all user hard filters remain active,
    - no sibling reference is trusted merely because it is nearby,
    - question text/evidence must independently support the detailed topic,
    - rescued rows are forced through the broad hybrid gate later.
    """

    bank_rows = postgres_hard_filtered_bank_rows()

    if bank_rows.empty:
        return [], {
            "metadata_rescue_scanned_rows": 0,
            "metadata_rescue_candidate_count": 0,
            "metadata_rescue_used": False,
        }

    topic_text = str(
        topic_row.get("detected_topic")
        or ""
    ).strip()

    direct_query = "\n".join(
        [
            "AQA GCSE Computer Science assessment question.",
            f"Detailed lesson topic: {topic_text}.",
        ]
    ).strip()

    direct_topic_vector = model.encode(
        [direct_query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0].astype(np.float32)

    # Do NOT prepend the stored official topic name here. The reason for this
    # recovery stage is precisely that the stored metadata may be inaccurate.
    question_texts = []

    for _, row in bank_rows.iterrows():
        question_texts.append(
            "\n".join(
                [
                    "AQA GCSE Computer Science assessment question.",
                    str(
                        row.get("question_text")
                        or ""
                    ).strip(),
                    str(
                        row.get("context_text")
                        or ""
                    ).strip(),
                ]
            ).strip()
        )

    question_vectors = model.encode(
        question_texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    evidence_scores = (
        question_vectors
        @ query_vector.astype(np.float32)
    )

    direct_scores = (
        question_vectors
        @ direct_topic_vector
    )

    lexical_scores = np.array(
        [
            metadata_rescue_lexical_coverage(
                topic_text=topic_text,
                question_text=(
                    str(row.get("question_text") or "")
                    + " "
                    + str(row.get("context_text") or "")
                ),
            )
            for _, row in bank_rows.iterrows()
        ],
        dtype=float,
    )

    rescue_scores = (
        METADATA_RESCUE_DIRECT_TOPIC_WEIGHT
        * np.clip(
            direct_scores,
            0.0,
            1.0,
        )
        + METADATA_RESCUE_EVIDENCE_WEIGHT
        * np.clip(
            evidence_scores,
            0.0,
            1.0,
        )
        + METADATA_RESCUE_LEXICAL_WEIGHT
        * np.clip(
            lexical_scores,
            0.0,
            1.0,
        )
    )

    ranked = bank_rows.copy()
    ranked[
        "_metadata_rescue_evidence_score"
    ] = evidence_scores
    ranked[
        "_metadata_rescue_direct_topic_score"
    ] = direct_scores
    ranked[
        "_metadata_rescue_lexical_coverage"
    ] = lexical_scores
    ranked[
        "_metadata_rescue_score"
    ] = rescue_scores

    requested_reference = str(
        topic_row["official_reference"]
    ).strip()

    # This recovery stage is specifically for metadata disagreement. Rows with
    # the same stored official reference should already have been found by the
    # exact PostgreSQL complement and therefore are not needed here.
    ranked = ranked[
        ranked[
            "official_reference"
        ].astype(str).str.strip().ne(
            requested_reference
        )
    ].copy()

    ranked = ranked[
        (
            ranked[
                "_metadata_rescue_score"
            ].astype(float)
            >= METADATA_RESCUE_MIN_SCORE
        )
        & (
            ranked[
                "_metadata_rescue_direct_topic_score"
            ].astype(float)
            >= METADATA_RESCUE_MIN_DIRECT_TOPIC_SCORE
        )
    ].copy()

    ranked = (
        ranked.sort_values(
            [
                "_metadata_rescue_score",
                "_metadata_rescue_direct_topic_score",
                "_metadata_rescue_evidence_score",
            ],
            ascending=False,
        )
        .head(
            METADATA_RESCUE_CANDIDATES_PER_TOPIC
        )
        .reset_index(drop=True)
    )

    output: list[dict[str, Any]] = []

    for _, row in ranked.iterrows():
        output.append(
            {
                "question_id": str(
                    row["question_id"]
                ),
                "agent1_topic_index": int(
                    topic_row[
                        "agent1_topic_index"
                    ]
                ),
                "detected_topic": (
                    topic_row["detected_topic"]
                ),
                "detected_concepts": (
                    topic_row[
                        "detected_concepts"
                    ]
                ),
                "agent1_role": (
                    topic_row["role"]
                ),
                "agent1_confidence": float(
                    topic_row["confidence"]
                ),
                "agent1_ranking_score": float(
                    topic_row[
                        "ranking_score"
                    ]
                ),
                "role_weight": float(
                    topic_row["role_weight"]
                ),
                "source_chunks": (
                    topic_row["source_chunks"]
                ),
                "query_evidence_source": (
                    topic_row[
                        "query_evidence_source"
                    ]
                ),
                "query_evidence": (
                    topic_row[
                        "query_evidence"
                    ]
                ),
                "agent1_official_reference": (
                    topic_row.get(
                        "agent1_official_reference",
                        topic_row[
                            "official_reference"
                        ],
                    )
                ),
                "reference_resolution_strategy": (
                    str(
                        topic_row.get(
                            "reference_resolution_strategy",
                            "exact_reference",
                        )
                    )
                    + "+metadata_mismatch_semantic_rescue"
                ),
                "requested_official_reference": (
                    requested_reference
                ),
                "requested_section_reference": (
                    topic_row[
                        "official_section_reference"
                    ]
                ),
                "retrieval_stage": (
                    "metadata_mismatch_semantic_rescue"
                ),
                "retrieval_backend": (
                    "postgresql_hard_filtered_metadata_rescue"
                ),
                "retrieval_relaxation": (
                    "stored_official_reference_metadata_only"
                ),
                # Retain the evidence-aware semantic score in the historical
                # semantic_score field. Direct-topic + lexical are recalculated
                # generically again in the normal hybrid stage.
                "semantic_score": float(
                    row[
                        "_metadata_rescue_evidence_score"
                    ]
                ),
                "metadata_rescue_score": float(
                    row[
                        "_metadata_rescue_score"
                    ]
                ),
                "metadata_rescue_direct_topic_score": float(
                    row[
                        "_metadata_rescue_direct_topic_score"
                    ]
                ),
                "metadata_rescue_lexical_coverage": float(
                    row[
                        "_metadata_rescue_lexical_coverage"
                    ]
                ),
                "metadata_reference_mismatch": True,
                "question_content_hash": (
                    row.get(
                        "question_content_hash"
                    )
                ),
                "embedding_text_hash": (
                    row.get(
                        "embedding_text_hash"
                    )
                ),
                # Preserve the DB metadata for audit. It is NOT treated as
                # authoritative for rescue acceptance.
                "official_reference": (
                    row.get(
                        "official_reference"
                    )
                ),
                "official_concept_name": (
                    row.get(
                        "official_concept_name"
                    )
                ),
                "official_section_reference": (
                    row.get(
                        "official_section_reference"
                    )
                ),
                "question_number": (
                    row.get(
                        "question_number"
                    )
                ),
                "question_text": (
                    row.get(
                        "question_text"
                    )
                ),
                "marks": int(
                    row.get("marks", 0)
                ),
                "has_code": bool(
                    row.get(
                        "has_code",
                        False,
                    )
                ),
                "has_visual": bool(
                    row.get(
                        "has_visual",
                        False,
                    )
                ),
                "paper_code": (
                    row.get(
                        "paper_code"
                    )
                ),
                "programming_language": (
                    row.get(
                        "programming_language"
                    )
                ),
                "review_status": (
                    row.get(
                        "review_status"
                    )
                ),
            }
        )

    return output, {
        "metadata_rescue_scanned_rows": int(
            len(bank_rows)
        ),
        "metadata_rescue_candidate_count": int(
            len(output)
        ),
        "metadata_rescue_used": bool(output),
        "metadata_rescue_version": (
            METADATA_MISMATCH_RESCUE_VERSION
        ),
    }


def postgres_rows_to_candidates(
    *,
    rows_df: pd.DataFrame,
    topic_row: pd.Series,
    query_vector: np.ndarray,
    limit: int,
    retrieval_relaxation: str | None,
) -> list[dict[str, Any]]:
    if rows_df.empty:
        return []

    # Reuse the canonical MiniLM question vectors already persisted by
    # Notebook 04 in Qdrant. Do not rebuild a second question representation
    # inside Notebook 05.
    candidate_vectors = qdrant_vectors_for_rows(
        rows_df
    )

    semantic_scores = (
        candidate_vectors
        @ query_vector.astype(
            np.float32
        )
    )

    ranked_rows = rows_df.copy()
    ranked_rows[
        "semantic_score"
    ] = semantic_scores

    # FINAL POLICY:
    # Do not apply the Qdrant score threshold to exact-reference PostgreSQL
    # rows. Metadata has already established the legal pool. Semantic
    # relevance is applied later according to the specific-vs-broad policy.

    ranked_rows = (
        ranked_rows.sort_values(
            "semantic_score",
            ascending=False,
        )
        .head(limit)
        .reset_index(drop=True)
    )

    output = []

    for _, row in ranked_rows.iterrows():
        output.append(
            {
                "question_id": str(
                    row["question_id"]
                ),
                "agent1_topic_index": int(
                    topic_row[
                        "agent1_topic_index"
                    ]
                ),
                "detected_topic": (
                    topic_row[
                        "detected_topic"
                    ]
                ),
                "agent1_role": (
                    topic_row["role"]
                ),
                "agent1_confidence": float(
                    topic_row["confidence"]
                ),
                "agent1_ranking_score": float(
                    topic_row[
                        "ranking_score"
                    ]
                ),
                "role_weight": float(
                    topic_row["role_weight"]
                ),
                "source_chunks": (
                    topic_row[
                        "source_chunks"
                    ]
                ),
                "query_evidence_source": (
                    topic_row[
                        "query_evidence_source"
                    ]
                ),
                "query_evidence": (
                    topic_row[
                        "query_evidence"
                    ]
                ),
                "agent1_official_reference": (
                    topic_row.get(
                        "agent1_official_reference",
                        topic_row["official_reference"],
                    )
                ),
                "reference_resolution_strategy": (
                    topic_row.get(
                        "reference_resolution_strategy",
                        "exact_reference",
                    )
                ),
                "requested_official_reference": (
                    topic_row[
                        "official_reference"
                    ]
                ),
                "requested_section_reference": (
                    topic_row[
                        "official_section_reference"
                    ]
                ),
                "retrieval_stage": (
                    "postgres_exact_official_reference"
                ),
                "retrieval_backend": (
                    "postgresql_local_minilm"
                ),
                "retrieval_relaxation": (
                    retrieval_relaxation
                ),
                "semantic_score": float(
                    row[
                        "semantic_score"
                    ]
                ),
                "question_content_hash": (
                    row.get(
                        "question_content_hash"
                    )
                ),
                "embedding_text_hash": (
                    row.get(
                        "embedding_text_hash"
                    )
                ),
                "official_reference": (
                    row.get(
                        "official_reference"
                    )
                ),
                "official_concept_name": (
                    row.get(
                        "official_concept_name"
                    )
                ),
                "official_section_reference": (
                    row.get(
                        "official_section_reference"
                    )
                ),
                "question_number": (
                    row.get(
                        "question_number"
                    )
                ),
                "question_text": (
                    row.get(
                        "question_text"
                    )
                ),
                "marks": int(
                    row.get("marks", 0)
                ),
                "has_code": bool(
                    row.get(
                        "has_code",
                        False,
                    )
                ),
                "has_visual": bool(
                    row.get(
                        "has_visual",
                        False,
                    )
                ),
                "paper_code": (
                    row.get(
                        "paper_code"
                    )
                ),
                "programming_language": (
                    row.get(
                        "programming_language"
                    )
                ),
                "review_status": (
                    row.get(
                        "review_status"
                    )
                ),
            }
        )

    return output


def retrieve_exact_candidates_for_topic(
    *,
    topic_row: pd.Series,
    query_vector: np.ndarray,
) -> tuple[
    list[dict[str, Any]],
    dict[str, Any],
]:
    """
    Build a complete exact-reference candidate pool.

    Qdrant remains the fast semantic source, but PostgreSQL exact-reference
    rows are ALWAYS searched as a complementary source. Previously PostgreSQL
    ran only when Qdrant returned zero points, which meant a valid PMT question
    could be absent simply because it fell outside Qdrant's small top-k window.

    Important:
      - paper/language/min-max/code/visual filters stay hard,
      - the official reference stays exact/canonical,
      - same-section/sibling-topic fallback remains disabled.
    """
    attempts = [
        {
            "name": "qdrant_strict",
            "include_specification_metadata": True,
            "include_optional_filters": True,
            "relaxation": None,
        },
        {
            "name": "qdrant_legacy_payload_compatibility",
            "include_specification_metadata": False,
            "include_optional_filters": True,
            "relaxation": "payload_metadata_compatibility",
        },
    ]

    qdrant_candidates: list[dict[str, Any]] = []
    qdrant_attempt_used: str | None = None
    qdrant_relaxation: str | None = None

    for attempt in attempts:
        points = search_points(
            query_vector,
            build_filter(
                official_reference=str(
                    topic_row["official_reference"]
                ),
                include_specification_metadata=(
                    attempt["include_specification_metadata"]
                ),
                include_optional_filters=(
                    attempt["include_optional_filters"]
                ),
            ),
            CANDIDATES_PER_TOPIC,
        )

        if not points:
            continue

        stage = (
            "exact_official_reference"
            if attempt["relaxation"] is None
            else "exact_official_reference_recovered"
        )

        qdrant_candidates = [
            point_to_candidate(
                point=point,
                topic_row=topic_row,
                stage=stage,
                retrieval_backend="qdrant",
                retrieval_relaxation=attempt["relaxation"],
            )
            for point in points
        ]

        qdrant_attempt_used = attempt["name"]
        qdrant_relaxation = attempt["relaxation"]
        break

    # Complement Qdrant with the full exact-reference PostgreSQL bank.
    # This is deliberately NOT a relaxed topic lookup.
    postgres_rows = postgres_exact_rows(
        topic_row=topic_row,
        include_optional_filters=True,
        maximum_rows=POSTGRES_EXACT_SCAN_ROWS,
    )

    postgres_candidates = postgres_rows_to_candidates(
        rows_df=postgres_rows,
        topic_row=topic_row,
        query_vector=query_vector,
        limit=POSTGRES_EXACT_CANDIDATES_PER_TOPIC,
        retrieval_relaxation=None,
    )

    # Stable union by question_id. Keep Qdrant's representation when the same
    # question occurs in both sources, but include PostgreSQL-only questions.
    combined: list[dict[str, Any]] = []
    seen_question_ids: set[str] = set()

    for candidate in [
        *qdrant_candidates,
        *postgres_candidates,
    ]:
        question_id = str(candidate.get("question_id") or "").strip()

        if not question_id or question_id in seen_question_ids:
            continue

        seen_question_ids.add(question_id)
        combined.append(candidate)

    metadata_rescue_event = {
        "metadata_rescue_scanned_rows": 0,
        "metadata_rescue_candidate_count": 0,
        "metadata_rescue_used": False,
        "metadata_rescue_version": (
            METADATA_MISMATCH_RESCUE_VERSION
        ),
    }

    # Generic recovery for stale/misaligned official-reference metadata.
    # IMPORTANT: this does not execute while an exact/canonical pool exists.
    # It also never relaxes any user-selected hard filter.
    if (
        not combined
        and ENABLE_METADATA_MISMATCH_RESCUE
    ):
        (
            metadata_rescue_candidates,
            metadata_rescue_event,
        ) = postgres_metadata_mismatch_rescue_candidates(
            topic_row=topic_row,
            query_vector=query_vector,
        )

        combined.extend(
            metadata_rescue_candidates
        )

    if (
        metadata_rescue_event.get(
            "metadata_rescue_used",
            False,
        )
    ):
        backend = (
            "postgresql_hard_filtered_metadata_rescue"
        )
        attempt_name = (
            "metadata_mismatch_semantic_rescue"
        )
        candidate_pool_strategy = (
            "zero_exact_pool_then_hard_filtered_metadata_rescue"
        )
        effective_relaxation = (
            "stored_official_reference_metadata_only"
        )
    elif qdrant_candidates and postgres_candidates:
        backend = "qdrant_plus_postgresql_exact_pool"
        attempt_name = (
            f"{qdrant_attempt_used}+postgres_exact_reference_complement"
        )
        candidate_pool_strategy = (
            "exact_reference_qdrant_postgres_union"
        )
        effective_relaxation = qdrant_relaxation
    elif qdrant_candidates:
        backend = "qdrant"
        attempt_name = (
            qdrant_attempt_used
            or "qdrant_exact_reference"
        )
        candidate_pool_strategy = (
            "exact_reference_qdrant_postgres_union"
        )
        effective_relaxation = qdrant_relaxation
    else:
        backend = "postgresql_local_minilm"
        attempt_name = "postgres_exact_reference"
        candidate_pool_strategy = (
            "exact_reference_qdrant_postgres_union"
        )
        effective_relaxation = qdrant_relaxation

    return (
        combined,
        {
            "topic": topic_row["detected_topic"],
            "official_reference": topic_row["official_reference"],
            "backend": backend,
            "attempt": attempt_name,
            "candidate_count": len(combined),
            "qdrant_candidate_count": len(qdrant_candidates),
            "postgres_candidate_count": len(postgres_candidates),
            "candidate_pool_strategy": (
                candidate_pool_strategy
            ),
            "retrieval_relaxation": (
                effective_relaxation
            ),
            **metadata_rescue_event,
        },
    )


def diagnose_topic_paper_availability(
    topic_row: pd.Series,
) -> dict[str, Any]:
    """Inspect other papers for diagnosis only; never return them as candidates."""
    requested_paper = request.get("paper_code")
    if not requested_paper:
        return {"status": "not_applicable", "available_paper_codes": []}

    unrestricted = postgres_exact_rows(
        topic_row=topic_row,
        include_optional_filters=False,
        maximum_rows=500,
    )

    if unrestricted.empty or "paper_code" not in unrestricted.columns:
        return {"status": "no_questions_any_paper", "available_paper_codes": []}

    paper_codes = sorted(
        {
            str(value).strip()
            for value in unrestricted["paper_code"].dropna().tolist()
            if str(value).strip()
        }
    )
    requested_family = paper_family(requested_paper)
    available_families = sorted(
        {paper_family(code) for code in paper_codes if paper_family(code)}
    )

    if any(paper_code_matches(code, requested_paper) for code in paper_codes):
        return {
            "status": "requested_paper_has_unfiltered_questions",
            "available_paper_codes": paper_codes,
            "available_paper_families": available_families,
        }

    if requested_family not in set(available_families) and len(available_families) == 1:
        return {
            "status": "other_paper_only",
            "available_paper_codes": paper_codes,
            "available_paper_families": available_families,
            "other_paper_family": available_families[0],
        }

    return {
        "status": "requested_paper_unavailable",
        "available_paper_codes": paper_codes,
        "available_paper_families": available_families,
    }


def hard_filter_candidate_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Final guard against any stale/legacy retrieval payload leaking past UI filters."""
    if frame.empty:
        return frame.copy()

    filtered = frame.copy()
    requested_paper = request.get("paper_code")
    if requested_paper and "paper_code" in filtered.columns:
        filtered = filtered[
            filtered["paper_code"].map(
                lambda value: paper_code_matches(value, requested_paper)
            )
        ].copy()

    if (
        should_apply_programming_language_filter()
        and "programming_language" in filtered.columns
    ):
        requested_language = str(request.get("programming_language") or "").strip().lower()
        filtered = filtered[
            filtered["programming_language"]
            .fillna("").astype(str).str.strip().str.lower()
            .eq(requested_language)
        ].copy()

    return filtered.reset_index(drop=True)



print(
    "Paper filter resolution:",
    requested_paper_debug_label(),
)

candidate_rows = []
retrieval_recovery_events = []

for position, topic_row in (
    validated_topics_df
    .reset_index(drop=True)
    .iterrows()
):
    if globals().get(
        "AGENT2_SYLLABUS_PAPER_BLOCKED",
        False,
    ):
        retrieval_recovery_events.append(
            {
                "agent1_topic_index": int(
                    topic_row[
                        "agent1_topic_index"
                    ]
                ),
                "detected_topic": str(
                    topic_row[
                        "detected_topic"
                    ]
                ),
                "candidate_count": 0,
                "recovery_used": False,
                "recovery_status": (
                    "skipped_before_retrieval_"
                    "because_topic_belongs_to_other_aqa_paper"
                ),
                "requested_paper_code": (
                    request.get(
                        "paper_code"
                    )
                ),
                "syllabus_paper_label": (
                    topic_row.get(
                        "syllabus_paper_label"
                    )
                ),
            }
        )
        continue

    (
        topic_candidates,
        recovery_event,
    ) = retrieve_exact_candidates_for_topic(
        topic_row=topic_row,
        query_vector=query_vectors[
            position
        ],
    )

    candidate_rows.extend(
        topic_candidates
    )

    retrieval_recovery_events.append(
        recovery_event
    )


retrieval_recovery_df = pd.DataFrame(
    retrieval_recovery_events
)

display(retrieval_recovery_df)


exact_candidates_df = pd.DataFrame(
    candidate_rows
)

# Preserve a stable schema for a legitimate zero-retrieval run. This is
# especially important when the syllabus pre-check blocks every topic before
# Qdrant/PostgreSQL retrieval.
if exact_candidates_df.empty:
    for column in [
        "agent1_topic_index",
        "detected_topic",
        "agent1_role",
        "official_reference",
        "retrieval_backend",
        "retrieval_relaxation",
        "question_number",
        "marks",
        "semantic_score",
        "question_text",
        "paper_code",
        "programming_language",
    ]:
        if column not in exact_candidates_df.columns:
            exact_candidates_df[column] = pd.Series(
                dtype="object"
            )

# User-selected paper/language filters are hard constraints.
# A final dataframe guard prevents any legacy payload compatibility path
# from leaking a question from the wrong paper into later ranking.
exact_candidates_df = hard_filter_candidate_frame(
    exact_candidates_df
)

# Report topic-level retrieval gaps without aborting the whole run.
retrieved_topic_indexes = set(
    exact_candidates_df.get(
        "agent1_topic_index",
        pd.Series(dtype=int),
    ).dropna().astype(int).tolist()
)

retrieval_availability_rows = []
for _, topic_row in validated_topics_df.iterrows():
    topic_index = int(topic_row["agent1_topic_index"])
    topic_name = str(topic_row["detected_topic"])
    available_count = int(
        (
            exact_candidates_df.get(
                "agent1_topic_index",
                pd.Series(dtype=int),
            )
            == topic_index
        ).sum()
    )
    retrieval_availability_rows.append(
        {
            "agent1_topic_index": topic_index,
            "detected_topic": topic_name,
            "agent1_official_reference": str(
                topic_row["agent1_official_reference"]
            ),
            "canonical_official_reference": str(
                topic_row["official_reference"]
            ),
            "retrieved_candidates": available_count,
        }
    )

    if available_count == 0:
        # The deterministic AQA specification pre-check has already produced
        # the user-facing paper mismatch message. Do not query the assessment
        # bank merely to rediscover the same mismatch.
        if globals().get(
            "AGENT2_SYLLABUS_PAPER_BLOCKED",
            False,
        ):
            continue

        diagnostic = diagnose_topic_paper_availability(topic_row)
        requested_paper = request.get("paper_code")

        if diagnostic.get("status") == "other_paper_only":
            PAPER_MISMATCH_TOPIC_INDEXES.add(topic_index)
            other_family = diagnostic.get("other_paper_family")
            other_label = paper_label(other_family)
            requested_label = paper_label(requested_paper)
            add_agent2_user_message(
                level="warning",
                code="topic_available_on_other_paper_only",
                topic=topic_name,
                requested=1,
                available=0,
                message=(
                    f"This topic appears only on {other_label} in the current "
                    f"AQA assessment bank, but {requested_label} was requested. "
                    f"The topic was skipped and no {other_label} question was "
                    "substituted."
                ),
                details={
                    "requested_paper_code": requested_paper,
                    "available_paper_codes": diagnostic.get(
                        "available_paper_codes", []
                    ),
                    "agent1_official_reference": str(
                        topic_row["agent1_official_reference"]
                    ),
                    "canonical_official_reference": str(
                        topic_row["official_reference"]
                    ),
                },
            )
        else:
            add_agent2_user_message(
                level="warning",
                code="no_retrieval_candidates_for_topic",
                topic=topic_name,
                requested=1,
                available=0,
                message=(
                    "No assessment questions matched this topic and the current "
                    "user filters. The topic will be skipped while other topics "
                    "continue; paper/language filters were not relaxed."
                ),
                details={
                    "requested_paper_code": requested_paper,
                    "paper_diagnostic": diagnostic,
                    "agent1_official_reference": str(
                        topic_row["agent1_official_reference"]
                    ),
                    "canonical_official_reference": str(
                        topic_row["official_reference"]
                    ),
                },
            )


topic_retrieval_availability_df = pd.DataFrame(
    retrieval_availability_rows
)

if not topic_retrieval_availability_df.empty:
    display(topic_retrieval_availability_df)

if exact_candidates_df.empty:
    # Zero retrieval results is a normal, user-actionable outcome, not a
    # notebook/infrastructure failure. Keep hard paper/language filters intact,
    # never substitute a wrong-paper question, and let the notebook complete
    # with current-run empty/status artifacts.
    AGENT2_NO_RETRIEVAL_CANDIDATES = True
    AGENT2_NO_SAFE_CANDIDATES = True

    missing_references = (
        validated_topics_df[
            [
                "detected_topic",
                "agent1_official_reference",
                "official_reference",
            ]
        ].to_dict(
            orient="records"
        )
    )

    requested_paper = request.get("paper_code")
    requested_paper_label = paper_label(requested_paper)

    # Per-topic diagnostics/messages were already produced above. Add one
    # concise run-level message for the frontend.
    all_topic_indexes = set(
        validated_topics_df.get(
            "agent1_topic_index",
            pd.Series(dtype=int),
        ).dropna().astype(int).tolist()
    )
    all_are_other_paper = bool(
        all_topic_indexes
        and all_topic_indexes.issubset(PAPER_MISMATCH_TOPIC_INDEXES)
    )

    if all_are_other_paper and requested_paper:
        run_message = (
            f"The AQA specification assigns all approved topics to the other paper, "
            f"not {requested_paper_label}. Retrieval was skipped for those "
            "topics and no wrong-paper question was substituted. Change the "
            "paper filter and run again."
        )
        run_code = "all_approved_topics_available_only_on_other_paper"
    else:
        run_message = (
            f"No assessment question matched the approved topics under the "
            f"current {requested_paper_label if requested_paper else 'paper'} "
            "and other hard filters. The notebook completed safely without "
            "substituting questions from another paper. Review the topic-level "
            "messages, change the paper/filter settings, or approve another topic."
        )
        run_code = "no_retrieval_candidates_for_current_hard_filters"

    add_agent2_user_message(
        level="warning",
        code=run_code,
        message=run_message,
        requested=int(request.get("number_of_questions", 0) or 0),
        available=0,
        details={
            "requested_paper_code": requested_paper,
            "requested_paper_label": requested_paper_label,
            "resolved_kb_paper_code": knowledge_base_paper_code(),
            "paper_mismatch_topic_indexes": sorted(PAPER_MISMATCH_TOPIC_INDEXES),
            "mapped_topics": missing_references,
            "assessment_generated": False,
        },
    )

    # Cell 35's resilient selector expects this name. It will convert the empty
    # pool into a no-assessment selection summary and write current-run status
    # files for the frontend.
    phase2_candidates_df = pd.DataFrame()

    print_agent2_user_messages(
        "AGENT 2 USER-FRIENDLY RETRIEVAL SUMMARY"
    )
else:
    AGENT2_NO_RETRIEVAL_CANDIDATES = False


missing_topic_candidate_rows = (
    retrieval_recovery_df[
        retrieval_recovery_df[
            "candidate_count"
        ]
        == 0
    ]
)

if not missing_topic_candidate_rows.empty:
    print(
        "Warning: one or more approved topics have no "
        "exact-reference questions after all recovery "
        "attempts."
    )

    display(
        missing_topic_candidate_rows
    )


print(
    f"Exact candidates: "
    f"{len(exact_candidates_df)}"
)

display(
    exact_candidates_df[
        [
            "detected_topic",
            "agent1_role",
            "official_reference",
            "retrieval_backend",
            "retrieval_relaxation",
            "question_number",
            "marks",
            "semantic_score",
            "question_text",
        ]
    ].head(50)
)


## 8. Controlled same-section fallback

Fallback runs only when a topic returns fewer candidates than the requested
question count. Fallback questions receive a ranking penalty.


In [ ]:
if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Skipped this candidate-processing stage because no questions matched the current hard filters.")
else:
    fallback_rows = []

    if ALLOW_SECTION_FALLBACK:
        for position, topic_row in (
            validated_topics_df
            .reset_index(drop=True)
            .iterrows()
        ):
            exact_count = int(
                (
                    exact_candidates_df[
                        "agent1_topic_index"
                    ]
                    == topic_row[
                        "agent1_topic_index"
                    ]
                ).sum()
            )

            if (
                exact_count
                >= request["number_of_questions"]
            ):
                continue

            section_points = search_points(
                query_vectors[position],
                build_filter(
                    section_reference=(
                        topic_row[
                            "official_section_reference"
                        ]
                    )
                ),
                FALLBACK_CANDIDATES_PER_TOPIC,
            )

            for point in section_points:
                payload = point.payload or {}

                if (
                    payload.get(
                        "official_reference"
                    )
                    == topic_row[
                        "official_reference"
                    ]
                ):
                    continue

                fallback_rows.append(
                    point_to_candidate(
                        point=point,
                        topic_row=topic_row,
                        stage=(
                            "same_section_fallback"
                        ),
                    )
                )


    fallback_candidates_df = pd.DataFrame(
        fallback_rows
    )

    all_candidates_df = pd.concat(
        [
            exact_candidates_df,
            fallback_candidates_df,
        ],
        ignore_index=True,
    )

    print(
        f"Fallback candidates: "
        f"{len(fallback_candidates_df)}"
    )
    print(
        f"Total raw candidates: "
        f"{len(all_candidates_df)}"
    )


## 9. Deterministic reranking and duplicate removal

The final score combines semantic similarity, Agent 1 confidence, Agent 1
ranking score, role priority and retrieval stage.

Duplicate questions are removed using content hash, embedding hash, or
normalised question text and marks.


In [ ]:
if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Skipped this candidate-processing stage because no questions matched the current hard filters.")
else:
    def final_score(
        row: pd.Series,
    ) -> float:
        exact_bonus = (
            EXACT_REFERENCE_BONUS
            if str(
                row.get(
                    "official_reference"
                )
            )
            == str(
                row.get(
                    "requested_official_reference"
                )
            )
            else 0.0
        )

        fallback_penalty = (
            SECTION_FALLBACK_PENALTY
            if "same_section_fallback"
            in str(
                row.get(
                    "retrieval_stage",
                    "",
                )
            )
            else 0.0
        )

        corrected_bonus = (
            HUMAN_CORRECTED_BONUS
            if row["review_status"]
            == "human_corrected"
            else 0.0
        )

        score = (
            0.55 * float(
                row["semantic_score"]
            )
            + 0.15 * float(
                row["agent1_confidence"]
            )
            + 0.10 * float(
                row["agent1_ranking_score"]
            )
            + 0.20 * float(
                row["role_weight"]
            )
            + exact_bonus
            + corrected_bonus
            - fallback_penalty
        )

        return round(score, 6)


    def normalise_text(value: Any) -> str:
        cleaned = re.sub(
            r"\s+",
            " ",
            str(value or "").lower(),
        ).strip()

        return re.sub(
            r"[^a-z0-9 ]+",
            "",
            cleaned,
        )


    def duplicate_key(
        row: pd.Series,
    ) -> str:
        content_hash = str(
            row.get(
                "question_content_hash"
            )
            or ""
        ).strip()

        if content_hash:
            return f"content:{content_hash}"

        embedding_hash = str(
            row.get(
                "embedding_text_hash"
            )
            or ""
        ).strip()

        if embedding_hash:
            return f"embedding:{embedding_hash}"

        return (
            f"text:{normalise_text(row['question_text'])}:"
            f"{int(row['marks'])}"
        )


    all_candidates_df["final_score"] = (
        all_candidates_df.apply(
            final_score,
            axis=1,
        )
    )

    all_candidates_df = (
        all_candidates_df.sort_values(
            [
                "final_score",
                "semantic_score",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    all_candidates_df["raw_rank"] = (
        np.arange(
            1,
            len(all_candidates_df) + 1,
        )
    )

    all_candidates_df["duplicate_key"] = (
        all_candidates_df.apply(
            duplicate_key,
            axis=1,
        )
    )

    raw_count = len(all_candidates_df)

    # IMPORTANT:
    # Exact duplicate removal is deliberately scoped to ONE Agent 1 topic.
    # The same underlying DB question is allowed to remain temporarily in
    # multiple detailed-topic pools. Cross-topic ownership is resolved only
    # after the hybrid detailed-topic relevance scores have been calculated.
    unique_candidates_df = (
        all_candidates_df.drop_duplicates(
            subset=[
                "agent1_topic_index",
                "duplicate_key",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

    unique_candidates_df["unique_rank"] = (
        np.arange(
            1,
            len(unique_candidates_df) + 1,
        )
    )

    duplicates_removed = (
        raw_count
        - len(unique_candidates_df)
    )

    print(f"Raw candidates:      {raw_count}")
    print(
        f"Within-topic exact duplicates removed:  "
        f"{duplicates_removed}"
    )
    print(
        f"Unique candidates:   "
        f"{len(unique_candidates_df)}"
    )

    display(
        unique_candidates_df[
            [
                "unique_rank",
                "detected_topic",
                "agent1_role",
                "retrieval_stage",
                "official_reference",
                "marks",
                "semantic_score",
                "final_score",
                "question_text",
            ]
        ].head(50)
    )


## 10. Phase 2 refinement — near-duplicate detection

The original duplicate logic is retained above. It removes exact content-hash,
embedding-hash and normalised-text duplicates.

The evaluated output showed that the same question may still appear in different PMT
topical packs with:

```text
different database IDs
different topical headings
small formatting differences
the same actual question and marks
```

This refinement therefore performs a second transparent duplicate pass.

### Comparison stages

```text
1. Remove topical headings and common footer fragments.
2. Compare cleaned question text.
3. Compare token overlap.
4. Compare MiniLM question-to-question embeddings.
5. Require the same marks unless configured otherwise.
6. Keep the higher-ranked version and record the rejected duplicate.
```

Every decision is stored in a manifest rather than being silently discarded.


In [ ]:
if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Skipped this candidate-processing stage because no questions matched the current hard filters.")
else:

    PMT_TOPIC_HEADING_PATTERN = re.compile(
        r"^\s*\d+(?:\.\d+)+\s+[A-Za-z].*$",
        flags=re.IGNORECASE,
    )

    DUPLICATE_NOISE_LINES = {
        "do not",
        "turn over",
        "outside the box",
        "do not write outside the box",
    }


    def canonical_duplicate_text(
        value: Any,
    ) -> str:
        raw_text = str(
            value
            or ""
        )

        retained_lines = []

        for line in raw_text.splitlines():
            cleaned_line = re.sub(
                r"\s+",
                " ",
                line.strip(),
            )

            if not cleaned_line:
                continue

            lowered_line = cleaned_line.lower()

            if (
                PMT_TOPIC_HEADING_PATTERN.match(
                    cleaned_line
                )
            ):
                continue

            if lowered_line in DUPLICATE_NOISE_LINES:
                continue

            retained_lines.append(
                cleaned_line
            )

        joined = " ".join(
            retained_lines
        ).lower()

        joined = re.sub(
            r"[^a-z0-9£≤≥←→+\-*/= ]+",
            " ",
            joined,
        )

        return re.sub(
            r"\s+",
            " ",
            joined,
        ).strip()


    def token_set(
        text_value: str,
    ) -> set[str]:
        return set(
            re.findall(
                r"[a-z0-9]+",
                text_value.lower(),
            )
        )


    def token_jaccard(
        left: str,
        right: str,
    ) -> float:
        left_tokens = token_set(left)
        right_tokens = token_set(right)

        if (
            not left_tokens
            and not right_tokens
        ):
            return 1.0

        union = (
            left_tokens
            | right_tokens
        )

        if not union:
            return 0.0

        return len(
            left_tokens
            & right_tokens
        ) / len(union)


    def lexical_similarity(
        left: str,
        right: str,
    ) -> float:
        return float(
            SequenceMatcher(
                None,
                left,
                right,
            ).ratio()
        )


    def detect_near_duplicates(
        candidates: pd.DataFrame,
    ) -> tuple[
        pd.DataFrame,
        pd.DataFrame,
    ]:
        working = (
            candidates.copy()
            .sort_values(
                [
                    "final_score",
                    "semantic_score",
                ],
                ascending=False,
            )
            .reset_index(drop=True)
        )

        working[
            "duplicate_canonical_text"
        ] = working[
            "question_text"
        ].map(
            canonical_duplicate_text
        )

        canonical_texts = working[
            "duplicate_canonical_text"
        ].tolist()

        if canonical_texts:
            # Lexical/Jaccard duplicate checks still use canonical_texts.
            # Near-duplicate semantic comparison should reflect the actual question
            # wording, not topic/context metadata from the enriched retrieval
            # vector. Reuse the question-focus representation from Qdrant.
            duplicate_vectors = qdrant_question_focus_vectors_for_rows(
                working
            )

        else:
            duplicate_vectors = np.empty(
                (
                    0,
                    VECTOR_SIZE,
                ),
                dtype=np.float32,
            )

        keep_indexes: list[int] = []
        decision_rows: list[
            dict[str, Any]
        ] = []

        for candidate_index, row in (
            working.iterrows()
        ):
            duplicate_of_index = None
            duplicate_method = None
            best_lexical = 0.0
            best_jaccard = 0.0
            best_semantic = 0.0

            for kept_index in keep_indexes:
                kept_row = working.loc[
                    kept_index
                ]

                # Do not let a broader Agent 1 topic remove a candidate from a
                # more detailed topic (or vice versa) before detailed-topic
                # relevance and ownership have been calculated.
                if int(
                    row["agent1_topic_index"]
                ) != int(
                    kept_row["agent1_topic_index"]
                ):
                    continue

                if (
                    REQUIRE_SAME_MARKS_FOR_NEAR_DUPLICATES
                    and int(row["marks"])
                    != int(kept_row["marks"])
                ):
                    continue

                left_text = row[
                    "duplicate_canonical_text"
                ]
                right_text = kept_row[
                    "duplicate_canonical_text"
                ]

                lexical_score = (
                    lexical_similarity(
                        left_text,
                        right_text,
                    )
                )

                jaccard_score = token_jaccard(
                    left_text,
                    right_text,
                )

                semantic_score = float(
                    np.dot(
                        duplicate_vectors[
                            candidate_index
                        ],
                        duplicate_vectors[
                            kept_index
                        ],
                    )
                )

                best_lexical = max(
                    best_lexical,
                    lexical_score,
                )
                best_jaccard = max(
                    best_jaccard,
                    jaccard_score,
                )
                best_semantic = max(
                    best_semantic,
                    semantic_score,
                )

                exact_cleaned_match = bool(
                    left_text
                    and left_text == right_text
                )

                strong_lexical_match = bool(
                    lexical_score
                    >= NEAR_DUPLICATE_LEXICAL_THRESHOLD
                    and jaccard_score
                    >= NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
                )

                strong_semantic_match = bool(
                    semantic_score
                    >= NEAR_DUPLICATE_SEMANTIC_THRESHOLD
                    and jaccard_score
                    >= 0.75
                )

                if exact_cleaned_match:
                    duplicate_of_index = kept_index
                    duplicate_method = (
                        "exact_cleaned_text"
                    )
                    break

                if strong_lexical_match:
                    duplicate_of_index = kept_index
                    duplicate_method = (
                        "lexical_and_token_overlap"
                    )
                    break

                if strong_semantic_match:
                    duplicate_of_index = kept_index
                    duplicate_method = (
                        "minilm_and_token_overlap"
                    )
                    break

            is_duplicate = (
                duplicate_of_index
                is not None
            )

            if not is_duplicate:
                keep_indexes.append(
                    candidate_index
                )

            duplicate_of_question_id = (
                str(
                    working.loc[
                        duplicate_of_index,
                        "question_id",
                    ]
                )
                if is_duplicate
                else None
            )

            decision_rows.append(
                {
                    "question_id": str(
                        row["question_id"]
                    ),
                    "agent1_topic_index": int(
                        row[
                            "agent1_topic_index"
                        ]
                    ),
                    "near_duplicate_status": (
                        "duplicate_removed"
                        if is_duplicate
                        else "retained"
                    ),
                    "duplicate_of_question_id": (
                        duplicate_of_question_id
                    ),
                    "duplicate_detection_method": (
                        duplicate_method
                    ),
                    "maximum_lexical_similarity": round(
                        best_lexical,
                        6,
                    ),
                    "maximum_token_jaccard": round(
                        best_jaccard,
                        6,
                    ),
                    "maximum_minilm_similarity": round(
                        best_semantic,
                        6,
                    ),
                    "duplicate_canonical_text": (
                        row[
                            "duplicate_canonical_text"
                        ]
                    ),
                }
            )

        decisions_df = pd.DataFrame(
            decision_rows
        )

        manifest_df = working.merge(
            decisions_df,
            on=[
                "question_id",
                "agent1_topic_index",
                "duplicate_canonical_text",
            ],
            how="left",
            validate="one_to_one",
        )

        retained_df = (
            manifest_df[
                manifest_df[
                    "near_duplicate_status"
                ]
                == "retained"
            ]
            .copy()
            .reset_index(drop=True)
        )

        retained_df[
            "refined_unique_rank"
        ] = np.arange(
            1,
            len(retained_df) + 1,
        )

        return (
            retained_df,
            manifest_df,
        )


    if ENABLE_NEAR_DUPLICATE_GATE:
        (
            near_unique_candidates_df,
            near_duplicate_manifest_df,
        ) = detect_near_duplicates(
            unique_candidates_df
        )

    else:
        near_unique_candidates_df = (
            unique_candidates_df.copy()
        )

        near_unique_candidates_df[
            "near_duplicate_status"
        ] = "retained"

        near_unique_candidates_df[
            "duplicate_of_question_id"
        ] = None

        near_unique_candidates_df[
            "duplicate_detection_method"
        ] = None

        near_unique_candidates_df[
            "refined_unique_rank"
        ] = np.arange(
            1,
            len(
                near_unique_candidates_df
            ) + 1,
        )

        near_duplicate_manifest_df = (
            near_unique_candidates_df.copy()
        )


    near_duplicates_removed = int(
        (
            near_duplicate_manifest_df[
                "near_duplicate_status"
            ]
            == "duplicate_removed"
        ).sum()
    )

    print(
        f"Within-topic exact duplicates removed earlier: "
        f"{duplicates_removed}"
    )

    print(
        f"Within-topic near duplicates removed in refinement: "
        f"{near_duplicates_removed}"
    )

    print(
        f"Candidates after refined deduplication: "
        f"{len(near_unique_candidates_df)}"
    )

    display(
        near_duplicate_manifest_df[
            [
                "question_id",
                "detected_topic",
                "official_reference",
                "marks",
                "near_duplicate_status",
                "duplicate_of_question_id",
                "duplicate_detection_method",
                "maximum_lexical_similarity",
                "maximum_token_jaccard",
                "maximum_minilm_similarity",
                "question_text",
            ]
        ].head(80)
    )


## 10. Final Phase 2 — hierarchical hybrid relevance + full precision validation

The original hybrid retrieval stays intact: metadata policy, enriched Qdrant semantic evidence, question-only direct topic fit, enriched BM25, quality filtering, hard filters and downstream selection are preserved.

This revision adds only downstream precision signals:

- same-query enriched-vs-question-only **context gap**,
- persistent Qdrant **assessed-skill** vectors,
- generalized parser-noise detection,
- cross-topic ownership with **`NO_OWNER` abstention**,
- a final deterministic relevance verifier before the unchanged selection routine.

The goal is to replace context-carried false positives with the next strongest valid candidate while still allowing the existing question-count/coverage selector to fill the requested assessment whenever enough verified candidates exist.


In [ ]:
if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Skipped this candidate-processing stage because no questions matched the current hard filters.")
else:

    INCOMPLETE_TRAILING_PATTERNS = [
        # Stable page/instruction fragments only. Parser-specific truncated
        # tokens are handled structurally below rather than by literal words.
        r"\bdo not[.!]?$",
        r"\bturn over[.!]?$",
        r"\boutside[.!]?$",
    ]

    # Deliberately narrow and reusable. These tokens are highly
    # likely to indicate an incomplete short final line when the
    # line is not a question.
    TRUNCATED_FINAL_TOKENS = {
        "and",
        "or",
        "the",
        "a",
        "an",
    }

    MAX_TRUNCATED_FINAL_LINE_WORDS = 8

    ISOLATED_NOISE_LINES = {
        "do not",
        "turn over",
        "outside",
        "do not write outside the box",
    }

    ASSESSMENT_COMMAND_VERBS = {
        "calculate", "choose", "circle", "compare", "complete",
        "construct", "define", "describe", "determine", "develop",
        "discuss", "draw", "evaluate", "explain", "give", "identify",
        "implement", "name", "outline", "select", "show", "state",
        "tick", "trace", "what", "which", "write",
    }

    # Standalone short tokens that can legitimately appear in code/MCQ material.
    SAFE_ORPHAN_TOKENS = {
        "a", "b", "c", "d", "x", "y", "i", "j", "k",
        "if", "or", "to", "in", "is", "no", "do", "by", "of",
        "as", "at", "be", "on", "yes", "true", "false", "none",
        "input", "output", "start", "stop",
    }


    def is_non_instruction_label(
        segment: str,
    ) -> bool:
        cleaned = re.sub(
            r"\s+",
            " ",
            str(segment or "").strip(),
        )

        if not cleaned:
            return True

        label_patterns = [
            r"^(?:figure|table|diagram|flowchart)\s+\d+[a-z]?$",
            r"^\d+(?:\.\d+)+\s+[A-Za-z].*$",
            r"^[A-Za-z]\s+[A-Za-z]\s+[A-Za-z](?:\s+[A-Za-z])?$",
        ]

        return any(
            re.fullmatch(
                pattern,
                cleaned,
                flags=re.IGNORECASE,
            )
            for pattern in label_patterns
        )


    def split_instruction_segments(
        raw_text: str,
    ) -> list[str]:
        """
        Build sentence-like segments from the full question text.

        This catches an incomplete instructional sentence even when a
        later label such as 'Figure 7' is the final line.
        """
        normalized = re.sub(
            r"\s+",
            " ",
            raw_text,
        ).strip()

        sentence_segments = re.split(
            r"(?<=[.!?])\s+",
            normalized,
        )

        raw_lines = [
            re.sub(
                r"\s+",
                " ",
                line.strip(),
            )
            for line in raw_text.splitlines()
            if line.strip()
        ]

        combined = []
        seen = set()

        for segment in [
            *sentence_segments,
            *raw_lines,
        ]:
            cleaned = segment.strip()

            if (
                not cleaned
                or is_non_instruction_label(
                    cleaned
                )
            ):
                continue

            key = cleaned.lower()

            if key in seen:
                continue

            seen.add(key)
            combined.append(cleaned)

        return combined


    def detect_question_quality_issues(
        question_text: Any,
    ) -> list[str]:
        raw_text = str(
            question_text
            or ""
        ).strip()

        if not raw_text:
            return [
                "missing_question_text"
            ]

        issues: list[str] = []

        normalized_text = re.sub(
            r"\s+",
            " ",
            raw_text,
        ).strip()

        word_count = len(
            re.findall(
                r"[A-Za-z0-9]+",
                normalized_text,
            )
        )

        lowered = normalized_text.lower()

        # A short exam question is not automatically a parser fragment.
        # Commands such as "Define...", "State...", "Name..." and "Give..."
        # are legitimate AQA prompts even when they contain fewer than the old
        # eight-word heuristic.
        question_tokens = re.findall(
            r"[a-z0-9]+",
            lowered,
        )

        has_exam_command = bool(
            set(question_tokens)
            & ASSESSMENT_COMMAND_VERBS
        )

        if (
            word_count
            < ABSOLUTE_MIN_QUESTION_WORD_COUNT
        ):
            issues.append(
                "question_text_too_short"
            )
        elif (
            word_count
            < MIN_QUESTION_WORD_COUNT
            and not has_exam_command
            and not normalized_text.rstrip().endswith("?")
        ):
            issues.append(
                "question_text_too_short_without_complete_exam_command"
            )

        if any(
            re.search(
                pattern,
                lowered,
            )
            for pattern in (
                INCOMPLETE_TRAILING_PATTERNS
            )
        ):
            issues.append(
                "incomplete_trailing_fragment"
            )

        suspicious_segments = []

        for segment in split_instruction_segments(
            raw_text
        ):
            segment_tokens = re.findall(
                r"[a-z0-9]+",
                segment.lower(),
            )

            if not segment_tokens:
                continue

            segment_is_question = (
                segment.rstrip()
                .endswith("?")
            )

            # Keep this deliberately narrow to avoid overfitting.
            # A sentence-like instructional segment ending in
            # "and." or "or." is highly likely to be truncated.
            dangling_connector = bool(
                len(segment_tokens) >= 4
                and segment_tokens[-1]
                in {
                    "and",
                    "or",
                }
                and not segment_is_question
                and re.search(
                    r"[.!]\s*$",
                    segment,
                )
            )

            if dangling_connector:
                suspicious_segments.append(
                    segment
                )

        if suspicious_segments:
            issues.append(
                "probable_truncated_instruction_segment"
            )

        normalized_lines = [
            re.sub(
                r"\s+",
                " ",
                line.strip().lower(),
            )
            for line in raw_text.splitlines()
            if line.strip()
        ]

        if any(
            line in ISOLATED_NOISE_LINES
            for line in normalized_lines
        ):
            issues.append(
                "isolated_footer_or_instruction_fragment"
            )

        # General parser-noise signal: a short orphan alphabetic line near the
        # end of an otherwise substantive question is suspicious when it is not
        # a label, exam command, common language token, or code/MCQ literal.
        content_lines = [
            re.sub(r"\s+", " ", line.strip())
            for line in raw_text.splitlines()
            if line.strip()
            and not is_non_instruction_label(line.strip())
        ]

        substantive_word_count = len(
            re.findall(r"[A-Za-z0-9]+", raw_text)
        )

        trailing_orphan_fragments = []
        for line in content_lines[-2:]:
            alpha_tokens = re.findall(r"[A-Za-z]+", line.casefold())
            if len(alpha_tokens) != 1:
                continue

            token = alpha_tokens[0]
            if (
                substantive_word_count >= 8
                and 2 <= len(token) <= 8
                and token not in SAFE_ORPHAN_TOKENS
                and token not in ASSESSMENT_COMMAND_VERBS
                and not line.rstrip().endswith("?")
                and not re.search(r"[=<>()[\]{}:+*/\\]", line)
            ):
                trailing_orphan_fragments.append(line)

        if trailing_orphan_fragments:
            issues.append(
                "probable_parser_or_truncated_orphan_fragment"
            )

        return sorted(set(issues))


    def extract_assessed_skill_text(question_text: Any) -> str:
        """
        Return the instruction/answer-demand portion of a subquestion.

        Parent stems are useful for retrieval, but the final relevance decision
        should focus on what the learner must actually do. Find the first exact
        AQA-style command verb in the ORIGINAL subquestion text and retain text
        from that command onward. Using the original order prevents earlier
        parent-stem lines from being reintroduced by sentence/line merging.
        If no command is found, fall back to the whole subquestion rather than
        inventing a classification.
        """
        raw_text = str(question_text or "").strip()
        if not raw_text:
            return ""

        command_pattern = re.compile(
            r"\\b(?:"
            + "|".join(
                sorted(
                    (re.escape(verb) for verb in ASSESSMENT_COMMAND_VERBS),
                    key=len,
                    reverse=True,
                )
            )
            + r")\\b",
            flags=re.IGNORECASE,
        )
        match = command_pattern.search(raw_text)

        focused = (
            raw_text[match.start():]
            if match is not None
            else raw_text
        )

        # Precision fix: MCQ option text must never create topic relevance by
        # itself. Keep only the question stem/command here; the correct option is
        # checked later against the mark scheme before final selection.
        mcq_lines = [
            re.sub(r"\s+", " ", line).strip()
            for line in focused.replace("\r", "\n").splitlines()
            if re.sub(r"\s+", " ", line).strip()
        ]
        option_pattern = re.compile(r"^[A-D](?:[.)])?\s+.+$", re.IGNORECASE)
        option_positions = [
            position for position, line in enumerate(mcq_lines)
            if option_pattern.match(line)
        ]
        if len(option_positions) >= 2:
            mcq_lines = mcq_lines[:min(option_positions)]
            focused = "\n".join(mcq_lines).strip()

        return re.sub(r"\s+", " ", focused).strip()


    def compute_shared_role_allocation_targets(
        *,
        requested_count: int,
        minimum_primary: int,
        minimum_supporting: int,
        has_primary: bool,
        has_supporting: bool,
    ) -> dict[str, int | float]:
        """
        Compute one dynamic role plan for the whole Notebook 05 run.

        The user's minima are respected first. When both roles exist, the
        remaining preference is DEFAULT_PRIMARY_PREFERRED_SHARE. This same
        plan is reused by retrieval quotas, warnings and final selection.
        """
        requested_count = int(max(0, requested_count))
        minimum_primary = int(max(0, minimum_primary))
        minimum_supporting = int(max(0, minimum_supporting))

        if requested_count <= 0:
            return {
                "preferred_primary_count": 0,
                "preferred_supporting_count": 0,
                "preferred_primary_share": 0.0,
            }

        if has_primary and has_supporting:
            preferred_primary = max(
                minimum_primary,
                int(
                    math.ceil(
                        requested_count
                        * DEFAULT_PRIMARY_PREFERRED_SHARE
                    )
                ),
            )

            # Preserve the user's Supporting minimum.
            preferred_primary = min(
                preferred_primary,
                max(
                    0,
                    requested_count - minimum_supporting,
                ),
            )
            preferred_supporting = (
                requested_count - preferred_primary
            )

            if preferred_supporting < minimum_supporting:
                preferred_supporting = minimum_supporting
                preferred_primary = max(
                    0,
                    requested_count - preferred_supporting,
                )

        elif has_primary:
            preferred_primary = requested_count
            preferred_supporting = 0

        elif has_supporting:
            preferred_primary = 0
            preferred_supporting = requested_count

        else:
            preferred_primary = 0
            preferred_supporting = 0

        return {
            "preferred_primary_count": int(preferred_primary),
            "preferred_supporting_count": int(preferred_supporting),
            "preferred_primary_share": (
                float(preferred_primary)
                / float(requested_count)
                if requested_count > 0
                else 0.0
            ),
        }


    def derive_topic_candidate_requirements(
        topics_frame: pd.DataFrame,
    ) -> dict[int, int]:
        """
        Derive per-topic candidate quotas from the frontend request.

        Order of intent:
          1. one candidate for each required Agent 1 topic (primary first),
          2. satisfy minimum primary/supporting counts,
          3. use remaining candidate quota on primary topics first, then
             supporting topics round-robin.

        These quotas influence adaptive thresholding only; they never allow a
        question that fails the quality/concept gates.
        """
        topic_rows = (
            topics_frame[
                [
                    "agent1_topic_index",
                    "detected_topic",
                    "role",
                    "official_reference",
                    "agent1_official_reference",
                    "ranking_score",
                    "role_weight",
                ]
            ]
            .drop_duplicates(subset=["agent1_topic_index"])
            .copy()
        )
        topic_rows["agent1_topic_index"] = topic_rows["agent1_topic_index"].astype(int)
        topic_rows["_role_order"] = topic_rows["role"].map(
            {"primary": 0, "supporting": 1}
        ).fillna(2)
        topic_rows = topic_rows.sort_values(
            ["_role_order", "agent1_topic_index"],
            ascending=[True, True],
        )

        requirements = {
            int(index): 0
            for index in topic_rows["agent1_topic_index"].tolist()
        }

        required_topic_indexes = {
            int(value)
            for value in request.get("required_agent1_topic_indexes", [])
        }

        for topic_index in required_topic_indexes:
            if topic_index in requirements:
                requirements[topic_index] = 1

        def distribute_role_requirement(role: str, required_total: int) -> None:
            role_indexes = [
                int(value)
                for value in topic_rows.loc[
                    topic_rows["role"].eq(role),
                    "agent1_topic_index",
                ].tolist()
            ]
            if not role_indexes:
                return
            already = sum(requirements[index] for index in role_indexes)
            remaining = max(0, int(required_total) - already)
            position = 0
            while remaining > 0:
                topic_index = role_indexes[position % len(role_indexes)]
                requirements[topic_index] += 1
                remaining -= 1
                position += 1

        distribute_role_requirement(
            "primary", int(request["minimum_primary_questions"])
        )
        distribute_role_requirement(
            "supporting", int(request["minimum_supporting_questions"])
        )

        primary_indexes = [
            int(value)
            for value in topic_rows.loc[
                topic_rows["role"].eq("primary"),
                "agent1_topic_index",
            ].tolist()
        ]
        supporting_indexes = [
            int(value)
            for value in topic_rows.loc[
                topic_rows["role"].eq("supporting"),
                "agent1_topic_index",
            ].tolist()
        ]

        shared_role_targets = compute_shared_role_allocation_targets(
            requested_count=int(request["number_of_questions"]),
            minimum_primary=int(request["minimum_primary_questions"]),
            minimum_supporting=int(request["minimum_supporting_questions"]),
            has_primary=bool(primary_indexes),
            has_supporting=bool(supporting_indexes),
        )

        preferred_primary_total = int(
            shared_role_targets["preferred_primary_count"]
        )
        preferred_supporting_total = int(
            shared_role_targets["preferred_supporting_count"]
        )

        current_primary_total = sum(
            requirements[index]
            for index in primary_indexes
        )
        current_supporting_total = sum(
            requirements[index]
            for index in supporting_indexes
        )

        remaining_primary = max(
            0,
            preferred_primary_total - current_primary_total,
        )
        remaining_supporting = max(
            0,
            preferred_supporting_total - current_supporting_total,
        )

        # Role-local round-robin allocation avoids one supporting topic taking
        # every remaining slot when several Supporting concepts were approved.
        position = 0
        while remaining_primary > 0 and primary_indexes:
            topic_index = primary_indexes[
                position % len(primary_indexes)
            ]
            requirements[topic_index] += 1
            remaining_primary -= 1
            position += 1

        position = 0
        while remaining_supporting > 0 and supporting_indexes:
            topic_index = supporting_indexes[
                position % len(supporting_indexes)
            ]
            requirements[topic_index] += 1
            remaining_supporting -= 1
            position += 1

        return requirements




    # ================================================================
    # FINAL PHASE 2 — HIERARCHICAL HYBRID RELEVANCE
    # ================================================================

    HYBRID_STOPWORDS = {
        "a", "an", "and", "are", "as", "at", "be", "by", "describe",
        "explain", "for", "from", "give", "how", "in", "is", "it",
        "of", "on", "or", "state", "that", "the", "their", "these",
        "this", "to", "two", "what", "when", "which", "with",
    }


    def hybrid_tokens(value: Any) -> list[str]:
        """
        Generic lexical tokenisation for topic alignment and BM25.

        No syllabus-topic keyword lists are used.
        """
        tokens = re.findall(
            r"[a-z0-9]+",
            str(value or "").casefold(),
        )
        return [
            token
            for token in tokens
            if token not in HYBRID_STOPWORDS
            and len(token) > 1
        ]



    OWNERSHIP_GENERIC_CONCEPT_TOKENS = {
        "algorithm", "program", "programming", "code", "computer",
        "computing", "data", "question", "assessment", "gcse", "aqa",
    }

    # ---------------------------------------------------------
    # OWNERSHIP-ONLY LEXICAL TOKEN NORMALISATION
    # ---------------------------------------------------------
    # Retrieval/BM25 tokenisation is intentionally NOT changed.
    #
    # These low-information tokens must never count as evidence that the
    # student's actual answer-demand assesses a detected syllabus concept.
    # In particular, exam boilerplate such as "Shade one lozenge" must not
    # make a topic beginning with "One- ..." look explicitly relevant.
    OWNERSHIP_LOW_INFORMATION_TOKENS = {
        # cardinal numbers are weak unigram ownership evidence
        "zero", "one", "two", "three", "four", "five",
        "six", "seven", "eight", "nine", "ten",
        "eleven", "twelve", "thirteen", "fourteen",
        "fifteen", "sixteen", "seventeen", "eighteen",
        "nineteen", "twenty",

        # common exam-response boilerplate
        "shade", "lozenge", "lozenges", "tick", "ticks",
        "circle", "box", "boxes",
    }


    def normalise_ownership_token(token: Any) -> str:
        """
        Conservative morphology normalisation used only for ownership lexical
        evidence. This allows e.g. `array` and `arrays` to match without
        changing Qdrant/BM25 retrieval behaviour.
        """
        value = str(token or "").casefold().strip()

        if (
            not value
            or value.isdigit()
            or value in OWNERSHIP_GENERIC_CONCEPT_TOKENS
            or value in OWNERSHIP_LOW_INFORMATION_TOKENS
        ):
            return ""

        if len(value) > 4 and value.endswith("ies"):
            return value[:-3] + "y"

        if (
            len(value) > 4
            and value.endswith("s")
            and not value.endswith("ss")
        ):
            return value[:-1]

        return value


    def ownership_lexical_tokens(value: Any) -> list[str]:
        """
        Content-bearing tokens for FINAL CONCEPT OWNERSHIP only.

        This deliberately removes cardinal-number/exam-instruction noise and
        lightly normalises singular/plural forms. It is generic and does not
        contain any syllabus-topic keyword list.
        """
        normalised = []

        for token in hybrid_tokens(value):
            cleaned = normalise_ownership_token(token)
            if cleaned:
                normalised.append(cleaned)

        return normalised


    def topic_concept_tokens(row: pd.Series) -> set[str]:
        parts = [
            str(row.get("detected_topic") or ""),
            str(row.get("official_concept_name") or ""),
        ]
        detected_concepts = row.get("detected_concepts")
        if isinstance(detected_concepts, list):
            parts.extend(str(value) for value in detected_concepts)

        return {
            token
            for token in hybrid_tokens(" ".join(parts))
            if token not in OWNERSHIP_GENERIC_CONCEPT_TOKENS
        }


    def assessed_skill_lexical_overlap_score(
        assessed_text: Any,
        concept_tokens: set[str],
    ) -> float:
        if not concept_tokens:
            return 0.0

        assessed_tokens = set(
            ownership_lexical_tokens(assessed_text)
        )
        normalised_concept_tokens = {
            cleaned
            for token in concept_tokens
            if (
                cleaned := normalise_ownership_token(token)
            )
        }

        if not assessed_tokens or not normalised_concept_tokens:
            return 0.0

        return float(
            len(
                assessed_tokens
                & normalised_concept_tokens
            )
            / max(
                1,
                len(normalised_concept_tokens),
            )
        )


    def canonical_name_alignment(
        detailed_topic: Any,
        canonical_topic: Any,
    ) -> float:
        """
        Conservative token-containment score.

        Examples with only word-order changes receive a high score, while a
        narrow detailed concept under a broad canonical title receives a low
        score. A low/uncertain score sends the topic through the safer broad
        retrieval path.
        """
        detailed = set(hybrid_tokens(detailed_topic))
        canonical = set(hybrid_tokens(canonical_topic))

        if not detailed or not canonical:
            return 0.0

        denominator = min(
            len(detailed),
            len(canonical),
        )

        if denominator <= 0:
            return 0.0

        return float(
            len(detailed & canonical)
            / denominator
        )


    def build_topic_policy_frame() -> pd.DataFrame:
        topic_frame = (
            validated_topics_df
            .drop_duplicates(subset=["agent1_topic_index"])
            .sort_values("agent1_topic_index")
            .copy()
        )

        reference_counts = (
            topic_frame["official_reference"]
            .astype(str)
            .value_counts()
            .to_dict()
        )

        topic_frame["canonical_reference_topic_count"] = (
            topic_frame["official_reference"]
            .astype(str)
            .map(reference_counts)
            .astype(int)
        )

        topic_frame["canonical_name_alignment"] = topic_frame.apply(
            lambda row: canonical_name_alignment(
                row.get("detected_topic"),
                row.get("official_concept_name"),
            ),
            axis=1,
        )

        topic_frame["exact_detailed_reference"] = (
            topic_frame["agent1_official_reference"]
            .astype(str)
            .eq(
                topic_frame["official_reference"]
                .astype(str)
            )
        )

        topic_frame["canonical_reference_shared"] = (
            topic_frame["canonical_reference_topic_count"]
            .gt(1)
        )

        topic_frame["metadata_trusted_specific"] = (
            topic_frame["exact_detailed_reference"]
            & ~topic_frame["canonical_parent_applied"].astype(bool)
            & ~topic_frame["canonical_reference_shared"]
            & (
                topic_frame["canonical_name_alignment"]
                >= SPECIFIC_CANONICAL_NAME_ALIGNMENT_MIN
            )
        )

        topic_frame["retrieval_pool_type"] = np.where(
            topic_frame["metadata_trusted_specific"],
            "metadata_trusted_specific",
            "broad_shared_hybrid_disambiguation",
        )

        topic_frame["retrieval_policy_reason"] = topic_frame.apply(
            lambda row: (
                "exact_unshared_reference_with_strong_canonical_name_alignment"
                if bool(row["metadata_trusted_specific"])
                else (
                    "canonical_parent_resolution"
                    if bool(row.get("canonical_parent_applied"))
                    else (
                        "canonical_reference_shared_by_multiple_agent1_topics"
                        if bool(row["canonical_reference_shared"])
                        else "detailed_topic_narrower_or_different_from_canonical_name"
                    )
                )
            ),
            axis=1,
        )

        return topic_frame


    topic_retrieval_policy_df = build_topic_policy_frame()

    print("Final per-topic retrieval policy:")
    display(
        topic_retrieval_policy_df[
            [
                "agent1_topic_index",
                "detected_topic",
                "role",
                "agent1_official_reference",
                "official_reference",
                "official_concept_name",
                "canonical_name_alignment",
                "canonical_reference_topic_count",
                "canonical_parent_applied",
                "retrieval_pool_type",
                "retrieval_policy_reason",
            ]
        ]
    )


    def semantic_recall_evidence_excerpt(
        row: pd.Series,
        maximum_characters: int = 1200,
    ) -> str:
        """
        Use Agent 1's own lesson/chunk evidence to broaden semantic recall.
        Topic text remains first so MiniLM truncation cannot hide the concept.
        """
        evidence = re.sub(
            r"\s+",
            " ",
            str(row.get("query_evidence") or ""),
        ).strip()
        return evidence[:maximum_characters]


    def phase2_direct_topic_query(
        row: pd.Series,
    ) -> str:
        """
        Compact precision query.

        Full Agent 1 chunk/lesson evidence is already included in the upstream
        Qdrant retrieval query. Direct-topic scoring stays compact so valid
        differently worded exam questions are not penalised by a long/noisy
        transcript representation.
        """
        return "\n".join(
            [
                "AQA GCSE Computer Science assessment question.",
                (
                    "Detailed lesson topic: "
                    f"{str(row.get('detected_topic') or '').strip()}."
                ),
                (
                    "Canonical syllabus concept: "
                    f"{str(row.get('official_concept_name') or '').strip()}."
                ),
            ]
        ).strip()


    def detected_topic_only_query(
        row: pd.Series,
    ) -> str:
        """
        Compact detailed-concept representation for precision/ownership scoring.
        Retrieval recall continues to use Agent 1 lesson evidence upstream.
        """
        detected_concepts = row.get("detected_concepts")
        if not isinstance(detected_concepts, list):
            detected_concepts = []

        concept_text = "; ".join(
            str(value).strip()
            for value in detected_concepts
            if str(value).strip()
        )

        return "\n".join(
            [
                "AQA GCSE Computer Science assessment question.",
                (
                    "Specific detected lesson concept: "
                    f"{str(row.get('detected_topic') or '').strip()}."
                ),
                (
                    f"Detected concept evidence: {concept_text}."
                    if concept_text
                    else ""
                ),
            ]
        ).strip()


    def canonical_topic_only_query(
        row: pd.Series,
    ) -> str:
        return "\n".join(
            [
                "AQA GCSE Computer Science assessment question.",
                (
                    "Canonical syllabus concept: "
                    f"{str(row.get('official_concept_name') or '').strip()}."
                ),
            ]
        ).strip()


    def detected_topic_specific_tokens(
        row: pd.Series,
    ) -> set[str]:
        parts = [str(row.get("detected_topic") or "")]
        detected_concepts = row.get("detected_concepts")
        if isinstance(detected_concepts, list):
            parts.extend(str(value) for value in detected_concepts)

        return set(
            ownership_lexical_tokens(
                " ".join(parts)
            )
        )


    def canonical_topic_specific_tokens(
        row: pd.Series,
    ) -> set[str]:
        return set(
            ownership_lexical_tokens(
                str(row.get("official_concept_name") or "")
            )
        )


    def same_official_reference_branch(
        left_reference: Any,
        right_reference: Any,
    ) -> bool:
        """
        Treat an exact reference and its dotted parent/child as the same
        conceptual branch for child-ownership competition. The competition is
        intended to find a genuinely different official syllabus concept, not
        reject a question because a neighbouring granularity of the same concept
        receives a marginally higher embedding score.
        """
        left = str(left_reference or "").strip()
        right = str(right_reference or "").strip()
        if not left or not right:
            return left == right
        return (
            left == right
            or left.startswith(f"{right}.")
            or right.startswith(f"{left}.")
        )


    def official_concept_competition_query(
        concept_name: Any,
    ) -> str:
        """
        Canonical concept-only representation used solely to ask which official
        AQA concept best owns a child's assessed skill. No lesson evidence,
        parent context, metadata strength or BM25 is included.
        """
        cleaned = str(concept_name or "").strip()
        return "\n".join(
            [
                "AQA GCSE Computer Science assessment question.",
                f"Detailed lesson topic: {cleaned}.",
                f"Canonical syllabus concept: {cleaned}.",
            ]
        ).strip()


    def phase2_question_only_text(
        row: pd.Series,
    ) -> str:
        return "\n".join(
            [
                "AQA GCSE Computer Science assessment question.",
                str(row.get("question_text") or "").strip(),
            ]
        ).strip()


    def phase2_bm25_query(
        row: pd.Series,
    ) -> str:
        """
        Build the lexical BM25 query from Agent 1 evidence rather than using
        only the short detected-topic label.

        Query components are intentionally generic and come only from the
        current Agent 1 handoff:
          - detected topic
          - detected concepts
          - topic-specific lesson evidence

        BM25 tokenisation already removes generic stopwords and de-duplicates
        query terms, so repeated words do not receive artificial extra weight.
        """
        detected_concepts = row.get("detected_concepts")

        if not isinstance(detected_concepts, list):
            detected_concepts = [
                str(row.get("detected_topic") or "").strip()
            ]

        parts = [
            str(row.get("detected_topic") or "").strip(),
            " ".join(
                str(value).strip()
                for value in detected_concepts
                if str(value).strip()
            ),
            str(row.get("query_evidence") or "").strip(),
        ]

        return "\n".join(
            part
            for part in parts
            if part
        ).strip()


    def bm25_scores(
        query_text: str,
        document_texts: list[str],
    ) -> np.ndarray:
        """
        Pure-Python BM25 over the current topic's candidate set.

        The score is later normalised to 0–1 within that topic pool.
        """
        from collections import Counter

        documents = [
            hybrid_tokens(text)
            for text in document_texts
        ]

        query_terms = list(
            dict.fromkeys(
                hybrid_tokens(query_text)
            )
        )

        document_count = len(documents)

        if document_count == 0:
            return np.zeros(0, dtype=float)

        if not query_terms:
            return np.zeros(
                document_count,
                dtype=float,
            )

        document_lengths = np.array(
            [len(tokens) for tokens in documents],
            dtype=float,
        )

        average_length = float(
            document_lengths.mean()
        ) if document_count else 0.0

        if average_length <= 0:
            average_length = 1.0

        frequencies = [
            Counter(tokens)
            for tokens in documents
        ]

        document_frequency = {
            term: sum(
                1
                for token_counts in frequencies
                if token_counts.get(term, 0) > 0
            )
            for term in query_terms
        }

        scores = np.zeros(
            document_count,
            dtype=float,
        )

        for position, token_counts in enumerate(frequencies):
            document_length = document_lengths[position]

            for term in query_terms:
                term_frequency = float(
                    token_counts.get(term, 0)
                )

                if term_frequency <= 0:
                    continue

                df = float(
                    document_frequency.get(term, 0)
                )

                idf = math.log(
                    1.0
                    + (
                        document_count
                        - df
                        + 0.5
                    )
                    / (
                        df
                        + 0.5
                    )
                )

                length_normaliser = (
                    1.0
                    - BM25_B
                    + BM25_B
                    * (
                        document_length
                        / average_length
                    )
                )

                denominator = (
                    term_frequency
                    + BM25_K1
                    * length_normaliser
                )

                scores[position] += (
                    idf
                    * (
                        term_frequency
                        * (BM25_K1 + 1.0)
                    )
                    / denominator
                )

        return scores


    def is_hierarchical_subquestion_number(value: Any) -> bool:
        """
        Return True when the stored AQA question number is structurally a
        child/subquestion (for example 04.5 or 14.1).

        The current assessment DB does not store an explicit provenance flag
        proving that an official reference was assigned independently to a
        child row. Therefore a dotted/child number is conservatively treated
        as parent-inherited metadata for *bypass* purposes only. The metadata
        is still retained as a normal 5% relevance signal.
        """
        normalized = re.sub(r"\s+", "", str(value or "").strip())
        if not normalized:
            return False

        return bool(
            re.fullmatch(r"\d+(?:\.\d+)+", normalized)
            or re.fullmatch(r"\d+\([a-z0-9]+\)", normalized, flags=re.IGNORECASE)
        )


    def add_hybrid_relevance_scores(
        candidates: pd.DataFrame,
    ) -> pd.DataFrame:
        working = candidates.copy()

        if working.empty:
            working["is_hierarchical_subquestion"] = pd.Series(dtype=bool)
            working["metadata_reference_scope"] = pd.Series(dtype=str)
            working["metadata_trusted_candidate_specific"] = pd.Series(dtype=bool)
            for column in [
                "direct_topic_semantic_score",
                "question_focus_semantic_score",
                "context_enriched_direct_topic_score",
                "context_dependency_gap",
                "context_dependency_ratio",
                "assessed_skill_text",
                "assessed_skill_semantic_score",
                "assessed_skill_official_concept_score",
                "best_competing_assessed_skill_score",
                "assessed_skill_lexical_overlap",
                "detected_topic_assessed_skill_score",
                "canonical_topic_assessed_skill_score",
                "detected_topic_specificity_delta",
                "detected_topic_specific_lexical_overlap",
                "ownership_core_fit_score",
                "context_dependency_flag",
                "bm25_lexical_score",
                "bm25_lexical_normalized",
                "metadata_strength_score",
                "hybrid_relevance_score",
                "semantic_gate_score",
            ]:
                working[column] = pd.Series(dtype=float)
            working["assessed_skill_official_owner_is_requested"] = pd.Series(dtype=bool)
            working["detected_topic_specificity_required"] = pd.Series(dtype=bool)
            working["exact_detected_concept_gate_passed"] = pd.Series(dtype=bool)
            working["exact_detected_concept_gate_reason"] = pd.Series(dtype=str)
            working["best_competing_official_reference"] = pd.Series(dtype=str)
            working["best_competing_official_concept"] = pd.Series(dtype=str)
            working["child_direct_evidence_passed"] = pd.Series(dtype=bool)
            working["child_direct_evidence_reason"] = pd.Series(dtype=str)
            return working

        policy_columns = [
            "agent1_topic_index",
            "canonical_reference_topic_count",
            "canonical_name_alignment",
            "exact_detailed_reference",
            "canonical_reference_shared",
            "metadata_trusted_specific",
            "retrieval_pool_type",
            "retrieval_policy_reason",
        ]

        working = working.merge(
            topic_retrieval_policy_df[
                policy_columns
            ],
            on="agent1_topic_index",
            how="left",
            validate="many_to_one",
        )

        # A metadata-mismatch rescue candidate can never use the
        # metadata-trusted-specific semantic bypass. Its stored reference did
        # not match the requested topic, so detailed-topic relevance must prove
        # the match through the normal broad hybrid gate.
        metadata_rescue_mask = (
            working.get(
                "retrieval_stage",
                pd.Series(
                    "",
                    index=working.index,
                ),
            )
            .astype(str)
            .eq(
                "metadata_mismatch_semantic_rescue"
            )
        )

        if metadata_rescue_mask.any():
            working.loc[
                metadata_rescue_mask,
                "metadata_trusted_specific",
            ] = False

            working.loc[
                metadata_rescue_mask,
                "retrieval_pool_type",
            ] = (
                "metadata_mismatch_hybrid_rescue"
            )

            working.loc[
                metadata_rescue_mask,
                "retrieval_policy_reason",
            ] = (
                "stored_reference_mismatch_requires_hybrid_proof"
            )

        # Candidate-level metadata scope. Topic-level exact metadata remains
        # useful evidence, but a child/subquestion cannot inherit the
        # metadata-trusted semantic bypass unless the DB can prove a
        # child-specific mapping. The current schema has no such provenance
        # field, so dotted child numbers take the normal hybrid path.
        working["is_hierarchical_subquestion"] = working[
            "question_number"
        ].map(is_hierarchical_subquestion_number)
        working["metadata_reference_scope"] = np.where(
            working["is_hierarchical_subquestion"].astype(bool),
            "parent_inherited_subquestion",
            "candidate_level_or_main_question",
        )
        working["metadata_trusted_candidate_specific"] = (
            working["metadata_trusted_specific"].astype(bool)
            & ~working["is_hierarchical_subquestion"].astype(bool)
        )

        topic_rows = (
            validated_topics_df
            .drop_duplicates(subset=["agent1_topic_index"])
            .sort_values("agent1_topic_index")
        )

        direct_queries = topic_rows.apply(
            phase2_direct_topic_query,
            axis=1,
        ).tolist()

        direct_query_vectors = model.encode(
            direct_queries,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        direct_vector_lookup = {
            int(topic_index): direct_query_vectors[position]
            for position, topic_index in enumerate(
                topic_rows["agent1_topic_index"].tolist()
            )
        }

        # Separate the exact Agent 1 concept from the broader canonical
        # syllabus concept. Broad metadata may help retrieval/ranking, but
        # cannot by itself prove that the assessed skill matches the detected
        # lesson concept.
        detected_only_vectors = model.encode(
            topic_rows.apply(
                detected_topic_only_query,
                axis=1,
            ).tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        canonical_only_vectors = model.encode(
            topic_rows.apply(
                canonical_topic_only_query,
                axis=1,
            ).tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        detected_only_vector_lookup = {
            int(topic_index): detected_only_vectors[position]
            for position, topic_index in enumerate(
                topic_rows["agent1_topic_index"].tolist()
            )
        }
        canonical_only_vector_lookup = {
            int(topic_index): canonical_only_vectors[position]
            for position, topic_index in enumerate(
                topic_rows["agent1_topic_index"].tolist()
            )
        }

        detected_specific_token_lookup = {
            int(row["agent1_topic_index"]): detected_topic_specific_tokens(row)
            for _, row in topic_rows.iterrows()
        }
        canonical_specific_token_lookup = {
            int(row["agent1_topic_index"]): canonical_topic_specific_tokens(row)
            for _, row in topic_rows.iterrows()
        }
        detailed_specificity_required_lookup = {
            int(row["agent1_topic_index"]): bool(
                detected_specific_token_lookup[int(row["agent1_topic_index"])]
                - canonical_specific_token_lookup[int(row["agent1_topic_index"])]
            )
            for _, row in topic_rows.iterrows()
        }

        # Structural child-ownership reference bank. This is built from the
        # already approved official mapping table, not from the current transcript
        # and not from hand-written topic rules. It gives single-topic runs a real
        # competing concept space without tuning medians or per-topic thresholds.
        official_concept_competition_df = (
            mapping_df[["official_reference", "official_concept_name"]]
            .dropna(subset=["official_reference", "official_concept_name"])
            .copy()
        )
        official_concept_competition_df["official_reference"] = (
            official_concept_competition_df["official_reference"]
            .astype(str)
            .str.strip()
        )
        official_concept_competition_df["official_concept_name"] = (
            official_concept_competition_df["official_concept_name"]
            .astype(str)
            .str.strip()
        )
        official_concept_competition_df = (
            official_concept_competition_df[
                official_concept_competition_df["official_reference"].ne("")
                & official_concept_competition_df["official_concept_name"].ne("")
            ]
            .drop_duplicates(subset=["official_reference"], keep="first")
            .sort_values("official_reference")
            .reset_index(drop=True)
        )

        if official_concept_competition_df.empty:
            official_concept_competition_df = (
                topic_rows[["official_reference", "official_concept_name"]]
                .dropna(subset=["official_reference", "official_concept_name"])
                .drop_duplicates(subset=["official_reference"], keep="first")
                .reset_index(drop=True)
            )

        if official_concept_competition_df.empty:
            raise RuntimeError(
                "No approved official concepts are available for child direct-evidence ownership."
            )

        official_concept_query_vectors = model.encode(
            official_concept_competition_df["official_concept_name"]
            .map(official_concept_competition_query)
            .tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        official_competition_references = (
            official_concept_competition_df["official_reference"]
            .astype(str)
            .tolist()
        )
        official_competition_names = (
            official_concept_competition_df["official_concept_name"]
            .astype(str)
            .tolist()
        )
        official_competition_index_by_reference = {
            reference: position
            for position, reference in enumerate(official_competition_references)
        }

        # Compare the SAME detailed-topic query against two stable Qdrant views:
        # (1) Notebook 04 enriched topic/subtopic/question/context vector, and
        # (2) whole-question-only vector. Their difference is a genuine context
        # dependency signal rather than a comparison of different queries.
        enriched_question_vectors = qdrant_vectors_for_rows(working)
        question_vectors = qdrant_question_focus_vectors_for_rows(working)

        working["context_enriched_direct_topic_score"] = [
            float(
                enriched_question_vectors[position]
                @ direct_vector_lookup[int(row["agent1_topic_index"])]
            )
            for position, (_, row) in enumerate(working.iterrows())
        ]

        working["question_focus_semantic_score"] = [
            float(
                question_vectors[position]
                @ direct_vector_lookup[int(row["agent1_topic_index"])]
            )
            for position, (_, row) in enumerate(working.iterrows())
        ]

        # Keep the existing hybrid direct-topic field backed by the question-only
        # representation. Metadata-rescue rows may override this later exactly as
        # before, while the audit-only question_focus score stays truly question-only.
        working["direct_topic_semantic_score"] = working[
            "question_focus_semantic_score"
        ].astype(float)

        working["context_dependency_gap"] = np.maximum(
            0.0,
            working["context_enriched_direct_topic_score"].astype(float)
            - working["question_focus_semantic_score"].astype(float),
        )
        working["context_dependency_ratio"] = (
            working["context_dependency_gap"].astype(float)
            / working["context_enriched_direct_topic_score"].astype(float)
            .abs().clip(lower=1e-6)
        ).clip(lower=0.0)

        # Deterministically focus on what the student is actually asked to do.
        working["assessed_skill_text"] = working["question_text"].map(
            extract_assessed_skill_text
        )
        assessed_skill_vectors = qdrant_assessed_skill_vectors_for_rows(
            working
        )
        working["assessed_skill_semantic_score"] = [
            float(
                assessed_skill_vectors[position]
                @ direct_vector_lookup[int(row["agent1_topic_index"])]
            )
            for position, (_, row) in enumerate(working.iterrows())
        ]

        working["detected_topic_assessed_skill_score"] = [
            float(
                assessed_skill_vectors[position]
                @ detected_only_vector_lookup[int(row["agent1_topic_index"])]
            )
            for position, (_, row) in enumerate(working.iterrows())
        ]
        working["canonical_topic_assessed_skill_score"] = [
            float(
                assessed_skill_vectors[position]
                @ canonical_only_vector_lookup[int(row["agent1_topic_index"])]
            )
            for position, (_, row) in enumerate(working.iterrows())
        ]
        working["detected_topic_specificity_delta"] = (
            working["detected_topic_assessed_skill_score"].astype(float)
            - working["canonical_topic_assessed_skill_score"].astype(float)
        )
        working["detected_topic_specific_lexical_overlap"] = [
            assessed_skill_lexical_overlap_score(
                row.get("assessed_skill_text"),
                detected_specific_token_lookup[int(row["agent1_topic_index"])],
            )
            for _, row in working.iterrows()
        ]
        working["detected_topic_specificity_required"] = [
            bool(
                detailed_specificity_required_lookup[
                    int(row["agent1_topic_index"])
                ]
            )
            for _, row in working.iterrows()
        ]

        exact_semantic_support = (
            working["detected_topic_assessed_skill_score"]
            .astype(float)
            .ge(BROAD_HYBRID_RELEVANCE_FLOOR)
            & working["detected_topic_specificity_delta"]
            .astype(float)
            .ge(-EXACT_CONCEPT_CANONICAL_TOLERANCE)
        )
        exact_lexical_support = (
            working["detected_topic_specific_lexical_overlap"]
            .astype(float)
            .gt(0.0)
        )
        working["exact_detected_concept_gate_passed"] = (
            ~working["detected_topic_specificity_required"].astype(bool)
            | exact_lexical_support
            | exact_semantic_support
        )
        working["exact_detected_concept_gate_reason"] = [
            (
                "canonical_and_detected_concept_have_no_extra_specificity"
                if not bool(required)
                else (
                    "specific_detected_concept_lexical_evidence"
                    if bool(lexical)
                    else (
                        "specific_detected_concept_semantically_supported"
                        if bool(semantic)
                        else "broad_canonical_fit_without_specific_detected_concept_support"
                    )
                )
            )
            for required, lexical, semantic in zip(
                working["detected_topic_specificity_required"].tolist(),
                exact_lexical_support.tolist(),
                exact_semantic_support.tolist(),
            )
        ]

        # ------------------------------------------------------------
        # CHILD DIRECT-EVIDENCE SEMANTIC OWNERSHIP
        # ------------------------------------------------------------
        # Compare the child's own assessed-skill vector against the official AQA
        # concept space. This is deliberately independent of parent context,
        # retrieval BM25 and metadata strength. No learned median or new absolute
        # threshold is introduced: the requested concept simply has to own the
        # assessed skill relative to genuinely different official concepts.
        requested_concept_scores: list[float] = []
        best_competing_scores: list[float] = []
        best_competing_references: list[str] = []
        best_competing_names: list[str] = []
        requested_is_official_owner: list[bool] = []

        for position, (_, row) in enumerate(working.iterrows()):
            assessed_vector = assessed_skill_vectors[position]
            requested_reference = str(row.get("official_reference") or "").strip()

            requested_index = official_competition_index_by_reference.get(
                requested_reference
            )
            if requested_index is None:
                requested_score = float(
                    assessed_vector
                    @ direct_vector_lookup[int(row["agent1_topic_index"])]
                )
            else:
                requested_score = float(
                    assessed_vector
                    @ official_concept_query_vectors[requested_index]
                )

            competitor_positions = [
                competitor_position
                for competitor_position, competitor_reference in enumerate(
                    official_competition_references
                )
                if not same_official_reference_branch(
                    requested_reference,
                    competitor_reference,
                )
            ]

            if competitor_positions:
                competitor_matrix = official_concept_query_vectors[
                    competitor_positions
                ]
                competitor_scores = competitor_matrix @ assessed_vector
                local_best = int(np.argmax(competitor_scores))
                best_position = competitor_positions[local_best]
                best_score = float(competitor_scores[local_best])
                best_reference = official_competition_references[best_position]
                best_name = official_competition_names[best_position]
                is_requested_owner = bool(
                    requested_score >= best_score - 1e-12
                )
            else:
                best_score = 0.0
                best_reference = ""
                best_name = ""
                is_requested_owner = True

            requested_concept_scores.append(requested_score)
            best_competing_scores.append(best_score)
            best_competing_references.append(best_reference)
            best_competing_names.append(best_name)
            requested_is_official_owner.append(is_requested_owner)

        working["assessed_skill_official_concept_score"] = (
            requested_concept_scores
        )
        working["best_competing_assessed_skill_score"] = (
            best_competing_scores
        )
        working["best_competing_official_reference"] = (
            best_competing_references
        )
        working["best_competing_official_concept"] = (
            best_competing_names
        )
        working["assessed_skill_official_owner_is_requested"] = (
            requested_is_official_owner
        )

        topic_concept_lookup = {
            int(row["agent1_topic_index"]): topic_concept_tokens(row)
            for _, row in topic_rows.iterrows()
        }
        working["assessed_skill_lexical_overlap"] = [
            assessed_skill_lexical_overlap_score(
                row.get("assessed_skill_text"),
                topic_concept_lookup[int(row["agent1_topic_index"])],
            )
            for _, row in working.iterrows()
        ]

        # Parent-inherited children must establish relevance from their own
        # assessed skill before metadata/context/BM25 are allowed to help rank
        # them. Direct child evidence is either explicit concept-token overlap,
        # or semantic support at the notebook's existing broad floor where the
        # requested official concept also wins the cross-concept ownership check.
        # BM25 is intentionally absent from this admissibility decision.
        child_mask = working["is_hierarchical_subquestion"].astype(bool)
        # Child lexical evidence must come from the DETAILED Agent 1 topic,
        # not from a broad canonical heading and not from exam boilerplate.
        child_lexical_evidence = (
            working[
                "detected_topic_specific_lexical_overlap"
            ]
            .astype(float)
            .gt(0.0)
        )
        child_semantic_evidence = (
            working["assessed_skill_semantic_score"]
            .astype(float)
            .ge(BROAD_HYBRID_RELEVANCE_FLOOR)
            & working["assessed_skill_official_owner_is_requested"]
            .astype(bool)
        )
        working["child_direct_evidence_passed"] = (
            ~child_mask
            | child_lexical_evidence
            | child_semantic_evidence
        )
        working["child_direct_evidence_reason"] = [
            (
                "not_parent_inherited_child"
                if not bool(is_child)
                else (
                    "explicit_child_concept_lexical_evidence"
                    if bool(has_lexical)
                    else (
                        "child_assessed_skill_owned_by_requested_official_concept"
                        if bool(has_semantic)
                        else (
                            "child_assessed_skill_prefers_competing_official_concept"
                            if float(skill_score) >= BROAD_HYBRID_RELEVANCE_FLOOR
                            else "child_assessed_skill_below_existing_broad_floor"
                        )
                    )
                )
            )
            for is_child, has_lexical, has_semantic, skill_score in zip(
                child_mask.tolist(),
                child_lexical_evidence.tolist(),
                child_semantic_evidence.tolist(),
                working["assessed_skill_semantic_score"].astype(float).tolist(),
            )
        ]

        # Reuse the existing 50/50 concept-combination convention: the question
        # as a whole and the assessed instruction must both contribute.
        working["ownership_core_fit_score"] = (
            0.50 * working["question_focus_semantic_score"].astype(float)
            + 0.50 * working["assessed_skill_semantic_score"].astype(float)
        )

        # No new magic threshold: context is considered dominant when the amount
        # of relevance gained from parent/context is larger than the semantic fit
        # of the assessed instruction itself.
        working["context_dependency_flag"] = (
            working["context_dependency_gap"].astype(float)
            > working["assessed_skill_semantic_score"].astype(float).clip(lower=0.0)
        )

        # Metadata-mismatch rescue is a deliberate exception: its stored topic
        # metadata may be wrong, so preserve the direct-topic score that was
        # already computed from question/context text without the stored topic
        # header. This avoids reintroducing the metadata error during Phase 2.
        metadata_rescue_score_mask = (
            working.get(
                "retrieval_stage",
                pd.Series("", index=working.index),
            )
            .astype(str)
            .eq("metadata_mismatch_semantic_rescue")
        )

        if (
            metadata_rescue_score_mask.any()
            and "metadata_rescue_direct_topic_score" in working.columns
        ):
            preserved_rescue_scores = pd.to_numeric(
                working.loc[
                    metadata_rescue_score_mask,
                    "metadata_rescue_direct_topic_score",
                ],
                errors="coerce",
            )

            valid_preserved_scores = (
                preserved_rescue_scores.notna()
            )

            if valid_preserved_scores.any():
                valid_indexes = preserved_rescue_scores.index[
                    valid_preserved_scores
                ]
                working.loc[
                    valid_indexes,
                    "direct_topic_semantic_score",
                ] = preserved_rescue_scores.loc[
                    valid_indexes
                ].astype(float)

        # Do not overwrite question_focus_semantic_score here: it is the true
        # whole-question-only Qdrant score used by context-gap/ownership audit.
        # The rescue-specific override remains confined to the existing hybrid
        # direct_topic_semantic_score field.

        working["bm25_lexical_score"] = 0.0
        working["bm25_lexical_normalized"] = 0.0

        bm25_query_lookup = {
            int(row["agent1_topic_index"]): phase2_bm25_query(row)
            for _, row in topic_rows.iterrows()
        }

        for topic_index, topic_candidates in (
            working.groupby(
                "agent1_topic_index",
                sort=False,
            )
        ):
            topic_index = int(topic_index)

            candidate_indexes = (
                topic_candidates.index.tolist()
            )

            raw_bm25 = bm25_scores(
                bm25_query_lookup[topic_index],
                topic_candidates["question_text"]
                .fillna("")
                .astype(str)
                .tolist(),
            )

            maximum_bm25 = (
                float(raw_bm25.max())
                if len(raw_bm25)
                and float(raw_bm25.max()) > 0
                else 0.0
            )

            normalised_bm25 = (
                raw_bm25 / maximum_bm25
                if maximum_bm25 > 0
                else np.zeros_like(raw_bm25)
            )

            working.loc[
                candidate_indexes,
                "bm25_lexical_score",
            ] = raw_bm25

            working.loc[
                candidate_indexes,
                "bm25_lexical_normalized",
            ] = normalised_bm25

        working["metadata_strength_score"] = np.where(
            working["exact_detailed_reference"].astype(bool),
            1.0,
            0.85,
        )

        # Metadata-rescue candidates have deliberately weak metadata support.
        # Their detailed-topic/evidence/lexical signals must carry the match.
        if metadata_rescue_mask.any():
            working.loc[
                metadata_rescue_mask,
                "metadata_strength_score",
            ] = METADATA_RESCUE_METADATA_STRENGTH

        evidence_signal = np.clip(
            working["semantic_score"].astype(float),
            0.0,
            1.0,
        )

        direct_signal = np.clip(
            working["direct_topic_semantic_score"].astype(float),
            0.0,
            1.0,
        )

        lexical_signal = np.clip(
            working["bm25_lexical_normalized"].astype(float),
            0.0,
            1.0,
        )

        metadata_signal = np.clip(
            working["metadata_strength_score"].astype(float),
            0.0,
            1.0,
        )

        working["hybrid_relevance_score"] = (
            HYBRID_EVIDENCE_SEMANTIC_WEIGHT
            * evidence_signal
            + HYBRID_DIRECT_TOPIC_WEIGHT
            * direct_signal
            + HYBRID_BM25_WEIGHT
            * lexical_signal
            + HYBRID_METADATA_WEIGHT
            * metadata_signal
        )

        # Backward-compatible alias used by some later audit/output cells.
        working["semantic_gate_score"] = (
            working["hybrid_relevance_score"]
        )

        # Preserve the earlier raw score for audit and use the final hybrid
        # relevance for downstream ranking.
        if "pre_hybrid_final_score" not in working.columns:
            working["pre_hybrid_final_score"] = (
                working["final_score"].astype(float)
            )

        working["final_score"] = (
            working["hybrid_relevance_score"]
            + 0.02
            * working["agent1_confidence"].astype(float)
            + 0.01
            * working["agent1_ranking_score"].astype(float)
        )

        return working


    phase2_gate_manifest_df = (
        add_hybrid_relevance_scores(
            near_unique_candidates_df
        )
    )

    phase2_gate_manifest_df[
        "question_quality_issues"
    ] = phase2_gate_manifest_df[
        "question_text"
    ].map(
        detect_question_quality_issues
    )

    phase2_gate_manifest_df[
        "question_quality_gate_passed"
    ] = phase2_gate_manifest_df[
        "question_quality_issues"
    ].map(
        lambda issues: len(issues) == 0
    )

    # A single relevance threshold exists only for broad/shared pools.
    phase2_gate_manifest_df[
        "phase2_semantic_threshold"
    ] = np.where(
        phase2_gate_manifest_df[
            "metadata_trusted_candidate_specific"
        ].astype(bool),
        0.0,
        BROAD_HYBRID_RELEVANCE_FLOOR,
    )

    phase2_gate_manifest_df[
        "adaptive_threshold"
    ] = phase2_gate_manifest_df[
        "phase2_semantic_threshold"
    ]

    phase2_gate_manifest_df[
        "threshold_reason"
    ] = np.where(
        phase2_gate_manifest_df[
            "metadata_trusted_candidate_specific"
        ].astype(bool),
        "candidate_specific_metadata_no_semantic_rejection",
        "single_stable_broad_hybrid_relevance_floor",
    )

    phase2_gate_manifest_df[
        "concept_gate_passed"
    ] = np.where(
        phase2_gate_manifest_df[
            "metadata_trusted_candidate_specific"
        ].astype(bool),
        True,
        (
            phase2_gate_manifest_df[
                "hybrid_relevance_score"
            ].astype(float)
            + 1e-12
            >= BROAD_HYBRID_RELEVANCE_FLOOR
        ),
    )

    phase2_gate_manifest_df[
        "phase2_gate_passed"
    ] = (
        phase2_gate_manifest_df[
            "concept_gate_passed"
        ].astype(bool)
        & phase2_gate_manifest_df[
            "question_quality_gate_passed"
        ].astype(bool)
        & phase2_gate_manifest_df[
            "child_direct_evidence_passed"
        ].astype(bool)
        & phase2_gate_manifest_df[
            "exact_detected_concept_gate_passed"
        ].astype(bool)
    )

    phase2_gate_manifest_df[
        "phase2_gate_mode"
    ] = np.select(
        [
            (
                phase2_gate_manifest_df[
                    "metadata_trusted_candidate_specific"
                ].astype(bool)
                & phase2_gate_manifest_df[
                    "phase2_gate_passed"
                ].astype(bool)
            ),
            (
                ~phase2_gate_manifest_df[
                    "metadata_trusted_candidate_specific"
                ].astype(bool)
                & phase2_gate_manifest_df[
                    "phase2_gate_passed"
                ].astype(bool)
            ),
        ],
        [
            "candidate_specific_metadata_rank_only",
            "broad_shared_single_hybrid_gate",
        ],
        default="rejected",
    )

    phase2_gate_manifest_df[
        "semantic_rescue_used"
    ] = False

    def final_phase2_rejection_reasons(
        row: pd.Series,
    ) -> list[str]:
        reasons = list(
            row.get("question_quality_issues")
            if isinstance(
                row.get("question_quality_issues"),
                list,
            )
            else []
        )

        if (
            not bool(
                row.get(
                    "metadata_trusted_candidate_specific",
                    False,
                )
            )
            and float(
                row.get(
                    "hybrid_relevance_score",
                    0.0,
                )
            )
            < BROAD_HYBRID_RELEVANCE_FLOOR
        ):
            reasons.append(
                "broad_pool_hybrid_relevance_below_floor"
            )

        if not bool(row.get("child_direct_evidence_passed", True)):
            reasons.append(
                "parent_inherited_child_missing_direct_evidence"
            )

        if not bool(row.get("exact_detected_concept_gate_passed", True)):
            reasons.append(
                "broad_syllabus_fit_without_exact_detected_concept_support"
            )

        return sorted(set(reasons))

    phase2_gate_manifest_df[
        "phase2_rejection_reasons"
    ] = phase2_gate_manifest_df.apply(
        final_phase2_rejection_reasons,
        axis=1,
    )

    # One row per topic for backward-compatible audit exports.
    profile_rows = []

    for _, topic_row in (
        topic_retrieval_policy_df
        .sort_values("agent1_topic_index")
        .iterrows()
    ):
        topic_index = int(
            topic_row["agent1_topic_index"]
        )

        topic_candidates = phase2_gate_manifest_df[
            phase2_gate_manifest_df[
                "agent1_topic_index"
            ].astype(int)
            == topic_index
        ]

        quality_safe = topic_candidates[
            topic_candidates[
                "question_quality_gate_passed"
            ].astype(bool)
        ]

        retained = topic_candidates[
            topic_candidates[
                "phase2_gate_passed"
            ].astype(bool)
        ]

        scores = (
            quality_safe[
                "hybrid_relevance_score"
            ].astype(float)
            if not quality_safe.empty
            else pd.Series(dtype=float)
        )

        threshold = (
            0.0
            if bool(
                topic_row[
                    "metadata_trusted_specific"
                ]
            )
            else BROAD_HYBRID_RELEVANCE_FLOOR
        )

        profile_rows.append(
            {
                "agent1_topic_index": topic_index,
                "detected_topic": topic_row["detected_topic"],
                "role": topic_row["role"],
                "official_reference": (
                    topic_row["agent1_official_reference"]
                ),
                "canonical_reference": (
                    topic_row["official_reference"]
                ),
                "retrieval_pool_type": (
                    topic_row["retrieval_pool_type"]
                ),
                "retrieval_policy_reason": (
                    topic_row["retrieval_policy_reason"]
                ),
                "canonical_name_alignment": float(
                    topic_row[
                        "canonical_name_alignment"
                    ]
                ),
                "candidate_count": int(
                    len(topic_candidates)
                ),
                "quality_safe_candidate_count": int(
                    len(quality_safe)
                ),
                "retained_candidate_count": int(
                    len(retained)
                ),
                "required_candidate_quota": int(
                    derive_topic_candidate_requirements(
                        validated_topics_df
                    ).get(
                        topic_index,
                        0,
                    )
                ),
                "minimum_score": (
                    float(scores.min())
                    if not scores.empty
                    else None
                ),
                "maximum_score": (
                    float(scores.max())
                    if not scores.empty
                    else None
                ),
                "median_score": (
                    float(scores.median())
                    if not scores.empty
                    else None
                ),
                "percentile_score": None,
                "top_score_margin_threshold": None,
                "distribution_threshold": threshold,
                "adaptive_threshold": threshold,
                "candidates_passing_threshold": int(
                    len(retained)
                ),
                "quota_adjustment_used": False,
                "quota_satisfied_above_floor": bool(
                    len(retained)
                    >= int(
                        derive_topic_candidate_requirements(
                            validated_topics_df
                        ).get(
                            topic_index,
                            0,
                        )
                    )
                ),
                "threshold_reason": (
                    topic_row["retrieval_policy_reason"]
                ),
            }
        )

    adaptive_threshold_profile_df = pd.DataFrame(
        profile_rows
    )

    phase2_candidates_df = (
        phase2_gate_manifest_df[
            phase2_gate_manifest_df[
                "phase2_gate_passed"
            ].astype(bool)
        ]
        .sort_values(
            [
                "hybrid_relevance_score",
                "final_score",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    phase2_candidates_df[
        "phase2_rank"
    ] = np.arange(
        1,
        len(phase2_candidates_df) + 1,
    )

    phase2_gate_manifest_df = (
        phase2_gate_manifest_df.drop(
            columns=["phase2_rank"],
            errors="ignore",
        )
        .merge(
            phase2_candidates_df[
                [
                    "question_id",
                    "agent1_topic_index",
                    "phase2_rank",
                ]
            ],
            on=[
                "question_id",
                "agent1_topic_index",
            ],
            how="left",
            validate="one_to_one",
        )
    )

    phase2_rejected_df = (
        phase2_gate_manifest_df[
            ~phase2_gate_manifest_df[
                "phase2_gate_passed"
            ].astype(bool)
        ]
        .copy()
    )

    adaptive_threshold_values = (
        adaptive_threshold_profile_df[
            "adaptive_threshold"
        ]
        .dropna()
        .astype(float)
    )

    adaptive_threshold_min = (
        float(adaptive_threshold_values.min())
        if not adaptive_threshold_values.empty
        else None
    )
    adaptive_threshold_max = (
        float(adaptive_threshold_values.max())
        if not adaptive_threshold_values.empty
        else None
    )
    adaptive_threshold_mean = (
        float(adaptive_threshold_values.mean())
        if not adaptive_threshold_values.empty
        else None
    )

    adaptive_pool_sufficient = bool(
        len(phase2_candidates_df)
        >= int(request["number_of_questions"])
    )
    phase2_rescue_used = False

    phase2_gate_summary = {
        "phase2_version": PHASE2_VERSION,
        "threshold_strategy": (
            "hierarchical_metadata_specific_vs_broad_single_hybrid_floor"
        ),
        "hybrid_retrieval_policy_version": (
            HYBRID_RETRIEVAL_POLICY_VERSION
        ),
        "child_metadata_scope_version": CHILD_METADATA_SCOPE_VERSION,
        "parent_inherited_subquestion_candidates": int(
            phase2_gate_manifest_df["is_hierarchical_subquestion"].astype(bool).sum()
        ),
        "candidate_specific_metadata_bypass_candidates": int(
            phase2_gate_manifest_df["metadata_trusted_candidate_specific"].astype(bool).sum()
        ),
        "specific_canonical_name_alignment_min": (
            SPECIFIC_CANONICAL_NAME_ALIGNMENT_MIN
        ),
        "broad_hybrid_relevance_floor": (
            BROAD_HYBRID_RELEVANCE_FLOOR
        ),
        "hybrid_evidence_semantic_weight": (
            HYBRID_EVIDENCE_SEMANTIC_WEIGHT
        ),
        "hybrid_direct_topic_weight": (
            HYBRID_DIRECT_TOPIC_WEIGHT
        ),
        "hybrid_bm25_weight": (
            HYBRID_BM25_WEIGHT
        ),
        "hybrid_metadata_weight": (
            HYBRID_METADATA_WEIGHT
        ),
        "bm25_k1": BM25_K1,
        "bm25_b": BM25_B,
        "adaptive_threshold_version": (
            ADAPTIVE_THRESHOLD_VERSION
        ),
        "legacy_fixed_thresholds_active": False,
        "adaptive_threshold_min": adaptive_threshold_min,
        "adaptive_threshold_max": adaptive_threshold_max,
        "adaptive_threshold_mean": adaptive_threshold_mean,
        "adaptive_pool_sufficient_without_rescue": (
            adaptive_pool_sufficient
        ),
        "quality_safe_rescue_enabled": False,
        "quality_safe_rescue_used": False,
        "rescue_semantic_floor": None,
        "rescued_candidate_count": 0,
        "baseline_unique_candidates": int(
            len(unique_candidates_df)
        ),
        "near_duplicates_removed": (
            near_duplicates_removed
        ),
        "refined_unique_candidates": int(
            len(near_unique_candidates_df)
        ),
        "phase2_eligible_candidates": int(
            len(phase2_candidates_df)
        ),
        "phase2_rejected_candidates": int(
            len(phase2_rejected_df)
        ),
        "semantic_gate_rejections": int(
            (
                ~phase2_gate_manifest_df[
                    "concept_gate_passed"
                ].astype(bool)
            ).sum()
        ),
        "question_quality_rejections": int(
            (
                ~phase2_gate_manifest_df[
                    "question_quality_gate_passed"
                ].astype(bool)
            ).sum()
        ),
        "metadata_trusted_specific_topics": int(
            topic_retrieval_policy_df[
                "metadata_trusted_specific"
            ].astype(bool).sum()
        ),
        "broad_shared_topics": int(
            (
                ~topic_retrieval_policy_df[
                    "metadata_trusted_specific"
                ].astype(bool)
            ).sum()
        ),
        "topics_using_chunk_evidence": int(
            len(validated_topics_df)
            - fallback_evidence_topic_count
        ),
        "topics_using_summary_fallback": (
            fallback_evidence_topic_count
        ),
    }

    print("Final hybrid retrieval profile:")
    display(adaptive_threshold_profile_df)

    print("Final Phase 2 gate summary:")
    display(pd.DataFrame([phase2_gate_summary]))

    audit_columns = [
        "unique_rank",
        "phase2_rank",
        "detected_topic",
        "agent1_role",
        "agent1_official_reference",
        "official_reference",
        "retrieval_pool_type",
        "metadata_reference_scope",
        "metadata_trusted_candidate_specific",
        "marks",
        "semantic_score",
        "direct_topic_semantic_score",
        "bm25_lexical_normalized",
        "metadata_strength_score",
        "hybrid_relevance_score",
        "phase2_semantic_threshold",
        "question_quality_gate_passed",
        "phase2_gate_passed",
        "phase2_gate_mode",
        "question_quality_issues",
        "phase2_rejection_reasons",
        "question_text",
    ]

    display(
        phase2_gate_manifest_df[
            [
                column
                for column in audit_columns
                if column in phase2_gate_manifest_df.columns
            ]
        ].head(120)
    )

    if not phase2_rejected_df.empty:
        print("Rejected candidate examples:")
        display(
            phase2_rejected_df[
                [
                    column
                    for column in [
                        "detected_topic",
                        "agent1_official_reference",
                        "official_reference",
                        "retrieval_pool_type",
                        "semantic_score",
                        "direct_topic_semantic_score",
                        "bm25_lexical_normalized",
                        "hybrid_relevance_score",
                        "phase2_rejection_reasons",
                        "question_text",
                    ]
                    if column in phase2_rejected_df.columns
                ]
            ].head(40)
        )


## 10A. Concept-fit audit

The previous implementation applied a second direct-concept rejection gate
after the semantic gate. That created multiple opportunities to discard a good
question.

In the final architecture, direct-topic semantic similarity is already part of
the single hybrid relevance score.

This stage therefore keeps the old direct-concept fields for output/database
compatibility and audit, but **does not apply a second rejection threshold**.

```text
Phase 2 hybrid gate result
        ↓
direct-concept score retained for audit
        ↓
no additional candidate rejection
        ↓
selection
```


### Structural child-admissibility rule (v1.5)

For a hierarchical child/subquestion whose topic metadata is inherited from a parent block, the child must establish relevance from its **own assessed-skill representation** before parent metadata, context, BM25, or the hybrid score may support/rank it. Direct evidence is either explicit concept-token overlap or semantic ownership by the requested official syllabus concept relative to other approved official concepts. The existing broad semantic floor is retained; no topic-specific medians, keyword blacklists, or new tuned thresholds are introduced. Non-child behavior and global retrieval weights remain unchanged.


In [ ]:
if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Skipped this candidate-processing stage because no questions matched the current hard filters.")
else:
    # ================================================================
    # PHASE 2B — AUDIT-ONLY DIRECT CONCEPT FIELDS
    # ================================================================
    # Direct-topic semantic similarity is already included in the single
    # hybrid score. Keep the historical concept-fit columns for downstream
    # output/database compatibility, but do NOT apply another rejection gate.

    concept_fit_manifest_df = (
        phase2_gate_manifest_df
        .copy()
    )

    concept_fit_manifest_df[
        "direct_concept_score"
    ] = concept_fit_manifest_df[
        "direct_topic_semantic_score"
    ].astype(float)

    concept_fit_manifest_df[
        "combined_concept_score"
    ] = (
        0.50
        * concept_fit_manifest_df[
            "semantic_score"
        ].astype(float)
        + 0.50
        * concept_fit_manifest_df[
            "direct_concept_score"
        ].astype(float)
    )

    concept_fit_manifest_df[
        "concept_fit_mark_margin"
    ] = 0.0

    concept_fit_manifest_df[
        "direct_concept_threshold"
    ] = 0.0

    concept_fit_manifest_df[
        "required_direct_concept_score"
    ] = 0.0

    concept_fit_manifest_df[
        "required_combined_concept_score"
    ] = 0.0

    concept_fit_manifest_df[
        "concept_fit_threshold_reason"
    ] = (
        "audit_only_direct_topic_signal_already_in_hybrid_score"
    )

    concept_fit_manifest_df[
        "concept_fit_gate_passed"
    ] = concept_fit_manifest_df[
        "phase2_gate_passed"
    ].astype(bool)

    concept_fit_manifest_df[
        "concept_fit_rescue_used"
    ] = False

    concept_fit_manifest_df[
        "phase2_base_gate_passed"
    ] = concept_fit_manifest_df[
        "phase2_gate_passed"
    ].astype(bool)

    concept_fit_profile_df = (
        concept_fit_manifest_df.groupby(
            "agent1_topic_index",
            sort=True,
        )
        .agg(
            candidate_count=(
                "question_id",
                "size",
            ),
            minimum_score=(
                "direct_concept_score",
                "min",
            ),
            maximum_score=(
                "direct_concept_score",
                "max",
            ),
        )
        .reset_index()
    )

    concept_fit_profile_df[
        "percentile_score"
    ] = None
    concept_fit_profile_df[
        "top_margin_threshold"
    ] = None
    concept_fit_profile_df[
        "direct_concept_threshold"
    ] = 0.0
    concept_fit_profile_df[
        "threshold_reason"
    ] = (
        "audit_only_no_second_rejection_gate"
    )

    phase2_gate_manifest_df = (
        concept_fit_manifest_df.copy()
    )

    phase2_candidates_df = (
        phase2_gate_manifest_df[
            phase2_gate_manifest_df[
                "phase2_gate_passed"
            ].astype(bool)
        ]
        .sort_values(
            [
                "hybrid_relevance_score",
                "final_score",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    # Re-apply UI hard filters after the final relevance stage.
    _phase2_count_before_hard_guard = len(
        phase2_candidates_df
    )

    phase2_candidates_df = hard_filter_candidate_frame(
        phase2_candidates_df
    )

    _phase2_wrong_filter_removed = (
        _phase2_count_before_hard_guard
        - len(phase2_candidates_df)
    )

    if _phase2_wrong_filter_removed > 0:
        add_agent2_user_message(
            level="warning",
            code="hard_filter_guard_removed_candidates",
            message=(
                f"Removed {_phase2_wrong_filter_removed} candidate(s) that "
                "did not match the selected paper/language filters. No "
                "cross-paper substitution was allowed."
            ),
            details={
                "requested_paper_code": request.get("paper_code"),
                "resolved_kb_paper_code": knowledge_base_paper_code(),
            },
        )

    phase2_candidates_df = (
        phase2_candidates_df.drop(
            columns=["phase2_rank"],
            errors="ignore",
        )
        .reset_index(drop=True)
    )

    phase2_candidates_df[
        "phase2_rank"
    ] = np.arange(
        1,
        len(phase2_candidates_df) + 1,
    )

    phase2_gate_manifest_df = (
        phase2_gate_manifest_df.drop(
            columns=["phase2_rank"],
            errors="ignore",
        )
        .merge(
            phase2_candidates_df[
                [
                    "question_id",
                    "agent1_topic_index",
                    "phase2_rank",
                ]
            ],
            on=[
                "question_id",
                "agent1_topic_index",
            ],
            how="left",
            validate="one_to_one",
        )
    )

    phase2_rejected_df = (
        phase2_gate_manifest_df[
            ~phase2_gate_manifest_df[
                "phase2_gate_passed"
            ].astype(bool)
        ]
        .copy()
    )

    # ================================================================
    # FINAL MCQ DISTRACTOR / CORRECT-ANSWER CONCEPT GATE
    # ================================================================
    # A topic name appearing only as an MCQ option must not make the question
    # relevant. The assessed-skill vector above already excludes option text.
    # Here we use the linked mark scheme to identify the correct option. A row
    # survives when the stem itself supports the assigned topic OR the correct
    # option supports it. A topic that appears only in a wrong option is rejected.

    MCQ_OPTION_LINE_PATTERN = re.compile(
        r"^\s*([A-D])(?:[.)])?\s+(.+?)\s*$",
        flags=re.IGNORECASE,
    )

    def _normalise_mcq_text(value: Any) -> str:
        return re.sub(
            r"[^a-z0-9]+",
            " ",
            str(value or "").casefold(),
        ).strip()

    def _parse_mcq_stem_options(value: Any) -> tuple[str, dict[str, str], bool]:
        lines = [
            re.sub(r"\s+", " ", line).strip()
            for line in str(value or "").replace("\r", "\n").splitlines()
            if re.sub(r"\s+", " ", line).strip()
        ]
        positions = []
        options: dict[str, str] = {}
        for position, line in enumerate(lines):
            match = MCQ_OPTION_LINE_PATTERN.match(line)
            if match:
                positions.append(position)
                options[match.group(1).upper()] = match.group(2).strip()
        is_mcq = len(positions) >= 2
        stem = "\n".join(lines[:min(positions)] if is_mcq else lines).strip()
        return stem, options, is_mcq

    def _extract_correct_option_from_guidance(
        guidance: Any,
        options: dict[str, str],
    ) -> tuple[str | None, str | None]:
        if not options:
            return None, None
        raw = str(guidance or "")
        # Prefer a mark-scheme line beginning with A/B/C/D and matching the
        # corresponding option text. This avoids treating examiner prose as an
        # answer letter.
        for raw_line in raw.replace("\r", "\n").splitlines():
            line = re.sub(r"\s+", " ", raw_line).strip()
            match = MCQ_OPTION_LINE_PATTERN.match(line.rstrip(";."))
            if not match:
                continue
            letter = match.group(1).upper()
            answer_text = match.group(2).strip().rstrip(";.")
            option_text = str(options.get(letter) or "").strip()
            if option_text and (
                _normalise_mcq_text(answer_text) == _normalise_mcq_text(option_text)
                or _normalise_mcq_text(option_text) in _normalise_mcq_text(answer_text)
            ):
                return letter, option_text
        return None, None

    if not phase2_candidates_df.empty:
        candidate_uuid_values = [
            uuid.UUID(value)
            for value in phase2_candidates_df["question_id"].astype(str).tolist()
        ]
        candidate_ms_query = (
            select(
                question_ms_links.c.question_id.label("question_id"),
                mark_schemes.c.marking_guidance,
                mark_schemes.c.marking_points,
            )
            .select_from(
                question_ms_links.join(
                    mark_schemes,
                    mark_schemes.c.id
                    == question_ms_links.c.mark_scheme_entry_id,
                )
            )
            .where(question_ms_links.c.question_id.in_(candidate_uuid_values))
        )
        with engine.connect() as connection:
            candidate_mcq_ms_df = pd.read_sql(candidate_ms_query, connection)
        candidate_mcq_ms_df["question_id"] = (
            candidate_mcq_ms_df["question_id"].astype(str)
        )
        candidate_mcq_ms_df = candidate_mcq_ms_df.drop_duplicates(
            subset=["question_id"], keep="first"
        )
        phase2_candidates_df = phase2_candidates_df.merge(
            candidate_mcq_ms_df,
            on="question_id",
            how="left",
            validate="many_to_one",
        )

        mcq_records = phase2_candidates_df["question_text"].map(
            _parse_mcq_stem_options
        )
        phase2_candidates_df["mcq_stem_text"] = [record[0] for record in mcq_records]
        phase2_candidates_df["mcq_options"] = [record[1] for record in mcq_records]
        phase2_candidates_df["multiple_choice_detected"] = [record[2] for record in mcq_records]

        correct_records = phase2_candidates_df.apply(
            lambda row: _extract_correct_option_from_guidance(
                row.get("marking_guidance"),
                row.get("mcq_options") if isinstance(row.get("mcq_options"), dict) else {},
            ),
            axis=1,
        )
        phase2_candidates_df["mcq_correct_option_letter"] = [record[0] for record in correct_records]
        phase2_candidates_df["mcq_correct_option_text"] = [record[1] for record in correct_records]

        approved_topic_token_lookup = {
            int(row["agent1_topic_index"]): topic_concept_tokens(row)
            for _, row in validated_topics_df.drop_duplicates(
                subset=["agent1_topic_index"]
            ).iterrows()
        }

        # Rebuild the approved-topic direct-query vectors in this notebook scope.
        # `direct_vector_lookup` used earlier lives inside add_hybrid_relevance_scores()
        # and is therefore not available here.
        mcq_topic_rows = (
            validated_topics_df
            .drop_duplicates(subset=["agent1_topic_index"])
            .sort_values("agent1_topic_index")
        )
        mcq_direct_queries = mcq_topic_rows.apply(
            phase2_direct_topic_query,
            axis=1,
        ).tolist()
        mcq_direct_query_vectors = model.encode(
            mcq_direct_queries,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)
        mcq_topic_vector_lookup = {
            int(topic_index): mcq_direct_query_vectors[position]
            for position, topic_index in enumerate(
                mcq_topic_rows["agent1_topic_index"].tolist()
            )
        }

        # Correct-option semantic evidence uses the same MiniLM model/query
        # vectors and the existing broad floor; no topic/question-specific rule.
        unique_correct_texts = sorted({
            str(value).strip()
            for value in phase2_candidates_df["mcq_correct_option_text"].dropna().tolist()
            if str(value).strip()
        })
        if unique_correct_texts:
            correct_vectors = model.encode(
                unique_correct_texts,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            ).astype(np.float32)
            correct_vector_lookup = dict(zip(unique_correct_texts, correct_vectors))
        else:
            correct_vector_lookup = {}

        correct_support_flags = []
        distractor_only_flags = []
        for _, row in phase2_candidates_df.iterrows():
            if not bool(row.get("multiple_choice_detected", False)):
                correct_support_flags.append(True)
                distractor_only_flags.append(False)
                continue

            topic_index = int(row["agent1_topic_index"])
            topic_tokens = approved_topic_token_lookup.get(topic_index, set())
            stem_supported = bool(
                float(row.get("assessed_skill_semantic_score", 0.0))
                >= BROAD_HYBRID_RELEVANCE_FLOOR
                or float(row.get("assessed_skill_lexical_overlap", 0.0)) > 0.0
            )

            correct_text = str(row.get("mcq_correct_option_text") or "").strip()
            correct_supported = False
            if correct_text:
                lexical_support = assessed_skill_lexical_overlap_score(
                    correct_text, topic_tokens
                ) > 0.0
                vector = correct_vector_lookup.get(correct_text)
                semantic_support = bool(
                    vector is not None
                    and float(vector @ mcq_topic_vector_lookup[topic_index])
                    >= BROAD_HYBRID_RELEVANCE_FLOOR
                )
                correct_supported = bool(lexical_support or semantic_support)

            wrong_option_support = False
            options = row.get("mcq_options") if isinstance(row.get("mcq_options"), dict) else {}
            correct_letter = str(row.get("mcq_correct_option_letter") or "").upper()
            for letter, option_text in options.items():
                if letter == correct_letter:
                    continue
                if assessed_skill_lexical_overlap_score(option_text, topic_tokens) > 0.0:
                    wrong_option_support = True
                    break

            # If the correct answer could not be resolved, an option-only topic
            # mention is not trusted; precision wins over forced coverage.
            option_only = bool(
                not stem_supported
                and (
                    (correct_text and not correct_supported and wrong_option_support)
                    or (not correct_text and wrong_option_support)
                )
            )
            correct_support_flags.append(bool(stem_supported or correct_supported))
            distractor_only_flags.append(option_only)

        phase2_candidates_df["mcq_correct_answer_supports_topic"] = correct_support_flags
        phase2_candidates_df["mcq_distractor_only_topic_conflict"] = distractor_only_flags
        before_mcq_gate = len(phase2_candidates_df)
        phase2_candidates_df = phase2_candidates_df[
            ~phase2_candidates_df["mcq_distractor_only_topic_conflict"].astype(bool)
            & phase2_candidates_df["mcq_correct_answer_supports_topic"].astype(bool)
        ].copy()
        mcq_gate_removed_count = int(before_mcq_gate - len(phase2_candidates_df))
        print(
            "Candidates removed by MCQ distractor/correct-answer concept gate: "
            f"{mcq_gate_removed_count}"
        )
    else:
        candidate_mcq_ms_df = pd.DataFrame()
        mcq_gate_removed_count = 0

    # ================================================================
    # FINAL CROSS-TOPIC QUESTION OWNERSHIP
    # ================================================================
    # A single PostgreSQL question can legitimately be retrieved for multiple
    # approved Agent 1 topics that share a broad canonical pool. It must not be
    # globally deduplicated before detailed-topic scoring.
    #
    # At this point every row has:
    #   - evidence-aware semantic similarity,
    #   - direct detailed-topic semantic similarity,
    #   - BM25 lexical relevance,
    #   - metadata strength,
    #   - question-quality status.
    #
    # We can therefore assign the physical question to the most appropriate
    # detailed Agent 1 topic and keep the losing topics' alternative candidates.

    phase2_candidates_before_ownership_df = (
        phase2_candidates_df.copy()
    )

    if not phase2_candidates_before_ownership_df.empty:
        ownership_working_df = (
            phase2_candidates_before_ownership_df
            .copy()
        )

        ownership_working_df[
            "topic_ownership_score"
        ] = (
            OWNERSHIP_DIRECT_TOPIC_WEIGHT
            * ownership_working_df[
                "ownership_core_fit_score"
            ].astype(float).clip(
                lower=0.0,
                upper=1.0,
            )
            + OWNERSHIP_BM25_WEIGHT
            * ownership_working_df[
                "bm25_lexical_normalized"
            ].astype(float).clip(
                lower=0.0,
                upper=1.0,
            )
            + OWNERSHIP_EVIDENCE_WEIGHT
            * ownership_working_df[
                "semantic_score"
            ].astype(float).clip(
                lower=0.0,
                upper=1.0,
            )
            + OWNERSHIP_METADATA_WEIGHT
            * ownership_working_df[
                "metadata_strength_score"
            ].astype(float).clip(
                lower=0.0,
                upper=1.0,
            )
        )

        # A physical question is allowed to abstain from ALL approved topics.
        # Non-child questions preserve the existing semantic/lexical credibility
        # rule. Parent-inherited children must additionally satisfy the single
        # child_direct_evidence_passed invariant computed from their own assessed
        # skill. Parent metadata, context and BM25 cannot create a credible owner.
        # -------------------------------------------------------------
        # FINAL ASSESSED-TASK OWNERSHIP GATE — ADAPTIVE ANCHOR VERSION
        # -------------------------------------------------------------
        # Parent/context may improve recall, but final ownership is based on the
        # student's actual answer-demand. Instead of a high fixed MiniLM cutoff,
        # explicit direct-task matches create a topic-local semantic anchor.
        detected_task_score = ownership_working_df[
            "detected_topic_assessed_skill_score"
        ].astype(float)

        specific_task_lexical = ownership_working_df[
            "detected_topic_specific_lexical_overlap"
        ].astype(float)

        requested_official_owner = ownership_working_df[
            "assessed_skill_official_owner_is_requested"
        ].astype(bool)

        explicit_task_anchor = (
            specific_task_lexical.gt(0.0)
            & ownership_working_df[
                "exact_detected_concept_gate_passed"
            ].astype(bool)
        )

        ownership_working_df[
            "final_assessed_task_explicit_anchor"
        ] = explicit_task_anchor

        anchor_floor_by_topic = (
            ownership_working_df.loc[
                explicit_task_anchor,
                [
                    "agent1_topic_index",
                    "detected_topic_assessed_skill_score",
                ],
            ]
            .groupby("agent1_topic_index")[
                "detected_topic_assessed_skill_score"
            ]
            .min()
            .to_dict()
        )

        anchor_count_by_topic = (
            ownership_working_df.loc[
                explicit_task_anchor,
                ["agent1_topic_index"],
            ]
            .groupby("agent1_topic_index")
            .size()
            .to_dict()
        )

        adaptive_floors = []
        anchor_counts = []

        for _, row in ownership_working_df.iterrows():
            topic_index = int(row["agent1_topic_index"])
            anchor_count = int(
                anchor_count_by_topic.get(topic_index, 0)
            )
            anchor_counts.append(anchor_count)

            if anchor_count > 0:
                weakest_anchor = float(
                    anchor_floor_by_topic[topic_index]
                )
                adaptive_floor = max(
                    BROAD_HYBRID_RELEVANCE_FLOOR,
                    weakest_anchor
                    - FINAL_ASSESSED_TASK_ANCHOR_TOLERANCE,
                )
            else:
                # Do not wipe out a topic just because its valid exam wording
                # contains no literal topic token.
                adaptive_floor = BROAD_HYBRID_RELEVANCE_FLOOR

            adaptive_floors.append(float(adaptive_floor))

        ownership_working_df[
            "final_assessed_task_adaptive_floor"
        ] = adaptive_floors
        ownership_working_df[
            "final_assessed_task_anchor_count"
        ] = anchor_counts

        semantic_equivalent_support = (
            requested_official_owner
            & detected_task_score.ge(
                ownership_working_df[
                    "final_assessed_task_adaptive_floor"
                ].astype(float)
            )
        )

        ownership_working_df[
            "final_assessed_task_gate_passed"
        ] = (
            ownership_working_df[
                "exact_detected_concept_gate_passed"
            ].astype(bool)
            & ownership_working_df[
                "child_direct_evidence_passed"
            ].astype(bool)
            & (
                explicit_task_anchor
                | semantic_equivalent_support
            )
        )

        ownership_working_df[
            "final_assessed_task_gate_reason"
        ] = [
            (
                "explicit_direct_assessed_task_evidence"
                if bool(explicit_anchor)
                else (
                    "semantic_equivalent_owned_by_requested_concept"
                    if bool(semantic_support)
                    else "actual_answer_demand_not_owned_by_detected_topic"
                )
            )
            for explicit_anchor, semantic_support in zip(
                explicit_task_anchor.tolist(),
                semantic_equivalent_support.tolist(),
            )
        ]

        ownership_working_df["ownership_credible_match"] = (
            ownership_working_df[
                "final_assessed_task_gate_passed"
            ].astype(bool)
        )
        ownership_working_df["_ownership_credible_priority"] = (
            ownership_working_df["ownership_credible_match"]
            .astype(bool)
            .astype(int)
        )

        # -------------------------------------------------------------
        # ROLE-AWARE OWNERSHIP TIE BREAK
        # -------------------------------------------------------------
        # Many programming questions legitimately assess more than one approved
        # Agent 1 topic (for example a program-execution task may also contain a
        # loop or a data structure). A Supporting label should not automatically
        # steal such a question from a Primary topic when both ownership fits are
        # essentially tied.
        #
        # Primary is therefore preferred ONLY when:
        #   1. it is already a credible owner, and
        #   2. its ownership score is within a small global margin of the best
        #      credible ownership score for that same physical question.
        #
        # This never turns a non-credible Primary match into an owner.
        credible_ownership_score = ownership_working_df[
            "topic_ownership_score"
        ].astype(float).where(
            ownership_working_df["ownership_credible_match"].astype(bool),
            -1.0,
        )

        ownership_working_df[
            "_best_credible_ownership_score"
        ] = credible_ownership_score.groupby(
            ownership_working_df["question_id"]
        ).transform("max")

        ownership_working_df[
            "_ownership_primary_tie_eligible"
        ] = (
            ownership_working_df["ownership_credible_match"].astype(bool)
            & ownership_working_df["agent1_role"].astype(str).eq("primary")
            & ownership_working_df["topic_ownership_score"].astype(float).ge(
                ownership_working_df[
                    "_best_credible_ownership_score"
                ].astype(float)
                - PRIMARY_OWNERSHIP_TIE_MARGIN
            )
        ).astype(int)

        # Only candidate-specific metadata can act as an ownership tie-break.
        # Parent-inherited child metadata remains ordinary evidence and cannot
        # override the child question's own assessed skill.
        ownership_working_df[
            "_ownership_specific_priority"
        ] = ownership_working_df[
            "metadata_trusted_candidate_specific"
        ].astype(bool).astype(int)

        ownership_sorted_df = (
            ownership_working_df.sort_values(
                [
                    "question_id",
                    "_ownership_credible_priority",
                    "_ownership_primary_tie_eligible",
                    "_ownership_specific_priority",
                    "topic_ownership_score",
                    "hybrid_relevance_score",
                    "final_score",
                ],
                ascending=[
                    True,
                    False,
                    False,
                    False,
                    False,
                    False,
                    False,
                ],
            )
            .reset_index(drop=True)
        )

        ownership_sorted_df[
            "question_topic_candidate_count"
        ] = ownership_sorted_df.groupby(
            "question_id"
        )[
            "agent1_topic_index"
        ].transform(
            "nunique"
        )

        ownership_sorted_df[
            "question_ownership_rank"
        ] = ownership_sorted_df.groupby(
            "question_id"
        ).cumcount() + 1

        ownership_sorted_df["question_has_credible_owner"] = (
            ownership_sorted_df.groupby("question_id")[
                "ownership_credible_match"
            ].transform("max").astype(bool)
        )

        ownership_sorted_df[
            "question_ownership_status"
        ] = np.select(
            [
                ~ownership_sorted_df["question_has_credible_owner"],
                ownership_sorted_df["question_ownership_rank"].eq(1),
            ],
            [
                "no_owner",
                "owner",
            ],
            default="owned_by_stronger_detailed_topic",
        )

        # Owner-vs-runner-up margin is audit evidence only; ambiguity between two
        # approved topics is not itself a reason to reject a relevant question.
        ownership_sorted_df["ownership_score_margin"] = 0.0
        for question_id, group in ownership_sorted_df.groupby("question_id", sort=False):
            scores = group["topic_ownership_score"].astype(float).tolist()
            margin = scores[0] - scores[1] if len(scores) > 1 else scores[0]
            ownership_sorted_df.loc[
                group.index, "ownership_score_margin"
            ] = float(margin)

        owner_lookup_df = (
            ownership_sorted_df[
                ownership_sorted_df[
                    "question_ownership_status"
                ].eq("owner")
            ][
                [
                    "question_id",
                    "agent1_topic_index",
                    "detected_topic",
                    "topic_ownership_score",
                    "ownership_core_fit_score",
                    "assessed_skill_semantic_score",
                    "ownership_score_margin",
                ]
            ]
            .rename(
                columns={
                    "agent1_topic_index": (
                        "owner_agent1_topic_index"
                    ),
                    "detected_topic": (
                        "owner_detected_topic"
                    ),
                    "topic_ownership_score": (
                        "owner_topic_ownership_score"
                    ),
                    "ownership_core_fit_score": (
                        "owner_core_fit_score"
                    ),
                    "assessed_skill_semantic_score": (
                        "owner_assessed_skill_score"
                    ),
                    "ownership_score_margin": (
                        "owner_score_margin"
                    ),
                }
            )
        )

        candidate_ownership_manifest_df = (
            ownership_sorted_df.merge(
                owner_lookup_df,
                on="question_id",
                how="left",
                validate="many_to_one",
            )
        )

        phase2_candidates_df = (
            candidate_ownership_manifest_df[
                candidate_ownership_manifest_df[
                    "question_ownership_status"
                ].eq("owner")
            ]
            .drop(
                columns=[
                    "_ownership_specific_priority",
                    "_ownership_credible_priority",
                    "_ownership_primary_tie_eligible",
                    "_best_credible_ownership_score",
                ],
                errors="ignore",
            )
            .sort_values(
                [
                    "hybrid_relevance_score",
                    "final_score",
                ],
                ascending=False,
            )
            .reset_index(drop=True)
        )

        phase2_candidates_df = (
            phase2_candidates_df.drop(
                columns=["phase2_rank"],
                errors="ignore",
            )
        )

        phase2_candidates_df[
            "phase2_rank"
        ] = np.arange(
            1,
            len(phase2_candidates_df) + 1,
        )

        ownership_conflict_count = int(
            candidate_ownership_manifest_df[
                "question_topic_candidate_count"
            ].gt(1).groupby(
                candidate_ownership_manifest_df[
                    "question_id"
                ]
            ).max().sum()
        )

        ownership_rows_reassigned = int(
            candidate_ownership_manifest_df[
                "question_ownership_status"
            ].eq(
                "owned_by_stronger_detailed_topic"
            ).sum()
        )

        no_owner_question_count = int(
            candidate_ownership_manifest_df[
                "question_ownership_status"
            ].eq("no_owner")
            .groupby(candidate_ownership_manifest_df["question_id"])
            .max()
            .sum()
        )

    else:
        candidate_ownership_manifest_df = (
            phase2_candidates_before_ownership_df
            .copy()
        )
        phase2_candidates_df = (
            phase2_candidates_before_ownership_df
            .copy()
        )
        ownership_conflict_count = 0
        ownership_rows_reassigned = 0
        no_owner_question_count = 0

    # Final invariant: each physical question can reach selection only once.
    if (
        not phase2_candidates_df.empty
        and phase2_candidates_df[
            "question_id"
        ].duplicated().any()
    ):
        raise RuntimeError(
            "Cross-topic ownership failed: duplicate question IDs remain "
            "before final assessment selection."
        )

    print(
        "Cross-topic question ownership conflicts resolved: "
        f"{ownership_conflict_count}"
    )
    print(
        "Candidate-topic rows reassigned to stronger detailed topics: "
        f"{ownership_rows_reassigned}"
    )
    print(
        "Physical questions with NO_OWNER across approved topics: "
        f"{no_owner_question_count}"
    )

    # ================================================================
    # FINAL QUALITY + RELEVANCE VERIFIER
    # ================================================================
    # The original quality gate already ran earlier. Re-run the same generalized
    # detector after ownership so downstream selection sees an explicit final
    # quality invariant. Then verify the assessed skill and context consistency.
    if not phase2_candidates_df.empty:
        phase2_candidates_df["final_quality_issues"] = (
            phase2_candidates_df["question_text"].map(
                detect_question_quality_issues
            )
        )
        phase2_candidates_df["final_quality_gate_passed"] = (
            phase2_candidates_df["final_quality_issues"].map(
                lambda issues: len(issues) == 0
            )
        )

        phase2_candidates_df["assessed_skill_supported"] = (
            phase2_candidates_df["assessed_skill_semantic_score"]
            .astype(float)
            .ge(BROAD_HYBRID_RELEVANCE_FLOOR)
            | phase2_candidates_df["assessed_skill_lexical_overlap"]
            .astype(float)
            .gt(0.0)
        )

        # Context may be strong, but it must not be the only reason a subquestion
        # appears relevant. A context-dominant row is still allowed when the
        # whole question itself independently clears the existing broad floor.
        phase2_candidates_df["context_consistency_passed"] = (
            ~phase2_candidates_df["context_dependency_flag"].astype(bool)
            | phase2_candidates_df["question_focus_semantic_score"]
            .astype(float)
            .ge(BROAD_HYBRID_RELEVANCE_FLOOR)
        )

        # Compatibility alias for downstream exports. The actual structural
        # decision is made once, earlier, as child_direct_evidence_passed. BM25,
        # parent metadata and context are intentionally excluded from that gate.
        phase2_candidates_df["child_assessed_skill_validation_passed"] = (
            phase2_candidates_df["child_direct_evidence_passed"].astype(bool)
        )

        phase2_candidates_df["final_relevance_verifier_passed"] = (
            phase2_candidates_df["final_quality_gate_passed"].astype(bool)
            & phase2_candidates_df["ownership_credible_match"].astype(bool)
            & phase2_candidates_df["assessed_skill_supported"].astype(bool)
            & phase2_candidates_df["context_consistency_passed"].astype(bool)
            & phase2_candidates_df["child_assessed_skill_validation_passed"].astype(bool)
            & phase2_candidates_df["exact_detected_concept_gate_passed"].astype(bool)
            & ~phase2_candidates_df.get(
                "mcq_distractor_only_topic_conflict",
                pd.Series(False, index=phase2_candidates_df.index),
            ).astype(bool)
            & phase2_candidates_df.get(
                "mcq_correct_answer_supports_topic",
                pd.Series(True, index=phase2_candidates_df.index),
            ).astype(bool)
        )

        phase2_candidates_df["final_relevance_verifier_reason"] = (
            phase2_candidates_df.apply(
                lambda row: (
                    "verified"
                    if bool(row.get("final_relevance_verifier_passed", False))
                    else ";".join(
                        reason
                        for reason, failed in [
                            ("final_quality_gate_failed", not bool(row.get("final_quality_gate_passed", False))),
                            ("no_credible_topic_owner", not bool(row.get("ownership_credible_match", False))),
                            ("assessed_skill_not_supported", not bool(row.get("assessed_skill_supported", False))),
                            ("context_dependency_not_independently_supported", not bool(row.get("context_consistency_passed", False))),
                            ("parent_inherited_subquestion_not_supported_by_child_skill", not bool(row.get("child_assessed_skill_validation_passed", False))),
                            ("broad_syllabus_fit_without_exact_detected_concept_support", not bool(row.get("exact_detected_concept_gate_passed", False))),
                            ("mcq_topic_only_in_distractor_or_wrong_option", bool(row.get("mcq_distractor_only_topic_conflict", False))),
                            ("mcq_correct_answer_does_not_support_topic", not bool(row.get("mcq_correct_answer_supports_topic", True))),
                        ]
                        if failed
                    )
                ),
                axis=1,
            )
        )

        final_verifier_manifest_df = phase2_candidates_df.copy()
        final_verifier_removed_count = int(
            (~phase2_candidates_df["final_relevance_verifier_passed"].astype(bool)).sum()
        )
        phase2_candidates_df = (
            phase2_candidates_df[
                phase2_candidates_df["final_relevance_verifier_passed"].astype(bool)
            ]
            .sort_values(
                ["hybrid_relevance_score", "final_score"],
                ascending=False,
            )
            .reset_index(drop=True)
        )
        phase2_candidates_df = phase2_candidates_df.drop(
            columns=["phase2_rank"], errors="ignore"
        )
        phase2_candidates_df["phase2_rank"] = np.arange(
            1, len(phase2_candidates_df) + 1
        )
    else:
        final_verifier_manifest_df = phase2_candidates_df.copy()
        final_verifier_removed_count = 0

    print(
        "Candidates removed by final assessed-skill/context verifier: "
        f"{final_verifier_removed_count}"
    )

    if (
        not candidate_ownership_manifest_df.empty
        and "question_topic_candidate_count"
        in candidate_ownership_manifest_df.columns
        and candidate_ownership_manifest_df[
            "question_topic_candidate_count"
        ].gt(1).any()
    ):
        print(
            "Questions retrieved by multiple Agent 1 topics "
            "(ownership audit):"
        )
        display(
            candidate_ownership_manifest_df[
                candidate_ownership_manifest_df[
                    "question_topic_candidate_count"
                ].gt(1)
            ][
                [
                    "question_id",
                    "detected_topic",
                    "agent1_role",
                    "agent1_official_reference",
                    "official_reference",
                    "retrieval_pool_type",
                    "metadata_reference_scope",
                    "metadata_trusted_candidate_specific",
                    "direct_topic_semantic_score",
                    "question_focus_semantic_score",
                    "assessed_skill_semantic_score",
                    "assessed_skill_official_concept_score",
                    "best_competing_assessed_skill_score",
                    "best_competing_official_reference",
                    "best_competing_official_concept",
                    "assessed_skill_official_owner_is_requested",
                    "assessed_skill_lexical_overlap",
                    "detected_topic_assessed_skill_score",
                    "canonical_topic_assessed_skill_score",
                    "detected_topic_specificity_delta",
                    "detected_topic_specific_lexical_overlap",
                    "exact_detected_concept_gate_passed",
                    "exact_detected_concept_gate_reason",
                    "child_direct_evidence_passed",
                    "child_direct_evidence_reason",
                    "context_enriched_direct_topic_score",
                    "context_dependency_gap",
                    "context_dependency_flag",
                    "ownership_core_fit_score",
                    "bm25_lexical_normalized",
                    "semantic_score",
                    "topic_ownership_score",
                    "final_assessed_task_gate_passed",
                    "final_assessed_task_gate_reason",
                    "final_assessed_task_explicit_anchor",
                    "final_assessed_task_adaptive_floor",
                    "final_assessed_task_anchor_count",
                    "_ownership_primary_tie_eligible",
                    "_best_credible_ownership_score",
                    "question_ownership_status",
                    "owner_detected_topic",
                    "question_text",
                ]
            ].head(80)
        )

    # ---------------------------------------------------------------
    # Per-topic quality-safe availability / shortfall audit
    # ---------------------------------------------------------------
    # The "requested" value is the quota derived from the assessment-level
    # request for that Agent 1 topic. If fewer safe questions survive, do not
    # fail and do not pull in rejected questions: warn and continue.
    topic_required_quota = derive_topic_candidate_requirements(
        validated_topics_df
    )

    quality_safe_topic_counts = (
        phase2_candidates_df.groupby("agent1_topic_index")
        .size()
        .to_dict()
        if not phase2_candidates_df.empty
        else {}
    )

    topic_question_availability_rows = []
    for _, topic_row in (
        validated_topics_df
        .drop_duplicates(subset=["agent1_topic_index"])
        .sort_values("agent1_topic_index")
        .iterrows()
    ):
        topic_index = int(topic_row["agent1_topic_index"])
        topic_name = str(topic_row["detected_topic"])
        requested_for_topic = int(
            topic_required_quota.get(topic_index, 0)
        )
        available_for_topic = int(
            quality_safe_topic_counts.get(topic_index, 0)
        )
        shortfall = max(
            0, requested_for_topic - available_for_topic
        )

        topic_question_availability_rows.append(
            {
                "agent1_topic_index": topic_index,
                "detected_topic": topic_name,
                "role": str(topic_row["role"]),
                "agent1_official_reference": str(
                    topic_row["agent1_official_reference"]
                ),
                "canonical_official_reference": str(
                    topic_row["official_reference"]
                ),
                "requested_quality_safe_questions": requested_for_topic,
                "available_quality_safe_questions": available_for_topic,
                "question_shortfall": shortfall,
                "availability_status": (
                    "available"
                    if shortfall == 0
                    else (
                        "none_available"
                        if available_for_topic == 0
                        else "partial"
                    )
                ),
            }
        )

        if shortfall > 0:
            if topic_index in PAPER_MISMATCH_TOPIC_INDEXES:
                # A more specific Paper 1/Paper 2 message was already emitted
                # during retrieval; do not duplicate it with a generic shortage.
                continue
            if available_for_topic == 0:
                friendly_message = (
                    "No quality-safe past-paper question remained for this "
                    "topic after the selected paper/filter, question-quality "
                    "and final retrieval checks. The topic will be skipped and "
                    "the rest of the assessment will still be generated."
                )
            else:
                friendly_message = (
                    f"Only {available_for_topic} quality-safe question(s) are "
                    f"available for this topic, while {requested_for_topic} "
                    "were requested by the assessment quota. Available valid "
                    "questions will be used; rejected/weak questions will not "
                    "be reintroduced."
                )

            add_agent2_user_message(
                level="warning",
                code="topic_question_shortfall",
                topic=topic_name,
                requested=requested_for_topic,
                available=available_for_topic,
                message=friendly_message,
                details={
                    "question_shortfall": shortfall,
                    "agent1_official_reference": str(
                        topic_row["agent1_official_reference"]
                    ),
                    "canonical_official_reference": str(
                        topic_row["official_reference"]
                    ),
                },
            )

    topic_question_availability_df = pd.DataFrame(
        topic_question_availability_rows
    )

    print("Per-topic quality-safe question availability:")
    display(topic_question_availability_df)

    if (
        not topic_question_availability_df.empty
        and topic_question_availability_df["question_shortfall"].gt(0).any()
    ):
        print_agent2_user_messages(
            "AGENT 2 PARTIAL-ASSESSMENT NOTICE"
        )

    concept_fit_removed_count = int(
        (
            phase2_gate_manifest_df["phase2_base_gate_passed"]
            & ~phase2_gate_manifest_df["phase2_gate_passed"]
        ).sum()
    )

    concept_fit_rescued_count = int(
        phase2_gate_manifest_df[
            "concept_fit_rescue_used"
        ].sum()
    )

    phase2_gate_summary.update(
        {
            "concept_fit_version": CONCEPT_FIT_VERSION,
            "concept_fit_enabled": ENABLE_DIRECT_CONCEPT_FIT_GATE,
            "concept_fit_score_percentile": (
                CONCEPT_FIT_SCORE_PERCENTILE
            ),
            "concept_fit_absolute_floor": (
                CONCEPT_FIT_ABSOLUTE_FLOOR
            ),
            "concept_fit_candidates_removed": (
                concept_fit_removed_count
            ),
            "concept_fit_reference_rescued": (
                concept_fit_rescued_count
            ),
            "phase2_eligible_candidates_after_concept_fit": int(
                len(phase2_candidates_df)
            ),
            "weak_candidate_replacement_strategy": (
                "remove_before_selection_and_select_next_best"
            ),
            "question_ownership_version": (
                QUESTION_OWNERSHIP_VERSION
            ),
            "cross_topic_question_conflicts": int(
                ownership_conflict_count
            ),
            "candidate_topic_rows_reassigned": int(
                ownership_rows_reassigned
            ),
            "candidate_count_after_ownership_and_final_verifier": int(
                len(phase2_candidates_df)
            ),
            "no_owner_question_count": int(no_owner_question_count),
            "final_relevance_verifier_version": FINAL_RELEVANCE_VERIFIER_VERSION,
            "child_metadata_scope_version": CHILD_METADATA_SCOPE_VERSION,
            "general_quality_gate_version": GENERAL_QUALITY_GATE_VERSION,
            "context_gap_version": CONTEXT_GAP_VERSION,
            "final_relevance_verifier_removed": int(final_verifier_removed_count),
        }
    )

    print("Direct concept-fit threshold profile:")
    display(concept_fit_profile_df)

    print("Candidates after generalized concept-fit refinement:")
    display(
        phase2_gate_manifest_df[
            [
                "detected_topic",
                "official_reference",
                "question_number",
                "marks",
                "semantic_score",
                "direct_concept_score",
                "combined_concept_score",
                "direct_concept_threshold",
                "concept_fit_mark_margin",
                "concept_fit_gate_passed",
                "concept_fit_rescue_used",
                "phase2_rejection_reasons",
                "question_text",
            ]
        ].head(100)
    )


## 10B. Retrieval HITL Phase 3 — Qdrant memory recall + compatibility diagnostics

This stage queries the dedicated Phase 2 retrieval-memory collection for every quality-safe candidate.
It validates recalled memories against the current approved AQA topic, role, lesson evidence and question.

**Safety rule:** Phase 3 is diagnostic only. `final_score`, `phase2_rank` and selection are not changed.
Every candidate receives `memory_rank_adjustment = 0.0`.


In [ ]:
# ============================================================================
# RETRIEVAL HITL PHASE 3 — MEMORY RECALL + COMPATIBILITY DIAGNOSTICS ONLY
# ============================================================================
# IMPORTANT:
# - Qdrant memory is queried here.
# - Compatible / incompatible historical feedback is diagnosed.
# - final_score is NOT modified.
# - phase2_rank is NOT modified.
# - selection logic is NOT modified.
# ============================================================================

RETRIEVAL_MEMORY_PHASE3_VERSION = (
    "agent2-retrieval-hitl-phase3-recall-v1.0.0"
)
RETRIEVAL_MEMORY_COLLECTION = (
    os.getenv("AGENT2_RETRIEVAL_MEMORY_COLLECTION", "").strip()
    or f"{AGENT2_COLLECTION}_retrieval_memory"
)
RETRIEVAL_MEMORY_TOP_K = 8
RETRIEVAL_MEMORY_OVERALL_SIMILARITY_MIN = 0.50
RETRIEVAL_MEMORY_TOPIC_SIMILARITY_MIN = 0.90
RETRIEVAL_MEMORY_EVIDENCE_SIMILARITY_MIN = 0.75
RETRIEVAL_MEMORY_QUESTION_SIMILARITY_MIN = 0.78
RETRIEVAL_MEMORY_RANKING_ADJUSTMENT_ENABLED = False


def _phase3_memory_normalize(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _phase3_memory_norm_label(value: Any) -> str:
    return _phase3_memory_normalize(value).casefold()


_phase3_embedding_cache: dict[str, np.ndarray] = {}
_phase3_feedback_source_cache: dict[int, dict[str, Any]] = {}


def _phase3_embed_text(value: Any) -> np.ndarray | None:
    text_value = _phase3_memory_normalize(value)
    if not text_value:
        return None
    cache_key = hashlib.sha256(text_value.encode("utf-8")).hexdigest()
    if cache_key not in _phase3_embedding_cache:
        vector = model.encode(
            [text_value],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )[0].astype(np.float32)
        if int(len(vector)) != VECTOR_SIZE:
            raise RuntimeError(
                "Unexpected Phase 3 memory embedding size: "
                f"{len(vector)}; expected {VECTOR_SIZE}."
            )
        _phase3_embedding_cache[cache_key] = vector
    return _phase3_embedding_cache[cache_key]


def _phase3_cosine_text(left: Any, right: Any) -> float | None:
    left_vector = _phase3_embed_text(left)
    right_vector = _phase3_embed_text(right)
    if left_vector is None or right_vector is None:
        return None
    return float(np.dot(left_vector, right_vector))


def _phase3_feedback_source(feedback_id: Any) -> dict[str, Any]:
    try:
        feedback_id_int = int(feedback_id)
    except (TypeError, ValueError):
        return {}

    if feedback_id_int in _phase3_feedback_source_cache:
        return _phase3_feedback_source_cache[feedback_id_int]

    try:
        with engine.connect() as connection:
            row = connection.execute(
                text(
                    """
                    SELECT
                        id,
                        transcript_evidence,
                        question_text,
                        detected_topic,
                        official_reference,
                        agent1_role,
                        concept_id,
                        decision,
                        reason
                    FROM agent2_retrieval_feedback
                    WHERE id = :feedback_id
                    """
                ),
                {"feedback_id": feedback_id_int},
            ).mappings().first()
    except Exception:
        row = None

    value = dict(row) if row is not None else {}
    _phase3_feedback_source_cache[feedback_id_int] = value
    return value


def _phase3_memory_search(
    *,
    query_vector: np.ndarray,
    official_reference: str,
    role: str,
) -> list[Any]:
    query_filter = models.Filter(
        must=[
            models.FieldCondition(
                key="official_reference",
                match=models.MatchValue(value=official_reference),
            ),
            models.FieldCondition(
                key="agent1_role",
                match=models.MatchValue(value=role),
            ),
        ]
    )
    try:
        result = qdrant_client.query_points(
            collection_name=RETRIEVAL_MEMORY_COLLECTION,
            query=query_vector.tolist(),
            query_filter=query_filter,
            limit=RETRIEVAL_MEMORY_TOP_K,
            with_payload=True,
            with_vectors=False,
        )
        return list(result.points if hasattr(result, "points") else result)
    except (AttributeError, TypeError):
        return list(
            qdrant_client.search(
                collection_name=RETRIEVAL_MEMORY_COLLECTION,
                query_vector=query_vector.tolist(),
                query_filter=query_filter,
                limit=RETRIEVAL_MEMORY_TOP_K,
                with_payload=True,
                with_vectors=False,
            )
        )


def _phase3_memory_query_text(row: pd.Series) -> str:
    return "\n".join(
        [
            "Agent 2 retrieval memory lookup",
            f"Approved lesson topic: {row.get('detected_topic')}",
            f"Official reference: {row.get('official_reference')}",
            f"Topic role: {row.get('agent1_role')}",
            "Lesson evidence:",
            str(row.get("query_evidence") or ""),
            "Current retrieved official question:",
            str(row.get("question_text") or ""),
        ]
    ).strip()


def _phase3_memory_candidate_diagnostic(
    *,
    point: Any,
    current_row: pd.Series,
) -> dict[str, Any]:
    payload = point.payload or {}
    feedback_source = _phase3_feedback_source(payload.get("feedback_id"))

    current_question_id = str(current_row.get("question_id") or "").strip()
    memory_question_id = str(payload.get("question_id") or "").strip()
    exact_question_id = bool(
        current_question_id
        and memory_question_id
        and current_question_id == memory_question_id
    )

    current_reference = str(current_row.get("official_reference") or "").strip()
    memory_reference = str(payload.get("official_reference") or "").strip()
    reference_compatible = bool(
        current_reference
        and memory_reference
        and current_reference == memory_reference
    )

    current_role = _phase3_memory_norm_label(current_row.get("agent1_role"))
    memory_role = _phase3_memory_norm_label(payload.get("agent1_role"))
    role_compatible = bool(current_role and memory_role and current_role == memory_role)

    current_topic = _phase3_memory_normalize(current_row.get("detected_topic"))
    memory_topic = _phase3_memory_normalize(payload.get("detected_topic"))
    topic_similarity = _phase3_cosine_text(current_topic, memory_topic)
    topic_exact = bool(
        _phase3_memory_norm_label(current_topic)
        and _phase3_memory_norm_label(current_topic)
        == _phase3_memory_norm_label(memory_topic)
    )
    topic_compatible = bool(
        topic_exact
        or (
            topic_similarity is not None
            and topic_similarity >= RETRIEVAL_MEMORY_TOPIC_SIMILARITY_MIN
        )
    )

    current_evidence = _phase3_memory_normalize(current_row.get("query_evidence"))
    historical_evidence = _phase3_memory_normalize(
        feedback_source.get("transcript_evidence")
    )
    evidence_similarity = _phase3_cosine_text(
        current_evidence,
        historical_evidence,
    )

    # When PostgreSQL evidence is unavailable, exact evidence hash is still a
    # safe fallback for the identical lesson context. Semantic generalisation
    # remains fail-closed until source evidence can be read.
    current_evidence_hash = (
        hashlib.sha256(current_evidence.casefold().encode("utf-8")).hexdigest()
        if current_evidence
        else ""
    )
    memory_evidence_hash = str(payload.get("lesson_evidence_hash") or "").strip()
    exact_evidence_hash = bool(
        current_evidence_hash
        and memory_evidence_hash
        and current_evidence_hash == memory_evidence_hash
    )
    evidence_compatible = bool(
        exact_evidence_hash
        or (
            evidence_similarity is not None
            and evidence_similarity >= RETRIEVAL_MEMORY_EVIDENCE_SIMILARITY_MIN
        )
    )

    current_question = _phase3_memory_normalize(current_row.get("question_text"))
    historical_question = _phase3_memory_normalize(
        payload.get("question_text")
        or feedback_source.get("question_text")
    )
    question_similarity = (
        1.0
        if exact_question_id
        else _phase3_cosine_text(current_question, historical_question)
    )
    question_compatible = bool(
        exact_question_id
        or (
            question_similarity is not None
            and question_similarity >= RETRIEVAL_MEMORY_QUESTION_SIMILARITY_MIN
        )
    )

    overall_similarity = float(getattr(point, "score", 0.0) or 0.0)
    overall_similarity_passed = bool(
        overall_similarity >= RETRIEVAL_MEMORY_OVERALL_SIMILARITY_MIN
    )

    rejection_reasons: list[str] = []
    if not reference_compatible:
        rejection_reasons.append("official_reference_mismatch")
    if not role_compatible:
        rejection_reasons.append("topic_role_mismatch")
    if not topic_compatible:
        rejection_reasons.append("topic_not_compatible")
    if not evidence_compatible:
        rejection_reasons.append("lesson_evidence_similarity_below_threshold")
    if not question_compatible:
        rejection_reasons.append("question_similarity_below_threshold")
    if not overall_similarity_passed:
        rejection_reasons.append("overall_memory_similarity_below_threshold")

    matched = not rejection_reasons

    return {
        "memory_point_id": str(getattr(point, "id", "")),
        "feedback_id": payload.get("feedback_id"),
        "decision": str(payload.get("decision") or ""),
        "reason": str(payload.get("reason") or ""),
        "memory_question_id": memory_question_id,
        "memory_question_text": historical_question,
        "memory_detected_topic": str(payload.get("detected_topic") or ""),
        "memory_official_reference": memory_reference,
        "memory_role": str(payload.get("agent1_role") or ""),
        "overall_similarity": round(overall_similarity, 6),
        "topic_similarity": (
            round(float(topic_similarity), 6)
            if topic_similarity is not None else None
        ),
        "evidence_similarity": (
            round(float(evidence_similarity), 6)
            if evidence_similarity is not None else (1.0 if exact_evidence_hash else None)
        ),
        "question_similarity": (
            round(float(question_similarity), 6)
            if question_similarity is not None else None
        ),
        "exact_question_id": exact_question_id,
        "exact_lesson_evidence_hash": exact_evidence_hash,
        "reference_compatible": reference_compatible,
        "role_compatible": role_compatible,
        "topic_compatible": topic_compatible,
        "evidence_compatible": evidence_compatible,
        "question_compatible": question_compatible,
        "overall_similarity_passed": overall_similarity_passed,
        "compatibility_status": "MATCHED" if matched else "REJECTED",
        "rejection_reasons": rejection_reasons,
    }


def _phase3_recall_for_candidate(row: pd.Series) -> dict[str, Any]:
    base = {
        "version": RETRIEVAL_MEMORY_PHASE3_VERSION,
        "memory_collection": RETRIEVAL_MEMORY_COLLECTION,
        "memory_collection_points": int(RETRIEVAL_MEMORY_POINT_COUNT),
        "ranking_adjustment_enabled": False,
        "ranking_adjustment": 0.0,
        "base_final_score": float(row.get("final_score") or 0.0),
        "score_after_memory": float(row.get("final_score") or 0.0),
        "rank_changed": False,
        "matched": False,
        "recall_status": "no_compatible_memory",
        "candidate_memories": [],
    }

    if not RETRIEVAL_MEMORY_AVAILABLE:
        base["recall_status"] = RETRIEVAL_MEMORY_UNAVAILABLE_REASON
        return base

    query_text = _phase3_memory_query_text(row)
    query_vector = _phase3_embed_text(query_text)
    if query_vector is None:
        base["recall_status"] = "missing_current_context"
        return base

    points = _phase3_memory_search(
        query_vector=query_vector,
        official_reference=str(row.get("official_reference") or "").strip(),
        role=_phase3_memory_norm_label(row.get("agent1_role")),
    )

    diagnostics = [
        _phase3_memory_candidate_diagnostic(
            point=point,
            current_row=row,
        )
        for point in points
    ]
    diagnostics.sort(
        key=lambda value: (
            value.get("compatibility_status") == "MATCHED",
            bool(value.get("exact_question_id")),
            float(value.get("evidence_similarity") or -1.0),
            float(value.get("question_similarity") or -1.0),
            float(value.get("overall_similarity") or -1.0),
        ),
        reverse=True,
    )
    base["candidate_memories"] = diagnostics[:RETRIEVAL_MEMORY_TOP_K]

    matched_rows = [
        value
        for value in diagnostics
        if value.get("compatibility_status") == "MATCHED"
    ]
    if not matched_rows:
        if diagnostics:
            best = diagnostics[0]
            base.update(
                {
                    "overall_similarity": best.get("overall_similarity"),
                    "topic_similarity": best.get("topic_similarity"),
                    "evidence_similarity": best.get("evidence_similarity"),
                    "question_similarity": best.get("question_similarity"),
                    "exact_question_id": best.get("exact_question_id"),
                    "reference_compatible": best.get("reference_compatible"),
                    "role_compatible": best.get("role_compatible"),
                    "topic_compatible": best.get("topic_compatible"),
                    "evidence_compatible": best.get("evidence_compatible"),
                    "question_compatible": best.get("question_compatible"),
                    "overall_similarity_passed": best.get("overall_similarity_passed"),
                    "rejection_reasons": best.get("rejection_reasons") or [],
                }
            )
        return base

    best = matched_rows[0]
    base.update(
        {
            "matched": True,
            "recall_status": "matched_compatible_memory",
            "memory_point_id": best.get("memory_point_id"),
            "feedback_id": best.get("feedback_id"),
            "decision": best.get("decision"),
            "reason": best.get("reason"),
            "overall_similarity": best.get("overall_similarity"),
            "topic_similarity": best.get("topic_similarity"),
            "evidence_similarity": best.get("evidence_similarity"),
            "question_similarity": best.get("question_similarity"),
            "exact_question_id": best.get("exact_question_id"),
            "reference_compatible": best.get("reference_compatible"),
            "role_compatible": best.get("role_compatible"),
            "topic_compatible": best.get("topic_compatible"),
            "evidence_compatible": best.get("evidence_compatible"),
            "question_compatible": best.get("question_compatible"),
            "overall_similarity_passed": best.get("overall_similarity_passed"),
            "rejection_reasons": [],
        }
    )
    return base


if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Phase 3 retrieval-memory recall skipped: no retrieval candidates.")
else:
    # Resolve collection safely. Missing/empty memory is a normal cold-start state.
    try:
        phase3_collection_names = {
            collection.name
            for collection in qdrant_client.get_collections().collections
        }
        RETRIEVAL_MEMORY_AVAILABLE = (
            RETRIEVAL_MEMORY_COLLECTION in phase3_collection_names
        )
        if RETRIEVAL_MEMORY_AVAILABLE:
            RETRIEVAL_MEMORY_POINT_COUNT = int(
                qdrant_client.count(
                    collection_name=RETRIEVAL_MEMORY_COLLECTION,
                    exact=True,
                ).count
            )
            if RETRIEVAL_MEMORY_POINT_COUNT <= 0:
                RETRIEVAL_MEMORY_AVAILABLE = False
                RETRIEVAL_MEMORY_UNAVAILABLE_REASON = "memory_collection_empty"
            else:
                RETRIEVAL_MEMORY_UNAVAILABLE_REASON = ""
        else:
            RETRIEVAL_MEMORY_POINT_COUNT = 0
            RETRIEVAL_MEMORY_UNAVAILABLE_REASON = "memory_collection_missing"
    except Exception as exc:
        RETRIEVAL_MEMORY_AVAILABLE = False
        RETRIEVAL_MEMORY_POINT_COUNT = 0
        RETRIEVAL_MEMORY_UNAVAILABLE_REASON = (
            "memory_collection_unavailable: "
            f"{type(exc).__name__}: {exc}"
        )

    # Hard safety snapshot: Phase 3 is forbidden from changing baseline scores/ranks.
    phase3_score_before = phase2_candidates_df["final_score"].astype(float).copy()
    phase3_rank_before = phase2_candidates_df["phase2_rank"].copy()

    phase3_memory_diagnostics = [
        _phase3_recall_for_candidate(row)
        for _, row in phase2_candidates_df.iterrows()
    ]

    phase2_candidates_df = phase2_candidates_df.copy()
    phase2_candidates_df["memory_recall_phase3"] = phase3_memory_diagnostics
    phase2_candidates_df["memory_recall_phase3_version"] = RETRIEVAL_MEMORY_PHASE3_VERSION
    phase2_candidates_df["memory_recall_collection"] = RETRIEVAL_MEMORY_COLLECTION
    phase2_candidates_df["memory_recall_collection_points"] = RETRIEVAL_MEMORY_POINT_COUNT
    phase2_candidates_df["memory_match_found"] = [
        bool(value.get("matched")) for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_match_decision"] = [
        value.get("decision") for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_match_overall_similarity"] = [
        value.get("overall_similarity") for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_match_evidence_similarity"] = [
        value.get("evidence_similarity") for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_match_question_similarity"] = [
        value.get("question_similarity") for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_match_topic_similarity"] = [
        value.get("topic_similarity") for value in phase3_memory_diagnostics
    ]
    phase2_candidates_df["memory_rank_adjustment"] = 0.0
    phase2_candidates_df["memory_adjusted_final_score"] = phase2_candidates_df["final_score"].astype(float)
    phase2_candidates_df["memory_ranking_adjustment_enabled"] = False

    # Explicit invariant: Phase 3 recall must not mutate ranking inputs.
    if not np.allclose(
        phase3_score_before.to_numpy(dtype=float),
        phase2_candidates_df["final_score"].to_numpy(dtype=float),
        atol=0.0,
        rtol=0.0,
    ):
        raise RuntimeError("Phase 3 memory recall changed final_score unexpectedly.")
    if not phase3_rank_before.reset_index(drop=True).equals(
        phase2_candidates_df["phase2_rank"].reset_index(drop=True)
    ):
        raise RuntimeError("Phase 3 memory recall changed phase2_rank unexpectedly.")

    phase3_memory_recall_df = phase2_candidates_df[
        [
            "phase2_rank",
            "detected_topic",
            "agent1_role",
            "official_reference",
            "question_id",
            "question_text",
            "final_score",
            "memory_match_found",
            "memory_match_decision",
            "memory_match_overall_similarity",
            "memory_match_evidence_similarity",
            "memory_match_question_similarity",
            "memory_match_topic_similarity",
            "memory_rank_adjustment",
            "memory_adjusted_final_score",
        ]
    ].copy()

    print(
        "Retrieval HITL Phase 3 memory recall:",
        {
            "collection": RETRIEVAL_MEMORY_COLLECTION,
            "memory_points": RETRIEVAL_MEMORY_POINT_COUNT,
            "candidates_checked": int(len(phase2_candidates_df)),
            "compatible_matches": int(phase2_candidates_df["memory_match_found"].sum()),
            "ranking_adjustment_enabled": False,
            "all_adjustments_zero": bool(
                phase2_candidates_df["memory_rank_adjustment"].eq(0.0).all()
            ),
        },
    )
    display(phase3_memory_recall_df)


## 10C. Retrieval HITL Phase 4 — bounded memory-aware ranking

Phase 4 activates the already-validated Phase 3 compatibility layer.

Safety policy:

- only Phase 2 quality-safe candidates are eligible;
- no incompatible/rejected Phase 3 memory can alter a score;
- exact question + exact lesson-context memory is preferred;
- conflicting compatible decisions fail closed with zero adjustment;
- `relevant` memory receives a small positive boost;
- exact-context `not_relevant` memory is excluded from final selection for that lesson only;
- broader compatible `not_relevant` memory receives a small negative penalty;
- the adjustment is confidence-scaled and hard-bounded;
- the original `final_score` and `phase2_rank` are preserved for audit;
- selection uses `memory_adjusted_final_score`, never a modified baseline score;
- exact-context suppressed questions remain in the knowledge base for different lesson contexts.

This is the first phase where retrieval memory is allowed to affect the final
question ranking/selection.


In [ ]:
# ============================================================================
# RETRIEVAL HITL PHASE 4 — BOUNDED MEMORY-AWARE RANKING
# ============================================================================
# Phase 3 owns recall/compatibility. Phase 4 is deliberately small:
# it converts only a compatible, non-conflicting memory decision into a
# bounded score adjustment. All Phase 2 quality/concept/paper gates remain
# unchanged and the original final_score is preserved for audit.
# ============================================================================

RETRIEVAL_MEMORY_PHASE4_VERSION = (
    "agent2-retrieval-hitl-phase4-contextual-suppression-v1.1.0"
)
RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST = 0.04
RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY = 0.04
RETRIEVAL_MEMORY_RANKING_ADJUSTMENT_ENABLED = True


def _phase4_matching_memory_rows(
    phase3_diagnostic: dict[str, Any],
) -> list[dict[str, Any]]:
    rows = [
        dict(value)
        for value in (phase3_diagnostic.get("candidate_memories") or [])
        if isinstance(value, dict)
        and str(value.get("compatibility_status") or "").strip().upper()
        == "MATCHED"
        and str(value.get("decision") or "").strip().casefold()
        in {"relevant", "not_relevant"}
    ]

    # Backward-compatible fallback if a Phase 3 diagnostic contains only its
    # winning memory rather than the full candidate list.
    if not rows and bool(phase3_diagnostic.get("matched")):
        decision = str(
            phase3_diagnostic.get("decision") or ""
        ).strip().casefold()
        if decision in {"relevant", "not_relevant"}:
            rows = [
                {
                    "memory_point_id": phase3_diagnostic.get("memory_point_id"),
                    "feedback_id": phase3_diagnostic.get("feedback_id"),
                    "decision": decision,
                    "reason": phase3_diagnostic.get("reason"),
                    "overall_similarity": phase3_diagnostic.get("overall_similarity"),
                    "topic_similarity": phase3_diagnostic.get("topic_similarity"),
                    "evidence_similarity": phase3_diagnostic.get("evidence_similarity"),
                    "question_similarity": phase3_diagnostic.get("question_similarity"),
                    "exact_question_id": phase3_diagnostic.get("exact_question_id"),
                    "exact_lesson_evidence_hash": phase3_diagnostic.get(
                        "exact_lesson_evidence_hash"
                    ),
                    "compatibility_status": "MATCHED",
                }
            ]

    return rows


def _phase4_memory_strength(memory_row: dict[str, Any]) -> float:
    """
    Confidence-scale the bounded adjustment.

    The weakest available compatibility signal controls the strength. Therefore
    a generalized semantic match near a Phase 3 threshold receives a smaller
    adjustment than an exact/super-strong contextual match.
    """
    values: list[float] = []
    for key in (
        "overall_similarity",
        "topic_similarity",
        "evidence_similarity",
        "question_similarity",
    ):
        value = memory_row.get(key)
        if value is None:
            continue
        try:
            values.append(float(value))
        except (TypeError, ValueError):
            continue

    if not values:
        return 0.0

    return float(np.clip(min(values), 0.0, 1.0))


def _phase4_adjustment_for_candidate(
    row: pd.Series,
) -> dict[str, Any]:
    phase3_diagnostic = row.get("memory_recall_phase3")
    if not isinstance(phase3_diagnostic, dict):
        phase3_diagnostic = {}

    base_score = float(row.get("final_score") or 0.0)
    base = {
        "version": RETRIEVAL_MEMORY_PHASE4_VERSION,
        "enabled": True,
        "applied": False,
        "status": "no_compatible_memory",
        "decision": None,
        "memory_point_id": None,
        "feedback_id": None,
        "memory_strength": 0.0,
        "max_positive_boost": RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST,
        "max_negative_penalty": RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY,
        "ranking_adjustment": 0.0,
        "base_final_score": base_score,
        "adjusted_final_score": base_score,
        "phase2_rank": int(row.get("phase2_rank") or 0),
        "phase4_rank": None,
        "rank_changed": False,
        "conflict_detected": False,
        "exact_context_preferred": False,
        "matched_memory_count": 0,
        "hard_suppressed": False,
        "selection_eligible_after_memory": True,
        "suppression_reason": None,
        "reason": None,
    }

    if not bool(phase3_diagnostic.get("matched")):
        status = str(
            phase3_diagnostic.get("recall_status") or "no_compatible_memory"
        ).strip()
        base["status"] = (
            "no_compatible_memory"
            if status == "matched_compatible_memory"
            else status
        )
        return base

    matched_rows = _phase4_matching_memory_rows(phase3_diagnostic)
    base["matched_memory_count"] = int(len(matched_rows))
    if not matched_rows:
        base["status"] = "compatible_memory_without_supported_decision"
        return base

    # Exact same question + exact same lesson evidence takes precedence over
    # broader semantic matches. This is the most context-specific human memory.
    exact_context_rows = [
        value
        for value in matched_rows
        if bool(value.get("exact_question_id"))
        and bool(value.get("exact_lesson_evidence_hash"))
    ]
    decision_pool = exact_context_rows if exact_context_rows else matched_rows
    base["exact_context_preferred"] = bool(exact_context_rows)

    decisions = {
        str(value.get("decision") or "").strip().casefold()
        for value in decision_pool
        if str(value.get("decision") or "").strip().casefold()
        in {"relevant", "not_relevant"}
    }

    # Conflicting human outcomes in equally eligible memory fail closed.
    if len(decisions) != 1:
        base["status"] = "conflicting_compatible_memories_no_adjustment"
        base["conflict_detected"] = True
        return base

    decision = next(iter(decisions))

    decision_pool = sorted(
        decision_pool,
        key=lambda value: (
            bool(value.get("exact_question_id")),
            bool(value.get("exact_lesson_evidence_hash")),
            float(value.get("evidence_similarity") or -1.0),
            float(value.get("question_similarity") or -1.0),
            float(value.get("overall_similarity") or -1.0),
        ),
        reverse=True,
    )
    chosen = decision_pool[0]

    strength = _phase4_memory_strength(chosen)
    if strength <= 0.0:
        base["status"] = "compatible_memory_zero_strength_no_adjustment"
        return base

    # Exact same question + exact same lesson evidence + a human Not Relevant
    # decision is authoritative for THIS lesson context. Do not merely lower the
    # score and then reintroduce the question because the candidate pool is
    # small. Keep the question in the knowledge base and diagnostics, but remove
    # it from final selection for this exact context.
    #
    # This is deliberately narrower than semantic negative memory:
    # - exact-context Not Relevant -> hard suppression for this lesson only;
    # - broader compatible Not Relevant -> bounded penalty only;
    # - incompatible/different lesson -> no effect.
    if exact_context_rows and decision == "not_relevant":
        base.update(
            {
                "applied": False,
                "status": "exact_context_not_relevant_suppressed",
                "decision": decision,
                "memory_point_id": chosen.get("memory_point_id"),
                "feedback_id": chosen.get("feedback_id"),
                "memory_strength": round(strength, 6),
                "ranking_adjustment": 0.0,
                "adjusted_final_score": base_score,
                "hard_suppressed": True,
                "selection_eligible_after_memory": False,
                "suppression_reason": (
                    chosen.get("reason")
                    or "Previously marked Not Relevant for the exact same question and lesson evidence."
                ),
                "reason": chosen.get("reason"),
            }
        )
        return base

    if decision == "relevant":
        adjustment = (
            RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST
            * strength
        )
        status = "relevant_memory_boost_applied"
    else:
        adjustment = -(
            RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY
            * strength
        )
        status = "not_relevant_memory_penalty_applied"

    adjusted_score = float(
        np.clip(
            base_score + adjustment,
            0.0,
            1.0,
        )
    )
    effective_adjustment = adjusted_score - base_score

    base.update(
        {
            "applied": bool(abs(effective_adjustment) > 1e-12),
            "status": status,
            "decision": decision,
            "memory_point_id": chosen.get("memory_point_id"),
            "feedback_id": chosen.get("feedback_id"),
            "memory_strength": round(strength, 6),
            "ranking_adjustment": round(float(effective_adjustment), 6),
            "adjusted_final_score": round(adjusted_score, 6),
            "reason": chosen.get("reason"),
        }
    )
    return base


if globals().get("AGENT2_NO_RETRIEVAL_CANDIDATES", False):
    print("Phase 4 memory-aware ranking skipped: no retrieval candidates.")
else:
    # Hard audit snapshot. Phase 4 may not mutate the baseline Phase 2 score.
    phase4_baseline_score_before = (
        phase2_candidates_df["final_score"].astype(float).copy()
    )
    phase4_baseline_rank_before = (
        phase2_candidates_df["phase2_rank"].copy()
    )

    phase4_diagnostics = [
        _phase4_adjustment_for_candidate(row)
        for _, row in phase2_candidates_df.iterrows()
    ]

    phase2_candidates_df = phase2_candidates_df.copy()
    phase2_candidates_df["memory_ranking_phase4"] = phase4_diagnostics
    phase2_candidates_df["memory_ranking_phase4_version"] = (
        RETRIEVAL_MEMORY_PHASE4_VERSION
    )
    phase2_candidates_df["memory_rank_adjustment"] = [
        float(value.get("ranking_adjustment") or 0.0)
        for value in phase4_diagnostics
    ]
    phase2_candidates_df["memory_adjusted_final_score"] = [
        float(value.get("adjusted_final_score") or 0.0)
        for value in phase4_diagnostics
    ]
    phase2_candidates_df["memory_adjustment_strength"] = [
        float(value.get("memory_strength") or 0.0)
        for value in phase4_diagnostics
    ]
    phase2_candidates_df["memory_ranking_adjustment_enabled"] = True
    phase2_candidates_df["memory_adjustment_applied"] = [
        bool(value.get("applied"))
        for value in phase4_diagnostics
    ]
    phase2_candidates_df["memory_hard_suppressed"] = [
        bool(value.get("hard_suppressed"))
        for value in phase4_diagnostics
    ]
    phase2_candidates_df["memory_selection_eligible"] = [
        bool(value.get("selection_eligible_after_memory", True))
        for value in phase4_diagnostics
    ]

    # Global diagnostic rank only. Final selection still applies all existing
    # topic-role/coverage/quality constraints, using the adjusted score as one
    # bounded quality input.
    phase4_ordered_indexes = (
        phase2_candidates_df
        .sort_values(
            [
                "memory_hard_suppressed",
                "memory_adjusted_final_score",
                "final_score",
                "phase2_rank",
            ],
            ascending=[True, False, False, True],
        )
        .index
        .tolist()
    )
    phase4_rank_lookup = {
        index: rank
        for rank, index in enumerate(
            phase4_ordered_indexes,
            start=1,
        )
    }
    phase2_candidates_df["phase4_rank"] = [
        int(phase4_rank_lookup[index])
        for index in phase2_candidates_df.index
    ]

    updated_phase4_diags = []
    for index, diagnostic in zip(
        phase2_candidates_df.index,
        phase4_diagnostics,
    ):
        value = dict(diagnostic)
        value["phase2_rank"] = int(
            phase2_candidates_df.at[index, "phase2_rank"]
        )
        value["phase4_rank"] = int(
            phase2_candidates_df.at[index, "phase4_rank"]
        )
        value["rank_changed"] = bool(
            value["phase2_rank"] != value["phase4_rank"]
        )
        updated_phase4_diags.append(value)

    phase2_candidates_df["memory_ranking_phase4"] = updated_phase4_diags

    # ---------------------------------------------------------------
    # Phase 4 hard safety invariants
    # ---------------------------------------------------------------
    # 1. Baseline score/rank remain intact for audit.
    if not np.allclose(
        phase4_baseline_score_before.to_numpy(dtype=float),
        phase2_candidates_df["final_score"].to_numpy(dtype=float),
        atol=0.0,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Phase 4 unexpectedly mutated baseline final_score."
        )
    if not phase4_baseline_rank_before.reset_index(drop=True).equals(
        phase2_candidates_df["phase2_rank"].reset_index(drop=True)
    ):
        raise RuntimeError(
            "Phase 4 unexpectedly mutated baseline phase2_rank."
        )

    # 2. Adjustment can never exceed the configured bounds.
    if (
        phase2_candidates_df["memory_rank_adjustment"].max()
        > RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST + 1e-12
        or phase2_candidates_df["memory_rank_adjustment"].min()
        < -RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY - 1e-12
    ):
        raise RuntimeError(
            "Phase 4 memory adjustment exceeded its hard bound."
        )

    # 3. An incompatible/no-memory candidate must remain unchanged.
    no_match_mask = ~phase2_candidates_df[
        "memory_match_found"
    ].astype(bool)
    if not phase2_candidates_df.loc[
        no_match_mask,
        "memory_rank_adjustment",
    ].eq(0.0).all():
        raise RuntimeError(
            "Phase 4 adjusted a candidate without a compatible Phase 3 memory."
        )

    # 4. Exact-context human rejection must be selection-ineligible and must
    # not mutate the baseline score. Suppression is a contextual selection
    # decision, not an infinite/hidden score penalty.
    hard_suppressed_mask = phase2_candidates_df[
        "memory_hard_suppressed"
    ].astype(bool)
    if hard_suppressed_mask.any():
        if not (
            ~phase2_candidates_df.loc[
                hard_suppressed_mask,
                "memory_selection_eligible",
            ].astype(bool)
        ).all():
            raise RuntimeError(
                "Phase 4 exact-context rejected candidate remained selection-eligible."
            )
        if not phase2_candidates_df.loc[
            hard_suppressed_mask,
            "memory_rank_adjustment",
        ].eq(0.0).all():
            raise RuntimeError(
                "Phase 4 hard suppression unexpectedly mutated the ranking score."
            )

    # 5. Phase 4 cannot rescue anything outside the existing Phase 2 safe pool.
    if "phase2_gate_passed" in phase2_candidates_df.columns:
        if not phase2_candidates_df[
            "phase2_gate_passed"
        ].astype(bool).all():
            raise RuntimeError(
                "Phase 4 received a candidate that did not pass Phase 2."
            )

    phase4_memory_ranking_df = phase2_candidates_df[
        [
            "phase2_rank",
            "phase4_rank",
            "detected_topic",
            "agent1_role",
            "official_reference",
            "question_id",
            "question_text",
            "final_score",
            "memory_match_found",
            "memory_match_decision",
            "memory_adjustment_strength",
            "memory_rank_adjustment",
            "memory_adjusted_final_score",
            "memory_adjustment_applied",
            "memory_hard_suppressed",
            "memory_selection_eligible",
        ]
    ].copy()

    print(
        "Retrieval HITL Phase 4 bounded ranking:",
        {
            "version": RETRIEVAL_MEMORY_PHASE4_VERSION,
            "max_positive_boost": RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST,
            "max_negative_penalty": RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY,
            "candidates_checked": int(len(phase2_candidates_df)),
            "adjustments_applied": int(
                phase2_candidates_df["memory_adjustment_applied"].sum()
            ),
            "positive_boosts": int(
                (phase2_candidates_df["memory_rank_adjustment"] > 0).sum()
            ),
            "negative_penalties": int(
                (phase2_candidates_df["memory_rank_adjustment"] < 0).sum()
            ),
            "exact_context_not_relevant_suppressed": int(
                phase2_candidates_df["memory_hard_suppressed"].sum()
            ),
            "global_diagnostic_rank_changes": int(
                (
                    phase2_candidates_df["phase4_rank"]
                    != phase2_candidates_df["phase2_rank"]
                ).sum()
            ),
        },
    )
    display(phase4_memory_ranking_df)


## 10D. Select a balanced question set after topic ownership

Dynamic programming first tries to satisfy the complete assessment request:

- requested number of questions;
- minimum primary and supporting questions;
- minimum distinct official references;
- every explicitly required official reference;
- total marks closest to the requested target;
- strongest combined retrieval and concept-fit score.

### Coverage-feasibility refinement

A valid quality-safe candidate pool can still make the requested coverage impossible.

Example:

```text
Approved Agent 1 topics: one primary topic only
Requested supporting questions: one
```

The previous implementation raised an exception even though relevant, quality-safe
questions were available.

The final approach is:

```text
Try strict coverage requirements
        ↓
If strict coverage is feasible:
    select the best strict combination
        ↓
If strict coverage is impossible:
    keep all quality gates active
    select the largest best-quality safe combination
    minimise documented coverage gaps
    set release status to needs_user_decision
```

The fallback never reintroduces questions rejected by the semantic, concept-fit,
duplicate or text-quality gates. It only prevents an impossible user configuration
from crashing the notebook.


### Ownership guarantee

Before selection, each physical question has exactly one detailed Agent 1
topic owner. If a question was relevant to multiple approved topics, ownership
is awarded using the detailed-topic relevance score and the other topics keep
their alternative candidates.

This means a penetration-testing question cannot disappear from the
`Penetration testing` topic merely because a broader security topic retrieved
the same DB question first.


### Metadata-mismatch rescue safety

A rescued question is **not** treated as an exact metadata match. Even if the
Agent 1 topic itself has an exact canonical reference, a question recovered
from another stored reference is forced into the broad hybrid path.

This prevents the old failure mode where a nearby/sibling question could be
accepted simply because its metadata was in the same area of the syllabus.
The question text and lesson evidence must independently support the detailed
Agent 1 topic.

### Generic precision/selection refinement

The final selector now applies four topic-agnostic rules:

1. **Exact detected concept before broad syllabus fit.** A candidate cannot
   survive only because it belongs to the same broad official section. When
   Agent 1's detected topic adds specificity beyond the canonical concept, the
   candidate's assessed skill must support that specific concept.

2. **Semantic subconcept diversity.** Within each topic, quality-safe candidates
   receive a small assessed-skill novelty preference. This encourages different
   facets/question styles without hardcoding syllabus topic names.

3. **Explicit role allocation.** Frontend minimum primary/supporting counts are
   hard constraints. After relevance and coverage, the selector prefers a mark
   distribution consistent with the requested primary-question share. If
   `target_total_marks` is supplied, it is optimized only after those constraints.

4. **Per-question PDF annotation.** Every selected question is labelled with the
   generated question number, detected topic, role, marks and original AQA source
   question number in the student and teacher PDFs.

No topic-specific keyword list, question ID or syllabus concept exception is added.


### Resilient role-minimum policy

Primary/supporting minima are handled as follows:

```text
try the exact requested role minima
        ↓
if feasible:
    enforce them
        ↓
if infeasible after all quality/relevance gates:
    do NOT relax quality
    do NOT insert weak questions
    select the best quality-safe partial combination
    report requested vs available vs selected counts
    mark the assessment as needs_user_decision / partial
    continue generating the available assessment
```

Only structurally invalid requests (for example, a minimum larger than the total
requested question count) remain fatal validation errors.


### Final generic refinement

Two final selection refinements are applied without any syllabus-topic-specific
rules:

1. **Primary-role balance**
   - If both Primary and Supporting roles are available, the preferred assessment
     balance is 50% Primary questions/marks, subject to the user's explicit
     minimums and topic-coverage requirements.
   - This is a preference only. If the quality-safe Primary pool is smaller, no
     rejected/weak question is restored; the safe partial assessment is returned
     with an explicit balance shortfall.
   - When one physical question credibly matches both a Primary and Supporting
     topic, Primary wins only when its ownership score is within a small global
     tie margin of the best credible owner.

2. **Actual assessed-skill directness**
   - Final selection explicitly scores the instruction/answer-demand the student
     must perform.
   - Direct detected-topic semantic fit, cross-concept ownership and lexical
     evidence are rewarded.
   - Context-dependent matches are penalised so a question does not rank highly
     merely because its surrounding parent stem contains the topic.
   - No topic names, question IDs or manually written concept keyword rules are
     used.

These refinements preserve all existing hard paper/language filters, Qdrant
retrieval, PostgreSQL QP/MS linking, question-quality gates, deduplication,
resilient partial-output behaviour and PDF annotations.


### Final lock-candidate refinements

The final retrieval logic applies three topic-agnostic refinements:

1. **Shared dynamic role allocation**
   - One function computes the preferred Primary/Supporting allocation from the
     user's number-of-questions and minimum-role controls.
   - The same plan is used for Phase 2 topic quotas, availability messages and
     final selection.
   - Therefore a 10-question request with both roles and Primary minimum 3 /
     Supporting minimum 1 targets approximately 5 Primary and 5 Supporting,
     rather than creating a separate internal Primary quota of 8.

2. **Agent 1 evidence-based semantic recall**
   - Direct topic queries now include an excerpt of Agent 1's actual chunk/lesson
     evidence in addition to the detected topic and canonical concept.
   - This helps semantically equivalent exam tasks survive retrieval even when
     they use different wording or application contexts.
   - No quality threshold is lowered and no hand-authored synonym list is used.

3. **Hard final assessed-task ownership**
   - Parent/context material may help retrieve a candidate.
   - Final ownership requires the actual student answer-demand to directly support
     the detailed Agent 1 concept.
   - A broad canonical match is insufficient when the answer-demand fits the
     broad concept materially better than the detailed detected topic.
   - This prevents context-only subquestions from surviving simply because an
     array, loop, search, network concept, etc. appears elsewhere in the parent.

Resilient partial output remains unchanged: if the shared preferred role target
cannot be achieved with quality-safe questions, the best safe assessment is still
generated and the exact shortfall is reported.


### Over-pruning correction

The final gate now separates recall from precision:

- Agent 1 chunk/lesson evidence remains in the upstream Qdrant retrieval query,
  so semantically equivalent exam questions can still enter the candidate pool.
- Compact topic/concept queries are used for precision and ownership scoring;
  long lesson evidence is not mixed into these direct scores.
- Final assessed-task ownership uses an adaptive topic-local anchor instead of
  a high fixed MiniLM cutoff.
- Explicit direct-task matches define the topic's anchor. Semantic equivalents
  may pass when the requested official concept owns the assessed skill and the
  score is close to that anchor.
- If no explicit anchor exists, the existing broad semantic floor plus official
  ownership is used rather than deleting the entire topic.
- The shared dynamic Primary/Supporting allocation remains unchanged.

No topic-specific keywords, question IDs or lowered quality thresholds were
introduced.


### Ownership lexical-noise root-cause fix

Runtime audit data exposed a lexical false-positive in the ownership gate.

The detected topic `One- and two-dimensional arrays` previously produced the
specific tokens `one`, `dimensional`, `arrays`. AQA multiple-choice boilerplate
such as `Shade one lozenge` therefore created non-zero "specific topic" lexical
overlap even when the actual assessed task was unrelated to arrays.

The ownership-only lexical path now:

- removes cardinal-number and exam-response boilerplate tokens;
- normalises simple singular/plural forms for ownership evidence only;
- keeps Qdrant/BM25 retrieval tokenisation unchanged;
- evaluates hierarchical child lexical evidence against the **detailed Agent 1
  topic tokens**, not the broad canonical topic heading.

As a result, generic `data structure` or Boolean subquestions cannot pass an
Array ownership gate merely because the word `one` occurs in AQA response
instructions.

No question IDs, topic-specific exceptions or new semantic thresholds are used.


In [ ]:
def select_questions(
    candidates: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """
    Select a quality-safe assessment while respecting frontend controls.

    Priority order:
      1. hard user filters are already enforced upstream and are never relaxed,
      2. primary Agent 1 topic coverage first,
      3. one question from each required supporting topic where available,
      4. minimum primary/supporting counts,
      5. requested question count,
      6. quality/ranking score.

    Total marks are informational only. Selection does not try to hit a
    target-total-marks value.

    If a topic is unavailable, the best safe partial assessment is returned and
    the frontend receives structured warnings. Rejected/weak questions are never
    reintroduced merely to satisfy coverage.
    """

    requested_count = int(request["number_of_questions"])
    minimum_primary = int(request["minimum_primary_questions"])
    minimum_supporting = int(request["minimum_supporting_questions"])
    minimum_distinct = int(request["minimum_distinct_official_references"])
    target_total_marks = request.get("target_total_marks")
    target_total_marks = (
        int(target_total_marks)
        if target_total_marks not in (None, "")
        else None
    )
    cover_all_topics = bool(
        request.get("cover_all_approved_topics", COVER_ALL_APPROVED_TOPICS)
    )

    global AGENT2_NO_SAFE_CANDIDATES

    if candidates.empty:
        AGENT2_NO_SAFE_CANDIDATES = True
        requested_paper = request.get("paper_code")
        requested_paper_label = paper_label(requested_paper)
        approved_topic_indexes_for_run = set(
            validated_topics_df.get("agent1_topic_index", pd.Series(dtype=int))
            .dropna().astype(int).tolist()
        )
        all_topics_are_paper_mismatches = bool(
            approved_topic_indexes_for_run
            and approved_topic_indexes_for_run.issubset(PAPER_MISMATCH_TOPIC_INDEXES)
        )
        approved_topic_names_for_run = (
            validated_topics_df.get("detected_topic", pd.Series(dtype=str))
            .dropna().astype(str).tolist()
        )
        if all_topics_are_paper_mismatches and requested_paper:
            friendly_message = (
                f"The AQA specification assigns all approved topics to the other paper, "
                f"not {requested_paper_label}. Retrieval was skipped and no "
                "wrong-paper question was substituted. Change the paper filter "
                "and run again."
            )
            message_code = "all_approved_topics_on_other_paper"
        else:
            friendly_message = (
                "No quality-safe questions remain after applying the selected "
                "paper/filter settings and the semantic/concept-fit checks. "
                "No weak or incompatible question was substituted. Revise the "
                "filters, approve another topic, or change the paper and run again."
            )
            message_code = "no_quality_safe_questions_for_current_request"
        add_agent2_user_message(
            level="warning", code=message_code, message=friendly_message,
            requested=requested_count, available=0,
            details={
                "requested_paper_code": requested_paper,
                "requested_paper_label": requested_paper_label,
                "approved_topics": approved_topic_names_for_run,
                "paper_mismatch_topic_indexes": sorted(PAPER_MISMATCH_TOPIC_INDEXES),
                "assessment_generated": False,
            },
        )
        empty_df = candidates.copy()
        for column in [
            "selected_rank","phase2_rank","detected_topic","agent1_role",
            "retrieval_stage","official_reference","question_number","marks",
            "semantic_score","direct_topic_semantic_score","bm25_lexical_normalized",
            "hybrid_relevance_score","retrieval_pool_type",
            "topic_ownership_score","question_ownership_status",
            "owner_detected_topic",
            "direct_concept_score","combined_concept_score",
            "direct_concept_threshold","concept_fit_mark_margin",
            "concept_fit_rescue_used","question_quality_issues","final_score",
            "question_text",
        ]:
            if column not in empty_df.columns:
                empty_df[column] = pd.Series(dtype="object")
        summary = {
            "run_status": AGENT2_RUN_STATUS,
            "assessment_generated": False,
            "assessment_release_status": "no_safe_assessment",
            "requires_user_decision": True,
            "user_message_count": int(len(AGENT2_USER_MESSAGES)),
            "user_messages": json_safe(AGENT2_USER_MESSAGES),
            "requested_questions": requested_count,
            "selected_questions": 0,
            "requested_question_count_met": False,
            "candidate_count_after_concept_fit": 0,
            "question_count_shortfall": requested_count,
            "selected_marks": 0,
            "coverage_selection_mode": "no_safe_candidates",
            "strict_coverage_possible": False,
            "strict_coverage_option_count": 0,
            "coverage_constraint_adaptation_applied": True,
            "coverage_requirements_met": False,
            "coverage_release_blockers": ["no_quality_safe_candidates_for_current_request"],
            "cover_all_approved_topics": cover_all_topics,
            "requested_agent1_topic_indexes": sorted(approved_topic_indexes_for_run),
            "effective_required_agent1_topic_indexes": [],
            "selected_agent1_topic_indexes": [],
            "selected_distinct_agent1_topics": 0,
            "missing_required_agent1_topic_indexes": sorted(approved_topic_indexes_for_run),
            "missing_required_agent1_topics": approved_topic_names_for_run,
            "preferred_primary_question_count_after_coverage": 0,
            "requested_minimum_primary_questions": minimum_primary,
            "effective_minimum_primary_questions": 0,
            "selected_primary_questions": 0,
            "primary_requirement_met": False,
            "requested_minimum_supporting_questions": minimum_supporting,
            "effective_minimum_supporting_questions": 0,
            "selected_supporting_questions": 0,
            "supporting_requirement_met": False,
            "requested_minimum_distinct_official_references": int(request.get("requested_minimum_distinct_official_references", minimum_distinct)),
            "configured_effective_minimum_distinct_official_references": minimum_distinct,
            "effective_minimum_distinct_official_references": 0,
            "selected_distinct_official_references": 0,
            "distinct_reference_requirement_met": False,
            "required_official_references": [],
            "effective_required_official_references": [],
            "unavailable_required_official_references": [],
            "missing_selected_required_official_references": [],
            "all_required_references_covered": False,
            "available_primary_candidates": 0,
            "available_supporting_candidates": 0,
            "available_official_references": [],
            "selected_official_references": [],
            "selected_canonical_agent2_references": [],
            "fallback_selected": 0,
            "phase2_threshold_strategy": (
            "hierarchical_hybrid_plus_post_relevance_topic_ownership"
        ),
            "adaptive_threshold_min": None,
            "adaptive_threshold_max": None,
            "adaptive_threshold_mean": None,
            "adaptive_pool_sufficient_without_rescue": False,
            "phase2_rescue_used": False,
            "weak_candidate_replacement_strategy": "never_replace_with_weak_or_wrong-paper_candidates",
        "retrieval_memory_phase4_version": (
            RETRIEVAL_MEMORY_PHASE4_VERSION
            if "RETRIEVAL_MEMORY_PHASE4_VERSION" in globals()
            else None
        ),
        "retrieval_memory_phase4_enabled": bool(
            globals().get(
                "RETRIEVAL_MEMORY_RANKING_ADJUSTMENT_ENABLED",
                False,
            )
        ),
        "selected_memory_adjustments_applied": 0,
        "selected_memory_positive_boosts": 0,
        "selected_memory_negative_penalties": 0,
        }
        return empty_df, summary

    working = candidates.copy()

    if "final_assessed_task_gate_passed" in working.columns:
        working = working[
            working["final_assessed_task_gate_passed"].astype(bool)
        ].copy()

    working["agent1_topic_index"] = working["agent1_topic_index"].astype(int)
    # Exact-concept precision is completed upstream. Selection adds a
    # small topic-local semantic novelty preference so different assessed
    # skills/facets are preferred when equally relevant alternatives exist.
    # This is generic MMR-style diversification; no topic names are hardcoded.
    # -------------------------------------------------------------
    # ACTUAL ASSESSED-SKILL DIRECTNESS
    # -------------------------------------------------------------
    # The final selector should prefer questions whose answer-demand itself
    # assesses the Agent 1 topic. Context, parent stems and metadata may still
    # help retrieve a question, but they should not outrank a question with a
    # stronger direct assessed-skill match.
    assessed_semantic = working.get(
        "detected_topic_assessed_skill_score",
        working.get(
            "assessed_skill_semantic_score",
            pd.Series(0.0, index=working.index),
        ),
    ).astype(float).clip(lower=0.0, upper=1.0)

    assessed_lexical = working.get(
        "detected_topic_specific_lexical_overlap",
        working.get(
            "assessed_skill_lexical_overlap",
            pd.Series(0.0, index=working.index),
        ),
    ).astype(float).clip(lower=0.0, upper=1.0)

    assessed_owner = working.get(
        "assessed_skill_official_owner_is_requested",
        pd.Series(True, index=working.index),
    ).astype(bool).astype(float)

    working["assessed_task_directness_score"] = (
        0.70 * assessed_semantic
        + 0.20 * assessed_owner
        + 0.10 * assessed_lexical
    ).clip(lower=0.0, upper=1.0)

    context_dependency_penalty = working.get(
        "context_dependency_flag",
        pd.Series(False, index=working.index),
    ).astype(bool).astype(float)

    # Phase 4 affects ranking only through this bounded effective score.
    # The original final_score remains untouched and auditable.
    working["_memory_effective_score"] = working.get(
        "memory_adjusted_final_score",
        working["final_score"],
    ).astype(float)

    working["_base_selection_quality_score"] = (
        working["_memory_effective_score"].astype(float)
        + 0.25 * working.get("combined_concept_score", 0.0).astype(float)
        + ASSESSED_TASK_DIRECTNESS_SELECTION_WEIGHT
        * working["assessed_task_directness_score"].astype(float)
        - CONTEXT_DEPENDENCY_SELECTION_PENALTY
        * context_dependency_penalty
    )

    working["subconcept_diversity_novelty_score"] = 1.0

    for topic_index, topic_group in working.groupby(
        "agent1_topic_index",
        sort=False,
    ):
        ordered_group = topic_group.sort_values(
            ["_base_selection_quality_score", "_memory_effective_score", "final_score"],
            ascending=[False, False, False],
        )
        ordered_indexes = ordered_group.index.tolist()
        if not ordered_indexes:
            continue

        diversity_texts = []
        for index in ordered_indexes:
            assessed_value = str(
                working.at[index, "assessed_skill_text"]
                if "assessed_skill_text" in working.columns
                else ""
            ).strip()
            diversity_texts.append(
                assessed_value
                or str(working.at[index, "question_text"] or "").strip()
            )

        # Reuse/persist assessed-skill question vectors through Qdrant.
        # No local/file embedding cache is introduced.
        diversity_vectors = qdrant_assessed_skill_vectors_for_rows(
            ordered_group
        )

        prior_vectors: list[np.ndarray] = []
        for local_position, index in enumerate(ordered_indexes):
            vector = diversity_vectors[local_position]
            if not prior_vectors:
                novelty = 1.0
            else:
                similarities = np.asarray(prior_vectors) @ vector
                novelty = float(
                    np.clip(
                        1.0 - float(np.max(similarities)),
                        0.0,
                        1.0,
                    )
                )
            working.at[index, "subconcept_diversity_novelty_score"] = novelty
            prior_vectors.append(vector)

    working["_selection_quality_score"] = (
        working["_base_selection_quality_score"].astype(float)
        + SUBCONCEPT_DIVERSITY_WEIGHT
        * working["subconcept_diversity_novelty_score"].astype(float)
    )

    # Keep a healthy number of candidates for every topic rather than taking a
    # global head() that could accidentally remove a lower-scoring topic pool.
    per_topic_limit = max(10, min(30, requested_count * 4))
    working = (
        working.sort_values(
            ["agent1_topic_index", "_selection_quality_score", "_memory_effective_score", "final_score"],
            ascending=[True, False, False, False],
        )
        .groupby("agent1_topic_index", group_keys=False)
        .head(per_topic_limit)
        .reset_index(drop=True)
    )

    maximum_selectable_count = min(requested_count, len(working))

    topic_metadata = (
        validated_topics_df
        .drop_duplicates(subset=["agent1_topic_index"])
        [["agent1_topic_index", "detected_topic", "role", "ranking_score"]]
        .copy()
    )
    topic_metadata["agent1_topic_index"] = topic_metadata[
        "agent1_topic_index"
    ].astype(int)
    topic_metadata["_role_order"] = topic_metadata["role"].map(
        {"primary": 0, "supporting": 1}
    ).fillna(2)
    topic_metadata = topic_metadata.sort_values(
        ["_role_order", "agent1_topic_index"], ascending=[True, True]
    ).reset_index(drop=True)

    approved_topic_indexes = [
        int(value) for value in topic_metadata["agent1_topic_index"].tolist()
    ]
    available_topic_indexes = sorted(
        set(working["agent1_topic_index"].astype(int).tolist())
    )
    available_topic_set = set(available_topic_indexes)

    primary_topic_indexes = [
        int(value)
        for value in topic_metadata.loc[
            topic_metadata["role"].eq("primary"), "agent1_topic_index"
        ].tolist()
        if int(value) in available_topic_set
    ]
    supporting_topic_indexes = [
        int(value)
        for value in topic_metadata.loc[
            topic_metadata["role"].eq("supporting"), "agent1_topic_index"
        ].tolist()
        if int(value) in available_topic_set
    ]

    requested_topic_indexes = [
        int(value)
        for value in request.get("requested_agent1_topic_indexes", approved_topic_indexes)
    ]

    if cover_all_topics:
        ordered_available_topics = [
            index for index in requested_topic_indexes if index in available_topic_set
        ]
        required_topic_indexes = set(
            ordered_available_topics[:maximum_selectable_count]
        )
    else:
        required_topic_indexes = {
            int(value)
            for value in request.get("required_agent1_topic_indexes", [])
            if int(value) in available_topic_set
        }

    # -------------------------------------------------------------
    # SHARED PRIMARY / SUPPORTING BALANCE TARGET
    # -------------------------------------------------------------
    # Use the exact same role plan that Phase 2 used for candidate quotas and
    # availability warnings. This keeps retrieval and final selection aligned.
    shared_role_targets = compute_shared_role_allocation_targets(
        requested_count=requested_count,
        minimum_primary=minimum_primary,
        minimum_supporting=minimum_supporting,
        has_primary=bool(primary_topic_indexes),
        has_supporting=bool(supporting_topic_indexes),
    )

    preferred_primary_count = int(
        shared_role_targets["preferred_primary_count"]
    )
    preferred_supporting_count = int(
        shared_role_targets["preferred_supporting_count"]
    )
    preferred_primary_question_share = float(
        shared_role_targets["preferred_primary_share"]
    )

    available_primary_candidates = int(
        working["agent1_role"].eq("primary").sum()
    )
    available_supporting_candidates = int(
        working["agent1_role"].eq("supporting").sum()
    )

    available_references = sorted(
        working["agent1_official_reference"]
        .dropna().astype(str).unique().tolist()
    )
    reference_bits = {
        reference: 1 << position
        for position, reference in enumerate(available_references)
    }
    topic_bits = {
        topic_index: 1 << position
        for position, topic_index in enumerate(available_topic_indexes)
    }

    required_references = {
        str(reference).strip()
        for reference in request.get("required_official_references", [])
        if str(reference).strip()
    }
    available_reference_set = set(available_references)
    unavailable_required_references = sorted(
        required_references - available_reference_set
    )
    effective_required_references = required_references & available_reference_set

    # Requested role minima are STRICT TARGETS when the quality-safe pool can
    # satisfy them. Candidate shortages are non-fatal: do not lower any quality
    # gate and do not insert weak questions. Instead, later selection chooses the
    # best safe partial combination and the frontend receives a structured warning.
    role_minimum_pool_shortage = bool(
        available_primary_candidates < minimum_primary
        or available_supporting_candidates < minimum_supporting
        or minimum_primary + minimum_supporting > maximum_selectable_count
    )

    effective_minimum_primary = min(
        minimum_primary,
        available_primary_candidates,
        maximum_selectable_count,
    )
    effective_minimum_supporting = min(
        minimum_supporting,
        available_supporting_candidates,
        maximum_selectable_count,
    )
    effective_minimum_distinct = min(
        minimum_distinct, len(available_references), maximum_selectable_count
    )

    # State = count, total_marks, primary_count, supporting_count,
    #         primary_marks, topic_mask, ref_mask
    states: dict[tuple[int, ...], tuple[float, tuple[int, ...]]] = {
        (0, 0, 0, 0, 0, 0, 0): (0.0, tuple())
    }

    for index, row in working.iterrows():
        next_states = dict(states)
        topic_index = int(row["agent1_topic_index"])
        reference = str(row["agent1_official_reference"])

        for state, value in states.items():
            (
                count,
                marks,
                primary,
                supporting,
                primary_marks,
                topic_mask,
                ref_mask,
            ) = state
            score, selected = value
            if count >= maximum_selectable_count:
                continue

            new_state = (
                count + 1,
                marks + int(row["marks"]),
                primary + int(row["agent1_role"] == "primary"),
                supporting + int(row["agent1_role"] == "supporting"),
                primary_marks
                + (
                    int(row["marks"])
                    if row["agent1_role"] == "primary"
                    else 0
                ),
                topic_mask | topic_bits[topic_index],
                ref_mask | reference_bits[reference],
            )
            new_value = (
                score + float(row["_selection_quality_score"]),
                selected + (index,),
            )
            previous = next_states.get(new_state)
            if previous is None or new_value[0] > previous[0]:
                next_states[new_state] = new_value

        states = next_states

    options: list[dict[str, Any]] = []
    primary_required_topics = required_topic_indexes.intersection(
        set(primary_topic_indexes)
    )
    supporting_required_topics = required_topic_indexes.intersection(
        set(supporting_topic_indexes)
    )

    for state, value in states.items():
        (
            count,
            marks,
            primary,
            supporting,
            primary_marks,
            topic_mask,
            ref_mask,
        ) = state
        score, selected = value
        if count == 0 or count > requested_count:
            continue

        selected_topics = {
            topic_index
            for topic_index, bit_value in topic_bits.items()
            if topic_mask & bit_value
        }
        selected_references = {
            reference
            for reference, bit_value in reference_bits.items()
            if ref_mask & bit_value
        }

        missing_primary_topics = primary_required_topics - selected_topics
        missing_supporting_topics = supporting_required_topics - selected_topics
        missing_required_topics = required_topic_indexes - selected_topics
        missing_required_references = required_references - selected_references

        primary_topic_gap = len(missing_primary_topics)
        supporting_topic_gap = len(missing_supporting_topics)
        required_topic_gap = len(missing_required_topics)
        primary_gap = max(0, minimum_primary - primary)
        supporting_gap = max(0, minimum_supporting - supporting)
        distinct_count = int(ref_mask.bit_count())
        distinct_gap = max(0, minimum_distinct - distinct_count)
        required_reference_gap = len(missing_required_references)
        primary_preference_gap = max(0, preferred_primary_count - primary)
        supporting_marks = max(0, int(marks) - int(primary_marks))
        primary_mark_share = (
            float(primary_marks) / float(marks)
            if int(marks) > 0
            else 0.0
        )
        primary_mark_share_gap = max(
            0.0,
            preferred_primary_question_share - primary_mark_share,
        )
        target_total_marks_gap = (
            abs(int(marks) - int(target_total_marks))
            if target_total_marks is not None
            else 0
        )

        requirements_met = bool(
            required_topic_gap == 0
            and primary_gap == 0
            and supporting_gap == 0
            and distinct_gap == 0
            and required_reference_gap == 0
        )

        options.append(
            {
                "count": count,
                "marks": marks,
                "primary_count": primary,
                "supporting_count": supporting,
                "distinct_reference_count": distinct_count,
                "distinct_topic_count": len(selected_topics),
                "score": score,
                "selected": selected,
                "requirements_met": requirements_met,
                "primary_topic_gap": primary_topic_gap,
                "supporting_topic_gap": supporting_topic_gap,
                "required_topic_gap": required_topic_gap,
                "primary_gap": primary_gap,
                "supporting_gap": supporting_gap,
                "distinct_reference_gap": distinct_gap,
                "required_reference_gap": required_reference_gap,
                "primary_preference_gap": primary_preference_gap,
                "primary_marks": int(primary_marks),
                "supporting_marks": int(supporting_marks),
                "primary_mark_share": float(primary_mark_share),
                "primary_mark_share_gap": float(primary_mark_share_gap),
                "target_total_marks_gap": int(target_total_marks_gap),
                "missing_required_topics": sorted(missing_required_topics),
                "missing_required_references": sorted(missing_required_references),
            }
        )

    options_df = pd.DataFrame(options)
    if options_df.empty:
        raise RuntimeError(
            "No quality-safe question combination could be generated."
        )

    strict_options = options_df[options_df["requirements_met"]].copy()
    strict_coverage_possible = not strict_options.empty

    # Question count is the main completion target.
    exact_count_options = options_df[
        options_df["count"].eq(requested_count)
    ].copy()

    if not exact_count_options.empty:
        selection_pool = exact_count_options
        selection_mode = "exact_question_count_user_controls"
    else:
        largest_count = int(options_df["count"].max())
        selection_pool = options_df[
            options_df["count"].eq(largest_count)
        ].copy()
        selection_mode = "best_quality_safe_partial_question_count"

    # The requested minimum primary/supporting counts are treated as
    # at-least requirements whenever the quality-safe pool can satisfy them.
    role_minimum_options = selection_pool[
        selection_pool["primary_gap"].eq(0)
        & selection_pool["supporting_gap"].eq(0)
    ].copy()

    if not role_minimum_options.empty:
        # Normal path: the exact frontend minima are feasible and therefore
        # remain enforced.
        selection_pool = role_minimum_options
        selection_mode += "_role_minima_strict_met"
    else:
        # Resilient partial path: keep every relevance/quality gate active and
        # choose the combination with the smallest role-count shortfall.
        # Never rescue or add a weak question merely to satisfy a quota.
        role_gap_total = (
            selection_pool["primary_gap"].astype(int)
            + selection_pool["supporting_gap"].astype(int)
        )
        best_role_gap = int(role_gap_total.min())
        selection_pool = selection_pool[
            role_gap_total.eq(best_role_gap)
        ].copy()
        selection_mode += "_best_quality_safe_partial_role_minima"

    # Once question count and role minima are protected, preserve the existing
    # coverage behaviour as strongly as the safe pool allows.
    strict_in_pool = selection_pool[
        selection_pool["requirements_met"]
    ].copy()

    if not strict_in_pool.empty:
        selection_pool = strict_in_pool
        selection_mode += "_full_coverage"

    # After strict role targets/coverage are protected, first move as close as
    # possible to the preferred Primary QUESTION count. This prevents a paper
    # from meeting only the minimum (for example 2 Primary out of 10) when
    # additional quality-safe Primary questions are available.
    if primary_topic_indexes and preferred_primary_count > 0:
        best_primary_count_gap = int(
            selection_pool["primary_preference_gap"].min()
        )
        selection_pool = selection_pool[
            selection_pool["primary_preference_gap"].eq(
                best_primary_count_gap
            )
        ].copy()
        selection_mode += "_preferred_primary_question_share"

    # Then prefer a Primary MARK share consistent with the question-share target.
    # This remains a preference and never weakens any relevance/quality gate.
    if primary_topic_indexes and preferred_primary_count > 0:
        mark_balanced_options = selection_pool[
            selection_pool["primary_mark_share_gap"].le(1e-12)
        ].copy()
        if not mark_balanced_options.empty:
            selection_pool = mark_balanced_options
            selection_mode += "_primary_mark_share_met"
        else:
            best_primary_mark_gap = float(
                selection_pool["primary_mark_share_gap"].min()
            )
            selection_pool = selection_pool[
                selection_pool["primary_mark_share_gap"].eq(
                    best_primary_mark_gap
                )
            ].copy()
            selection_mode += "_best_feasible_primary_mark_share"

    # Optional frontend target marks are optimized only after relevance,
    # hard role minima and coverage. Weak candidates are never rescued to
    # hit the target.
    if target_total_marks is not None and not selection_pool.empty:
        best_target_gap = int(
            selection_pool["target_total_marks_gap"].min()
        )
        selection_pool = selection_pool[
            selection_pool["target_total_marks_gap"].eq(best_target_gap)
        ].copy()
        selection_mode += "_target_total_marks_optimized"

    # Among otherwise valid combinations, prefer better allocation/coverage,
    # then the strongest quality/ranking score.
    best = (
        selection_pool.sort_values(
            [
                "primary_gap",
                "supporting_gap",
                "primary_topic_gap",
                "supporting_topic_gap",
                "required_topic_gap",
                "distinct_reference_gap",
                "required_reference_gap",
                "primary_preference_gap",
                "primary_mark_share_gap",
                "target_total_marks_gap",
                "score",
                "distinct_topic_count",
            ],
            ascending=[
                True, True, True, True, True, True, True, True,
                True, True, False, False
            ],
        )
        .iloc[0]
    )

    selected_df = working.loc[list(best["selected"])].copy()
    selected_df["_role_order"] = selected_df["agent1_role"].map(
        {"primary": 0, "supporting": 1}
    ).fillna(2)
    selected_df = selected_df.sort_values(
        ["_role_order", "agent1_topic_index", "_selection_quality_score"],
        ascending=[True, True, False],
    ).reset_index(drop=True)
    selected_df["selected_rank"] = np.arange(1, len(selected_df) + 1)
    selected_df = selected_df.drop(
        columns=["_role_order", "_selection_quality_score", "_memory_effective_score"], errors="ignore"
    )

    selected_marks = int(selected_df["marks"].sum())
    requested_count_met = bool(len(selected_df) == requested_count)

    selected_primary_marks = int(
        selected_df.loc[
            selected_df["agent1_role"].eq("primary"),
            "marks",
        ].sum()
    )
    selected_supporting_marks = int(
        selected_df.loc[
            selected_df["agent1_role"].eq("supporting"),
            "marks",
        ].sum()
    )
    selected_primary_mark_share = (
        float(selected_primary_marks) / float(selected_marks)
        if selected_marks > 0
        else 0.0
    )

    selected_primary_count = int(selected_df["agent1_role"].eq("primary").sum())
    selected_supporting_count = int(selected_df["agent1_role"].eq("supporting").sum())
    selected_topic_set = set(selected_df["agent1_topic_index"].astype(int).tolist())
    selected_reference_set = set(
        selected_df["agent1_official_reference"].dropna().astype(str).tolist()
    )
    selected_distinct_count = len(selected_reference_set)

    missing_selected_required_topics = required_topic_indexes - selected_topic_set
    missing_selected_required_references = required_references - selected_reference_set

    primary_requirement_met = selected_primary_count >= minimum_primary
    supporting_requirement_met = selected_supporting_count >= minimum_supporting
    distinct_requirement_met = selected_distinct_count >= minimum_distinct
    required_topics_met = not missing_selected_required_topics
    required_references_met = not missing_selected_required_references

    strict_coverage_requirements_met = bool(
        primary_requirement_met
        and supporting_requirement_met
        and distinct_requirement_met
        and required_topics_met
        and required_references_met
    )

    coverage_release_blockers: list[str] = []
    if not required_topics_met:
        coverage_release_blockers.append("approved_agent1_topic_not_covered")
    if not primary_requirement_met:
        coverage_release_blockers.append("minimum_primary_coverage_not_met")
    if not supporting_requirement_met:
        coverage_release_blockers.append("minimum_supporting_coverage_not_met")
    if not distinct_requirement_met:
        coverage_release_blockers.append("minimum_distinct_reference_coverage_not_met")
    if not required_references_met:
        coverage_release_blockers.append("required_official_reference_not_covered")

    semantic_rescue_selected = bool(selected_df["semantic_rescue_used"].any())
    concept_fit_rescue_selected = bool(selected_df["concept_fit_rescue_used"].any())

    provisional_release_ready = bool(
        requested_count_met
        and strict_coverage_requirements_met
        and not semantic_rescue_selected
        and not concept_fit_rescue_selected
    )
    assessment_release_status = (
        "provisional_evaluation_ready"
        if provisional_release_ready
        else "needs_user_decision"
    )

    missing_topic_names = [
        str(
            topic_metadata.loc[
                topic_metadata["agent1_topic_index"].eq(topic_index),
                "detected_topic",
            ].iloc[0]
        )
        for topic_index in sorted(missing_selected_required_topics)
        if topic_metadata["agent1_topic_index"].eq(topic_index).any()
    ]

    if missing_topic_names:
        add_agent2_user_message(
            level="warning",
            code="approved_topics_not_represented_in_final_assessment",
            message=(
                "The assessment was generated without some approved topics "
                "because no quality-safe combination could represent all of them "
                "under the current user filters."
            ),
            details={"missing_topics": missing_topic_names},
        )

    if not primary_requirement_met:
        add_agent2_user_message(
            level="warning",
            code="minimum_primary_questions_not_met",
            message=(
                f"At least {minimum_primary} primary-topic question(s) were requested, "
                f"but only {selected_primary_count} quality-safe primary question(s) "
                "could be selected. The assessment continued with the available questions."
            ),
            requested=minimum_primary,
            available=selected_primary_count,
        )

    if not supporting_requirement_met:
        add_agent2_user_message(
            level="warning",
            code="minimum_supporting_questions_not_met",
            message=(
                f"At least {minimum_supporting} supporting-topic question(s) were requested, "
                f"but only {selected_supporting_count} quality-safe supporting question(s) "
                "could be selected. The assessment continued with the available questions."
            ),
            requested=minimum_supporting,
            available=selected_supporting_count,
        )

    summary = {
        "run_status": AGENT2_RUN_STATUS,
        "user_message_count": int(len(AGENT2_USER_MESSAGES)),
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
        "requested_questions": requested_count,
        "selected_questions": int(len(selected_df)),
        "requested_question_count_met": requested_count_met,
        "candidate_count_after_concept_fit": int(len(candidates)),
        "question_count_shortfall": max(0, requested_count - len(selected_df)),
        "selected_marks": selected_marks,
        "target_total_marks": target_total_marks,
        "target_total_marks_gap": (
            abs(selected_marks - target_total_marks)
            if target_total_marks is not None
            else None
        ),
        "selected_primary_marks": selected_primary_marks,
        "selected_supporting_marks": selected_supporting_marks,
        "selected_primary_mark_share": selected_primary_mark_share,
        "preferred_primary_count": int(preferred_primary_count),
        "preferred_supporting_count": int(preferred_supporting_count),
        "preferred_primary_question_share": preferred_primary_question_share,
        "shared_role_allocation_version": SHARED_ROLE_ALLOCATION_VERSION,
        "primary_question_balance_gap": max(
            0,
            int(preferred_primary_count) - int(selected_primary_count),
        ),
        "final_primary_balance_version": FINAL_PRIMARY_BALANCE_VERSION,
        "final_assessed_task_ownership_version": (
            FINAL_ASSESSED_TASK_OWNERSHIP_VERSION
        ),
        "role_allocation_version": ROLE_ALLOCATION_VERSION,
        "subconcept_diversity_version": SUBCONCEPT_DIVERSITY_VERSION,
        "role_minimum_pool_shortage": role_minimum_pool_shortage,
        "role_minimum_partial_output_used": bool(
            not primary_requirement_met or not supporting_requirement_met
        ),
        "assessment_release_status": assessment_release_status,
        "semantic_rescue_selected": semantic_rescue_selected,
        "concept_fit_rescue_selected": concept_fit_rescue_selected,
        "requires_user_decision": bool(not provisional_release_ready),
        "coverage_selection_mode": selection_mode,
        "strict_coverage_possible": strict_coverage_possible,
        "strict_coverage_option_count": int(len(strict_options)),
        "coverage_constraint_adaptation_applied": bool(
            not strict_coverage_possible
            or not primary_requirement_met
            or not supporting_requirement_met
        ),
        "reference_diversity_adaptation_applied": bool(
            request.get("reference_diversity_adapted", False)
        ),
        "coverage_requirements_met": strict_coverage_requirements_met,
        "coverage_release_blockers": coverage_release_blockers,
        "cover_all_approved_topics": cover_all_topics,
        "requested_agent1_topic_indexes": requested_topic_indexes,
        "effective_required_agent1_topic_indexes": sorted(required_topic_indexes),
        "selected_agent1_topic_indexes": sorted(selected_topic_set),
        "selected_distinct_agent1_topics": len(selected_topic_set),
        "missing_required_agent1_topic_indexes": sorted(
            missing_selected_required_topics
        ),
        "missing_required_agent1_topics": missing_topic_names,
        "preferred_primary_question_count_after_coverage": preferred_primary_count,
        "requested_minimum_primary_questions": minimum_primary,
        "effective_minimum_primary_questions": effective_minimum_primary,
        "selected_primary_questions": selected_primary_count,
        "primary_requirement_met": primary_requirement_met,
        "requested_minimum_supporting_questions": minimum_supporting,
        "effective_minimum_supporting_questions": effective_minimum_supporting,
        "selected_supporting_questions": selected_supporting_count,
        "supporting_requirement_met": supporting_requirement_met,
        "requested_minimum_distinct_official_references": int(
            request.get("requested_minimum_distinct_official_references", minimum_distinct)
        ),
        "configured_effective_minimum_distinct_official_references": minimum_distinct,
        "effective_minimum_distinct_official_references": effective_minimum_distinct,
        "selected_distinct_official_references": selected_distinct_count,
        "distinct_reference_requirement_met": distinct_requirement_met,
        "required_official_references": sorted(required_references),
        "effective_required_official_references": sorted(effective_required_references),
        "unavailable_required_official_references": unavailable_required_references,
        "missing_selected_required_official_references": sorted(
            missing_selected_required_references
        ),
        "all_required_references_covered": required_references_met,
        "available_primary_candidates": available_primary_candidates,
        "available_supporting_candidates": available_supporting_candidates,
        "available_official_references": available_references,
        "selected_official_references": sorted(selected_reference_set),
        "selected_canonical_agent2_references": sorted(
            selected_df["official_reference"].dropna().astype(str).unique().tolist()
        ),
        "fallback_selected": int(
            selected_df["retrieval_stage"].eq("same_section_fallback").sum()
        ),
        "phase2_threshold_strategy": (
            "hierarchical_hybrid_plus_post_relevance_topic_ownership"
        ),
        "adaptive_threshold_min": adaptive_threshold_min,
        "adaptive_threshold_max": adaptive_threshold_max,
        "adaptive_threshold_mean": adaptive_threshold_mean,
        "adaptive_pool_sufficient_without_rescue": adaptive_pool_sufficient,
        "phase2_rescue_used": phase2_rescue_used,
        "weak_candidate_replacement_strategy": (
            "per_topic_rank_then_cross_topic_ownership_then_coverage_selection"
        ),
        "question_ownership_version": QUESTION_OWNERSHIP_VERSION,
        "final_relevance_verifier_version": FINAL_RELEVANCE_VERIFIER_VERSION,
        "general_quality_gate_version": GENERAL_QUALITY_GATE_VERSION,
        "context_gap_version": CONTEXT_GAP_VERSION,
    }

    summary["minimum_primary_questions"] = minimum_primary
    summary["minimum_supporting_questions"] = minimum_supporting
    summary["minimum_distinct_official_references"] = minimum_distinct

    return selected_df, summary


# -------------------------------------------------------------------------
# Phase 4 exact-context negative-memory suppression
# -------------------------------------------------------------------------
# Keep suppressed candidates in phase2_candidates_df for audit/diagnostics,
# but do not allow them into final assessment selection for the exact same
# question + lesson-evidence context. Different lessons remain unaffected
# because Phase 3 must first establish exact contextual compatibility.
_phase4_hard_suppression_mask = phase2_candidates_df.get(
    "memory_hard_suppressed",
    pd.Series(False, index=phase2_candidates_df.index),
).fillna(False).astype(bool)

phase4_suppressed_candidates_df = phase2_candidates_df[
    _phase4_hard_suppression_mask
].copy()

phase4_selection_candidates_df = phase2_candidates_df[
    ~_phase4_hard_suppression_mask
].copy()

phase4_exact_context_suppressed_count = int(
    len(phase4_suppressed_candidates_df)
)

if phase4_exact_context_suppressed_count:
    add_agent2_user_message(
        level="warning",
        code="exact_context_not_relevant_questions_suppressed",
        message=(
            f"{phase4_exact_context_suppressed_count} question(s) previously marked "
            "Not Relevant for the exact same lesson context were excluded from final "
            "selection. They were not reintroduced merely to fill the requested "
            "question count. The questions remain available for different lesson contexts."
        ),
        requested=int(request["number_of_questions"]),
        available=int(len(phase4_selection_candidates_df)),
        details={
            "suppressed_question_ids": (
                phase4_suppressed_candidates_df["question_id"]
                .astype(str)
                .tolist()
                if "question_id" in phase4_suppressed_candidates_df.columns
                else []
            ),
            "suppression_policy": (
                "exact_question_plus_exact_lesson_evidence_not_relevant"
            ),
        },
    )

(
    selected_candidates_df,
    selection_summary,
) = select_questions(
    phase4_selection_candidates_df
)

selection_summary["phase4_exact_context_suppressed_count"] = (
    phase4_exact_context_suppressed_count
)
selection_summary["phase4_candidates_before_suppression"] = int(
    len(phase2_candidates_df)
)
selection_summary["phase4_candidates_after_suppression"] = int(
    len(phase4_selection_candidates_df)
)
selection_summary["phase4_suppressed_question_ids"] = (
    phase4_suppressed_candidates_df["question_id"].astype(str).tolist()
    if "question_id" in phase4_suppressed_candidates_df.columns
    else []
)
selection_summary["phase4_suppression_policy"] = (
    "exact_context_not_relevant_excluded_from_final_selection"
)
selection_summary["phase4_rejected_questions_reintroduced"] = False
selection_summary["user_message_count"] = int(len(AGENT2_USER_MESSAGES))
selection_summary["user_messages"] = json_safe(AGENT2_USER_MESSAGES)

# Absolute final paper-family guard before any downstream PostgreSQL bundle,
# CSV, JSON or PDF can be produced.
_requested_paper_for_final_guard = request.get("paper_code")
if (
    _requested_paper_for_final_guard
    and not selected_candidates_df.empty
    and "paper_code" in selected_candidates_df.columns
):
    _selected_paper_mask = selected_candidates_df["paper_code"].map(
        lambda value: paper_code_matches(
            value, _requested_paper_for_final_guard
        )
    )
    _wrong_paper_selected_count = int((~_selected_paper_mask).sum())
    if _wrong_paper_selected_count:
        selected_candidates_df = selected_candidates_df[_selected_paper_mask].copy()
        selected_candidates_df = selected_candidates_df.reset_index(drop=True)
        selected_candidates_df["selected_rank"] = np.arange(
            1, len(selected_candidates_df) + 1
        )
        AGENT2_RUN_STATUS = "partial_success"
        selection_summary["run_status"] = AGENT2_RUN_STATUS
        selection_summary["selected_questions"] = int(len(selected_candidates_df))
        selection_summary["selected_marks"] = int(
            selected_candidates_df["marks"].sum()
        ) if not selected_candidates_df.empty else 0
        selection_summary["requested_question_count_met"] = bool(
            len(selected_candidates_df) == int(request["number_of_questions"])
        )
        add_agent2_user_message(
            level="warning",
            code="final_wrong_paper_guard_triggered",
            message=(
                f"Removed {_wrong_paper_selected_count} selected question(s) "
                f"that did not belong to {paper_label(_requested_paper_for_final_guard)}. "
                "No question from another paper was substituted."
            ),
            details={
                "requested_paper_code": _requested_paper_for_final_guard,
                "resolved_kb_paper_code": knowledge_base_paper_code(),
            },
        )
        if selected_candidates_df.empty:
            AGENT2_NO_SAFE_CANDIDATES = True
            selection_summary["assessment_generated"] = False
            selection_summary["assessment_release_status"] = "no_safe_assessment"


display(
    pd.DataFrame(
        [selection_summary]
    )
)


_selection_audit_columns = [
    "selected_rank",
    "phase2_rank",
    "detected_topic",
    "agent1_role",
    "retrieval_stage",
    "official_reference",
    "question_number",
    "marks",
    "semantic_score",
    "direct_topic_semantic_score",
    "question_focus_semantic_score",
    "context_enriched_direct_topic_score",
    "context_dependency_gap",
    "context_dependency_flag",
    "assessed_skill_semantic_score",
    "assessed_skill_official_concept_score",
    "best_competing_assessed_skill_score",
    "best_competing_official_reference",
    "best_competing_official_concept",
    "assessed_skill_official_owner_is_requested",
    "assessed_skill_lexical_overlap",
    "child_direct_evidence_passed",
    "child_direct_evidence_reason",
    "ownership_core_fit_score",
    "ownership_credible_match",
    "final_quality_gate_passed",
    "assessed_skill_supported",
    "context_consistency_passed",
    "final_relevance_verifier_passed",
    "final_relevance_verifier_reason",
    "bm25_lexical_normalized",
    "hybrid_relevance_score",
    "topic_ownership_score",
    "owner_detected_topic",
    "semantic_gate_score",
    "direct_concept_score",
    "combined_concept_score",
    "direct_concept_threshold",
    "concept_fit_mark_margin",
    "concept_fit_rescue_used",
    "question_quality_issues",
    "final_score",
    "question_text",
]

# Safe for zero-candidate runs produced by paper/topic routing.
display(
    selected_candidates_df.reindex(
        columns=_selection_audit_columns
    )
)


if selection_summary[
    "coverage_constraint_adaptation_applied"
]:
    print(
        "\nCOVERAGE FEASIBILITY WARNING"
    )

    print(
        "The original coverage requirements cannot be met "
        "by the current quality-safe candidate pool."
    )

    print(
        "No rejected or weak candidate was reintroduced."
    )

    print(
        "Selection mode: "
        f"{selection_summary['coverage_selection_mode']}"
    )

    print(
        "Coverage blockers: "
        f"{selection_summary['coverage_release_blockers']}"
    )

    print(
        "Requested primary/supporting/distinct minima: "
        f"{selection_summary['requested_minimum_primary_questions']}/"
        f"{selection_summary['requested_minimum_supporting_questions']}/"
        f"{selection_summary['requested_minimum_distinct_official_references']}"
    )

    print(
        "Selected primary/supporting/distinct counts: "
        f"{selection_summary['selected_primary_questions']}/"
        f"{selection_summary['selected_supporting_questions']}/"
        f"{selection_summary['selected_distinct_official_references']}"
    )


if not selection_summary[
    "requested_question_count_met"
]:
    print(
        "\nQUESTION COUNT FEASIBILITY WARNING"
    )

    print(
        "The strict quality-safe pool cannot supply the full "
        "requested count. Weak candidates were not reintroduced."
    )

    print(
        f"Requested: "
        f"{selection_summary['requested_questions']}"
    )

    print(
        f"Selected: "
        f"{selection_summary['selected_questions']}"
    )


if AGENT2_USER_MESSAGES:
    print_agent2_user_messages(
        "AGENT 2 USER-FRIENDLY RUN SUMMARY"
    )


if AGENT2_NO_SAFE_CANDIDATES:
    no_assessment_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    no_assessment_summary_path = OUTPUT_DIR / f"agent2_no_assessment_summary_{no_assessment_timestamp}.json"
    no_assessment_payload = {
        "run_status": AGENT2_RUN_STATUS,
        "assessment_generated": False,
        "reason": selection_summary.get("assessment_release_status", "no_safe_assessment"),
        "request": json_safe(request),
        "syllabus_paper_check": json_safe(
            AGENT2_SYLLABUS_PAPER_CHECK
        ),
        "selection_summary": json_safe(selection_summary),
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
    }
    no_assessment_summary_path.write_text(
        json.dumps(no_assessment_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    # IMPORTANT FRONTEND CONTRACT:
    # Always create current-run artifacts even when no assessment can be
    # generated. Otherwise a frontend that discovers the latest successful
    # agent2_selected_questions_*.csv can accidentally display a stale Paper 2
    # assessment after a Paper 1 no-result run.
    no_assessment_selected_csv_path = (
        OUTPUT_DIR / f"agent2_selected_questions_{no_assessment_timestamp}.csv"
    )
    selected_candidates_df.head(0).to_csv(
        no_assessment_selected_csv_path, index=False
    )

    no_assessment_release_path = (
        OUTPUT_DIR / f"agent2_assessment_release_readiness_{no_assessment_timestamp}.json"
    )
    no_assessment_release_payload = {
        "run_status": AGENT2_RUN_STATUS,
        "assessment_generated": False,
        "final_release_status": "no_safe_assessment",
        "assessment_release_status": "no_safe_assessment",
        "requires_user_decision": True,
        "requested_paper_code": request.get("paper_code"),
        "resolved_kb_paper_code": knowledge_base_paper_code(),
        "requested_paper_label": paper_label(request.get("paper_code")),
        "selected_questions": 0,
        "selected_marks": 0,
        "release_blockers": (
            ["syllabus_paper_mismatch"]
            if globals().get(
                "AGENT2_SYLLABUS_PAPER_BLOCKED",
                False,
            )
            else [
                "no_quality_safe_questions_for_current_request"
            ]
        ),
        "syllabus_paper_check": json_safe(
            AGENT2_SYLLABUS_PAPER_CHECK
        ),
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
    }
    no_assessment_release_path.write_text(
        json.dumps(no_assessment_release_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    no_assessment_package_path = (
        OUTPUT_DIR / f"agent2_assessment_package_{no_assessment_timestamp}.json"
    )
    no_assessment_package_payload = {
        "run_status": AGENT2_RUN_STATUS,
        "assessment_generated": False,
        "request": json_safe(request),
        "syllabus_paper_check": json_safe(
            AGENT2_SYLLABUS_PAPER_CHECK
        ),
        "questions": [],
        "selection_summary": json_safe(selection_summary),
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
        "output_files": {
            "selected_questions_csv": str(no_assessment_selected_csv_path),
            "release_readiness_json": str(no_assessment_release_path),
            "combined_questions_and_answers_pdf": None,
        },
    }
    no_assessment_package_path.write_text(
        json.dumps(no_assessment_package_payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    current_run_status_path = OUTPUT_DIR / "agent2_current_run.json"
    current_run_status_path.write_text(
        json.dumps(
            {
                "timestamp": no_assessment_timestamp,
                "assessment_generated": False,
                "run_status": AGENT2_RUN_STATUS,
                "requested_paper_code": request.get("paper_code"),
                "resolved_kb_paper_code": knowledge_base_paper_code(),
                "selected_questions_csv": str(no_assessment_selected_csv_path),
                "release_readiness_json": str(no_assessment_release_path),
                "assessment_package_json": str(no_assessment_package_path),
                "combined_questions_and_answers_pdf": None,
            },
            indent=2, ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("\nASSESSMENT NOT GENERATED — USER ACTION NEEDED")
    print("-" * 46)
    print("Notebook 05 completed safely. No incompatible or weak question was inserted into the assessment.")
    print_agent2_user_messages("AGENT 2 USER-FRIENDLY RUN SUMMARY")
    print(f"Run summary written to: {no_assessment_summary_path}")


## 10E. Retrieval HITL Phase 5 — end-to-end evaluation + policy lock

Phase 5 does **not** add another ranking heuristic. It validates and freezes the
Phase 3/4 self-improving retrieval policy before the pipeline is considered
locked.

Automated checks cover:

- Phase 3 compatibility must gate every Phase 4 effect;
- baseline `final_score` / `phase2_rank` remain auditable;
- boosts/penalties stay inside the frozen ±0.04 bound;
- incompatible/no-memory candidates remain unchanged;
- exact-context `Not Relevant` candidates are selection-ineligible and never
  reintroduced to fill quota;
- conflicting compatible memories fail closed;
- selected questions are all memory-selection eligible;
- deterministic synthetic regression cases exercise positive boost, contextual
  hard rejection, broader negative penalty, conflict no-op, and no-memory no-op;
- validated Phase 3 thresholds and Phase 4 bounds are snapshotted as a frozen
  policy configuration with automatic threshold tuning disabled.

The evaluation report is written to JSON + CSV and copied into the assessment
`selection_summary`, so Streamlit can display the lock status for every run.


In [ ]:
# ============================================================================
# RETRIEVAL HITL PHASE 5 — END-TO-END EVALUATION + POLICY LOCK
# ============================================================================
# This phase does not alter retrieval or ranking. It validates the Phase 3/4
# policy and freezes the currently tested thresholds/bounds for audit.
# ============================================================================

RETRIEVAL_MEMORY_PHASE5_VERSION = (
    "agent2-retrieval-hitl-phase5-evaluation-lock-v1.0.0"
)

PHASE5_FROZEN_POLICY = {
    "phase3_version": RETRIEVAL_MEMORY_PHASE3_VERSION,
    "phase3_overall_similarity_min": RETRIEVAL_MEMORY_OVERALL_SIMILARITY_MIN,
    "phase3_topic_similarity_min": RETRIEVAL_MEMORY_TOPIC_SIMILARITY_MIN,
    "phase3_evidence_similarity_min": RETRIEVAL_MEMORY_EVIDENCE_SIMILARITY_MIN,
    "phase3_question_similarity_min": RETRIEVAL_MEMORY_QUESTION_SIMILARITY_MIN,
    "phase4_version": RETRIEVAL_MEMORY_PHASE4_VERSION,
    "phase4_max_positive_boost": RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST,
    "phase4_max_negative_penalty": RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY,
    "exact_context_not_relevant_policy": "hard_suppress_for_same_lesson_only",
    "broader_not_relevant_policy": "bounded_penalty_only",
    "relevant_policy": "bounded_confidence_scaled_boost",
    "conflict_policy": "fail_closed_zero_adjustment",
    "incompatible_memory_policy": "zero_adjustment",
    "automatic_threshold_tuning": False,
}

# Freeze the tested values explicitly so accidental later drift is visible.
PHASE5_EXPECTED_FROZEN_VALUES = {
    "phase3_overall_similarity_min": 0.50,
    "phase3_topic_similarity_min": 0.90,
    "phase3_evidence_similarity_min": 0.75,
    "phase3_question_similarity_min": 0.78,
    "phase4_max_positive_boost": 0.04,
    "phase4_max_negative_penalty": 0.04,
}


def _phase5_bool_series(
    frame: pd.DataFrame,
    column: str,
    default: bool = False,
) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=bool)
    return frame[column].fillna(default).astype(bool)


def _phase5_float_series(
    frame: pd.DataFrame,
    column: str,
    default: float = 0.0,
) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce").fillna(default).astype(float)


def _phase5_diag_status(row: pd.Series) -> str:
    diag = row.get("memory_ranking_phase4")
    if not isinstance(diag, dict):
        return ""
    return str(diag.get("status") or "").strip()


def _phase5_policy_case(
    *,
    decision: str | None,
    matched: bool = True,
    exact_question: bool = False,
    exact_lesson: bool = False,
    second_decision: str | None = None,
) -> dict[str, Any]:
    """Create a deterministic synthetic Phase 3 diagnostic for regression tests."""
    candidates: list[dict[str, Any]] = []
    if decision is not None:
        candidates.append(
            {
                "compatibility_status": "MATCHED" if matched else "REJECTED",
                "decision": decision,
                "overall_similarity": 1.0,
                "topic_similarity": 1.0,
                "evidence_similarity": 1.0,
                "question_similarity": 1.0,
                "exact_question_id": bool(exact_question),
                "exact_lesson_evidence_hash": bool(exact_lesson),
                "reason": "phase5 synthetic regression memory",
                "memory_point_id": f"phase5-{decision}-1",
                "feedback_id": 1,
            }
        )
    if second_decision is not None:
        candidates.append(
            {
                "compatibility_status": "MATCHED",
                "decision": second_decision,
                "overall_similarity": 1.0,
                "topic_similarity": 1.0,
                "evidence_similarity": 1.0,
                "question_similarity": 1.0,
                "exact_question_id": bool(exact_question),
                "exact_lesson_evidence_hash": bool(exact_lesson),
                "reason": "phase5 synthetic conflicting memory",
                "memory_point_id": f"phase5-{second_decision}-2",
                "feedback_id": 2,
            }
        )

    return {
        "matched": bool(matched and candidates),
        "recall_status": (
            "matched_compatible_memory"
            if matched and candidates
            else "no_compatible_memory"
        ),
        "candidate_memories": candidates,
    }


def _phase5_run_synthetic_policy_tests() -> list[dict[str, Any]]:
    base = {
        "final_score": 0.50,
        "phase2_rank": 1,
    }

    cases: list[tuple[str, dict[str, Any], Any]] = []

    relevant = pd.Series(
        {
            **base,
            "memory_recall_phase3": _phase5_policy_case(
                decision="relevant",
                matched=True,
                exact_question=True,
                exact_lesson=True,
            ),
        }
    )
    relevant_result = _phase4_adjustment_for_candidate(relevant)
    cases.append(
        (
            "exact_context_relevant_gets_positive_bounded_boost",
            relevant_result,
            lambda value: (
                value.get("status") == "relevant_memory_boost_applied"
                and 0.0 < float(value.get("ranking_adjustment") or 0.0)
                <= RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST + 1e-12
                and not bool(value.get("hard_suppressed"))
            ),
        )
    )

    exact_negative = pd.Series(
        {
            **base,
            "memory_recall_phase3": _phase5_policy_case(
                decision="not_relevant",
                matched=True,
                exact_question=True,
                exact_lesson=True,
            ),
        }
    )
    exact_negative_result = _phase4_adjustment_for_candidate(exact_negative)
    cases.append(
        (
            "exact_context_not_relevant_is_hard_suppressed",
            exact_negative_result,
            lambda value: (
                value.get("status") == "exact_context_not_relevant_suppressed"
                and bool(value.get("hard_suppressed"))
                and not bool(value.get("selection_eligible_after_memory", True))
                and abs(float(value.get("ranking_adjustment") or 0.0)) <= 1e-12
            ),
        )
    )

    broad_negative = pd.Series(
        {
            **base,
            "memory_recall_phase3": _phase5_policy_case(
                decision="not_relevant",
                matched=True,
                exact_question=False,
                exact_lesson=False,
            ),
        }
    )
    broad_negative_result = _phase4_adjustment_for_candidate(broad_negative)
    cases.append(
        (
            "broader_compatible_not_relevant_gets_bounded_penalty",
            broad_negative_result,
            lambda value: (
                value.get("status") == "not_relevant_memory_penalty_applied"
                and -RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY - 1e-12
                <= float(value.get("ranking_adjustment") or 0.0)
                < 0.0
                and not bool(value.get("hard_suppressed"))
            ),
        )
    )

    conflict = pd.Series(
        {
            **base,
            "memory_recall_phase3": _phase5_policy_case(
                decision="relevant",
                second_decision="not_relevant",
                matched=True,
                exact_question=False,
                exact_lesson=False,
            ),
        }
    )
    conflict_result = _phase4_adjustment_for_candidate(conflict)
    cases.append(
        (
            "conflicting_compatible_memories_fail_closed",
            conflict_result,
            lambda value: (
                value.get("status") == "conflicting_compatible_memories_no_adjustment"
                and bool(value.get("conflict_detected"))
                and abs(float(value.get("ranking_adjustment") or 0.0)) <= 1e-12
            ),
        )
    )

    no_memory = pd.Series(
        {
            **base,
            "memory_recall_phase3": _phase5_policy_case(
                decision=None,
                matched=False,
            ),
        }
    )
    no_memory_result = _phase4_adjustment_for_candidate(no_memory)
    cases.append(
        (
            "no_compatible_memory_is_no_op",
            no_memory_result,
            lambda value: (
                abs(float(value.get("ranking_adjustment") or 0.0)) <= 1e-12
                and abs(float(value.get("adjusted_final_score") or 0.0) - 0.50)
                <= 1e-12
                and not bool(value.get("hard_suppressed"))
            ),
        )
    )

    results: list[dict[str, Any]] = []
    for name, value, predicate in cases:
        passed = bool(predicate(value))
        results.append(
            {
                "check": name,
                "passed": passed,
                "status": value.get("status"),
                "decision": value.get("decision"),
                "ranking_adjustment": value.get("ranking_adjustment"),
                "hard_suppressed": value.get("hard_suppressed"),
                "selection_eligible_after_memory": value.get(
                    "selection_eligible_after_memory"
                ),
            }
        )
    return results


phase5_candidate_df = (
    phase2_candidates_df.copy()
    if isinstance(globals().get("phase2_candidates_df"), pd.DataFrame)
    else pd.DataFrame()
)
phase5_selected_df = (
    selected_candidates_df.copy()
    if isinstance(globals().get("selected_candidates_df"), pd.DataFrame)
    else pd.DataFrame()
)

phase5_adjustments = _phase5_float_series(
    phase5_candidate_df,
    "memory_rank_adjustment",
)
phase5_base_scores = _phase5_float_series(
    phase5_candidate_df,
    "final_score",
)
phase5_adjusted_scores = _phase5_float_series(
    phase5_candidate_df,
    "memory_adjusted_final_score",
)
phase5_matches = _phase5_bool_series(
    phase5_candidate_df,
    "memory_match_found",
)
phase5_suppressed = _phase5_bool_series(
    phase5_candidate_df,
    "memory_hard_suppressed",
)
phase5_selection_eligible = _phase5_bool_series(
    phase5_candidate_df,
    "memory_selection_eligible",
    default=True,
)

phase5_status_series = pd.Series(
    [
        _phase5_diag_status(row)
        for _, row in phase5_candidate_df.iterrows()
    ],
    index=phase5_candidate_df.index,
    dtype="object",
)

if phase5_candidate_df.empty:
    phase5_adjustment_bounds_ok = True
    phase5_adjusted_score_math_ok = True
    phase5_no_match_zero_ok = True
    phase5_suppression_state_ok = True
    phase5_status_signs_ok = True
else:
    phase5_adjustment_bounds_ok = bool(
        phase5_adjustments.between(
            -RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY - 1e-12,
            RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST + 1e-12,
        ).all()
    )
    expected_adjusted = np.clip(
        phase5_base_scores.to_numpy(dtype=float)
        + phase5_adjustments.to_numpy(dtype=float),
        0.0,
        1.0,
    )
    phase5_adjusted_score_math_ok = bool(
        np.allclose(
            expected_adjusted,
            phase5_adjusted_scores.to_numpy(dtype=float),
            atol=1e-6,
            rtol=0.0,
        )
    )
    phase5_no_match_zero_ok = bool(
        (phase5_adjustments.loc[~phase5_matches].abs() <= 1e-12).all()
    )
    phase5_suppression_state_ok = bool(
        (
            (~phase5_selection_eligible.loc[phase5_suppressed])
            & (phase5_adjustments.loc[phase5_suppressed].abs() <= 1e-12)
        ).all()
    )

    sign_checks: list[bool] = []
    for index, status in phase5_status_series.items():
        adjustment = float(phase5_adjustments.loc[index])
        if status == "relevant_memory_boost_applied":
            sign_checks.append(adjustment > 0.0)
        elif status == "not_relevant_memory_penalty_applied":
            sign_checks.append(adjustment < 0.0)
        elif status in {
            "exact_context_not_relevant_suppressed",
            "conflicting_compatible_memories_no_adjustment",
            "no_compatible_memory",
            "memory_collection_missing",
            "memory_collection_empty",
            "missing_current_context",
        }:
            sign_checks.append(abs(adjustment) <= 1e-12)
    phase5_status_signs_ok = bool(all(sign_checks)) if sign_checks else True

phase5_selected_ids = set(
    phase5_selected_df.get(
        "question_id",
        pd.Series(dtype="object"),
    ).astype(str).tolist()
)
phase5_suppressed_ids = set(
    phase5_candidate_df.loc[
        phase5_suppressed,
        "question_id",
    ].astype(str).tolist()
    if "question_id" in phase5_candidate_df.columns
    else []
)
phase5_no_suppressed_selected = bool(
    phase5_selected_ids.isdisjoint(phase5_suppressed_ids)
)

if phase5_selected_df.empty:
    phase5_selected_memory_eligible = True
else:
    phase5_selected_memory_eligible = bool(
        _phase5_bool_series(
            phase5_selected_df,
            "memory_selection_eligible",
            default=True,
        ).all()
    )

phase5_baseline_audit_fields_present = bool(
    phase5_candidate_df.empty
    or {
        "final_score",
        "phase2_rank",
        "memory_adjusted_final_score",
        "memory_rank_adjustment",
        "phase4_rank",
    }.issubset(set(phase5_candidate_df.columns))
)

phase5_frozen_values_unchanged = bool(
    all(
        abs(float(PHASE5_FROZEN_POLICY[key]) - float(expected)) <= 1e-12
        for key, expected in PHASE5_EXPECTED_FROZEN_VALUES.items()
    )
    and PHASE5_FROZEN_POLICY["automatic_threshold_tuning"] is False
)

phase5_synthetic_tests = _phase5_run_synthetic_policy_tests()
phase5_synthetic_all_passed = bool(
    all(bool(row.get("passed")) for row in phase5_synthetic_tests)
)

phase5_checks = [
    {
        "check": "phase3_and_phase4_diagnostics_available_for_candidate_layer",
        "passed": bool(
            phase5_candidate_df.empty
            or {
                "memory_recall_phase3",
                "memory_ranking_phase4",
            }.issubset(set(phase5_candidate_df.columns))
        ),
        "critical": True,
    },
    {
        "check": "baseline_audit_fields_preserved",
        "passed": phase5_baseline_audit_fields_present,
        "critical": True,
    },
    {
        "check": "memory_adjustment_bound_respected",
        "passed": phase5_adjustment_bounds_ok,
        "critical": True,
    },
    {
        "check": "adjusted_score_equals_clipped_baseline_plus_adjustment",
        "passed": phase5_adjusted_score_math_ok,
        "critical": True,
    },
    {
        "check": "incompatible_or_no_memory_candidates_receive_zero_adjustment",
        "passed": phase5_no_match_zero_ok,
        "critical": True,
    },
    {
        "check": "phase4_status_and_adjustment_signs_are_consistent",
        "passed": phase5_status_signs_ok,
        "critical": True,
    },
    {
        "check": "hard_suppressed_candidates_are_selection_ineligible_and_unscored",
        "passed": phase5_suppression_state_ok,
        "critical": True,
    },
    {
        "check": "no_exact_context_rejected_question_is_selected",
        "passed": phase5_no_suppressed_selected,
        "critical": True,
    },
    {
        "check": "all_selected_questions_are_memory_selection_eligible",
        "passed": phase5_selected_memory_eligible,
        "critical": True,
    },
    {
        "check": "frozen_thresholds_and_bounds_have_not_drifted",
        "passed": phase5_frozen_values_unchanged,
        "critical": True,
    },
    {
        "check": "synthetic_policy_regression_suite_passed",
        "passed": phase5_synthetic_all_passed,
        "critical": True,
    },
]

phase5_critical_passed = int(
    sum(
        1
        for row in phase5_checks
        if bool(row.get("critical")) and bool(row.get("passed"))
    )
)
phase5_critical_total = int(
    sum(1 for row in phase5_checks if bool(row.get("critical")))
)
phase5_lock_passed = bool(
    phase5_critical_passed == phase5_critical_total
)

phase5_run_metrics = {
    "candidate_count": int(len(phase5_candidate_df)),
    "selected_count": int(len(phase5_selected_df)),
    "compatible_memory_count": int(phase5_matches.sum()) if len(phase5_matches) else 0,
    "adjusted_candidate_count": int((phase5_adjustments.abs() > 1e-12).sum())
    if len(phase5_adjustments)
    else 0,
    "positive_boost_count": int((phase5_adjustments > 1e-12).sum())
    if len(phase5_adjustments)
    else 0,
    "negative_penalty_count": int((phase5_adjustments < -1e-12).sum())
    if len(phase5_adjustments)
    else 0,
    "exact_context_suppressed_count": int(phase5_suppressed.sum())
    if len(phase5_suppressed)
    else 0,
    "suppressed_question_ids": sorted(phase5_suppressed_ids),
    "selected_suppressed_overlap": sorted(
        phase5_selected_ids.intersection(phase5_suppressed_ids)
    ),
}

phase5_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
phase5_report_path = (
    OUTPUT_DIR
    / f"agent2_retrieval_phase5_evaluation_{phase5_timestamp}.json"
)
phase5_checks_path = (
    OUTPUT_DIR
    / f"agent2_retrieval_phase5_checks_{phase5_timestamp}.csv"
)

phase5_evaluation = {
    "version": RETRIEVAL_MEMORY_PHASE5_VERSION,
    "status": "locked" if phase5_lock_passed else "not_locked",
    "lock_passed": phase5_lock_passed,
    "critical_checks_passed": phase5_critical_passed,
    "critical_checks_total": phase5_critical_total,
    "policy_frozen": phase5_lock_passed,
    "automatic_threshold_tuning": False,
    "frozen_policy": PHASE5_FROZEN_POLICY,
    "run_metrics": phase5_run_metrics,
    "checks": phase5_checks,
    "synthetic_regression_tests": phase5_synthetic_tests,
    "manual_validation_protocol": [
        "same lesson/context positive-control memory should match and affect selection",
        "different lesson/context negative-control memory should fail Phase 3 and have zero Phase 4 effect",
    ],
    "report_path": str(phase5_report_path),
    "checks_csv_path": str(phase5_checks_path),
}

phase5_report_path.write_text(
    json.dumps(
        phase5_evaluation,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)
pd.DataFrame(phase5_checks).to_csv(
    phase5_checks_path,
    index=False,
)

selection_summary["phase5_evaluation"] = phase5_evaluation
selection_summary["phase5_lock_passed"] = phase5_lock_passed
selection_summary["phase5_policy_frozen"] = phase5_lock_passed
selection_summary["phase5_evaluation_report_path"] = str(phase5_report_path)
selection_summary["phase5_checks_csv_path"] = str(phase5_checks_path)

if not phase5_lock_passed:
    AGENT2_RUN_STATUS = "partial_success"
    selection_summary["run_status"] = AGENT2_RUN_STATUS
    add_agent2_user_message(
        level="warning",
        code="retrieval_phase5_lock_not_passed",
        message=(
            "Retrieval Phase 5 safety evaluation did not pass every critical check. "
            "The current assessment remains auditable, but the self-improving retrieval "
            "policy is NOT considered locked for this run."
        ),
        details={
            "failed_checks": [
                row["check"]
                for row in phase5_checks
                if bool(row.get("critical")) and not bool(row.get("passed"))
            ]
        },
    )

print(
    "Retrieval HITL Phase 5 evaluation:",
    {
        "version": RETRIEVAL_MEMORY_PHASE5_VERSION,
        "lock_status": phase5_evaluation["status"],
        "critical_checks": f"{phase5_critical_passed}/{phase5_critical_total}",
        "synthetic_tests_passed": phase5_synthetic_all_passed,
        "policy_frozen": phase5_evaluation["policy_frozen"],
        "report_path": str(phase5_report_path),
    },
)
display(pd.DataFrame(phase5_checks))
display(pd.DataFrame(phase5_synthetic_tests))


## 11. Fetch full questions and mark schemes from PostgreSQL

Qdrant provides IDs and ranking metadata. PostgreSQL provides the complete
source-of-truth assessment records.


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    selected_ids = (
        selected_candidates_df[
            "question_id"
        ]
        .astype(str)
        .tolist()
    )

    selected_uuids = [
        uuid.UUID(value)
        for value in selected_ids
    ]


    bundle_query = (
        select(
            questions.c.id.label(
                "question_id"
            ),
            questions.c.question_uid,
            questions.c.question_number,
            questions.c.question_text,
            questions.c.context_text,
            questions.c.marks,
            questions.c.page_start,
            questions.c.page_end,
            questions.c.has_code,
            questions.c.has_visual,
            questions.c.visual_page_numbers,
            questions.c.review_status,

            QUESTION_DOCUMENT_FK_COLUMN.label(
                "question_document_id"
            ),
            DOCUMENT_PATH_COLUMN.label(
                "source_pdf_path"
            ),
            (
                DOCUMENT_FILE_NAME_COLUMN
                if DOCUMENT_FILE_NAME_COLUMN is not None
                else DOCUMENT_PATH_COLUMN
            ).label(
                "source_file_name"
            ),
            (
                DOCUMENT_SOURCE_URL_COLUMN
                if DOCUMENT_SOURCE_URL_COLUMN is not None
                else DOCUMENT_PATH_COLUMN
            ).label(
                "source_document_url"
            ),

            questions.c.official_reference,
            questions.c.official_concept_name,
            questions.c.official_section_reference,
            questions.c.official_section_name,

            topics.c.pmt_subtopic_code,
            topics.c.pmt_subtopic_name,
            topics.c.paper_code,
            topics.c.programming_language,

            question_ms_links.c.match_method,
            question_ms_links.c.match_confidence,

            mark_schemes.c.id.label(
                "mark_scheme_id"
            ),
            mark_schemes.c.mark_scheme_uid,
            mark_schemes.c.maximum_marks,
            mark_schemes.c.marking_guidance,
            mark_schemes.c.marking_points,
            mark_schemes.c.acceptable_answers,
            mark_schemes.c.rejected_answers,
            mark_schemes.c.additional_guidance,
            mark_schemes.c.assessment_objectives,
        )
        .select_from(
            questions
            .join(
                topics,
                questions.c.topic_id
                == topics.c.id,
            )
            .join(
                documents,
                documents.c.id
                == QUESTION_DOCUMENT_FK_COLUMN,
            )
            .join(
                question_ms_links,
                question_ms_links.c.question_id
                == questions.c.id,
            )
            .join(
                mark_schemes,
                mark_schemes.c.id
                == question_ms_links
                .c.mark_scheme_entry_id,
            )
        )
        .where(
            questions.c.id.in_(
                selected_uuids
            )
        )
    )

    with engine.connect() as connection:
        bundles_df = pd.read_sql(
            bundle_query,
            connection,
        )

    bundles_df["question_id"] = (
        bundles_df[
            "question_id"
        ].astype(str)
    )

    final_df = (
        selected_candidates_df.merge(
            bundles_df,
            on="question_id",
            how="left",
            suffixes=(
                "_retrieval",
                "_postgres",
            ),
            validate="one_to_one",
        )
        .sort_values(
            "selected_rank"
        )
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # CANONICAL MARK-SCHEME FIELDS AFTER MERGE
    # ---------------------------------------------------------
    # Some MS fields (notably marking_guidance / marking_points) are already
    # attached upstream for MCQ quality checks. When the complete PostgreSQL
    # QP/MS bundle is merged here, pandas therefore suffixes the duplicate
    # names to *_retrieval and *_postgres.
    #
    # Downstream cells intentionally use the unsuffixed canonical field names.
    # Re-create those names here, preferring the complete PostgreSQL bundle as
    # the source of truth. This is schema-generic and avoids one-off fixes for
    # a single field.
    mark_scheme_canonical_fields = [
        "mark_scheme_id",
        "mark_scheme_uid",
        "maximum_marks",
        "marking_guidance",
        "marking_points",
        "acceptable_answers",
        "rejected_answers",
        "additional_guidance",
        "assessment_objectives",
    ]

    for field in mark_scheme_canonical_fields:
        postgres_field = f"{field}_postgres"
        retrieval_field = f"{field}_retrieval"

        if postgres_field in final_df.columns:
            final_df[field] = final_df[postgres_field]
        elif field in final_df.columns:
            # Already canonical because it existed on only one side of merge.
            pass
        elif retrieval_field in final_df.columns:
            # Defensive fallback only. PostgreSQL is preferred whenever present.
            final_df[field] = final_df[retrieval_field]

    required_canonical_ms_columns = [
        "mark_scheme_id",
        "maximum_marks",
        "marking_guidance",
    ]
    missing_canonical_ms_columns = [
        column
        for column in required_canonical_ms_columns
        if column not in final_df.columns
    ]
    if missing_canonical_ms_columns:
        raise RuntimeError(
            "Canonical mark-scheme columns are missing after the final bundle "
            f"merge: {missing_canonical_ms_columns}"
        )

    required_merged_columns = [
        "source_pdf_path",
        "question_document_id",
        "paper_code_postgres",
        "programming_language_postgres",
        "question_number_postgres",
        "question_text_postgres",
        "marks_postgres",
        "has_code_postgres",
        "has_visual_postgres",
        "official_reference_postgres",
        "official_concept_name_postgres",
    ]

    missing_merged_columns = [
        column
        for column in required_merged_columns
        if column not in final_df.columns
    ]

    if missing_merged_columns:
        raise RuntimeError(
            "Expected merged columns are missing: "
            f"{missing_merged_columns}"
        )

    if final_df["mark_scheme_id"].isna().any():
        raise RuntimeError(
            "A selected question has no mark scheme."
        )

    print(
        "Canonical MS fields ready:",
        {
            field: field in final_df.columns
            for field in required_canonical_ms_columns
        },
    )

    print(
        f"Complete QP/MS bundles: "
        f"{len(final_df)}"
    )

    display(
        final_df[
            [
                "selected_rank",
                "detected_topic",
                "agent1_role",
                "official_reference_postgres",
                "question_number_postgres",
                "marks_postgres",
                "question_text_postgres",
                "maximum_marks",
                "marking_guidance",
            ]
        ]
    )


## 11A. Generalized subquestion-context boundary refinement

Parsed topical PDFs sometimes attach shared or neighbouring text to the wrong
subquestion. The active rule does not look for particular question numbers.
It splits context into semantic segments and keeps a segment only when:

```text
it contains a Figure/Table/Diagram reference used by the question; or
its MiniLM similarity to the current question clears a bounded threshold.
```

A segment referencing a different figure is removed unless its semantic match
is exceptionally strong. The original context remains stored for audit.


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    # ================================================================
    # CONTEXT REFINEMENT — GENERAL SEMANTIC AND REFERENCE BOUNDARIES
    # ================================================================

    REFERENCE_LABEL_PATTERN = re.compile(
        r"\b(?:Figure|Table|Diagram)\s+\d+[A-Za-z]?\b",
        flags=re.IGNORECASE,
    )

    CONTEXT_DEPENDENCY_PATTERN = re.compile(
        (
            r"\b(?:Figure|Table|Diagram)\s+\d+[A-Za-z]?\b|"
            r"\bshown\s+(?:above|below|in)\b|"
            r"\bthe\s+(?:following|given)\b|"
            r"\bthis\s+(?:algorithm|program|code|table|diagram)\b"
        ),
        flags=re.IGNORECASE,
    )


    def extract_reference_labels(value: Any) -> set[str]:
        return {
            re.sub(r"\s+", " ", match).lower().strip()
            for match in REFERENCE_LABEL_PATTERN.findall(
                str(value or "")
            )
        }


    def split_context_segments(value: Any) -> list[str]:
        text_value = str(value or "").replace("\r", "\n").strip()
        if not text_value:
            return []

        paragraphs = [
            re.sub(r"\s+", " ", part).strip()
            for part in re.split(r"\n\s*\n+", text_value)
            if re.sub(r"\s+", " ", part).strip()
        ]

        if len(paragraphs) > 1:
            return paragraphs

        # When the parser removed blank lines, split on sentence starts
        # while keeping figure/table labels with their neighbouring text.
        sentences = [
            re.sub(r"\s+", " ", part).strip()
            for part in re.split(
                r"(?<=[.!?])\s+(?=[A-Z])",
                paragraphs[0],
            )
            if re.sub(r"\s+", " ", part).strip()
        ]

        return sentences or paragraphs


    def refine_context_for_question(
        question_text: Any,
        context_text: Any,
    ) -> dict[str, Any]:
        original_context = str(context_text or "").strip()
        question_value = str(question_text or "").strip()

        if not ENABLE_CONTEXT_REFINEMENT or not original_context:
            return {
                "original_context": original_context,
                "refined_context": original_context,
                "status": (
                    "disabled" if original_context else "empty"
                ),
                "kept_segments": [],
                "removed_segments": [],
                "segment_scores": [],
            }

        segments = split_context_segments(original_context)
        if not segments:
            return {
                "original_context": original_context,
                "refined_context": "",
                "status": "empty_after_segmentation",
                "kept_segments": [],
                "removed_segments": [],
                "segment_scores": [],
            }

        embeddings = model.encode(
            [question_value] + segments,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float32)

        question_vector = embeddings[0]
        segment_scores = embeddings[1:] @ question_vector

        question_references = extract_reference_labels(question_value)
        requires_context = bool(
            CONTEXT_DEPENDENCY_PATTERN.search(question_value)
        )

        if requires_context:
            score_threshold = max(
                CONTEXT_RELEVANCE_ABSOLUTE_FLOOR,
                min(
                    0.62,
                    float(np.percentile(segment_scores, 55.0)),
                ),
            )
        else:
            score_threshold = SELF_CONTAINED_CONTEXT_MINIMUM_SCORE

        kept = []
        removed = []
        score_records = []

        for segment, score in zip(segments, segment_scores):
            segment_references = extract_reference_labels(segment)
            reference_match = bool(
                question_references
                and question_references.intersection(
                    segment_references
                )
            )
            reference_mismatch = bool(
                question_references
                and segment_references
                and not reference_match
            )

            keep = bool(
                reference_match
                or (
                    not reference_mismatch
                    and float(score) >= score_threshold
                )
                or (
                    reference_mismatch
                    and float(score)
                    >= CONTEXT_REFERENCE_MISMATCH_OVERRIDE_SCORE
                )
            )

            record = {
                "segment": segment,
                "score": round(float(score), 6),
                "reference_match": reference_match,
                "reference_mismatch": reference_mismatch,
                "kept": keep,
            }
            score_records.append(record)

            if keep:
                kept.append(segment)
            else:
                removed.append(segment)

        refined_context = "\n\n".join(kept).strip()

        return {
            "original_context": original_context,
            "refined_context": refined_context,
            "status": (
                "unchanged"
                if refined_context == original_context
                else (
                    "trimmed"
                    if refined_context
                    else "removed_as_unrelated"
                )
            ),
            "kept_segments": kept,
            "removed_segments": removed,
            "segment_scores": score_records,
        }


    context_refinement_records = []

    for _, row in final_df.iterrows():
        result = refine_context_for_question(
            row.get("question_text_postgres"),
            row.get("context_text"),
        )
        context_refinement_records.append(
            {
                "question_id": str(row["question_id"]),
                "context_text_original": result["original_context"],
                "context_text_refined": result["refined_context"],
                "context_refinement_status": result["status"],
                "context_kept_segments": result["kept_segments"],
                "context_removed_segments": result["removed_segments"],
                "context_segment_scores": result["segment_scores"],
            }
        )

    context_refinement_df = pd.DataFrame(
        context_refinement_records
    )

    final_df = final_df.merge(
        context_refinement_df,
        on="question_id",
        how="left",
        validate="one_to_one",
    )

    # Downstream student-facing exports use the refined context. The
    # original parsed context is still retained in a separate column.
    final_df["context_text"] = final_df[
        "context_text_refined"
    ]

    context_refinement_summary = {
        "version": CONTEXT_REFINEMENT_VERSION,
        "selected_questions": int(len(final_df)),
        "unchanged": int(
            (final_df["context_refinement_status"] == "unchanged").sum()
        ),
        "trimmed": int(
            (final_df["context_refinement_status"] == "trimmed").sum()
        ),
        "removed_as_unrelated": int(
            (
                final_df["context_refinement_status"]
                == "removed_as_unrelated"
            ).sum()
        ),
    }

    display(pd.DataFrame([context_refinement_summary]))
    display(
        final_df[
            [
                "selected_rank",
                "question_number_postgres",
                "context_refinement_status",
                "context_text_original",
                "context_text_refined",
            ]
        ]
    )

## 12. Notebook 07 — source-question cropping and multi-page rendering

Notebook 05 now sends the selected question IDs, source PDF paths,
assigned page ranges and source-document question inventory to Notebook 07.

Notebook 07 returns a JSON render manifest. Notebook 05 merges that manifest
into `final_df` and continues with the existing assessment packaging.

In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get(
        "AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN",
        False,
    ):
        print(
            "Downstream assessment rendering/export was skipped because no "
            "quality-safe questions match the current request. See the "
            "user-friendly run summary above."
        )
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True

else:
    # ================================================================
    # NOTEBOOK 05 -> NOTEBOOK 07 RENDER CONTRACT
    # ================================================================

    RUN_TIMESTAMP = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )


    def notebook7_json_safe(
        value: Any,
    ) -> Any:
        return json_safe(
            value
        )


    # ---------------------------------------------------------------
    # Build an inventory of all parsed questions from the selected source
    # documents. Notebook 07 uses this only to find the NEXT question
    # boundary on each source page.
    # ---------------------------------------------------------------
    selected_document_ids = [
        value
        for value
        in final_df[
            "question_document_id"
        ].dropna().unique()
    ]

    inventory_query = select(
        questions.c.id.label(
            "question_id"
        ),
        QUESTION_DOCUMENT_FK_COLUMN.label(
            "question_document_id"
        ),
        questions.c.question_number,
        questions.c.question_text,
        questions.c.page_start,
        questions.c.page_end,
        questions.c.visual_page_numbers,
    ).where(
        QUESTION_DOCUMENT_FK_COLUMN.in_(
            selected_document_ids
        )
    )

    with engine.connect() as connection:
        notebook7_inventory_df = pd.read_sql(
            inventory_query,
            connection,
        )

    if not notebook7_inventory_df.empty:
        notebook7_inventory_df[
            "question_id"
        ] = notebook7_inventory_df[
            "question_id"
        ].astype(str)

        notebook7_inventory_df[
            "question_document_id"
        ] = notebook7_inventory_df[
            "question_document_id"
        ].astype(str)


    # ---------------------------------------------------------------
    # Selected-question render request.
    # ---------------------------------------------------------------
    render_questions = []

    for _, row in final_df.iterrows():
        render_questions.append(
            {
                "question_id": str(
                    row["question_id"]
                ),
                "selected_rank": int(
                    row["selected_rank"]
                ),
                "detected_topic": (
                    row.get(
                        "detected_topic"
                    )
                ),
                "agent1_role": (
                    row.get(
                        "agent1_role"
                    )
                ),
                "question_document_id": str(
                    row.get(
                        "question_document_id"
                    )
                    or ""
                ),
                "question_number": (
                    row.get(
                        "question_number_postgres"
                    )
                ),
                "question_text": (
                    row.get(
                        "question_text_postgres"
                    )
                ),
                "context_text": (
                    row.get(
                        "context_text"
                    )
                    or ""
                ),
                "marks": int(
                    row.get(
                        "marks_postgres",
                        0,
                    )
                ),
                "source_pdf_path": (
                    row.get(
                        "source_pdf_path"
                    )
                ),
                "source_file_name": (
                    row.get(
                        "source_file_name"
                    )
                ),
                "page_start": (
                    row.get(
                        "page_start"
                    )
                ),
                "page_end": (
                    row.get(
                        "page_end"
                    )
                ),
                "visual_page_numbers": (
                    row.get(
                        "visual_page_numbers"
                    )
                ),
                "has_visual": bool(
                    row.get(
                        "has_visual_postgres",
                        False,
                    )
                ),
                "has_code": bool(
                    row.get(
                        "has_code_postgres",
                        False,
                    )
                ),
                "paper_code": (
                    row.get(
                        "paper_code_postgres"
                    )
                ),
            }
        )

    render_request_payload = {
        "schema_version": (
            "agent2-question-render-request-v1.0.0"
        ),
        "generated_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "integration_version": (
            NOTEBOOK7_RENDERER_INTEGRATION_VERSION
        ),
        "run_timestamp": (
            RUN_TIMESTAMP
        ),
        "project_root": str(
            PROJECT_ROOT.resolve()
        ),
        "output_dir": str(
            OUTPUT_DIR.resolve()
        ),
        "render_dpi": int(
            VISUAL_RENDER_DPI
        ),
        "database_page_numbers_are_one_based": bool(
            DATABASE_PAGE_NUMBERS_ARE_ONE_BASED
        ),
        "source_page_search_radius": int(
            SOURCE_PAGE_SEARCH_RADIUS
        ),
        "dependency_page_search_radius": int(
            SOURCE_PAGE_SEARCH_RADIUS
        ),
        "display_rendered_images": bool(
            DISPLAY_RENDERED_IMAGES_IN_NOTEBOOK
        ),
        "allow_full_pdf_fallback": bool(
            ALLOW_NOTEBOOK7_FULL_PDF_FALLBACK
        ),
        "questions": (
            notebook7_json_safe(
                render_questions
            )
        ),
        "document_inventory": (
            notebook7_json_safe(
                notebook7_inventory_df.to_dict(
                    orient="records"
                )
            )
        ),
    }

    render_request_path = (
        OUTPUT_DIR
        / (
            "agent2_question_render_request_"
            f"{RUN_TIMESTAMP}.json"
        )
    )

    render_manifest_json_path = (
        OUTPUT_DIR
        / (
            "agent2_visual_render_manifest_"
            f"{RUN_TIMESTAMP}.json"
        )
    )

    render_request_path.write_text(
        json.dumps(
            render_request_payload,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    # ---------------------------------------------------------------
    # Resolve Notebook 07.
    # ---------------------------------------------------------------
    notebook7_candidates = [
        (
            PROJECT_ROOT
            / "notebooks"
            / NOTEBOOK7_RENDERER_FILE
        ),
        (
            PROJECT_ROOT
            / NOTEBOOK7_RENDERER_FILE
        ),
        (
            Path.cwd()
            / NOTEBOOK7_RENDERER_FILE
        ),
        (
            Path.cwd()
            / "notebooks"
            / NOTEBOOK7_RENDERER_FILE
        ),
    ]

    notebook7_path = next(
        (
            candidate.resolve()
            for candidate
            in notebook7_candidates
            if candidate.is_file()
        ),
        None,
    )

    if notebook7_path is None:
        raise RuntimeError(
            "Notebook 07 renderer could not be found. Expected "
            f"{NOTEBOOK7_RENDERER_FILE} inside Agent2/notebooks."
        )

    notebook7_execution_dir = (
        OUTPUT_DIR
        / "notebook7_execution"
    )

    notebook7_execution_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    executed_notebook7_name = (
        "07_question_visual_cropping_and_multipage_rendering_"
        f"executed_{RUN_TIMESTAMP}.ipynb"
    )

    notebook7_log_path = (
        OUTPUT_DIR
        / (
            "agent2_notebook7_render_log_"
            f"{RUN_TIMESTAMP}.txt"
        )
    )


    # ---------------------------------------------------------------
    # Execute Notebook 07 with explicit request/manifest paths.
    # ---------------------------------------------------------------
    child_environment = os.environ.copy()

    child_environment[
        "AGENT2_RENDER_REQUEST_PATH"
    ] = str(
        render_request_path.resolve()
    )

    child_environment[
        "AGENT2_RENDER_MANIFEST_PATH"
    ] = str(
        render_manifest_json_path.resolve()
    )

    notebook7_command = [
        sys.executable,
        "-m",
        "jupyter",
        "nbconvert",
        "--to",
        "notebook",
        "--execute",
        str(
            notebook7_path
        ),
        "--output",
        executed_notebook7_name,
        "--output-dir",
        str(
            notebook7_execution_dir.resolve()
        ),
        (
            "--ExecutePreprocessor.timeout="
            f"{NOTEBOOK7_EXECUTION_TIMEOUT_SECONDS}"
        ),
    ]

    print(
        "Executing Notebook 07 renderer:"
    )
    print(
        notebook7_path
    )
    print(
        "Render request:"
    )
    print(
        render_request_path
    )

    # Avoid buffering the entire nbconvert output in memory. The worker log
    # is written directly to disk while Notebook 05 waits for completion.
    with notebook7_log_path.open(
        "w",
        encoding="utf-8",
    ) as notebook7_log_handle:
        notebook7_log_handle.write(
            "COMMAND\n"
            + " ".join(
                notebook7_command
            )
            + "\n\nOUTPUT\n"
        )
        notebook7_log_handle.flush()

        notebook7_process = subprocess.run(
            notebook7_command,
            cwd=str(
                PROJECT_ROOT.resolve()
            ),
            env=child_environment,
            stdout=notebook7_log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            check=False,
        )

    if (
        notebook7_process.returncode
        != 0
    ):
        raise RuntimeError(
            "Notebook 07 visual renderer failed. "
            f"See {notebook7_log_path}"
        )

    if not render_manifest_json_path.is_file():
        raise RuntimeError(
            "Notebook 07 completed but did not create its render manifest."
        )


    # ---------------------------------------------------------------
    # Merge Notebook 07 output into final_df.
    # ---------------------------------------------------------------
    render_manifest_payload = json.loads(
        render_manifest_json_path.read_text(
            encoding="utf-8"
        )
    )

    visual_render_records = (
        render_manifest_payload.get(
            "records",
            [],
        )
    )

    visual_render_df = pd.DataFrame(
        visual_render_records
    )

    if visual_render_df.empty:
        raise RuntimeError(
            "Notebook 07 returned an empty visual render manifest."
        )

    visual_render_df[
        "question_id"
    ] = visual_render_df[
        "question_id"
    ].astype(str)

    if visual_render_df[
        "question_id"
    ].duplicated().any():
        raise RuntimeError(
            "Notebook 07 returned duplicate question IDs."
        )

    expected_render_ids = set(
        final_df[
            "question_id"
        ].astype(str)
    )

    returned_render_ids = set(
        visual_render_df[
            "question_id"
        ].astype(str)
    )

    if (
        expected_render_ids
        != returned_render_ids
    ):
        missing = sorted(
            expected_render_ids
            - returned_render_ids
        )

        unexpected = sorted(
            returned_render_ids
            - expected_render_ids
        )

        raise RuntimeError(
            "Notebook 07 render-manifest question IDs do not match "
            f"selected questions. Missing={missing}, unexpected={unexpected}"
        )

    final_df = final_df.merge(
        visual_render_df,
        on="question_id",
        how="left",
        validate="one_to_one",
    )

    # ===============================================================
    # GENERIC POST-RENDER VISUAL SAFETY LAYER
    # ===============================================================
    # Notebook 07 deliberately favours recall when locating visual regions.
    # Before student PDF assembly, Notebook 05 applies conservative structural
    # rules so neighbouring/sibling source material cannot leak into a selected
    # question.
    #
    # No question IDs, topic names, paper names or syllabus-specific rules are
    # used here.

    def visual_source_page_from_path(
        path_value: Any,
    ) -> int | None:
        match = re.search(
            r"source_(\d+)\.(?:png|jpg|jpeg|webp)$",
            str(path_value or ""),
            flags=re.IGNORECASE,
        )
        return int(match.group(1)) if match else None


    def visual_text_is_missing(
        value: Any,
    ) -> bool:
        if value is None:
            return True

        if (
            isinstance(value, float)
            and np.isnan(value)
        ):
            return True

        text_value = str(value).strip()
        return (
            not text_value
            or text_value.casefold() == "nan"
        )


    def visual_normalised_label(
        value: Any,
    ) -> str:
        return re.sub(
            r"\s+",
            " ",
            str(value or "").casefold(),
        ).strip()


    def visual_required_labels_are_in_question(
        row: pd.Series,
    ) -> bool:
        labels = (
            row.get("required_dependency_labels")
            if isinstance(
                row.get("required_dependency_labels"),
                list,
            )
            else []
        )

        if not labels:
            return False

        question_text_value = visual_normalised_label(
            row.get("question_text_postgres")
        )

        return all(
            visual_normalised_label(label)
            in question_text_value
            for label in labels
            if visual_normalised_label(label)
        )


    def visual_question_has_explicit_external_dependency(
        row: pd.Series,
    ) -> bool:
        """
        Detect learner-facing wording that explicitly depends on material
        outside the selected question crop.

        This is intentionally conservative and cue-based. Generic technical
        uses of words such as "table" do not count unless they are phrased as
        an external reference (for example "use Table 2" or "shown below").
        """
        question_text_value = visual_normalised_label(
            row.get("question_text_postgres")
        )
        if not question_text_value:
            return False

        dependency_patterns = [
            r"\b(?:refer\s+to|use|using|study|look\s+at|see)\s+(?:the\s+)?(?:figure|table|diagram|graph|chart|image)\b",
            r"\b(?:figure|table|diagram|graph|chart|image)\s+[a-z0-9][a-z0-9.:-]*\b",
            r"\b(?:shown|given|displayed)\s+(?:in\s+)?(?:the\s+)?(?:figure|table|diagram|graph|chart|image|above|below)\b",
            r"\b(?:figure|table|diagram|graph|chart|image|code)\s+(?:above|below)\b",
            r"\b(?:above|below)\s+(?:figure|table|diagram|graph|chart|image|code)\b",
        ]
        return any(
            re.search(pattern, question_text_value)
            for pattern in dependency_patterns
        )


    def visual_selected_question_is_self_contained(
        row: pd.Series,
        question_images: list[str],
    ) -> bool:
        """
        Safe self-contained release rule for an unanchored parent fallback.

        A verified selected-question crop may stand alone when:
        - the selected question crop exists,
        - no required dependency is unresolved,
        - no separate context/refined-context text is required, and
        - either every declared dependency label is already inside the
          selected question text OR no dependency labels are declared and the
          question wording has no explicit external visual/context cue.

        This fixes the generic edge case where an unnecessary unanchored
        parent-context fallback previously blocked a self-contained question.
        It does not weaken fail-closed handling for genuine Figure/Table/etc.
        dependencies.
        """
        if not question_images:
            return False

        missing_labels = (
            row.get("missing_dependency_labels")
            if isinstance(
                row.get("missing_dependency_labels"),
                list,
            )
            else []
        )
        if missing_labels:
            return False

        if not visual_text_is_missing(row.get("context_text")):
            return False
        if not visual_text_is_missing(row.get("context_text_refined")):
            return False

        labels = (
            row.get("required_dependency_labels")
            if isinstance(
                row.get("required_dependency_labels"),
                list,
            )
            else []
        )

        if labels:
            return visual_required_labels_are_in_question(row)

        return not visual_question_has_explicit_external_dependency(row)


    def visual_dependency_fallback_is_unanchored(
        row: pd.Series,
    ) -> bool:
        details = (
            row.get("dependency_resolution_details")
            if isinstance(
                row.get("dependency_resolution_details"),
                list,
            )
            else []
        )

        structural_records = [
            detail
            for detail in details
            if isinstance(detail, dict)
            and str(
                detail.get("status") or ""
            ).casefold()
            == "structural_parent_context_crop"
        ]

        if not structural_records:
            return False

        return any(
            (
                not bool(
                    detail.get("parent_anchor_found")
                )
                and not bool(
                    detail.get("caption_anchor_found")
                )
                and "fallback"
                in str(
                    detail.get("start_method") or ""
                ).casefold()
            )
            for detail in structural_records
        )


    def visual_filter_question_crops_to_pages(
        row: pd.Series,
        allowed_pages: set[int],
    ) -> tuple[list[str], list[dict[str, Any]]]:
        images = (
            list(row.get("cropped_question_images"))
            if isinstance(
                row.get("cropped_question_images"),
                list,
            )
            else []
        )

        rectangles = (
            list(row.get("question_region_crop_rectangles"))
            if isinstance(
                row.get("question_region_crop_rectangles"),
                list,
            )
            else []
        )

        kept_images = [
            path_value
            for path_value in images
            if (
                visual_source_page_from_path(path_value)
                in allowed_pages
            )
        ]

        kept_rectangles = [
            rectangle
            for rectangle in rectangles
            if (
                isinstance(rectangle, dict)
                and int(
                    rectangle.get(
                        "source_page_number",
                        -1,
                    )
                )
                in allowed_pages
            )
        ]

        return kept_images, kept_rectangles


    def visual_drop_tiny_continuation_slivers(
        images: list[str],
        rectangles: list[dict[str, Any]],
    ) -> tuple[list[str], list[dict[str, Any]], list[int]]:
        """
        Keep selected-question crops, but remove a continuation crop when the
        structural interval on that source page is only a tiny sliver before
        the next sibling/top-level boundary.
        """
        rectangle_by_page = {
            int(rectangle["source_page_number"]): rectangle
            for rectangle in rectangles
            if (
                isinstance(rectangle, dict)
                and rectangle.get("source_page_number") is not None
            )
        }

        candidate_page_heights = [
            float(rectangle.get("y1", 0.0))
            for rectangle in rectangles
            if isinstance(rectangle, dict)
        ]
        estimated_page_height = max(
            candidate_page_heights,
            default=0.0,
        )

        minimum_height = (
            estimated_page_height
            * VISUAL_MIN_CONTINUATION_HEIGHT_RATIO
        )

        kept_images = []
        dropped_pages = []

        for path_value in images:
            source_page = visual_source_page_from_path(
                path_value
            )
            rectangle = rectangle_by_page.get(
                source_page
            )

            if (
                rectangle
                and str(
                    rectangle.get("role") or ""
                ).casefold()
                == "selected_question_continuation"
            ):
                height = max(
                    0.0,
                    float(
                        rectangle.get("y1", 0.0)
                    )
                    - float(
                        rectangle.get("y0", 0.0)
                    ),
                )

                if (
                    estimated_page_height > 0.0
                    and height < minimum_height
                ):
                    dropped_pages.append(
                        int(source_page)
                    )
                    continue

            kept_images.append(
                path_value
            )

        kept_rectangles = [
            rectangle
            for rectangle in rectangles
            if int(
                rectangle.get(
                    "source_page_number",
                    -1,
                )
            )
            not in set(dropped_pages)
        ]

        return (
            kept_images,
            kept_rectangles,
            dropped_pages,
        )


    def sanitize_visual_render_record(
        row: pd.Series,
    ) -> pd.Series:
        row = row.copy()
        actions = []

        question_images = (
            list(row.get("cropped_question_images"))
            if isinstance(
                row.get("cropped_question_images"),
                list,
            )
            else []
        )

        question_rectangles = (
            list(row.get("question_region_crop_rectangles"))
            if isinstance(
                row.get("question_region_crop_rectangles"),
                list,
            )
            else []
        )

        assigned_pages = {
            int(page)
            for page in (
                row.get("assigned_source_pages")
                if isinstance(
                    row.get("assigned_source_pages"),
                    list,
                )
                else []
            )
            if page is not None
        }

        structural_parent = row.get(
            "structural_parent_question_number"
        )

        top_level_question = (
            structural_parent is None
            or (
                isinstance(
                    structural_parent,
                    float,
                )
                and np.isnan(
                    structural_parent
                )
            )
            or not str(
                structural_parent
            ).strip()
        )

        single_page_database_assignment = bool(
            len(assigned_pages) == 1
            and not bool(
                row.get(
                    "multi_page_question",
                    False,
                )
            )
        )

        # -------------------------------------------------------
        # RULE 1 — single-page top-level hard containment
        # -------------------------------------------------------
        # A top-level question whose database assignment is one page must never
        # expand across following sibling/top-level questions merely because the
        # next top-level structural marker is several pages away.
        if (
            top_level_question
            and single_page_database_assignment
            and question_images
        ):
            (
                bounded_images,
                bounded_rectangles,
            ) = visual_filter_question_crops_to_pages(
                row,
                assigned_pages,
            )

            if (
                bounded_images
                and len(bounded_images)
                < len(question_images)
            ):
                question_images = bounded_images
                question_rectangles = (
                    bounded_rectangles
                )
                actions.append(
                    "single_page_top_level_clamped_to_database_assignment"
                )

        # -------------------------------------------------------
        # RULE 2 — remove tiny continuation boundary slivers
        # -------------------------------------------------------
        (
            question_images,
            question_rectangles,
            tiny_pages,
        ) = visual_drop_tiny_continuation_slivers(
            question_images,
            question_rectangles,
        )

        if tiny_pages:
            actions.append(
                "tiny_continuation_sliver_removed:"
                + ",".join(
                    str(page)
                    for page in tiny_pages
                )
            )

        row["cropped_question_images"] = (
            question_images
        )
        row[
            "question_region_crop_rectangles"
        ] = question_rectangles

        row["source_page_numbers"] = list(
            dict.fromkeys(
                page
                for page in (
                    visual_source_page_from_path(
                        path_value
                    )
                    for path_value in question_images
                )
                if page is not None
            )
        )

        # -------------------------------------------------------
        # RULE 3 — reject unanchored parent-context fallback
        # -------------------------------------------------------
        # If the renderer could not find either the parent anchor or requested
        # figure/table caption, do not put the fallback context crop into the
        # student PDF. When the selected question already contains all required
        # dependency labels and no separate refined context was needed, treat the
        # selected question region as self-contained.
        dependency_images = (
            list(row.get("dependency_crop_images"))
            if isinstance(
                row.get("dependency_crop_images"),
                list,
            )
            else []
        )

        if (
            dependency_images
            and visual_dependency_fallback_is_unanchored(
                row
            )
        ):
            selected_question_is_self_contained = (
                visual_selected_question_is_self_contained(
                    row,
                    question_images,
                )
            )

            if selected_question_is_self_contained:
                row["dependency_crop_images"] = []
                row[
                    "dependency_crop_rectangles"
                ] = []
                row[
                    "dependency_source_pages"
                ] = []
                row[
                    "dependency_render_status"
                ] = (
                    "self_contained_selected_question"
                )
                row[
                    "resolved_dependency_labels"
                ] = list(
                    row.get(
                        "required_dependency_labels"
                    )
                    if isinstance(
                        row.get(
                            "required_dependency_labels"
                        ),
                        list,
                    )
                    else []
                )
                row[
                    "missing_dependency_labels"
                ] = []
                row[
                    "visual_dependency_complete"
                ] = True
                actions.append(
                    "unanchored_dependency_fallback_removed_self_contained"
                )
            else:
                # Never silently render an unrelated fallback crop.
                row["dependency_crop_images"] = []
                row[
                    "dependency_crop_rectangles"
                ] = []
                row[
                    "dependency_source_pages"
                ] = []
                row[
                    "dependency_render_status"
                ] = (
                    "unanchored_dependency_rejected"
                )
                row[
                    "visual_dependency_complete"
                ] = False
                row[
                    "student_release_eligible"
                ] = False
                actions.append(
                    "unanchored_dependency_fallback_rejected"
                )

        # Rebuild compatibility image list from the sanitised components.
        dependency_images = (
            list(row.get("dependency_crop_images"))
            if isinstance(
                row.get("dependency_crop_images"),
                list,
            )
            else []
        )

        full_page_images = (
            list(row.get("full_page_images"))
            if isinstance(
                row.get("full_page_images"),
                list,
            )
            else []
        )

        safe_rendered_images = []

        for path_value in (
            dependency_images
            + question_images
            + full_page_images
        ):
            if path_value not in safe_rendered_images:
                safe_rendered_images.append(
                    path_value
                )

        row["rendered_page_images"] = (
            safe_rendered_images
        )
        row["rendered_page_count"] = int(
            len(safe_rendered_images)
        )
        row[
            "visual_post_render_sanitization_actions"
        ] = actions
        row[
            "visual_post_render_sanitization_version"
        ] = VISUAL_POST_RENDER_SANITIZATION_VERSION

        return row


    final_df = final_df.apply(
        sanitize_visual_render_record,
        axis=1,
    )

    visual_sanitization_audit_df = final_df[
        [
            "selected_rank",
            "question_id",
            "question_number_postgres",
            "assigned_source_pages",
            "source_page_numbers",
            "dependency_render_status",
            "dependency_crop_images",
            "cropped_question_images",
            "rendered_page_count",
            "student_release_eligible",
            "visual_dependency_complete",
            "visual_post_render_sanitization_actions",
        ]
    ].copy()

    visual_sanitization_manifest_path = (
        OUTPUT_DIR
        / (
            "agent2_visual_render_sanitization_"
            f"{RUN_TIMESTAMP}.csv"
        )
    )

    visual_sanitization_audit_df.to_csv(
        visual_sanitization_manifest_path,
        index=False,
    )

    print(
        "Visual post-render sanitisation applied:"
    )
    display(
        visual_sanitization_audit_df
    )

    visual_manifest_path = (
        Path(
            render_manifest_payload.get(
                "csv_path",
                render_manifest_json_path.with_suffix(
                    ".csv"
                ),
            )
        )
    )

    VISUAL_IMAGE_ROOT = Path(
        render_manifest_payload.get(
            "image_root",
            (
                OUTPUT_DIR
                / "visual_question_crops"
                / RUN_TIMESTAMP
            ),
        )
    )


    # ---------------------------------------------------------------
    # Compatibility summary used by the existing package/PDF/audit cells.
    # ---------------------------------------------------------------
    selected_visual_questions = int(
        final_df[
            "has_visual_postgres"
        ].astype(bool).sum()
    )

    successfully_rendered_visual_questions = int(
        final_df[
            "visual_render_status"
        ].eq(
            "rendered_dependency_complete"
        ).sum()
    )

    visual_render_failures = int(
        final_df[
            "visual_render_status"
        ].eq(
            "failed"
        ).sum()
    )

    rendered_page_total = int(
        final_df[
            "rendered_page_count"
        ].fillna(
            0
        ).sum()
    )

    successful_crop_statuses = {
        "cropped_verified_with_dependencies",
        "multi_page_cropped_verified_with_dependencies",
    }

    visual_crop_success_count = int(
        final_df[
            "question_region_crop_status"
        ].isin(
            successful_crop_statuses
        ).sum()
    )

    visual_full_page_fallback_count = int(
        final_df[
            "question_region_crop_status"
        ].eq(
            "full_page_fallback"
        ).sum()
    )

    verified_source_page_count = int(
        final_df[
            "source_page_match_status"
        ].eq(
            "verified_question_page"
        ).sum()
    )

    final_crop_verification_failure_count = int(
        final_df[
            "question_region_crop_status"
        ].isin(
            [
                "anchor_verification_failed",
                "required_dependency_unresolved",
                "failed",
            ]
        ).sum()
    )

    visual_render_summary_df = pd.DataFrame(
        [
            {
                "visual_rendering_version": (
                    VISUAL_RENDERING_VERSION
                ),
                "notebook7_integration_version": (
                    NOTEBOOK7_RENDERER_INTEGRATION_VERSION
                ),
                "notebook7_renderer_version": (
                    render_manifest_payload.get(
                        "summary",
                        {},
                    ).get(
                        "renderer_version"
                    )
                ),
                "selected_questions": int(
                    len(
                        final_df
                    )
                ),
                "effective_visual_questions": int(
                    final_df[
                        "visual_render_required"
                    ].astype(bool).sum()
                ),
                "multi_page_questions": int(
                    final_df[
                        "multi_page_question"
                    ].astype(bool).sum()
                ),
                "selected_visual_questions": (
                    selected_visual_questions
                ),
                "verified_source_pages": (
                    verified_source_page_count
                ),
                "visual_questions_rendered": (
                    successfully_rendered_visual_questions
                ),
                "visual_render_failures": (
                    visual_render_failures
                ),
                "question_regions_cropped": (
                    visual_crop_success_count
                ),
                "safe_full_page_fallbacks": (
                    visual_full_page_fallback_count
                ),
                "total_images_created": (
                    rendered_page_total
                ),
            }
        ]
    )

    display(
        visual_render_summary_df
    )

    display(
        final_df[
            [
                column
                for column in [
                    "selected_rank",
                    "question_id",
                    "question_number_postgres",
                    "database_visual_flag",
                    "visual_render_required",
                    "multi_page_question",
                    "required_dependency_labels",
                    "resolved_dependency_labels",
                    "missing_dependency_labels",
                    "dependency_source_pages",
                    "dependency_crop_images",
                    "dependency_resolution_details",
                    "structured_response_required",
                    "assigned_source_pages",
                    "searched_source_pages",
                    "matched_source_page",
                    "source_page_numbers",
                    "source_page_match_status",
                    "source_page_match_score",
                    "source_page_token_coverage",
                    "source_page_question_number_match",
                    "final_crop_anchor_verified",
                    "structured_layout_verified",
                    "resolved_dependency_labels",
                    "missing_dependency_labels",
                    "visual_dependency_complete",
                    "student_release_eligible",
                    "visual_render_status",
                    "question_region_crop_status",
                    "question_region_crop_method",
                    "rendered_page_images",
                    "visual_render_error",
                ]
                if column
                in final_df.columns
            ]
        ]
    )

    print(
        "Notebook 07 render manifest:"
    )
    print(
        render_manifest_json_path
    )

    print(
        "Notebook 07 execution log:"
    )
    print(
        notebook7_log_path
    )

## 13. Phase 3 — structured mark-scheme cleanup

### Purpose

The raw AQA marking guidance remains unchanged and remains the source of truth.

Phase 3 creates a cleaner secondary representation for the interface and evaluation:

```text
raw marking guidance              preserved
legacy structured fields          retained for comparison
Phase 3 structured fields         newly generated
```

### Deterministic classification rules

```text
Mark A / Mark B / 1 mark / Program Logic
    → marking points

A. / Accept / Allow
    → acceptable answers

R. / Reject / Do not accept
    → rejected answers

I. / Note to examiners / Maximum / Refer to
    → additional guidance

Python Example / C# Example / VB.NET Example / Correct table is
    → worked examples

AO1 / AO2 / AO3
    → assessment objectives
```

### Safety and Human-in-the-Loop behaviour

```text
raw guidance is never overwritten
low-confidence cleanup is marked review_recommended
cleaned fields are stored separately
the interface should display raw guidance prominently
Phase 3 fields are a navigational aid, not a replacement for the source
```

### Release rule

A result using lesson-summary fallback is marked `evaluation_ready` rather than
`ready_for_release`. Actual Agent 1 source chunk text must be tested before student
release.


### Phase 3 issue found during evaluation

The initial Phase 3 parser classified every extracted PDF line independently. AQA
mark-scheme instructions are often wrapped across multiple PDF lines, for example:

```text
Note to examiners: Check vertically as well as horizontally for the
effect of duplicate values.
```

The first line was classified as `additional_guidance`, while its continuation was
incorrectly placed in `marking_points`.

### Final decision

The active parser now uses logical blocks:

```text
1. clean extracted lines;
2. split inline A. / I. / R. markers;
3. detect explicit Mark / A. / R. / I. / Note / Example boundaries;
4. attach unmarked wrapped lines to the active block;
5. preserve code and table line breaks inside marking points and examples;
6. join prose continuation lines inside guidance and answer blocks;
7. export block counts, merged-line counts and block audit records.
```

Raw `marking_guidance` remains unchanged and is still the source of truth. This is
a general continuation-block approach rather than a rule for one specific question.


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    def phase3_json_safe(value: Any) -> Any:
        if value is None:
            return None
        if isinstance(value, (bool, int, str)):
            return value
        if isinstance(value, float):
            return value if math.isfinite(value) else None
        if isinstance(value, datetime):
            return value.isoformat()
        if isinstance(value, uuid.UUID):
            return str(value)
        if isinstance(value, dict):
            return {str(k): phase3_json_safe(v) for k, v in value.items()}
        if isinstance(value, (list, tuple, set)):
            return [phase3_json_safe(v) for v in value]
        if hasattr(value, "item"):
            try:
                return phase3_json_safe(value.item())
            except (TypeError, ValueError):
                pass
        try:
            if pd.isna(value):
                return None
        except (TypeError, ValueError):
            pass
        return str(value)


    PHASE3_EXAMPLE_HEADER_PATTERN = re.compile(
        r"^(?:(?:Python|C#|VB\.NET|Java|Pseudocode)\s+)?Example\s+\d+\b.*$",
        flags=re.IGNORECASE | re.MULTILINE,
    )
    PHASE3_CORRECT_OUTPUT_PATTERN = re.compile(
        r"^Correct\s+(?:table|answer|output|trace)\b.*$",
        flags=re.IGNORECASE | re.MULTILINE,
    )
    PHASE3_ACCEPT_PATTERN = re.compile(
        r"^(?:A\.|Accept(?:able)?\b|Allow\b)", flags=re.IGNORECASE
    )
    PHASE3_REJECT_PATTERN = re.compile(
        r"^(?:R\.|Reject\b|Do\s+not\s+accept\b)", flags=re.IGNORECASE
    )
    PHASE3_GUIDANCE_PATTERN = re.compile(
        r"^(?:I\.|Note\s+to\s+examiners\b|Note\b|Maximum\b|Max\b|Refer\s+to\b)",
        flags=re.IGNORECASE,
    )
    PHASE3_MARKING_START_PATTERN = re.compile(
        r"^(?:\d+\s+marks?\b|Mark(?:\s+[A-Z]\b|\s+is\b)|Program\s+(?:Design|Logic)\b)",
        flags=re.IGNORECASE,
    )
    PHASE3_AO_PATTERN = re.compile(r"\bAO[123]\b", flags=re.IGNORECASE)
    PHASE3_INLINE_MARKER_PATTERN = re.compile(r"(?<=;)\s+(?=[AIR]\.\s)", flags=re.IGNORECASE)
    PHASE3_TRAILING_MARKER_PATTERN = re.compile(r"^(.*?);\s*([AIR]\.)\s*$", flags=re.IGNORECASE)
    PHASE3_NOISE_LINES = {"do not", "outsid", "outside", "bo", "turn over"}


    def clean_ms_line(value: Any) -> str:
        return re.sub(r"\s+", " ", str(value or "").strip())


    def strip_ms_prefix(value: str) -> str:
        return re.sub(
            r"^(?:A\.|R\.|Accept(?:able)?|Allow|Reject|Do\s+not\s+accept)\s*[:\-]?\s*",
            "",
            value,
            flags=re.IGNORECASE,
        ).strip()


    def expand_mark_scheme_lines(raw_text: str) -> tuple[list[str], int, list[str]]:
        expanded: list[str] = []
        split_count = 0
        dangling: list[str] = []
        pending: str | None = None

        for raw_line in raw_text.splitlines():
            line = clean_ms_line(raw_line)
            if not line:
                continue
            if pending is not None:
                line = f"{pending} {line}".strip()
                pending = None

            trailing = PHASE3_TRAILING_MARKER_PATTERN.match(line)
            if trailing:
                main = clean_ms_line(trailing.group(1) + ";")
                if main:
                    expanded.append(main)
                pending = trailing.group(2).upper()
                split_count += 1
                continue

            parts = (
                [clean_ms_line(p) for p in PHASE3_INLINE_MARKER_PATTERN.split(line) if clean_ms_line(p)]
                if PHASE3_SPLIT_INLINE_MARKERS
                else [line]
            )
            split_count += max(0, len(parts) - 1)
            expanded.extend(parts)

        if pending is not None:
            dangling.append(pending)
            expanded.append(pending)

        return expanded, split_count, dangling


    def classify_explicit_ms_line(line: str) -> str | None:
        if PHASE3_EXAMPLE_HEADER_PATTERN.match(line) or PHASE3_CORRECT_OUTPUT_PATTERN.match(line):
            return "worked_example"
        if PHASE3_ACCEPT_PATTERN.match(line):
            return "acceptable_answer"
        if PHASE3_REJECT_PATTERN.match(line):
            return "rejected_answer"
        if PHASE3_GUIDANCE_PATTERN.match(line):
            return "additional_guidance"
        if PHASE3_MARKING_START_PATTERN.match(line):
            return "marking_point"
        return None


    def block_text(category: str, lines: list[str]) -> str:
        if category in {"additional_guidance", "acceptable_answer", "rejected_answer"}:
            return clean_ms_line(" ".join(lines))
        return "\n".join(lines).strip()


    def parse_mark_scheme_guidance(raw_guidance: Any) -> dict[str, Any]:
        raw_text = str(raw_guidance or "").strip()
        empty = {
            "phase3_cleanup_status": "review_recommended",
            "phase3_rule_confidence": 0.0,
            "phase3_marking_points": [],
            "phase3_acceptable_answers": [],
            "phase3_rejected_answers": [],
            "phase3_additional_guidance": [],
            "phase3_worked_examples": [],
            "phase3_assessment_objectives": [],
            "phase3_review_reasons": ["missing_raw_marking_guidance"],
            "phase3_parser_noise_lines": [],
            "phase3_raw_guidance_preserved": True,
            "phase3_block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
            "phase3_block_count": 0,
            "phase3_continuation_lines_merged": 0,
            "phase3_inline_markers_split": 0,
            "phase3_implicit_marking_block_count": 0,
            "phase3_ambiguous_lines": [],
            "phase3_dangling_inline_markers": [],
            "phase3_block_audit": [],
        }
        if not raw_text:
            return empty

        lines, inline_split, dangling = expand_mark_scheme_lines(raw_text)
        objectives = sorted({m.upper() for m in PHASE3_AO_PATTERN.findall(raw_text)})
        noise = [line for line in lines if line.lower() in PHASE3_NOISE_LINES]

        marking_points: list[str] = []
        acceptable: list[str] = []
        rejected: list[str] = []
        guidance: list[str] = []
        examples: list[dict[str, Any]] = []
        block_audit: list[dict[str, Any]] = []
        ambiguous: list[str] = []
        continuation_count = 0
        implicit_count = 0

        current_block: dict[str, Any] | None = None
        current_example: dict[str, Any] | None = None

        def flush_block() -> None:
            nonlocal current_block
            if current_block is None:
                return
            category = current_block["category"]
            lines_in_block = current_block["lines"]
            value = block_text(category, lines_in_block)
            if value:
                if category == "marking_point":
                    marking_points.append(value)
                elif category == "acceptable_answer":
                    cleaned = strip_ms_prefix(value)
                    if cleaned:
                        acceptable.append(cleaned)
                elif category == "rejected_answer":
                    cleaned = strip_ms_prefix(value)
                    if cleaned:
                        rejected.append(cleaned)
                elif category == "additional_guidance":
                    guidance.append(value)
                block_audit.append({
                    "category": category,
                    "text": value,
                    "source_line_count": len(lines_in_block),
                    "continuation_line_count": max(0, len(lines_in_block)-1),
                    "implicit_start": bool(current_block.get("implicit_start", False)),
                })
            current_block = None

        def flush_example() -> None:
            nonlocal current_example
            if current_example is None:
                return
            content = "\n".join(current_example["lines"]).strip()
            examples.append({"header": current_example["header"], "content": content})
            block_audit.append({
                "category": "worked_example",
                "text": current_example["header"] + (("\n"+content) if content else ""),
                "source_line_count": 1 + len(current_example["lines"]),
                "continuation_line_count": len(current_example["lines"]),
                "implicit_start": False,
            })
            current_example = None

        for line in lines:
            category = classify_explicit_ms_line(line)

            if category == "worked_example":
                flush_example()
                flush_block()
                current_example = {"header": line, "lines": []}
                continue

            if current_example is not None:
                if category in {"acceptable_answer", "rejected_answer", "additional_guidance", "marking_point"}:
                    flush_example()
                else:
                    current_example["lines"].append(line)
                    continuation_count += 1
                    continue

            if category is not None:
                flush_block()
                current_block = {"category": category, "lines": [line], "implicit_start": False}
                continue

            if PHASE3_MERGE_WRAPPED_LINES and current_block is not None:
                current_block["lines"].append(line)
                continuation_count += 1
                continue

            current_block = {"category": "marking_point", "lines": [line], "implicit_start": True}
            implicit_count += 1

        flush_example()
        flush_block()

        def dedupe(values: list[Any]) -> list[Any]:
            output, seen = [], set()
            for value in values:
                key = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
                if key not in seen:
                    seen.add(key)
                    output.append(value)
            return output

        marking_points = dedupe(marking_points)
        acceptable = dedupe(acceptable)
        rejected = dedupe(rejected)
        guidance = dedupe(guidance)
        examples = dedupe(examples)
        block_audit = dedupe(block_audit)

        raw_has_examples = bool(
            PHASE3_EXAMPLE_HEADER_PATTERN.search(raw_text)
            or PHASE3_CORRECT_OUTPUT_PATTERN.search(raw_text)
        )

        confidence = 0.58
        confidence += 0.08 if objectives else 0.0
        confidence += 0.08 if marking_points else 0.0
        confidence += 0.08 if (acceptable or rejected or guidance) else 0.0
        confidence += 0.10 if (not raw_has_examples or examples) else 0.0
        confidence += 0.05 if continuation_count > 0 else 0.0
        confidence += 0.03 if (inline_split > 0 or not PHASE3_SPLIT_INLINE_MARKERS) else 0.0
        confidence -= 0.20 if noise else 0.0
        confidence -= 0.12 if dangling else 0.0
        confidence -= min(0.20, 0.05*len(ambiguous))
        confidence = round(min(0.97, max(0.0, confidence)), 4)

        review: list[str] = []
        if confidence < PHASE3_MIN_RULE_CONFIDENCE:
            review.append("rule_confidence_below_threshold")
        if noise:
            review.append("parser_noise_detected_in_raw_guidance")
        if raw_has_examples and not examples:
            review.append("worked_example_header_not_segmented")
        if dangling:
            review.append("dangling_inline_marker_detected")
        if PHASE3_REVIEW_ON_AMBIGUOUS_BLOCKS and ambiguous:
            review.append("ambiguous_unassigned_mark_scheme_lines")

        return {
            "phase3_cleanup_status": "structured_ready" if not review else "review_recommended",
            "phase3_rule_confidence": confidence,
            "phase3_marking_points": marking_points,
            "phase3_acceptable_answers": acceptable,
            "phase3_rejected_answers": rejected,
            "phase3_additional_guidance": guidance,
            "phase3_worked_examples": examples,
            "phase3_assessment_objectives": objectives,
            "phase3_review_reasons": review,
            "phase3_parser_noise_lines": noise,
            "phase3_raw_guidance_preserved": True,
            "phase3_block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
            "phase3_block_count": len(block_audit),
            "phase3_continuation_lines_merged": continuation_count,
            "phase3_inline_markers_split": inline_split,
            "phase3_implicit_marking_block_count": implicit_count,
            "phase3_ambiguous_lines": ambiguous,
            "phase3_dangling_inline_markers": dangling,
            "phase3_block_audit": block_audit,
        }


    phase3_records = []

    for _, row in final_df.iterrows():
        parsed = parse_mark_scheme_guidance(
            row[
                "marking_guidance"
            ]
        )

        legacy_payload = {
            "marking_points": (
                row["marking_points"]
            ),
            "acceptable_answers": (
                row["acceptable_answers"]
            ),
            "rejected_answers": (
                row["rejected_answers"]
            ),
            "additional_guidance": (
                row["additional_guidance"]
            ),
            "assessment_objectives": (
                row["assessment_objectives"]
            ),
        }

        cleaned_payload = {
            "marking_points": (
                parsed[
                    "phase3_marking_points"
                ]
            ),
            "acceptable_answers": (
                parsed[
                    "phase3_acceptable_answers"
                ]
            ),
            "rejected_answers": (
                parsed[
                    "phase3_rejected_answers"
                ]
            ),
            "additional_guidance": (
                parsed[
                    "phase3_additional_guidance"
                ]
            ),
            "assessment_objectives": (
                parsed[
                    "phase3_assessment_objectives"
                ]
            ),
        }

        parsed[
            "question_id"
        ] = str(
            row["question_id"]
        )

        parsed[
            "mark_scheme_id"
        ] = row[
            "mark_scheme_id"
        ]

        parsed[
            "phase3_legacy_classification_changed"
        ] = bool(
            json.dumps(
                phase3_json_safe(
                    legacy_payload
                ),
                sort_keys=True,
                ensure_ascii=False,
            )
            != json.dumps(
                phase3_json_safe(
                    cleaned_payload
                ),
                sort_keys=True,
                ensure_ascii=False,
            )
        )

        phase3_records.append(
            parsed
        )


    phase3_cleanup_df = pd.DataFrame(
        phase3_records
    )

    final_df = final_df.merge(
        phase3_cleanup_df,
        on=[
            "question_id",
            "mark_scheme_id",
        ],
        how="left",
        validate="one_to_one",
    )


    phase3_review_count = int(
        (
            final_df[
                "phase3_cleanup_status"
            ]
            == "review_recommended"
        ).sum()
    )

    phase3_ready_count = int(
        (
            final_df[
                "phase3_cleanup_status"
            ]
            == "structured_ready"
        ).sum()
    )

    phase3_changes_count = int(
        final_df[
            "phase3_legacy_classification_changed"
        ].sum()
    )

    phase3_mean_confidence = float(
        final_df[
            "phase3_rule_confidence"
        ].mean()
    )


    all_topics_use_actual_chunk_evidence = bool(
        fallback_evidence_topic_count
        == 0
    )


    selected_optional_filter_relaxation_count = int(
        selected_candidates_df[
            "retrieval_relaxation"
        ].fillna("").astype(str).str.contains(
            "optional_filters_relaxed",
            regex=False,
        ).sum()
    )


    release_blockers = list(
        dict.fromkeys(
            selection_summary.get(
                "coverage_release_blockers",
                [],
            )
        )
    )

    if selected_optional_filter_relaxation_count > 0:
        release_blockers.append(
            "optional_retrieval_filters_relaxed"
        )

    if not selection_summary[
        "requested_question_count_met"
    ]:
        release_blockers.append(
            "requested_question_count_not_achieved"
        )

    if selection_summary[
        "semantic_rescue_selected"
    ]:
        release_blockers.append(
            "semantic_rescue_selected"
        )

    if selection_summary[
        "concept_fit_rescue_selected"
    ]:
        release_blockers.append(
            "direct_concept_fit_rescue_selected"
        )

    if (
        REQUIRE_ACTUAL_AGENT1_CHUNK_EVIDENCE_FOR_RELEASE
        and not all_topics_use_actual_chunk_evidence
    ):
        release_blockers.append(
            "actual_agent1_chunk_evidence_not_used"
        )

    if phase3_review_count > 0:
        release_blockers.append(
            "phase3_mark_scheme_review_recommended"
        )


    hard_decision_blockers = {
        "requested_question_count_not_achieved",
        "semantic_rescue_selected",
        "direct_concept_fit_rescue_selected",
        "optional_retrieval_filters_relaxed",
        "minimum_primary_coverage_not_met",
        "minimum_supporting_coverage_not_met",
        "minimum_distinct_reference_coverage_not_met",
        "required_official_reference_not_covered",
        "approved_agent1_topic_not_covered",
    }

    if hard_decision_blockers.intersection(
        release_blockers
    ):
        final_release_status = (
            "needs_user_decision"
        )

    elif release_blockers:
        final_release_status = (
            "evaluation_ready"
        )

    else:
        final_release_status = (
            "ready_for_release"
        )


    selection_summary[
        "assessment_release_status"
    ] = final_release_status

    selection_summary[
        "release_blockers"
    ] = release_blockers

    selection_summary[
        "requires_user_decision"
    ] = bool(
        final_release_status
        == "needs_user_decision"
    )

    selection_summary[
        "all_topics_use_actual_chunk_evidence"
    ] = (
        all_topics_use_actual_chunk_evidence
    )

    selection_summary[
        "phase3_review_count"
    ] = phase3_review_count


    phase3_summary = {
        "phase3_version": PHASE3_VERSION,
        "phase3_enabled": (
            ENABLE_PHASE3_MARK_SCHEME_CLEANUP
        ),
        "raw_guidance_source_of_truth": (
            PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH
        ),
        "selected_mark_schemes": int(
            len(final_df)
        ),
        "structured_ready_count": (
            phase3_ready_count
        ),
        "review_recommended_count": (
            phase3_review_count
        ),
        "legacy_classification_changed_count": (
            phase3_changes_count
        ),
        "mean_rule_confidence": round(
            phase3_mean_confidence,
            4,
        ),
        "block_parser_version": PHASE3_BLOCK_PARSER_VERSION,
        "total_blocks_created": int(final_df["phase3_block_count"].sum()),
        "total_continuation_lines_merged": int(
            final_df["phase3_continuation_lines_merged"].sum()
        ),
        "total_inline_markers_split": int(
            final_df["phase3_inline_markers_split"].sum()
        ),
        "total_ambiguous_lines": int(
            final_df["phase3_ambiguous_lines"].map(len).sum()
        ),
        "actual_agent1_chunk_evidence_used": (
            all_topics_use_actual_chunk_evidence
        ),
        "selected_optional_filter_relaxation_count": (
            selected_optional_filter_relaxation_count
        ),
        "final_release_status": (
            final_release_status
        ),
        "release_blockers": (
            release_blockers
        ),
    }


    phase3_manifest_path = (
        OUTPUT_DIR
        / (
            "agent2_phase3_mark_scheme_cleanup_"
            f"{RUN_TIMESTAMP}.csv"
        )
    )

    phase3_json_path = (
        OUTPUT_DIR
        / (
            "agent2_phase3_mark_scheme_cleanup_"
            f"{RUN_TIMESTAMP}.json"
        )
    )


    phase3_export_df = (
        phase3_cleanup_df.copy()
    )

    for column in [
        "phase3_marking_points",
        "phase3_acceptable_answers",
        "phase3_rejected_answers",
        "phase3_additional_guidance",
        "phase3_worked_examples",
        "phase3_assessment_objectives",
        "phase3_review_reasons",
        "phase3_parser_noise_lines",
        "phase3_ambiguous_lines",
        "phase3_dangling_inline_markers",
        "phase3_block_audit",
    ]:
        phase3_export_df[
            column
        ] = phase3_export_df[
            column
        ].map(
            lambda value: json.dumps(
                phase3_json_safe(value),
                ensure_ascii=False,
            )
        )

    phase3_export_df.to_csv(
        phase3_manifest_path,
        index=False,
    )

    phase3_json_path.write_text(
        json.dumps(
            phase3_json_safe(
                {
                    "summary": (
                        phase3_summary
                    ),
                    "records": (
                        phase3_cleanup_df
                        .to_dict(
                            orient="records"
                        )
                    ),
                }
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    display(
        pd.DataFrame(
            [phase3_summary]
        )
    )

    display(
        final_df[
            [
                "selected_rank",
                "question_id",
                "mark_scheme_id",
                "phase3_cleanup_status",
                "phase3_rule_confidence",
                "phase3_block_parser_version",
                "phase3_block_count",
                "phase3_continuation_lines_merged",
                "phase3_inline_markers_split",
                "phase3_implicit_marking_block_count",
                "phase3_ambiguous_lines",
                "phase3_legacy_classification_changed",
                "phase3_review_reasons",
                "phase3_marking_points",
                "phase3_acceptable_answers",
                "phase3_rejected_answers",
                "phase3_additional_guidance",
                "phase3_worked_examples",
                "phase3_assessment_objectives",
            ]
        ]
    )

    print(
        "Final assessment release status: "
        f"{final_release_status}"
    )

    print(
        "Release blockers: "
        f"{release_blockers}"
    )

    print(
        "Phase 3 manifest saved:"
    )
    print(phase3_manifest_path)
    print(phase3_json_path)


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    # Phase 3 block-aware parser validation
    wrapped_guidance_test = parse_mark_scheme_guidance(
        """
        Mark is for AO2 (apply)
        The first iteration structure processes the rows;
        Note to examiners: award both marks if the student
        answers are correct but the opposite way around,
        and rows are given for the second answer
        """
    )
    expected_guidance = (
        "Note to examiners: award both marks if the student "
        "answers are correct but the opposite way around, "
        "and rows are given for the second answer"
    )

    inline_marker_test = parse_mark_scheme_guidance(
        """
        Mark F for resetting pos to 0; A. if the index could go out of range.
        Mark D for checking the input; I.
        data validation attempts
        """
    )

    phase3_block_parser_validation = {
        "wrapped_guidance_merged_correctly": (
            expected_guidance in wrapped_guidance_test["phase3_additional_guidance"]
        ),
        "wrapped_guidance_not_in_marking_points": all(
            "answers are correct but the opposite way around" not in value
            for value in wrapped_guidance_test["phase3_marking_points"]
        ),
        "inline_accept_split_correctly": any(
            "if the index could go out of range" in value.lower()
            for value in inline_marker_test["phase3_acceptable_answers"]
        ),
        "trailing_guidance_marker_merged": any(
            "data validation attempts" in value.lower()
            for value in inline_marker_test["phase3_additional_guidance"]
        ),
    }
    display(pd.DataFrame([
        {"check": key, "passed": bool(value)}
        for key, value in phase3_block_parser_validation.items()
    ]))
    if not all(phase3_block_parser_validation.values()):
        raise RuntimeError("Phase 3 block-aware parser validation failed.")
    print("Phase 3 block-aware parser validation: passed")

## 13A. Group selected subquestions by source parent

Subquestions from the same source parent may share one figure, table or scenario.
The display layer groups them using only:

```text
source question document
normalized parent part of question_number
```

For example, source numbers `04.1` and `04.2` share parent `04`. The retrieval,
marks and mark-scheme records remain individual; grouping changes only the
student/teacher presentation and prevents repeated shared context.


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    # ================================================================
    # SOURCE-PARENT SUBQUESTION GROUPING
    # ================================================================


    def normalized_parent_question_number(value: Any) -> str:
        text_value = str(value or "").strip()
        numeric_parts = re.findall(r"\d+", text_value)

        if len(numeric_parts) >= 2:
            return ".".join(numeric_parts[:-1])

        # A question without a subpart separator remains its own parent.
        return text_value or "unlabelled"


    def alphabetic_part_label(position: int) -> str:
        value = position
        output = ""
        while True:
            value, remainder = divmod(value, 26)
            output = chr(ord("a") + remainder) + output
            if value == 0:
                return output
            value -= 1


    final_df["source_parent_question_number"] = (
        final_df["question_number_postgres"].map(
            normalized_parent_question_number
        )
    )
    final_df["source_parent_group_key"] = final_df.apply(
        lambda row: (
            f"{row.get('question_document_id')}::"
            f"{row.get('source_parent_question_number')}"
        ),
        axis=1,
    )

    # Only repeated parent keys become a multi-part presentation group.
    group_sizes = final_df.groupby(
        "source_parent_group_key"
    )["question_id"].transform("count")
    final_df["is_shared_parent_group"] = bool(
        GROUP_SHARED_PARENT_SUBQUESTIONS
    ) & (group_sizes > 1)

    presentation_group_keys = []
    for _, row in final_df.iterrows():
        if bool(row["is_shared_parent_group"]):
            presentation_group_keys.append(
                row["source_parent_group_key"]
            )
        else:
            presentation_group_keys.append(
                f"single::{row['question_id']}"
            )
    final_df["presentation_group_key"] = presentation_group_keys

    ordered_group_keys = (
        final_df.groupby("presentation_group_key")["selected_rank"]
        .min()
        .sort_values()
        .index
        .tolist()
    )
    group_rank_lookup = {
        key: position + 1
        for position, key in enumerate(ordered_group_keys)
    }
    final_df["display_group_rank"] = final_df[
        "presentation_group_key"
    ].map(group_rank_lookup)

    part_labels = {}
    for group_key, group_rows in final_df.groupby(
        "presentation_group_key",
        sort=False,
    ):
        ordered = group_rows.sort_values(
            ["question_number_postgres", "selected_rank"]
        )
        if len(ordered) == 1:
            part_labels[str(ordered.iloc[0]["question_id"])] = None
        else:
            for position, (_, member) in enumerate(ordered.iterrows()):
                part_labels[str(member["question_id"])] = (
                    alphabetic_part_label(position)
                )

    final_df["display_part_label"] = final_df["question_id"].map(
        part_labels
    )

    question_group_records = []
    for group_key in ordered_group_keys:
        group_rows = final_df[
            final_df["presentation_group_key"] == group_key
        ].sort_values(
            ["question_number_postgres", "selected_rank"]
        )

        unique_contexts = [
            value
            for value in dict.fromkeys(
                str(value or "").strip()
                for value in group_rows["context_text"].tolist()
            )
            if value
        ]

        image_paths = []
        dependency_image_paths = []
        question_crop_paths = []
        full_page_paths = []

        for _, member in group_rows.iterrows():
            # Required figures/tables are deliberately placed before the
            # selected question crop in the student assessment.
            for path_value in (
                member.get("dependency_crop_images")
                if isinstance(member.get("dependency_crop_images"), list)
                else []
            ):
                if path_value not in dependency_image_paths:
                    dependency_image_paths.append(path_value)
                if path_value not in image_paths:
                    image_paths.append(path_value)

            for path_value in (
                member.get("cropped_question_images")
                if isinstance(member.get("cropped_question_images"), list)
                else []
            ):
                if path_value not in question_crop_paths:
                    question_crop_paths.append(path_value)
                if path_value not in image_paths:
                    image_paths.append(path_value)

            # Safe fallback for renderer records that contain only the combined
            # compatibility field.
            if not (
                isinstance(member.get("dependency_crop_images"), list)
                or isinstance(member.get("cropped_question_images"), list)
            ):
                for path_value in (
                    member.get("rendered_page_images")
                    if isinstance(member.get("rendered_page_images"), list)
                    else []
                ):
                    if path_value not in image_paths:
                        image_paths.append(path_value)

            for path_value in (
                member.get("full_page_images")
                if isinstance(member.get("full_page_images"), list)
                else []
            ):
                if path_value not in full_page_paths:
                    full_page_paths.append(path_value)

        question_group_records.append(
            {
                "presentation_group_key": group_key,
                "display_group_rank": int(
                    group_rows["display_group_rank"].iloc[0]
                ),
                "source_parent_question_number": (
                    group_rows[
                        "source_parent_question_number"
                    ].iloc[0]
                ),
                "is_shared_parent_group": bool(
                    group_rows["is_shared_parent_group"].iloc[0]
                ),
                "member_question_ids": (
                    group_rows["question_id"].astype(str).tolist()
                ),
                "member_source_numbers": (
                    group_rows[
                        "question_number_postgres"
                    ].astype(str).tolist()
                ),
                "group_total_marks": int(
                    group_rows["marks_postgres"].sum()
                ),
                "shared_contexts": unique_contexts,
                "dependency_image_paths": dependency_image_paths,
                "question_crop_paths": question_crop_paths,
                "student_image_paths": image_paths,
                "teacher_full_page_paths": full_page_paths,
            }
        )

    question_group_manifest_df = pd.DataFrame(
        question_group_records
    ).sort_values("display_group_rank")

    question_group_summary = {
        "version": QUESTION_GROUPING_VERSION,
        "selected_question_records": int(len(final_df)),
        "presentation_groups": int(len(question_group_manifest_df)),
        "shared_parent_groups": int(
            question_group_manifest_df[
                "is_shared_parent_group"
            ].sum()
        ),
    }

    display(pd.DataFrame([question_group_summary]))
    display(question_group_manifest_df)

In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    # ============================================================
    # RELEASE COMPLETENESS / SAFE-POOL EXHAUSTION SUMMARY
    # Add this AFTER final_df / selected_candidates_df is ready
    # and BEFORE JSON / TXT / MD / PDF export cells
    # ============================================================

    import pandas as pd
    from copy import deepcopy

    def _first_existing_column(df, candidates, default=None):
        for c in candidates:
            if c in df.columns:
                return c
        return default

    def build_release_completeness_summary(
        selected_df: pd.DataFrame,
        request_config: dict,
        safe_pool_df: pd.DataFrame | None = None,
    ):
        selected_df = selected_df.copy()

        requested_questions = int(request_config.get("number_of_questions", 0) or 0)
        marks_col = _first_existing_column(
            selected_df,
            ["marks", "maximum_marks", "max_marks", "question_marks"],
            default=None,
        )

        released_questions = int(len(selected_df))
        released_marks = (
            int(pd.to_numeric(selected_df[marks_col], errors="coerce").fillna(0).sum())
            if marks_col is not None and not selected_df.empty
            else 0
        )

        safe_pool_questions = released_questions
        safe_pool_marks = released_marks

        if safe_pool_df is not None and not safe_pool_df.empty:
            safe_pool_df = safe_pool_df.copy()
            safe_marks_col = _first_existing_column(
                safe_pool_df,
                ["marks", "maximum_marks", "max_marks", "question_marks"],
                default=None,
            )
            safe_pool_questions = int(len(safe_pool_df))
            safe_pool_marks = (
                int(pd.to_numeric(safe_pool_df[safe_marks_col], errors="coerce").fillna(0).sum())
                if safe_marks_col is not None
                else 0
            )

        unmet_requirements = []

        if released_questions < requested_questions:
            unmet_requirements.append("requested_question_count_not_met")

        # Distinguish between "pool exhausted" vs "combination not found".
        # Total marks are informational only and do not affect completeness.
        if unmet_requirements:
            if safe_pool_questions < requested_questions:
                reason = "safe candidate pool exhausted after all verification gates"
            else:
                reason = "no quality-safe combination satisfied all approved constraints"
        else:
            reason = "request fully satisfied"

        if released_questions == 0:
            release_status = "no_safe_release"
        elif unmet_requirements:
            release_status = "partial_safe_release"
        else:
            release_status = "full_release"

        summary = {
            "requested_questions": requested_questions,
            "released_questions": released_questions,
            "released_marks": released_marks,
            "safe_pool_questions_available": safe_pool_questions,
            "safe_pool_marks_available": safe_pool_marks,
            "safe_pool_exhausted": bool(unmet_requirements),
            "unmet_requirements": unmet_requirements,
            "release_status": release_status,
            "reason": reason,
        }

        return summary


    # ------------------------------------------------------------------
    # Figure out which dataframe should be treated as the final safe pool
    # Use the most downstream verified pool available in the notebook.
    # ------------------------------------------------------------------
    SAFE_POOL_DF = None

    for candidate_name in [
        "final_verified_candidates_df",
        "verified_candidates_df",
        "enforced_candidates_df",
        "refill_safe_pool_df",
        "phase2_candidates_df",
    ]:
        if candidate_name in globals():
            candidate_obj = globals()[candidate_name]
            if isinstance(candidate_obj, pd.DataFrame):
                SAFE_POOL_DF = candidate_obj.copy()
                print(f"Using safe pool dataframe: {candidate_name}")
                break

    # ------------------------------------------------------------------
    # Figure out which dataframe is the final selected release
    # ------------------------------------------------------------------
    FINAL_SELECTED_DF = None

    for selected_name in [
        "final_df",
        "selected_candidates_df",
        "selected_df",
    ]:
        if selected_name in globals():
            selected_obj = globals()[selected_name]
            if isinstance(selected_obj, pd.DataFrame):
                FINAL_SELECTED_DF = selected_obj.copy()
                print(f"Using selected dataframe: {selected_name}")
                break

    if FINAL_SELECTED_DF is None:
        raise RuntimeError("Could not locate final selected dataframe for release summary.")

    # ------------------------------------------------------------------
    # Figure out assessment request object
    # ------------------------------------------------------------------
    REQUEST_CONFIG = None

    for request_name in [
        "ASSESSMENT_REQUEST",
        "assessment_request",
        "REQUEST_JSON",
        "request_config",
    ]:
        if request_name in globals():
            request_obj = globals()[request_name]
            if isinstance(request_obj, dict):
                REQUEST_CONFIG = deepcopy(request_obj)
                print(f"Using request config: {request_name}")
                break

    if REQUEST_CONFIG is None:
        raise RuntimeError("Could not locate assessment request config.")

    # ------------------------------------------------------------------
    # Build summary
    # ------------------------------------------------------------------
    release_completeness_summary = build_release_completeness_summary(
        selected_df=FINAL_SELECTED_DF,
        request_config=REQUEST_CONFIG,
        safe_pool_df=SAFE_POOL_DF,
    )

    print("\n" + "=" * 70)
    print("RELEASE COMPLETENESS SUMMARY")
    print("=" * 70)
    for k, v in release_completeness_summary.items():
        print(f"{k}: {v}")

    release_summary_df = pd.DataFrame(
        [
            {"metric": "requested_questions", "value": release_completeness_summary["requested_questions"]},
            {"metric": "released_questions", "value": release_completeness_summary["released_questions"]},
            {"metric": "released_marks", "value": release_completeness_summary["released_marks"]},
            {"metric": "safe_pool_questions_available", "value": release_completeness_summary["safe_pool_questions_available"]},
            {"metric": "safe_pool_marks_available", "value": release_completeness_summary["safe_pool_marks_available"]},
            {"metric": "safe_pool_exhausted", "value": release_completeness_summary["safe_pool_exhausted"]},
            {"metric": "release_status", "value": release_completeness_summary["release_status"]},
            {"metric": "reason", "value": release_completeness_summary["reason"]},
            {"metric": "unmet_requirements", "value": ", ".join(release_completeness_summary["unmet_requirements"]) or "None"},
        ]
    )

    display(release_summary_df)

## 14. Build and export the final assessment package

Outputs:

```text
retrieval candidates CSV
selected questions CSV
complete assessment JSON
readable assessment Markdown
plain-text evaluation report
```


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    def json_safe(value: Any) -> Any:
        if value is None:
            return None

        if isinstance(
            value,
            (
                bool,
                int,
                str,
            ),
        ):
            return value

        if isinstance(value, float):
            return (
                value
                if math.isfinite(value)
                else None
            )

        if isinstance(value, datetime):
            return value.isoformat()

        if isinstance(value, uuid.UUID):
            return str(value)

        if isinstance(value, dict):
            return {
                str(key): json_safe(item)
                for key, item
                in value.items()
            }

        if isinstance(
            value,
            (
                list,
                tuple,
                set,
            ),
        ):
            return [
                json_safe(item)
                for item in value
            ]

        if hasattr(value, "item"):
            try:
                return json_safe(
                    value.item()
                )
            except (
                TypeError,
                ValueError,
            ):
                pass

        try:
            if pd.isna(value):
                return None
        except (
            TypeError,
            ValueError,
        ):
            pass

        return str(value)


    question_packages = []

    for _, row in final_df.iterrows():
        question_packages.append(
            {
                "rank": int(
                    row["selected_rank"]
                ),
                "question_id": (
                    row["question_id"]
                ),
                "topic": {
                    "detected_topic": (
                        row["detected_topic"]
                    ),
                    "detected_concepts": (
                        row.get(
                            "detected_concepts",
                            [],
                        )
                    ),
                    "role": (
                        row["agent1_role"]
                    ),
                    "official_reference": (
                        row[
                            "official_reference_postgres"
                        ]
                    ),
                    "official_concept_name": (
                        row[
                            "official_concept_name_postgres"
                        ]
                    ),
                    "section_reference": (
                        row[
                            "official_section_reference_postgres"
                        ]
                    ),
                    "pmt_subtopic_code": (
                        row["pmt_subtopic_code"]
                    ),
                    "pmt_subtopic_name": (
                        row["pmt_subtopic_name"]
                    ),
                },
                "presentation": {
                    "grouping_version": QUESTION_GROUPING_VERSION,
                    "display_group_rank": int(row["display_group_rank"]),
                    "display_part_label": row["display_part_label"],
                    "source_parent_question_number": row[
                        "source_parent_question_number"
                    ],
                    "is_shared_parent_group": bool(
                        row["is_shared_parent_group"]
                    ),
                },
                "question": {
                    "number": (
                        row[
                            "question_number_postgres"
                        ]
                    ),
                    "text": (
                        row[
                            "question_text_postgres"
                        ]
                    ),
                    "context": row["context_text"],
                    "original_context": row["context_text_original"],
                    "context_refinement": {
                        "version": CONTEXT_REFINEMENT_VERSION,
                        "status": row["context_refinement_status"],
                        "removed_segments": row[
                            "context_removed_segments"
                        ],
                        "segment_scores": row["context_segment_scores"],
                    },
                    "marks": int(
                        row["marks_postgres"]
                    ),
                    "has_code": bool(
                        row["has_code_postgres"]
                    ),
                    "has_visual": bool(
                        row["has_visual_postgres"]
                    ),
                    "paper_code": (
                        row["paper_code_postgres"]
                    ),
                    "programming_language": (
                        row["programming_language_postgres"]
                    ),
                    "source_pdf_path": (
                        row[
                            "resolved_source_pdf_path"
                        ]
                    ),
                    "source_page_numbers": (
                        row[
                            "source_page_numbers"
                        ]
                    ),
                    "rendered_page_images": row[
                        "rendered_page_images"
                    ],
                    "cropped_question_images": row[
                        "cropped_question_images"
                    ],
                    "full_page_images": row["full_page_images"],
                    "question_region_crop": {
                        "version": QUESTION_REGION_CROP_VERSION,
                        "status": row["question_region_crop_status"],
                        "methods": row["question_region_crop_method"],
                        "rectangles": row[
                            "question_region_crop_rectangles"
                        ],
                    },
                    "visual_render_status": (
                        row[
                            "visual_render_status"
                        ]
                    ),
                    "visual_render_error": (
                        row[
                            "visual_render_error"
                        ]
                    ),
                },
                "mark_scheme": {
                    "id": str(
                        row["mark_scheme_id"]
                    ),
                    "maximum_marks": int(
                        row["maximum_marks"]
                    ),
                    "marking_guidance": (
                        row[
                            "marking_guidance"
                        ]
                    ),
                    "raw_marking_guidance": (
                        row[
                            "marking_guidance"
                        ]
                    ),
                    "legacy_structured_fields": {
                        "marking_points": (
                            row[
                                "marking_points"
                            ]
                        ),
                        "acceptable_answers": (
                            row[
                                "acceptable_answers"
                            ]
                        ),
                        "rejected_answers": (
                            row[
                                "rejected_answers"
                            ]
                        ),
                        "additional_guidance": (
                            row[
                                "additional_guidance"
                            ]
                        ),
                        "assessment_objectives": (
                            row[
                                "assessment_objectives"
                            ]
                        ),
                    },
                    "phase3_structured": {
                        "version": (
                            PHASE3_VERSION
                        ),
                        "cleanup_status": (
                            row[
                                "phase3_cleanup_status"
                            ]
                        ),
                        "rule_confidence": float(
                            row[
                                "phase3_rule_confidence"
                            ]
                        ),
                        "marking_points": (
                            row[
                                "phase3_marking_points"
                            ]
                        ),
                        "acceptable_answers": (
                            row[
                                "phase3_acceptable_answers"
                            ]
                        ),
                        "rejected_answers": (
                            row[
                                "phase3_rejected_answers"
                            ]
                        ),
                        "additional_guidance": (
                            row[
                                "phase3_additional_guidance"
                            ]
                        ),
                        "worked_examples": (
                            row[
                                "phase3_worked_examples"
                            ]
                        ),
                        "assessment_objectives": (
                            row[
                                "phase3_assessment_objectives"
                            ]
                        ),
                        "review_reasons": (
                            row[
                                "phase3_review_reasons"
                            ]
                        ),
                        "block_parser_version": row["phase3_block_parser_version"],
                        "block_count": int(row["phase3_block_count"]),
                        "continuation_lines_merged": int(
                            row["phase3_continuation_lines_merged"]
                        ),
                        "inline_markers_split": int(
                            row["phase3_inline_markers_split"]
                        ),
                        "implicit_marking_block_count": int(
                            row["phase3_implicit_marking_block_count"]
                        ),
                        "ambiguous_lines": row["phase3_ambiguous_lines"],
                        "block_audit": row["phase3_block_audit"],
                        "raw_guidance_preserved": bool(
                            row[
                                "phase3_raw_guidance_preserved"
                            ]
                        ),
                        "legacy_classification_changed": bool(
                            row[
                                "phase3_legacy_classification_changed"
                            ]
                        ),
                    },
                },
                "retrieval": {
                    "agent1_topic_index": int(
                        row["agent1_topic_index"]
                    ),
                    "stage": (
                        row["retrieval_stage"]
                    ),
                    "backend": (
                        row.get(
                            "retrieval_backend"
                        )
                    ),
                    "relaxation": (
                        row.get(
                            "retrieval_relaxation"
                        )
                    ),
                    "semantic_score": float(row["semantic_score"]),
                    "direct_concept_score": float(
                        row["direct_concept_score"]
                    ),
                    "combined_concept_score": float(
                        row["combined_concept_score"]
                    ),
                    "direct_concept_threshold": float(
                        row["direct_concept_threshold"]
                    ),
                    "concept_fit_mark_margin": float(
                        row["concept_fit_mark_margin"]
                    ),
                    "concept_fit_gate_passed": bool(
                        row["concept_fit_gate_passed"]
                    ),
                    "concept_fit_rescue_used": bool(
                        row["concept_fit_rescue_used"]
                    ),
                    "final_score": float(
                        row["final_score"]
                    ),
                    "memory_adjusted_final_score": float(
                        row.get(
                            "memory_adjusted_final_score",
                            row["final_score"],
                        )
                    ),
                    "phase2_rank": int(
                        row.get("phase2_rank") or 0
                    ),
                    "phase4_rank": int(
                        row.get("phase4_rank")
                        or row.get("phase2_rank")
                        or 0
                    ),
                    "agent1_confidence": float(
                        row[
                            "agent1_confidence"
                        ]
                    ),
                    "source_chunks": (
                        row["source_chunks"]
                    ),
                    "query_evidence_source": (
                        row["query_evidence_source"]
                    ),
                    "query_evidence": (
                        row["query_evidence"]
                    ),
                    "phase2_version": PHASE2_VERSION,
                    "phase2_semantic_threshold": (
                        row["phase2_semantic_threshold"]
                    ),
                    "concept_gate_passed": bool(
                        row["concept_gate_passed"]
                    ),
                    "question_quality_gate_passed": bool(
                        row[
                            "question_quality_gate_passed"
                        ]
                    ),
                    "phase2_gate_passed": bool(
                        row["phase2_gate_passed"]
                    ),
                    "phase2_gate_mode": (
                        row["phase2_gate_mode"]
                    ),
                    "semantic_rescue_used": bool(
                        row[
                            "semantic_rescue_used"
                        ]
                    ),
                    "question_quality_issues": (
                        row["question_quality_issues"]
                    ),
                    "phase2_rejection_reasons": (
                        row["phase2_rejection_reasons"]
                    ),
                    "qp_ms_match_method": (
                        row["match_method"]
                    ),
                    "qp_ms_match_confidence": (
                        float(
                            row[
                                "match_confidence"
                            ]
                        )
                    ),
                    "memory_recall_phase3": json_safe(
                        row.get(
                            "memory_recall_phase3",
                            {
                                "version": RETRIEVAL_MEMORY_PHASE3_VERSION,
                                "matched": False,
                                "recall_status": "not_available",
                                "ranking_adjustment_enabled": False,
                                "ranking_adjustment": 0.0,
                                "base_final_score": float(row["final_score"]),
                                "score_after_memory": float(row["final_score"]),
                                "rank_changed": False,
                            },
                        )
                    ),
                    "memory_ranking_phase4": json_safe(
                        row.get(
                            "memory_ranking_phase4",
                            {
                                "version": RETRIEVAL_MEMORY_PHASE4_VERSION,
                                "enabled": True,
                                "applied": False,
                                "status": "not_available",
                                "ranking_adjustment": 0.0,
                                "base_final_score": float(row["final_score"]),
                                "adjusted_final_score": float(
                                    row.get(
                                        "memory_adjusted_final_score",
                                        row["final_score"],
                                    )
                                ),
                                "phase2_rank": int(
                                    row.get("phase2_rank") or 0
                                ),
                                "phase4_rank": int(
                                    row.get("phase4_rank")
                                    or row.get("phase2_rank")
                                    or 0
                                ),
                                "rank_changed": False,
                            },
                        )
                    ),
                },
            }
        )


    retrieval_summary = {
        "run_status": AGENT2_RUN_STATUS,
        "retrieval_memory_phase3": {
            "version": RETRIEVAL_MEMORY_PHASE3_VERSION,
            "enabled": True,
            "memory_collection": RETRIEVAL_MEMORY_COLLECTION,
            "memory_collection_points": int(RETRIEVAL_MEMORY_POINT_COUNT),
            "candidates_checked": int(len(phase2_candidates_df)),
            "candidate_compatible_matches": int(
                phase2_candidates_df["memory_match_found"].sum()
            ),
            "selected_compatible_matches": int(
                final_df.get(
                    "memory_match_found",
                    pd.Series(dtype=bool),
                ).fillna(False).astype(bool).sum()
            ),
            "ranking_adjustment_enabled": False,
            "compatibility_only": True,
        },
        "retrieval_memory_phase4": {
            "version": RETRIEVAL_MEMORY_PHASE4_VERSION,
            "enabled": True,
            "max_positive_boost": RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST,
            "max_negative_penalty": RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY,
            "candidates_checked": int(len(phase2_candidates_df)),
            "candidate_adjustments_applied": int(
                phase2_candidates_df[
                    "memory_adjustment_applied"
                ].fillna(False).astype(bool).sum()
            ),
            "candidate_positive_boosts": int(
                (
                    phase2_candidates_df[
                        "memory_rank_adjustment"
                    ].fillna(0.0).astype(float)
                    > 0.0
                ).sum()
            ),
            "candidate_negative_penalties": int(
                (
                    phase2_candidates_df[
                        "memory_rank_adjustment"
                    ].fillna(0.0).astype(float)
                    < 0.0
                ).sum()
            ),
            "candidate_exact_context_suppressions": int(
                phase2_candidates_df.get(
                    "memory_hard_suppressed",
                    pd.Series(False, index=phase2_candidates_df.index),
                ).fillna(False).astype(bool).sum()
            ),
            "suppressed_question_ids": json_safe(
                selection_summary.get(
                    "phase4_suppressed_question_ids",
                    [],
                )
            ),
            "selected_adjustments_applied": int(
                final_df.get(
                    "memory_adjustment_applied",
                    pd.Series(False, index=final_df.index),
                ).fillna(False).astype(bool).sum()
            ),
            "selected_positive_boosts": int(
                (
                    final_df.get(
                        "memory_rank_adjustment",
                        pd.Series(0.0, index=final_df.index),
                    ).fillna(0.0).astype(float)
                    > 0.0
                ).sum()
            ),
            "selected_negative_penalties": int(
                (
                    final_df.get(
                        "memory_rank_adjustment",
                        pd.Series(0.0, index=final_df.index),
                    ).fillna(0.0).astype(float)
                    < 0.0
                ).sum()
            ),
            "global_diagnostic_rank_changes": int(
                (
                    phase2_candidates_df[
                        "phase4_rank"
                    ].astype(int)
                    != phase2_candidates_df[
                        "phase2_rank"
                    ].astype(int)
                ).sum()
            ),
            "selection_uses_memory_adjusted_score": True,
            "baseline_final_score_preserved": True,
        },
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
        "syllabus_paper_check": json_safe(
            AGENT2_SYLLABUS_PAPER_CHECK
        ),
        "unresolved_agent1_topics": json_safe(
            unknown_topics_df.to_dict(orient="records")
            if isinstance(unknown_topics_df, pd.DataFrame)
            else []
        ),
        "topic_question_availability": json_safe(
            topic_question_availability_df.to_dict(orient="records")
            if "topic_question_availability_df" in globals()
            and isinstance(topic_question_availability_df, pd.DataFrame)
            else []
        ),
        "visual_rendering_version": (
            VISUAL_RENDERING_VERSION
        ),
        "selected_visual_questions": (
            selected_visual_questions
        ),
        "visual_questions_rendered": (
            successfully_rendered_visual_questions
        ),
        "visual_render_failures": (
            visual_render_failures
        ),
        "rendered_page_images": (
            rendered_page_total
        ),
        **phase2_gate_summary,
        "raw_candidates": len(
            all_candidates_df
        ),
        "duplicates_removed": (
            duplicates_removed
        ),
        "near_duplicates_removed": (
            near_duplicates_removed
        ),
        "unique_candidates": len(
            unique_candidates_df
        ),
        "refined_unique_candidates": len(
            near_unique_candidates_df
        ),
        **selection_summary,
        **phase3_summary,
    }


    assessment_package = json_safe(
        {
            "generated_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
            "retrieval_version": (
                RETRIEVAL_VERSION
            ),
            "specification": {
                "code": SPECIFICATION_CODE,
                "version": (
                    SPECIFICATION_VERSION
                ),
            },
            "embedding_model": MODEL_NAME,
            "qdrant_collection": (
                AGENT2_COLLECTION
            ),
            "retrieval_hitl": {
                "phase_version": RETRIEVAL_MEMORY_PHASE4_VERSION,
                "capture_enabled": True,
                "postgres_storage": "agent2_retrieval_feedback",
                "qdrant_memory_enabled": True,
                "qdrant_memory_collection_env": (
                    "AGENT2_RETRIEVAL_MEMORY_COLLECTION"
                ),
                "qdrant_memory_default": (
                    f"{AGENT2_COLLECTION}_retrieval_memory"
                ),
                "memory_embedding_model": MODEL_NAME,
                "memory_vector_size": 384,
                "phase3_compatibility_guard_enabled": True,
                "ranking_adjustment_enabled": True,
                "max_positive_boost": RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST,
                "max_negative_penalty": RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY,
                "baseline_final_score_preserved": True,
                "exact_context_not_relevant_suppression_enabled": True,
                "suppression_policy": (
                    "Exact same question + exact same lesson evidence + Not Relevant "
                    "is excluded from final selection for that lesson only. Broader "
                    "compatible negative memory receives only the bounded penalty."
                ),
                "note": (
                    "Phase 2 stores contextual human feedback. Phase 3 recalls and "
                    "validates compatible memory. Phase 4 boosts compatible Relevant "
                    "memory, penalises broader compatible Not Relevant memory, and "
                    "suppresses exact-context Not Relevant questions from final selection. "
                    "Different/incompatible lesson contexts receive no effect."
                ),
            },
            "syllabus_paper_check": json_safe(
                AGENT2_SYLLABUS_PAPER_CHECK
            ),
            "retrieval_recovery": {
                "strategy": (
                    "qdrant_strict_then_compatibility_"
                    "then_postgresql_local_minilm"
                ),
                "events": (
                    retrieval_recovery_df
                    .to_dict(
                        orient="records"
                    )
                ),
            },
            "phase_1_visual_rendering": {
                "version": (
                    VISUAL_RENDERING_VERSION
                ),
                "strategy": (
                    "notebook7_verified_question_crop_with_multipage_and_safe_fallback"
                ),
                "enabled": (
                    ENABLE_VISUAL_PAGE_RENDERING
                ),
                "render_dpi": (
                    VISUAL_RENDER_DPI
                ),
                "manifest": (
                    visual_manifest_path
                    .relative_to(
                        OUTPUT_DIR
                    )
                    .as_posix()
                ),
                "crop_version": QUESTION_REGION_CROP_VERSION,
                "question_regions_cropped": visual_crop_success_count,
                "full_page_fallbacks": visual_full_page_fallback_count,
                "known_limitation": (
                    "Notebook 07 retains the assigned source page(s) when a reliable "
                    "question boundary cannot be detected. Multi-page questions "
                    "are rendered as ordered page crops."
                ),
            },
            "phase_2_concept_quality_gate": {
                "version": PHASE2_VERSION,
                "refinement_version": (
                    PHASE2_REFINEMENT_VERSION
                ),
                "near_duplicate_gate": {
                    "enabled": (
                        ENABLE_NEAR_DUPLICATE_GATE
                    ),
                    "lexical_threshold": (
                        NEAR_DUPLICATE_LEXICAL_THRESHOLD
                    ),
                    "token_jaccard_threshold": (
                        NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
                    ),
                    "semantic_threshold": (
                        NEAR_DUPLICATE_SEMANTIC_THRESHOLD
                    ),
                    "near_duplicates_removed": (
                        near_duplicates_removed
                    ),
                },
                "threshold_strategy": (
                    "per_topic_adaptive"
                ),
                "adaptive_threshold_version": (
                    ADAPTIVE_THRESHOLD_VERSION
                ),
                "legacy_fixed_threshold_history": {
                    "strict_threshold": (
                        LEGACY_FIXED_STRICT_THRESHOLD
                    ),
                    "relaxed_threshold": (
                        LEGACY_FIXED_RELAXED_THRESHOLD
                    ),
                    "active": (
                        LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
                    ),
                },
                "adaptive_configuration": {
                    "score_percentile": (
                        ADAPTIVE_SCORE_PERCENTILE
                    ),
                    "top_score_margin": (
                        ADAPTIVE_TOP_SCORE_MARGIN
                    ),
                    "absolute_minimum_score": (
                        ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
                    ),
                    "maximum_allowed_threshold": (
                        ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
                    ),
                },
                "adaptive_threshold_summary": {
                    "minimum": (
                        adaptive_threshold_min
                    ),
                    "maximum": (
                        adaptive_threshold_max
                    ),
                    "mean": (
                        adaptive_threshold_mean
                    ),
                    "pool_sufficient_without_rescue": (
                        adaptive_pool_sufficient
                    ),
                },
                "quality_safe_rescue": {
                    "enabled": (
                        ENABLE_QUALITY_SAFE_TOPIC_RESCUE
                    ),
                    "used": (
                        phase2_rescue_used
                    ),
                    "minimum_semantic_score": (
                        MIN_TOPIC_RESCUE_SEMANTIC_SCORE
                    ),
                    "selected_rescue_candidate": (
                        selection_summary[
                            "semantic_rescue_selected"
                        ]
                    ),
                },
                "quality_gate_enabled": (
                    ENABLE_QUESTION_TEXT_QUALITY_GATE
                ),
                "minimum_question_word_count": (
                    MIN_QUESTION_WORD_COUNT
                ),
                "known_limitations": [
                    (
                        "Threshold remains provisional until "
                        "Notebook 06 evaluation."
                    ),
                    (
                        "Lesson summary is used when actual "
                        "Agent 1 chunk text is unavailable."
                    ),
                ],
            },
            "context_refinement": {
                "version": CONTEXT_REFINEMENT_VERSION,
                "summary": context_refinement_summary,
            },
            "question_grouping": {
                "version": QUESTION_GROUPING_VERSION,
                "summary": question_group_summary,
            },
            "phase_3_mark_scheme_cleanup": {
                "version": (
                    PHASE3_VERSION
                ),
                "enabled": (
                    ENABLE_PHASE3_MARK_SCHEME_CLEANUP
                ),
                "raw_guidance_source_of_truth": (
                    PHASE3_RAW_GUIDANCE_IS_SOURCE_OF_TRUTH
                ),
                "minimum_rule_confidence": (
                    PHASE3_MIN_RULE_CONFIDENCE
                ),
                "manifest": (
                    phase3_manifest_path
                    .relative_to(
                        OUTPUT_DIR
                    )
                    .as_posix()
                ),
                "json_report": (
                    phase3_json_path
                    .relative_to(
                        OUTPUT_DIR
                    )
                    .as_posix()
                ),
                "summary": (
                    phase3_summary
                ),
            },
            "lesson_summary": (
                LESSON_SUMMARY
            ),
            "assessment_request": request,
            "agent1_topics": (
                validated_topics_df[
                    [
                        "detected_topic",
                        "detected_concepts",
                        "source_detected_topic_count",
                        "role",
                        "official_reference",
                        "confidence",
                        "ranking_score",
                        "source_chunks",
                        "query_evidence_source",
                        "query_evidence_item_count",
                    ]
                ]
                .to_dict(
                    orient="records"
                )
            ),
            "retrieval_summary": (
                retrieval_summary
            ),
            "question_groups": json_safe(
                question_group_manifest_df.to_dict(orient="records")
            ),
            "questions": question_packages,
        }
    )


    timestamp = RUN_TIMESTAMP

    candidates_path = (
        OUTPUT_DIR
        / f"agent2_retrieval_candidates_{timestamp}.csv"
    )

    selected_path = (
        OUTPUT_DIR
        / f"agent2_selected_questions_{timestamp}.csv"
    )

    package_path = (
        OUTPUT_DIR
        / f"agent2_assessment_package_{timestamp}.json"
    )

    markdown_path = (
        OUTPUT_DIR
        / f"agent2_assessment_package_{timestamp}.md"
    )

    text_report_path = (
        OUTPUT_DIR
        / f"agent2_assessment_evaluation_{timestamp}.txt"
    )

    phase2_gate_manifest_path = (
        OUTPUT_DIR
        / f"agent2_phase2_gate_manifest_{timestamp}.csv"
    )

    phase2_query_evidence_path = (
        OUTPUT_DIR
        / f"agent2_phase2_query_evidence_{timestamp}.csv"
    )

    near_duplicate_manifest_path = (
        OUTPUT_DIR
        / f"agent2_phase2_near_duplicate_manifest_{timestamp}.csv"
    )

    adaptive_threshold_profile_path = (
        OUTPUT_DIR
        / f"agent2_phase2_adaptive_threshold_profile_{timestamp}.csv"
    )

    release_readiness_path = (
        OUTPUT_DIR
        / f"agent2_assessment_release_readiness_{timestamp}.json"
    )

    concept_fit_manifest_path = (
        OUTPUT_DIR
        / f"agent2_concept_fit_manifest_{timestamp}.csv"
    )
    concept_fit_profile_path = (
        OUTPUT_DIR
        / f"agent2_concept_fit_profile_{timestamp}.csv"
    )
    context_refinement_manifest_path = (
        OUTPUT_DIR
        / f"agent2_context_refinement_manifest_{timestamp}.json"
    )
    question_group_manifest_path = (
        OUTPUT_DIR
        / f"agent2_question_group_manifest_{timestamp}.json"
    )


    phase2_gate_manifest_df.to_csv(
        candidates_path,
        index=False,
    )

    phase2_gate_manifest_df.to_csv(
        phase2_gate_manifest_path,
        index=False,
    )

    phase2_query_evidence_df.to_csv(
        phase2_query_evidence_path,
        index=False,
    )

    near_duplicate_manifest_df.to_csv(
        near_duplicate_manifest_path,
        index=False,
    )

    adaptive_threshold_profile_df.to_csv(
        adaptive_threshold_profile_path,
        index=False,
    )

    phase2_gate_manifest_df.to_csv(
        concept_fit_manifest_path,
        index=False,
    )
    concept_fit_profile_df.to_csv(
        concept_fit_profile_path,
        index=False,
    )
    context_refinement_manifest_path.write_text(
        json.dumps(
            json_safe(
                context_refinement_df.to_dict(orient="records")
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    question_group_manifest_path.write_text(
        json.dumps(
            json_safe(
                question_group_manifest_df.to_dict(orient="records")
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    release_readiness_payload = {
        "run_status": AGENT2_RUN_STATUS,
        "user_messages": json_safe(AGENT2_USER_MESSAGES),
        "partial_output_generated": bool(
            AGENT2_RUN_STATUS == "partial_success"
        ),
        "topic_question_availability": json_safe(
            topic_question_availability_df.to_dict(orient="records")
            if "topic_question_availability_df" in globals()
            and isinstance(topic_question_availability_df, pd.DataFrame)
            else []
        ),
        "assessment_release_status": (
            selection_summary[
                "assessment_release_status"
            ]
        ),
        "requires_user_decision": (
            selection_summary[
                "requires_user_decision"
            ]
        ),
        "selected_marks": (
            selection_summary[
                "selected_marks"
            ]
        ),
        "requested_question_count_met": selection_summary[
            "requested_question_count_met"
        ],
        "question_count_shortfall": selection_summary[
            "question_count_shortfall"
        ],
        "concept_fit_rescue_selected": selection_summary[
            "concept_fit_rescue_selected"
        ],
        "semantic_rescue_used_in_pool": (
            phase2_rescue_used
        ),
        "semantic_rescue_selected": (
            selection_summary[
                "semantic_rescue_selected"
            ]
        ),
        "threshold_strategy": (
            "per_topic_adaptive"
        ),
        "adaptive_threshold_version": (
            ADAPTIVE_THRESHOLD_VERSION
        ),
        "adaptive_threshold_min": (
            adaptive_threshold_min
        ),
        "adaptive_threshold_max": (
            adaptive_threshold_max
        ),
        "adaptive_threshold_mean": (
            adaptive_threshold_mean
        ),
        "rescue_semantic_floor": (
            MIN_TOPIC_RESCUE_SEMANTIC_SCORE
        ),
        "required_official_references": (
            selection_summary[
                "required_official_references"
            ]
        ),
        "selected_official_references": (
            selection_summary[
                "selected_official_references"
            ]
        ),
        "all_required_references_covered": (
            selection_summary[
                "all_required_references_covered"
            ]
        ),
        "all_topics_use_actual_chunk_evidence": (
            selection_summary[
                "all_topics_use_actual_chunk_evidence"
            ]
        ),
        "phase3_review_count": (
            selection_summary[
                "phase3_review_count"
            ]
        ),
        "release_blockers": (
            selection_summary[
                "release_blockers"
            ]
        ),
    }

    release_readiness_path.write_text(
        json.dumps(
            json_safe(
                release_readiness_payload
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # Fixed-name pointer for the frontend. This is overwritten on every run
    # so the UI never has to guess which timestamped outputs belong to the
    # current execution.
    current_run_status_path = OUTPUT_DIR / "agent2_current_run.json"
    current_run_status_path.write_text(
        json.dumps(
            {
                "timestamp": timestamp,
                "assessment_generated": True,
                "run_status": selection_summary.get("run_status", "success"),
                "requested_paper_code": request.get("paper_code"),
                "resolved_kb_paper_code": knowledge_base_paper_code(),
                "selected_questions_csv": str(selected_path),
                "release_readiness_json": str(release_readiness_path),
                "assessment_package_json": str(package_path),
                "combined_questions_and_answers_pdf": None,
            },
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    final_df.to_csv(
        selected_path,
        index=False,
    )

    package_path.write_text(
        json.dumps(
            assessment_package,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    markdown_lines = [
        "# Generated Assessment",
        "",
        (
            f"**Total marks:** "
            f"{selection_summary['selected_marks']}"
        ),
        "",
    ]

    for question in question_packages:
        markdown_lines.extend(
            [
                (
                    f"## Question {question['rank']} "
                    f"({question['question']['marks']} marks)"
                ),
                "",
                (
                    f"**Topic:** "
                    f"{question['topic']['detected_topic']} "
                    f"({question['topic']['official_reference']})"
                ),
                "",
                question["question"]["text"],
                "",
            ]
        )

        context_text = str(
            question["question"].get(
                "context"
            )
            or ""
        ).strip()

        if context_text:
            markdown_lines.extend(
                [
                    "**Context**",
                    "",
                    context_text,
                    "",
                ]
            )

        rendered_images = (
            question["question"].get(
                "rendered_page_images"
            )
            or []
        )

        if rendered_images:
            markdown_lines.extend(
                [
                    "### Original question page image(s)",
                    "",
                    (
                        "The following page image(s) were rendered "
                        "from the original cached Question Paper PDF."
                    ),
                    "",
                ]
            )

            for rendered_image in rendered_images:
                markdown_lines.extend(
                    [
                        (
                            f"![Original question page]"
                            f"({rendered_image})"
                        ),
                        "",
                    ]
                )

        phase3_ms = (
            question[
                "mark_scheme"
            ][
                "phase3_structured"
            ]
        )

        markdown_lines.extend(
            [
                "### Mark scheme — raw guidance",
                "",
                str(
                    question[
                        "mark_scheme"
                    ][
                        "raw_marking_guidance"
                    ]
                    or ""
                ),
                "",
                "### Phase 3 structured view",
                "",
                (
                    f"**Cleanup status:** "
                    f"{phase3_ms['cleanup_status']}"
                ),
                "",
                (
                    f"**Rule confidence:** "
                    f"{phase3_ms['rule_confidence']}"
                ),
                "",
                f"**Block parser:** {phase3_ms['block_parser_version']}",
                "",
                f"**Wrapped lines merged:** {phase3_ms['continuation_lines_merged']}",
                "",
                f"**Inline markers split:** {phase3_ms['inline_markers_split']}",
                "",
                "**Marking points**",
                "",
                json.dumps(
                    phase3_ms[
                        "marking_points"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "**Acceptable answers**",
                "",
                json.dumps(
                    phase3_ms[
                        "acceptable_answers"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "**Rejected answers**",
                "",
                json.dumps(
                    phase3_ms[
                        "rejected_answers"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "**Additional guidance**",
                "",
                json.dumps(
                    phase3_ms[
                        "additional_guidance"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "**Worked examples**",
                "",
                json.dumps(
                    phase3_ms[
                        "worked_examples"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "---",
                "",
            ]
        )


    markdown_path.write_text(
        "\n".join(markdown_lines),
        encoding="utf-8",
    )


    # ---------------------------------------------------------
    # Plain-text evaluation report
    # ---------------------------------------------------------
    text_lines = [
        "=" * 78,
        "AGENT 2 — NOTEBOOK 05 RETRIEVAL EVALUATION REPORT",
        "=" * 78,
        "",
        f"Generated at (UTC): {assessment_package['generated_at_utc']}",
        f"Specification: {SPECIFICATION_CODE}",
        f"Specification version: {SPECIFICATION_VERSION}",
        f"Retrieval version: {RETRIEVAL_VERSION}",
        f"Embedding model: {MODEL_NAME}",
        f"Qdrant collection: {AGENT2_COLLECTION}",
        "",
        "-" * 78,
        "AGENT 1 DETECTED TOPICS",
        "-" * 78,
    ]

    for topic_index, topic in enumerate(
        assessment_package["agent1_topics"],
        start=1,
    ):
        text_lines.extend(
            [
                f"Topic {topic_index}",
                f"  Canonical topic: {topic['detected_topic']}",
                (
                    "  Detected concepts: "
                    f"{topic.get('detected_concepts', [])}"
                ),
                f"  Role: {topic['role']}",
                f"  Official reference: {topic['official_reference']}",
                f"  Confidence: {topic['confidence']}",
                f"  Ranking score: {topic['ranking_score']}",
                f"  Source chunks: {topic['source_chunks']}",
                "",
            ]
        )

    text_lines.extend(
        [
            "-" * 78,
            "ASSESSMENT REQUEST",
            "-" * 78,
        ]
    )

    for key, value in request.items():
        text_lines.append(f"{key}: {value}")

    text_lines.extend(
        [
            "",
            f"Lesson summary: {LESSON_SUMMARY}",
            "",
            "-" * 78,
            "RETRIEVAL SUMMARY",
            "-" * 78,
        ]
    )

    for key, value in retrieval_summary.items():
        text_lines.append(f"{key}: {value}")

    text_lines.extend(
        [
            "",
            "-" * 78,
            "SELECTED QUESTIONS AND MARK SCHEMES",
            "-" * 78,
            "",
        ]
    )

    for question in question_packages:
        text_lines.extend(
            [
                "=" * 78,
                (
                    f"QUESTION {question['rank']} "
                    f"({question['question']['marks']} marks)"
                ),
                "=" * 78,
                f"Question ID: {question['question_id']}",
                (
                    "Canonical topic: "
                    f"{question['topic']['detected_topic']}"
                ),
                (
                    "Detected concepts: "
                    f"{question['topic'].get('detected_concepts', [])}"
                ),
                f"Topic role: {question['topic']['role']}",
                (
                    "Official reference: "
                    f"{question['topic']['official_reference']}"
                ),
                (
                    "Official concept: "
                    f"{question['topic']['official_concept_name']}"
                ),
                (
                    "PMT source topic: "
                    f"{question['topic']['pmt_subtopic_code']} — "
                    f"{question['topic']['pmt_subtopic_name']}"
                ),
                f"Paper code: {question['question']['paper_code']}",
                (
                    "Programming language: "
                    f"{question['question']['programming_language']}"
                ),
                (
                    "Contains code: "
                    f"{question['question']['has_code']}"
                ),
                (
                    "Contains visual: "
                    f"{question['question']['has_visual']}"
                ),
                (
                    "Visual render status: "
                    f"{question['question']['visual_render_status']}"
                ),
                (
                    "Original source pages: "
                    f"{question['question']['source_page_numbers']}"
                ),
                (
                    "Rendered page images: "
                    f"{question['question']['rendered_page_images']}"
                ),
                (
                    "Visual rendering error: "
                    f"{question['question']['visual_render_error']}"
                ),
                "",
                "QUESTION TEXT",
                "-" * 78,
                str(question["question"]["text"] or ""),
                "",
            ]
        )

        context_text = str(
            question["question"].get("context") or ""
        ).strip()

        if context_text:
            text_lines.extend(
                [
                    "CONTEXT",
                    "-" * 78,
                    context_text,
                    "",
                ]
            )

        mark_scheme = question["mark_scheme"]

        text_lines.extend(
            [
                "MARK SCHEME",
                "-" * 78,
                (
                    "Maximum marks: "
                    f"{mark_scheme['maximum_marks']}"
                ),
                "",
                "Marking guidance:",
                str(mark_scheme["marking_guidance"] or ""),
                "",
                "PHASE 3 STRUCTURED MARK SCHEME",
                "-" * 78,
                (
                    "Cleanup status: "
                    f"{mark_scheme['phase3_structured']['cleanup_status']}"
                ),
                (
                    "Rule confidence: "
                    f"{mark_scheme['phase3_structured']['rule_confidence']}"
                ),
                (
                    "Review reasons: "
                    f"{mark_scheme['phase3_structured']['review_reasons']}"
                ),
                (
                    "Block parser version: "
                    f"{mark_scheme['phase3_structured']['block_parser_version']}"
                ),
                (
                    "Logical blocks created: "
                    f"{mark_scheme['phase3_structured']['block_count']}"
                ),
                (
                    "Wrapped continuation lines merged: "
                    f"{mark_scheme['phase3_structured']['continuation_lines_merged']}"
                ),
                (
                    "Inline markers split: "
                    f"{mark_scheme['phase3_structured']['inline_markers_split']}"
                ),
                (
                    "Ambiguous lines: "
                    f"{mark_scheme['phase3_structured']['ambiguous_lines']}"
                ),
                "",
                "Marking points:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "marking_points"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "Acceptable answers:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "acceptable_answers"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "Rejected answers:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "rejected_answers"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "Additional guidance:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "additional_guidance"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "Worked examples:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "worked_examples"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                "Assessment objectives:",
                json.dumps(
                    mark_scheme[
                        "phase3_structured"
                    ][
                        "assessment_objectives"
                    ],
                    indent=2,
                    ensure_ascii=False,
                ),
                "",
                (
                    "Legacy structured fields retained "
                    "in JSON/CSV for comparison."
                ),
                "",
                "RETRIEVAL EVIDENCE",
                "-" * 78,
                (
                    "Retrieval stage: "
                    f"{question['retrieval']['stage']}"
                ),
                (
                    "Retrieval backend: "
                    f"{question['retrieval'].get('backend')}"
                ),
                (
                    "Retrieval relaxation: "
                    f"{question['retrieval'].get('relaxation')}"
                ),
                (
                    "Semantic score: "
                    f"{question['retrieval']['semantic_score']:.6f}"
                ),
                (
                    "Final score: "
                    f"{question['retrieval']['final_score']:.6f}"
                ),
                (
                    "Agent 1 confidence: "
                    f"{question['retrieval']['agent1_confidence']:.6f}"
                ),
                (
                    "Source chunks: "
                    f"{question['retrieval']['source_chunks']}"
                ),
                (
                    "Query evidence source: "
                    f"{question['retrieval']['query_evidence_source']}"
                ),
                (
                    "Phase 2 threshold: "
                    f"{question['retrieval']['phase2_semantic_threshold']}"
                ),
                (
                    "Concept gate passed: "
                    f"{question['retrieval']['concept_gate_passed']}"
                ),
                (
                    "Question-quality gate passed: "
                    f"{question['retrieval']['question_quality_gate_passed']}"
                ),
                (
                    "Phase 2 gate mode: "
                    f"{question['retrieval']['phase2_gate_mode']}"
                ),
                (
                    "Semantic rescue used: "
                    f"{question['retrieval']['semantic_rescue_used']}"
                ),
                (
                    "Question-quality issues: "
                    f"{question['retrieval']['question_quality_issues']}"
                ),
                (
                    "QP/MS match method: "
                    f"{question['retrieval']['qp_ms_match_method']}"
                ),
                (
                    "QP/MS match confidence: "
                    f"{question['retrieval']['qp_ms_match_confidence']:.6f}"
                ),
                "",
            ]
        )

    text_lines.extend(
        [
            "=" * 78,
            "END OF REPORT",
            "=" * 78,
        ]
    )

    text_report_path.write_text(
        "\n".join(text_lines),
        encoding="utf-8",
    )


    print("Saved:")
    print(candidates_path)
    print(selected_path)
    print(package_path)
    print(markdown_path)
    print(text_report_path)
    print(visual_manifest_path)
    print(VISUAL_IMAGE_ROOT)
    print(phase2_gate_manifest_path)
    print(phase2_query_evidence_path)
    print(near_duplicate_manifest_path)
    print(adaptive_threshold_profile_path)
    print(release_readiness_path)
    print(phase3_manifest_path)
    print(phase3_json_path)


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    # ================================================================
    # PDF EXPORT — STUDENT PAPER, TEACHER MARK SCHEME AND AUDIT COPY
    # ================================================================

    from xml.sax.saxutils import escape
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import mm
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.platypus import (
        HRFlowable,
        Image as PDFImage,
        PageBreak,
        Paragraph,
        SimpleDocTemplate,
        Spacer,
    )
    from IPython.display import FileLink

    PDF_OUTPUT_DIR = Path(OUTPUT_DIR)
    PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


    def first_existing_path(paths: list[Path]) -> Path | None:
        return next((path for path in paths if path.exists()), None)


    regular_font_path = first_existing_path(
        [
            Path(r"C:\Windows\Fonts\arial.ttf"),
            Path(r"C:\Windows\Fonts\calibri.ttf"),
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
        ]
    )
    bold_font_path = first_existing_path(
        [
            Path(r"C:\Windows\Fonts\arialbd.ttf"),
            Path(r"C:\Windows\Fonts\calibrib.ttf"),
            Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"),
        ]
    )

    if regular_font_path and bold_font_path:
        pdfmetrics.registerFont(
            TTFont("Agent2Regular", str(regular_font_path))
        )
        pdfmetrics.registerFont(
            TTFont("Agent2Bold", str(bold_font_path))
        )
        BODY_FONT = "Agent2Regular"
        BOLD_FONT = "Agent2Bold"
    else:
        BODY_FONT = "Helvetica"
        BOLD_FONT = "Helvetica-Bold"

    SUPERSCRIPT_TRANSLATION = str.maketrans(
        "0123456789-+()",
        "⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺⁽⁾",
    )


    def display_text(value: Any) -> str:
        """Normalize presentation only; never mutate raw database text."""
        text_value = str(value if value is not None else "")
        text_value = text_value.replace("\r\n", "\n").replace("\r", "\n")
        # Presentation-only cleanup for missing parser labels such as "(nan)".
        # Raw database text remains unchanged in the audit package.
        text_value = re.sub(
            r"^\s*\(?nan\)?(?:\s+|$)",
            "",
            text_value,
            flags=re.IGNORECASE,
        )
        text_value = re.sub(
            r"[\x00-\x08\x0B\x0C\x0E-\x1F]",
            "",
            text_value,
        )
        text_value = text_value.replace("<=", "≤").replace(">=", "≥")
        text_value = text_value.replace("->", "→").replace("<-", "←")
        text_value = re.sub(r";{2,}", ";", text_value)

        def exponent_replace(match: re.Match) -> str:
            return (
                match.group(1)
                + match.group(2).translate(SUPERSCRIPT_TRANSLATION)
            )

        text_value = re.sub(
            r"\b(\d+)\^(\-?\d+)\b",
            exponent_replace,
            text_value,
        )

        # Some PDF extractors flatten a superscript in an examiner's
        # alternative formula such as "// 2⁶ - 1" into "// 26 - 1".
        # The contextual // and trailing arithmetic operator keep this
        # display-only repair narrowly scoped without question IDs.
        text_value = re.sub(
            r"(?<=//\s)(2)(\d{1,2})(?=\s*[-+]\s*\d)",
            lambda match: (
                match.group(1)
                + match.group(2).translate(SUPERSCRIPT_TRANSLATION)
            ),
            text_value,
        )

        text_value = re.sub(r"\n{3,}", "\n\n", text_value)
        return text_value.strip()


    def pdf_text(value: Any) -> str:
        return escape(display_text(value)).replace("\n", "<br/>")


    def resolve_pdf_image_path(value: Any) -> Path | None:
        image_path = Path(str(value or ""))
        candidates = []
        if image_path.is_absolute():
            candidates.append(image_path)
        candidates.extend(
            [
                PDF_OUTPUT_DIR / image_path,
                PROJECT_ROOT / image_path,
                Path.cwd() / image_path,
            ]
        )
        for candidate in candidates:
            if candidate.exists() and candidate.is_file():
                return candidate.resolve()
        return None


    def scaled_pdf_image(
        image_path: Path,
        reserved_vertical_mm: float = 48.0,
    ) -> PDFImage:
        image = PDFImage(str(image_path))
        available_width = A4[0] - (36 * mm)
        available_height = A4[1] - (float(reserved_vertical_mm) * mm)
        scale = min(
            available_width / image.drawWidth,
            available_height / image.drawHeight,
            1.0,
        )
        image.drawWidth *= scale
        image.drawHeight *= scale
        image.hAlign = "CENTER"
        return image


    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "Agent2TitleV2",
        parent=styles["Title"],
        fontName=BOLD_FONT,
        fontSize=20,
        leading=24,
        alignment=1,
        spaceAfter=14,
    )
    heading_style = ParagraphStyle(
        "Agent2HeadingV2",
        parent=styles["Heading1"],
        fontName=BOLD_FONT,
        fontSize=14,
        leading=18,
        spaceBefore=8,
        spaceAfter=7,
    )
    subheading_style = ParagraphStyle(
        "Agent2SubheadingV2",
        parent=styles["Heading2"],
        fontName=BOLD_FONT,
        fontSize=11,
        leading=14,
        spaceBefore=6,
        spaceAfter=5,
    )
    body_style = ParagraphStyle(
        "Agent2BodyV2",
        parent=styles["BodyText"],
        fontName=BODY_FONT,
        fontSize=9.5,
        leading=14,
        spaceAfter=6,
    )
    small_style = ParagraphStyle(
        "Agent2SmallV2",
        parent=styles["BodyText"],
        fontName=BODY_FONT,
        fontSize=8,
        leading=11,
        textColor=colors.HexColor("#555555"),
        spaceAfter=4,
    )


    def add_page_number(canvas, document) -> None:
        canvas.saveState()
        canvas.setFont(BODY_FONT, 8)
        canvas.setFillColor(colors.HexColor("#666666"))
        canvas.drawCentredString(
            A4[0] / 2,
            9 * mm,
            f"Page {document.page}",
        )
        canvas.restoreState()


    def mark_label(marks: int) -> str:
        return "mark" if int(marks) == 1 else "marks"


    def grouped_rows(
        frame: pd.DataFrame | None = None,
    ) -> list[pd.DataFrame]:
        source_frame = (
            final_df
            if frame is None
            else frame
        )

        groups = []

        for group_rank in sorted(
            source_frame[
                "display_group_rank"
            ].unique()
        ):
            groups.append(
                source_frame[
                    source_frame[
                        "display_group_rank"
                    ]
                    == group_rank
                ].sort_values(
                    [
                        "question_number_postgres",
                        "selected_rank",
                    ]
                )
            )

        return groups


    def append_unique_images(
        story: list[Any],
        paths: list[str],
        global_seen_hashes: set[str] | None = None,
        reserved_vertical_mm: float = 48.0,
    ) -> None:
        # De-duplicate both repeated paths and byte-identical rendered source
        # images across the entire student paper, not only inside one group.
        seen_paths = set()
        global_seen_hashes = global_seen_hashes if global_seen_hashes is not None else set()
        for path_value in paths:
            if path_value in seen_paths:
                continue
            seen_paths.add(path_value)
            resolved = resolve_pdf_image_path(path_value)
            if resolved is None:
                continue
            try:
                image_hash = hashlib.sha256(resolved.read_bytes()).hexdigest()
            except OSError:
                image_hash = str(resolved.resolve())
            if image_hash in global_seen_hashes:
                continue
            global_seen_hashes.add(image_hash)
            story.append(
                scaled_pdf_image(
                    resolved,
                    reserved_vertical_mm=reserved_vertical_mm,
                )
            )
            story.append(Spacer(1, 7))


    SUCCESSFUL_VISUAL_RENDER_STATUSES = {
        "rendered_dependency_complete",
    }


    def student_visual_image_paths(row: pd.Series) -> list[str]:
        """
        Return source images in dependency-safe order:

            required Figure/Table crop(s)
            → selected-question crop(s)

        ``rendered_page_images`` remains a compatibility fallback.
        """
        ordered_paths = []

        for field_name in (
            "dependency_crop_images",
            "cropped_question_images",
        ):
            raw_paths = (
                row.get(field_name)
                if isinstance(row.get(field_name), list)
                else []
            )

            for path_value in raw_paths:
                if (
                    path_value not in ordered_paths
                    and resolve_pdf_image_path(path_value) is not None
                ):
                    ordered_paths.append(
                        path_value
                    )

        if not ordered_paths:
            raw_paths = (
                row.get("rendered_page_images")
                if isinstance(row.get("rendered_page_images"), list)
                else []
            )

            for path_value in raw_paths:
                if (
                    path_value not in ordered_paths
                    and resolve_pdf_image_path(path_value) is not None
                ):
                    ordered_paths.append(
                        path_value
                    )

        return ordered_paths


    def student_uses_image_only(row: pd.Series) -> bool:
        """
        A validated visual question is shown from the original source image only.

        This prevents the student paper from displaying extracted/generated text
        and then repeating the same question as an image.
        """
        return bool(
            row.get(
                "visual_render_required"
            )
        ) and bool(
            row.get(
                "visual_dependency_complete"
            )
        ) and (
            str(
                row.get(
                    "visual_render_status"
                )
            )
            in SUCCESSFUL_VISUAL_RENDER_STATUSES
        ) and bool(
            student_visual_image_paths(
                row
            )
        )


    student_release_df = final_df[
        final_df[
            "student_release_eligible"
        ].astype(bool)
    ].copy()


    student_release_marks = int(
        student_release_df[
            "marks_postgres"
        ].sum()
    )


    student_dependency_exclusions_df = final_df[
        ~final_df[
            "student_release_eligible"
        ].astype(bool)
    ].copy()


    def append_question_annotation(
        story: list[Any],
        row: pd.Series,
        *,
        mark_scheme: bool = False,
    ) -> None:
        selected_rank = int(row.get("selected_rank") or 0)
        detected_topic = str(row.get("detected_topic") or "").strip()
        role = str(row.get("agent1_role") or "").strip().title()
        marks_value = row.get("marks_postgres")
        if marks_value is None or (
            isinstance(marks_value, float) and np.isnan(marks_value)
        ):
            marks_value = row.get("marks", 0)
        marks = int(marks_value)

        title = (
            f"Question {selected_rank}"
            + (" Mark Scheme" if mark_scheme else "")
            + f" — Topic: {detected_topic}"
            + f" — Role: {role}"
            + f" — {marks} {mark_label(marks)}"
        )
        story.append(
            Paragraph(
                pdf_text(title),
                subheading_style if mark_scheme else heading_style,
            )
        )

        source_number = str(
            row.get("question_number_postgres")
            or row.get("question_number")
            or ""
        ).strip()
        if source_number and source_number.casefold() != "nan":
            story.append(
                Paragraph(
                    pdf_text(
                        "Original AQA question: "
                        + source_number
                    ),
                    small_style,
                )
            )


    def student_story() -> list[Any]:
        seen_student_image_hashes: set[str] = set()
        story = [
            Paragraph("AQA GCSE Computer Science", title_style),
            Paragraph("Student Question Paper", heading_style),
            Paragraph(
                pdf_text(
                    f"Specification: {SPECIFICATION_CODE}"
                ),
                body_style,
            ),
            Paragraph(
                pdf_text(
                    "Answer all questions. "
                    f"Total marks: {student_release_marks}"
                ),
                body_style,
            ),
            PageBreak(),
        ]

        for group in grouped_rows(
            student_release_df
        ):
            group = group.copy()
            group["_student_image_only"] = group.apply(
                student_uses_image_only,
                axis=1,
            )

            all_rows_image_only = bool(
                len(group) > 0
                and group["_student_image_only"].all()
            )

            # A shared source parent can contain several selected subquestions.
            # Label each selected question separately using selected_rank.
            image_rows = group[
                group["_student_image_only"]
            ]
            text_rows = group[
                ~group["_student_image_only"]
            ]

            unique_contexts = [
                value
                for value in dict.fromkeys(
                    str(value or "").strip()
                    for value in text_rows["context_text"].tolist()
                )
                if value
            ]
            if len(unique_contexts) == 1:
                story.append(
                    Paragraph(pdf_text(unique_contexts[0]), body_style)
                )

            group_images = []

            # Image-only questions are annotated before their shared source
            # crop/page, so a full-page visual can never hide the generated
            # question number/topic/role label.
            for _, image_row in image_rows.iterrows():
                append_question_annotation(
                    story,
                    image_row,
                    mark_scheme=False,
                )
                for path_value in student_visual_image_paths(image_row):
                    if path_value not in group_images:
                        group_images.append(path_value)

            for _, row in text_rows.iterrows():
                append_question_annotation(
                    story,
                    row,
                    mark_scheme=False,
                )

                part = row.get("display_part_label")
                marks = int(row["marks_postgres"])
                prefix = f"({part}) " if part else ""

                if len(unique_contexts) != 1:
                    context_value = str(row.get("context_text") or "").strip()
                    if context_value:
                        story.append(
                            Paragraph(pdf_text(context_value), body_style)
                        )

                story.append(
                    Paragraph(
                        pdf_text(
                            prefix
                            + str(row["question_text_postgres"])
                            + f"   [{marks} {mark_label(marks)}]"
                        ),
                        body_style,
                    )
                )

            # Visual questions are presented once, using only the validated
            # original crop/source image. Extracted text remains available in
            # JSON and teacher/audit outputs through the unchanged final_df.
            append_unique_images(
                story,
                group_images,
                global_seen_hashes=seen_student_image_hashes,
                reserved_vertical_mm=78.0,
            )

            story.append(HRFlowable(width="100%", thickness=0.6))
            story.append(PageBreak())

        return story


    def teacher_story(include_retrieval_evidence: bool) -> list[Any]:
        subtitle = (
            "Teacher Mark Scheme and Retrieval Audit"
            if include_retrieval_evidence
            else "Teacher Mark Scheme"
        )
        story = [
            Paragraph("AQA GCSE Computer Science", title_style),
            Paragraph(subtitle, heading_style),
            Paragraph(
                "Raw marking guidance is retained as the source of truth.",
                body_style,
            ),
            PageBreak(),
        ]

        for group in grouped_rows():
            for _, row in group.iterrows():
                append_question_annotation(
                    story,
                    row,
                    mark_scheme=True,
                )
                part = row.get("display_part_label")
                if part:
                    story.append(
                        Paragraph(
                            pdf_text(f"Source part ({part})"),
                            small_style,
                        )
                    )
                story.append(
                    Paragraph(
                        pdf_text(row["question_text_postgres"]),
                        small_style,
                    )
                )

                structured_sections = [
                    ("Marking points", row["phase3_marking_points"]),
                    ("Acceptable answers", row["phase3_acceptable_answers"]),
                    ("Rejected answers", row["phase3_rejected_answers"]),
                    ("Additional guidance", row["phase3_additional_guidance"]),
                ]
                for section_title, values in structured_sections:
                    if not isinstance(values, list) or not values:
                        continue
                    story.append(Paragraph(section_title, subheading_style))
                    for value in values:
                        story.append(
                            Paragraph(
                                pdf_text(f"- {value}"),
                                body_style,
                            )
                        )

                worked_examples = row["phase3_worked_examples"]
                if isinstance(worked_examples, list) and worked_examples:
                    story.append(Paragraph("Worked examples", subheading_style))
                    for example in worked_examples:
                        story.append(
                            Paragraph(
                                pdf_text(example.get("header", "Example")),
                                body_style,
                            )
                        )
                        story.append(
                            Paragraph(
                                pdf_text(example.get("content", "")),
                                small_style,
                            )
                        )

                story.append(
                    Paragraph(
                        "Raw marking guidance (source of truth)",
                        subheading_style,
                    )
                )
                story.append(
                    Paragraph(
                        pdf_text(row["marking_guidance"]),
                        body_style,
                    )
                )

                if include_retrieval_evidence:
                    story.append(Paragraph("Retrieval evidence", subheading_style))
                    evidence_lines = [
                        f"Canonical topic: {row['detected_topic']}",
                        (
                            "Detected concepts: "
                            f"{row.get('detected_concepts', [])}"
                        ),
                        f"Official reference: {row['official_reference_postgres']}",
                        f"Evidence-aware semantic score: {float(row['semantic_score']):.6f}",
                        f"Direct-topic semantic score: {float(row.get('direct_topic_semantic_score', 0.0)):.6f}",
                        f"BM25 lexical score (normalised): {float(row.get('bm25_lexical_normalized', 0.0)):.6f}",
                        f"Hybrid relevance score: {float(row.get('hybrid_relevance_score', 0.0)):.6f}",
                        f"Retrieval pool type: {row.get('retrieval_pool_type', 'unknown')}",
                        f"Phase 2 gate mode: {row.get('phase2_gate_mode', 'unknown')}",
                        f"Direct concept score (audit only): {float(row['direct_concept_score']):.6f}",
                        f"Context status: {row['context_refinement_status']}",
                        (
                            "Effective visual required: "
                            f"{bool(row['visual_render_required'])}"
                        ),
                        (
                            "Required dependencies: "
                            f"{row['required_dependency_labels']}"
                        ),
                        (
                            "Resolved dependencies: "
                            f"{row['resolved_dependency_labels']}"
                        ),
                        (
                            "Missing dependencies: "
                            f"{row['missing_dependency_labels']}"
                        ),
                        (
                            "Structured response required: "
                            f"{bool(row['structured_response_required'])}"
                        ),
                        (
                            "Structured layout verified: "
                            f"{bool(row['structured_layout_verified'])}"
                        ),
                        (
                            "Visual dependency complete: "
                            f"{bool(row['visual_dependency_complete'])}"
                        ),
                        (
                            "Student release eligible: "
                            f"{bool(row['student_release_eligible'])}"
                        ),
                        f"Crop status: {row['question_region_crop_status']}",
                    ]
                    story.append(
                        Paragraph(
                            pdf_text("\n".join(evidence_lines)),
                            small_style,
                        )
                    )

            story.append(HRFlowable(width="100%", thickness=0.6))
            story.append(PageBreak())

        return story


    def build_pdf(path: Path, story: list[Any], title: str) -> None:
        document = SimpleDocTemplate(
            str(path),
            pagesize=A4,
            rightMargin=18 * mm,
            leftMargin=18 * mm,
            topMargin=18 * mm,
            bottomMargin=18 * mm,
            title=title,
            author="Agent 2",
        )
        document.build(
            story,
            onFirstPage=add_page_number,
            onLaterPages=add_page_number,
        )


    def verify_pdf(path: Path) -> dict[str, Any]:
        if not path.exists() or path.stat().st_size == 0:
            raise RuntimeError(f"PDF was not created: {path}")

        with fitz.open(str(path)) as document:
            if document.page_count < 1:
                raise RuntimeError(f"PDF contains no pages: {path}")
            for page_index in sorted({0, document.page_count - 1}):
                page = document.load_page(page_index)
                pixmap = page.get_pixmap(
                    matrix=fitz.Matrix(1.0, 1.0),
                    alpha=False,
                )
                if pixmap.width <= 0 or pixmap.height <= 0:
                    raise RuntimeError(
                        f"PDF render verification failed: {path}"
                    )
            return {
                "path": str(path),
                "bytes": int(path.stat().st_size),
                "page_count": int(document.page_count),
                "render_verified": True,
            }


    student_pdf_path = (
        PDF_OUTPUT_DIR
        / f"agent2_student_question_paper_{RUN_TIMESTAMP}.pdf"
    )
    teacher_pdf_path = (
        PDF_OUTPUT_DIR
        / f"agent2_teacher_mark_scheme_{RUN_TIMESTAMP}.pdf"
    )
    combined_pdf_path = (
        PDF_OUTPUT_DIR
        / f"agent2_questions_and_answers_{RUN_TIMESTAMP}.pdf"
    )

    build_pdf(
        student_pdf_path,
        student_story(),
        "Agent 2 Student Question Paper",
    )
    build_pdf(
        teacher_pdf_path,
        teacher_story(include_retrieval_evidence=False),
        "Agent 2 Teacher Mark Scheme",
    )
    combined_story = student_story()
    combined_story.append(PageBreak())
    combined_story.extend(
        teacher_story(include_retrieval_evidence=True)
    )
    build_pdf(
        combined_pdf_path,
        combined_story,
        "Agent 2 Questions, Answers and Retrieval Audit",
    )

    pdf_verification = {
        "version": PDF_EXPORT_VERSION,
        "student_pdf": verify_pdf(student_pdf_path),
        "teacher_pdf": verify_pdf(teacher_pdf_path),
        "combined_audit_pdf": verify_pdf(combined_pdf_path),
    }

    assessment_package["output_files"] = {
        **assessment_package.get("output_files", {}),
        "student_question_paper_pdf": str(student_pdf_path),
        "teacher_mark_scheme_pdf": str(teacher_pdf_path),
        "combined_questions_answers_audit_pdf": str(combined_pdf_path),
    }
    assessment_package["pdf_export"] = pdf_verification
    package_path.write_text(
        json.dumps(
            json_safe(assessment_package),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    print("PDF outputs created and render-verified:")
    display(pd.DataFrame(pdf_verification).T)
    display(FileLink(str(student_pdf_path)))
    display(FileLink(str(teacher_pdf_path)))
    display(FileLink(str(combined_pdf_path)))

## 15. Store retrieval audit logs

The audit tables record the Agent 1 input, request, candidates, scores and final
selections. These records will support Notebook 06 retrieval evaluation.


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:

    AUDIT_SQL = [
        """
        CREATE TABLE IF NOT EXISTS assessment_retrieval_runs
        (
            id UUID PRIMARY KEY,
            retrieval_version VARCHAR(120) NOT NULL,
            specification_code VARCHAR(20) NOT NULL,
            specification_version VARCHAR(120) NOT NULL,
            embedding_model VARCHAR(200) NOT NULL,
            qdrant_collection VARCHAR(200) NOT NULL,
            agent1_input JSONB NOT NULL,
            assessment_request JSONB NOT NULL,
            lesson_summary TEXT,
            raw_candidate_count INTEGER NOT NULL,
            duplicate_count INTEGER NOT NULL,
            unique_candidate_count INTEGER NOT NULL,
            selected_question_count INTEGER NOT NULL,
            selected_total_marks INTEGER NOT NULL,
            status VARCHAR(40) NOT NULL,
            started_at TIMESTAMPTZ NOT NULL,
            completed_at TIMESTAMPTZ
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS assessment_retrieval_results
        (
            id UUID PRIMARY KEY,
            retrieval_run_id UUID NOT NULL
                REFERENCES assessment_retrieval_runs(id)
                ON DELETE CASCADE,
            question_id UUID NOT NULL
                REFERENCES assessment_topical_questions(id)
                ON DELETE CASCADE,
            detected_topic TEXT NOT NULL,
            agent1_role VARCHAR(30) NOT NULL,
            requested_official_reference VARCHAR(20) NOT NULL,
            retrieval_stage VARCHAR(50) NOT NULL,
            semantic_score DOUBLE PRECISION NOT NULL,
            final_score DOUBLE PRECISION NOT NULL,
            raw_rank INTEGER,
            unique_rank INTEGER,
            selected BOOLEAN NOT NULL,
            selected_rank INTEGER,
            created_at TIMESTAMPTZ NOT NULL
        )
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase2_version VARCHAR(120)
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase2_refinement_version VARCHAR(120)
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase2_configuration JSONB
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS near_duplicate_count INTEGER
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase2_eligible_count INTEGER
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase2_rejected_count INTEGER
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS assessment_release_status VARCHAR(50)
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS marks_difference INTEGER
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS query_evidence_source VARCHAR(80)
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS phase2_semantic_threshold DOUBLE PRECISION
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS concept_gate_passed BOOLEAN
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS question_quality_gate_passed BOOLEAN
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS phase2_gate_passed BOOLEAN
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS question_quality_issues JSONB
        """,
        """
        ALTER TABLE assessment_retrieval_results
        ADD COLUMN IF NOT EXISTS phase2_rejection_reasons JSONB
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase3_version VARCHAR(120)
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS phase3_review_count INTEGER
        """,
        """
        ALTER TABLE assessment_retrieval_runs
        ADD COLUMN IF NOT EXISTS final_release_blockers JSONB
        """,
        """
        CREATE TABLE IF NOT EXISTS
            assessment_mark_scheme_cleanup_results
        (
            id UUID PRIMARY KEY,
            retrieval_run_id UUID NOT NULL
                REFERENCES assessment_retrieval_runs(id)
                ON DELETE CASCADE,
            question_id UUID NOT NULL
                REFERENCES assessment_topical_questions(id)
                ON DELETE CASCADE,
            mark_scheme_id UUID NOT NULL,
            cleanup_version VARCHAR(120) NOT NULL,
            cleanup_status VARCHAR(50) NOT NULL,
            rule_confidence DOUBLE PRECISION NOT NULL,
            structured_payload JSONB NOT NULL,
            review_reasons JSONB NOT NULL,
            created_at TIMESTAMPTZ NOT NULL
        )
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_assessment_ms_cleanup_run
        ON assessment_mark_scheme_cleanup_results
        (
            retrieval_run_id,
            cleanup_status
        )
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_assessment_retrieval_results_run
        ON assessment_retrieval_results
        (
            retrieval_run_id,
            selected,
            selected_rank
        )
        """,
        """
        CREATE TABLE IF NOT EXISTS agent2_retrieval_feedback
        (
            id BIGSERIAL PRIMARY KEY,
            feedback_event_id UUID UNIQUE NOT NULL,
            package_fingerprint TEXT NOT NULL,
            pipeline_run_id TEXT NOT NULL,
            package_generated_at_utc TIMESTAMPTZ,
            retrieval_version TEXT,
            question_id TEXT NOT NULL,
            selected_rank INTEGER,
            agent1_topic_index INTEGER,
            concept_id TEXT,
            detected_topic TEXT,
            official_reference TEXT,
            agent1_role TEXT,
            transcript_evidence TEXT,
            transcript_evidence_source TEXT,
            semantic_score DOUBLE PRECISION,
            base_final_score DOUBLE PRECISION,
            retrieval_stage TEXT,
            query_evidence_source TEXT,
            paper_code TEXT,
            question_number TEXT,
            question_marks INTEGER,
            question_text TEXT,
            decision TEXT NOT NULL
                CHECK (decision IN ('relevant', 'not_relevant')),
            reason TEXT,
            reviewed_by TEXT NOT NULL DEFAULT 'streamlit',
            phase_version TEXT NOT NULL,
            memory_eligible BOOLEAN NOT NULL DEFAULT FALSE,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_status TEXT NOT NULL DEFAULT 'not_indexed'
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_key TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_point_id UUID
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_context_hash TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_text TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_collection TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_embedding_model TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_vector_size INTEGER
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_phase_version TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_promoted_at TIMESTAMPTZ
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_error TEXT
        """,
        """
        ALTER TABLE agent2_retrieval_feedback
        ADD COLUMN IF NOT EXISTS memory_superseded_by_feedback_id BIGINT
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_agent2_retrieval_feedback_package
        ON agent2_retrieval_feedback
        (
            package_fingerprint,
            created_at DESC
        )
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_agent2_retrieval_feedback_run_question
        ON agent2_retrieval_feedback
        (
            pipeline_run_id,
            question_id,
            created_at DESC
        )
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_agent2_retrieval_feedback_memory_status
        ON agent2_retrieval_feedback
        (
            memory_status,
            created_at DESC
        )
        """,
        """
        CREATE INDEX IF NOT EXISTS
            ix_agent2_retrieval_feedback_memory_key
        ON agent2_retrieval_feedback
        (
            memory_key,
            created_at DESC
        )
        """,
    ]


    with engine.begin() as connection:
        for statement in AUDIT_SQL:
            connection.execute(
                text(statement)
            )

    print(
        "Retrieval audit + Phase 2 HITL/Qdrant-memory metadata created/verified."
    )


    retrieval_run_id = None

    if STORE_RETRIEVAL_LOGS:
        audit_metadata = MetaData()

        retrieval_runs = Table(
            "assessment_retrieval_runs",
            audit_metadata,
            autoload_with=engine,
        )

        retrieval_results = Table(
            "assessment_retrieval_results",
            audit_metadata,
            autoload_with=engine,
        )

        mark_scheme_cleanup_results = Table(
            "assessment_mark_scheme_cleanup_results",
            audit_metadata,
            autoload_with=engine,
        )

        retrieval_run_id = uuid.uuid4()

        now_utc = datetime.now(
            timezone.utc
        )

        selected_rank_lookup = {
            str(row["question_id"]): int(
                row["selected_rank"]
            )
            for _, row
            in selected_candidates_df.iterrows()
        }

        selected_ids_set = set(
            selected_rank_lookup
        )

        phase2_configuration = {
            "threshold_strategy": (
                "hierarchical_metadata_specific_vs_broad_single_hybrid_floor"
            ),
            "hybrid_retrieval_policy_version": (
                HYBRID_RETRIEVAL_POLICY_VERSION
            ),
            "specific_canonical_name_alignment_min": (
                SPECIFIC_CANONICAL_NAME_ALIGNMENT_MIN
            ),
            "broad_hybrid_relevance_floor": (
                BROAD_HYBRID_RELEVANCE_FLOOR
            ),
            "hybrid_weights": {
                "lesson_evidence_semantic": HYBRID_EVIDENCE_SEMANTIC_WEIGHT,
                "direct_topic_semantic": HYBRID_DIRECT_TOPIC_WEIGHT,
                "bm25_lexical": HYBRID_BM25_WEIGHT,
                "metadata_strength": HYBRID_METADATA_WEIGHT,
            },
            "bm25": {
                "k1": BM25_K1,
                "b": BM25_B,
                "query_version": BM25_QUERY_VERSION,
            },
            "adaptive_threshold_version": (
                ADAPTIVE_THRESHOLD_VERSION
            ),
            "legacy_fixed_strict_threshold": (
                LEGACY_FIXED_STRICT_THRESHOLD
            ),
            "legacy_fixed_relaxed_threshold": (
                LEGACY_FIXED_RELAXED_THRESHOLD
            ),
            "legacy_fixed_thresholds_active": (
                LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
            ),
            "adaptive_score_percentile": (
                ADAPTIVE_SCORE_PERCENTILE
            ),
            "adaptive_top_score_margin": (
                ADAPTIVE_TOP_SCORE_MARGIN
            ),
            "adaptive_absolute_minimum_score": (
                ADAPTIVE_ABSOLUTE_MINIMUM_SCORE
            ),
            "adaptive_maximum_allowed_threshold": (
                ADAPTIVE_MAXIMUM_ALLOWED_THRESHOLD
            ),
            "adaptive_threshold_min": (
                adaptive_threshold_min
            ),
            "adaptive_threshold_max": (
                adaptive_threshold_max
            ),
            "adaptive_threshold_mean": (
                adaptive_threshold_mean
            ),
            "adaptive_threshold_profile": (
                adaptive_threshold_profile_df
                .to_dict(
                    orient="records"
                )
            ),
            "adaptive_pool_sufficient_without_rescue": (
                adaptive_pool_sufficient
            ),
            "near_duplicate_lexical_threshold": (
                NEAR_DUPLICATE_LEXICAL_THRESHOLD
            ),
            "near_duplicate_token_jaccard_threshold": (
                NEAR_DUPLICATE_TOKEN_JACCARD_THRESHOLD
            ),
            "near_duplicate_semantic_threshold": (
                NEAR_DUPLICATE_SEMANTIC_THRESHOLD
            ),
            "exact_detected_concept_gate": {
                "enabled": ENABLE_EXACT_DETECTED_CONCEPT_GATE,
                "version": EXACT_DETECTED_CONCEPT_GATE_VERSION,
                "canonical_tolerance": EXACT_CONCEPT_CANONICAL_TOLERANCE,
            },
            "subconcept_diversity": {
                "version": SUBCONCEPT_DIVERSITY_VERSION,
                "weight": SUBCONCEPT_DIVERSITY_WEIGHT,
                "vector_source": "qdrant_assessed_skill_vectors",
            },
            "role_allocation_version": ROLE_ALLOCATION_VERSION,
            "pdf_question_annotation_version": PDF_QUESTION_ANNOTATION_VERSION,
            "topic_coverage_mode": (
                request[
                    "topic_coverage_mode"
                ]
            ),
            "required_official_references": (
                request[
                    "required_official_references"
                ]
            ),
        }

        with Session(engine) as session:
            session.execute(
                retrieval_runs.insert().values(
                    id=retrieval_run_id,
                    retrieval_version=(
                        RETRIEVAL_VERSION
                    ),
                    specification_code=(
                        SPECIFICATION_CODE
                    ),
                    specification_version=(
                        SPECIFICATION_VERSION
                    ),
                    embedding_model=MODEL_NAME,
                    qdrant_collection=(
                        AGENT2_COLLECTION
                    ),
                    agent1_input=json_safe(
                        AGENT1_TOPIC_OUTPUT
                    ),
                    assessment_request=json_safe(
                        request
                    ),
                    lesson_summary=(
                        LESSON_SUMMARY
                    ),
                    raw_candidate_count=len(
                        all_candidates_df
                    ),
                    duplicate_count=(
                        duplicates_removed
                        + near_duplicates_removed
                    ),
                    unique_candidate_count=len(
                        near_unique_candidates_df
                    ),
                    selected_question_count=len(
                        selected_candidates_df
                    ),
                    selected_total_marks=int(
                        selected_candidates_df[
                            "marks"
                        ].sum()
                    ),
                    phase2_version=(
                        PHASE2_VERSION
                    ),
                    phase2_refinement_version=(
                        PHASE2_REFINEMENT_VERSION
                    ),
                    phase2_configuration=json_safe(
                        phase2_configuration
                    ),
                    near_duplicate_count=(
                        near_duplicates_removed
                    ),
                    phase2_eligible_count=len(
                        phase2_candidates_df
                    ),
                    phase2_rejected_count=len(
                        phase2_rejected_df
                    ),
                    assessment_release_status=(
                        selection_summary[
                            "assessment_release_status"
                        ]
                    ),
                    marks_difference=None,
                    phase3_version=(
                        PHASE3_VERSION
                    ),
                    phase3_review_count=(
                        phase3_review_count
                    ),
                    final_release_blockers=json_safe(
                        selection_summary[
                            "release_blockers"
                        ]
                    ),
                    status=(
                        "partial_success"
                        if AGENT2_RUN_STATUS == "partial_success"
                        else "completed"
                    ),
                    started_at=now_utc,
                    completed_at=datetime.now(
                        timezone.utc
                    ),
                )
            )

            for _, row in (
                phase2_gate_manifest_df
                .iterrows()
            ):
                question_id = str(
                    row["question_id"]
                )

                session.execute(
                    retrieval_results
                    .insert()
                    .values(
                        id=uuid.uuid4(),
                        retrieval_run_id=(
                            retrieval_run_id
                        ),
                        question_id=uuid.UUID(
                            question_id
                        ),
                        detected_topic=(
                            row["detected_topic"]
                        ),
                        agent1_role=(
                            row["agent1_role"]
                        ),
                        requested_official_reference=(
                            row[
                                "requested_official_reference"
                            ]
                        ),
                        retrieval_stage=(
                            row[
                                "retrieval_stage"
                            ]
                        ),
                        semantic_score=float(
                            row["semantic_score"]
                        ),
                        final_score=float(
                            row["final_score"]
                        ),
                        raw_rank=int(
                            row["raw_rank"]
                        ),
                        unique_rank=int(
                            row["unique_rank"]
                        ),
                        selected=(
                            question_id
                            in selected_ids_set
                        ),
                        selected_rank=(
                            selected_rank_lookup.get(
                                question_id
                            )
                        ),
                        query_evidence_source=(
                            row[
                                "query_evidence_source"
                            ]
                        ),
                        phase2_semantic_threshold=(
                            float(
                                row[
                                    "phase2_semantic_threshold"
                                ]
                            )
                            if pd.notna(
                                row[
                                    "phase2_semantic_threshold"
                                ]
                            )
                            else None
                        ),
                        concept_gate_passed=bool(
                            row[
                                "concept_gate_passed"
                            ]
                        ),
                        question_quality_gate_passed=bool(
                            row[
                                "question_quality_gate_passed"
                            ]
                        ),
                        phase2_gate_passed=bool(
                            row[
                                "phase2_gate_passed"
                            ]
                        ),
                        question_quality_issues=json_safe(
                            row[
                                "question_quality_issues"
                            ]
                        ),
                        phase2_rejection_reasons=json_safe(
                            row[
                                "phase2_rejection_reasons"
                            ]
                        ),
                        created_at=now_utc,
                    )
                )

            if (
                PHASE3_STORE_AUDIT_RECORDS
            ):
                for _, row in (
                    final_df.iterrows()
                ):
                    session.execute(
                        mark_scheme_cleanup_results
                        .insert()
                        .values(
                            id=uuid.uuid4(),
                            retrieval_run_id=(
                                retrieval_run_id
                            ),
                            question_id=uuid.UUID(
                                str(
                                    row[
                                        "question_id"
                                    ]
                                )
                            ),
                            mark_scheme_id=uuid.UUID(
                                str(
                                    row[
                                        "mark_scheme_id"
                                    ]
                                )
                            ),
                            cleanup_version=(
                                PHASE3_VERSION
                            ),
                            cleanup_status=(
                                row[
                                    "phase3_cleanup_status"
                                ]
                            ),
                            rule_confidence=float(
                                row[
                                    "phase3_rule_confidence"
                                ]
                            ),
                            structured_payload=json_safe(
                                {
                                    "marking_points": (
                                        row[
                                            "phase3_marking_points"
                                        ]
                                    ),
                                    "acceptable_answers": (
                                        row[
                                            "phase3_acceptable_answers"
                                        ]
                                    ),
                                    "rejected_answers": (
                                        row[
                                            "phase3_rejected_answers"
                                        ]
                                    ),
                                    "additional_guidance": (
                                        row[
                                            "phase3_additional_guidance"
                                        ]
                                    ),
                                    "worked_examples": (
                                        row[
                                            "phase3_worked_examples"
                                        ]
                                    ),
                                    "assessment_objectives": (
                                        row[
                                            "phase3_assessment_objectives"
                                        ]
                                    ),
                                    "block_parser_version": row[
                                        "phase3_block_parser_version"
                                    ],
                                    "block_count": int(row["phase3_block_count"]),
                                    "continuation_lines_merged": int(
                                        row["phase3_continuation_lines_merged"]
                                    ),
                                    "inline_markers_split": int(
                                        row["phase3_inline_markers_split"]
                                    ),
                                    "implicit_marking_block_count": int(
                                        row["phase3_implicit_marking_block_count"]
                                    ),
                                    "ambiguous_lines": row[
                                        "phase3_ambiguous_lines"
                                    ],
                                    "block_audit": row["phase3_block_audit"],
                                }
                            ),
                            review_reasons=json_safe(
                                row[
                                    "phase3_review_reasons"
                                ]
                            ),
                            created_at=now_utc,
                        )
                    )

            session.commit()

        print(
            f"Retrieval audit run stored: "
            f"{retrieval_run_id}"
        )

    else:
        print(
            "Retrieval logging disabled."
        )


## 16. Final completion checks


In [ ]:
if globals().get("AGENT2_NO_SAFE_CANDIDATES", False):
    if not globals().get("AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN", False):
        print("Downstream assessment rendering/export was skipped because no quality-safe questions match the current request. See the user-friendly run summary above.")
        AGENT2_DOWNSTREAM_SKIP_NOTICE_SHOWN = True
else:
    actual_agent1_chunk_evidence_used = bool(
        fallback_evidence_topic_count
        == 0
    )

    phase3_manifest_created = bool(
        phase3_manifest_path.exists()
    )

    phase3_json_created = bool(
        phase3_json_path.exists()
    )

    phase3_raw_guidance_preserved = bool(
        final_df[
            "phase3_raw_guidance_preserved"
        ].all()
    )

    phase3_block_parser_current = bool(
        final_df["phase3_block_parser_version"].eq(
            PHASE3_BLOCK_PARSER_VERSION
        ).all()
    )
    phase3_block_metrics_present = bool(
        final_df[
            [
                "phase3_block_count",
                "phase3_continuation_lines_merged",
                "phase3_inline_markers_split",
                "phase3_implicit_marking_block_count",
            ]
        ].notna().all().all()
    )
    phase3_no_ambiguous_lines = bool(
        final_df["phase3_ambiguous_lines"].map(len).sum() == 0
    )
    phase3_block_validation_passed = bool(
        all(phase3_block_parser_validation.values())
    )

    phase3_cleanup_rows_complete = bool(
        len(
            phase3_cleanup_df
        )
        == len(
            final_df
        )
    )

    phase3_selected_review_count = int(
        (
            final_df[
                "phase3_cleanup_status"
            ]
            == "review_recommended"
        ).sum()
    )

    selected_candidate_ids = set(
        selected_candidates_df[
            "question_id"
        ].astype(str)
    )

    selected_near_duplicate_rows = (
        near_duplicate_manifest_df[
            near_duplicate_manifest_df[
                "question_id"
            ].astype(str).isin(
                selected_candidate_ids
            )
        ]
    )

    selected_near_duplicates_removed = int(
        (
            selected_near_duplicate_rows[
                "near_duplicate_status"
            ]
            == "duplicate_removed"
        ).sum()
    )

    selected_reference_set = set(
        selected_candidates_df[
            "official_reference"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    required_reference_set = set(
        request.get(
            "required_official_references",
            [],
        )
    )

    all_required_references_covered = bool(
        required_reference_set.issubset(
            selected_reference_set
        )
    )

    assessment_ready_for_release = bool(
        selection_summary[
            "assessment_release_status"
        ]
        == "ready_for_release"
    )

    selected_semantic_rescue_count = int(
        selected_candidates_df[
            "semantic_rescue_used"
        ].sum()
    )

    semantic_rescue_decision_consistent = bool(
        (
            selected_semantic_rescue_count == 0
            and selection_summary[
                "assessment_release_status"
            ]
            in {
                "ready_for_release",
                "needs_user_decision",
            }
        )
        or (
            selected_semantic_rescue_count > 0
            and selection_summary[
                "assessment_release_status"
            ]
            == "needs_user_decision"
        )
    )

    selected_phase2_gate_passed = bool(
        selected_candidates_df[
            "phase2_gate_passed"
        ].all()
    )

    selected_exact_detected_concept_passed = bool(
        selected_candidates_df[
            "exact_detected_concept_gate_passed"
        ].astype(bool).all()
    )

    selected_concept_or_rescue_passed = bool(
        (
            selected_candidates_df[
                "concept_gate_passed"
            ]
            | selected_candidates_df[
                "semantic_rescue_used"
            ]
        ).all()
    )

    selected_adaptive_thresholds_present = bool(
        selected_candidates_df[
            "phase2_semantic_threshold"
        ].notna().all()
    )

    selected_adaptive_thresholds_within_bounds = bool(
        (
            selected_candidates_df[
                "phase2_semantic_threshold"
            ]
            >= 0.0
        ).all()
        and (
            selected_candidates_df[
                "phase2_semantic_threshold"
            ]
            <= 1.0
        ).all()
    )

    adaptive_profile_complete = bool(
        len(
            adaptive_threshold_profile_df
        )
        == validated_topics_df[
            "agent1_topic_index"
        ].nunique()
    )

    legacy_fixed_thresholds_inactive = bool(
        not LEGACY_FIXED_THRESHOLD_APPROACH_ENABLED
    )

    selected_quality_gate_passed = bool(
        selected_candidates_df[
            "question_quality_gate_passed"
        ].all()
    )

    selected_quality_issue_count = int(
        selected_candidates_df[
            "question_quality_issues"
        ].map(len).sum()
    )

    selected_supporting_count = int(
        (
            selected_candidates_df["agent1_role"]
            == "supporting"
        ).sum()
    )

    selected_distinct_reference_count = int(
        selected_candidates_df[
            "official_reference"
        ].nunique()
    )

    selected_visual_mask = (
        final_df[
            "visual_render_required"
        ].astype(bool)
    )

    visual_image_lists = (
        final_df.loc[
            selected_visual_mask,
            "rendered_page_images",
        ]
    )

    all_required_visuals_rendered = bool(
        final_df.loc[
            selected_visual_mask,
            "visual_render_status",
        ].isin(
            [
                "rendered_dependency_complete",
            ]
        ).all()
    )

    all_rendered_image_files_exist = bool(
        all(
            (
                OUTPUT_DIR
                / relative_path
            ).exists()
            for image_list
            in visual_image_lists
            if isinstance(image_list, list)
            for relative_path in image_list
        )
    )

    selected_texts = (
        final_df[
            "question_text_postgres"
        ].map(normalise_text)
    )

    # --------------------------------------------------------------
    # Notebook 07 dependency-resolution readiness
    # --------------------------------------------------------------
    # Define this BEFORE the final checks dictionary so the release-readiness
    # checks can always reference it.
    if (
        "missing_dependency_labels"
        in final_df.columns
    ):
        selected_required_dependencies_complete = bool(
            final_df[
                "missing_dependency_labels"
            ].map(
                lambda value: (
                    len(value) == 0
                    if isinstance(
                        value,
                        list,
                    )
                    else True
                )
            ).all()
        )
    else:
        # Compatibility only. Runs using Notebook 07 v1.1 will have this
        # column; other visual safety checks remain active independently.
        selected_required_dependencies_complete = True

    # --------------------------------------------------------------
    # Retrieval HITL Phase 4 bounded-ranking readiness
    # --------------------------------------------------------------
    phase4_adjustment_bound_respected = bool(
        final_df.get(
            "memory_rank_adjustment",
            pd.Series(0.0, index=final_df.index),
        ).fillna(0.0).astype(float).between(
            -RETRIEVAL_MEMORY_NEGATIVE_MAX_PENALTY - 1e-12,
            RETRIEVAL_MEMORY_POSITIVE_MAX_BOOST + 1e-12,
        ).all()
    )

    phase4_baseline_score_preserved = bool(
        "final_score" in final_df.columns
        and "memory_adjusted_final_score" in final_df.columns
    )

    phase4_incompatible_candidates_unchanged = bool(
        (
            final_df.loc[
                ~final_df.get(
                    "memory_match_found",
                    pd.Series(False, index=final_df.index),
                ).fillna(False).astype(bool),
                "memory_rank_adjustment",
            ]
            .fillna(0.0)
            .astype(float)
            .abs()
            <= 1e-12
        ).all()
    )

    phase4_no_exact_context_rejected_question_selected = bool(
        not selected_candidates_df.get(
            "memory_hard_suppressed",
            pd.Series(False, index=selected_candidates_df.index),
        ).fillna(False).astype(bool).any()
    )

    checks = {
        "agent1_topics_validated": bool(
            not validated_topics_df.empty
            and validated_topics_df[
                "official_reference"
            ].notna().all()
            and (
                not request.get("paper_code")
                or validated_topics_df[
                    "syllabus_paper_matches_request"
                ].astype(bool).all()
            )
        ),
        "all_official_references_known": (
            unknown_topics_df.empty
        ),
        "qdrant_collection_available": (
            AGENT2_COLLECTION
            in collection_names
        ),
        "vector_size_is_384": (
            VECTOR_SIZE
            == EXPECTED_VECTOR_SIZE
        ),
        "phase4_adjustment_bound_respected": (
            phase4_adjustment_bound_respected
        ),
        "phase4_baseline_score_preserved": (
            phase4_baseline_score_preserved
        ),
        "phase4_incompatible_candidates_unchanged": (
            phase4_incompatible_candidates_unchanged
        ),
        "phase4_no_exact_context_rejected_question_selected": (
            phase4_no_exact_context_rejected_question_selected
        ),
        "exact_candidates_found": (
            len(exact_candidates_df) > 0
        ),
        "quality_safe_selection_not_empty": (
            len(selected_candidates_df) > 0
        ),
        "question_count_shortfall_documented": (
            selection_summary["requested_question_count_met"]
            or (
                selection_summary["question_count_shortfall"] > 0
                and "requested_question_count_not_achieved"
                in selection_summary.get("release_blockers", [])
            )
        ),
        "all_selected_have_mark_schemes": (
            final_df[
                "mark_scheme_id"
            ].notna().all()
        ),
        "no_duplicate_selected_questions": (
            selected_texts.nunique()
            == len(final_df)
        ),
        "candidate_csv_created": (
            candidates_path.exists()
        ),
        "selected_csv_created": (
            selected_path.exists()
        ),
        "json_package_created": (
            package_path.exists()
        ),
        "markdown_package_created": (
            markdown_path.exists()
        ),
        "text_evaluation_report_created": (
            text_report_path.exists()
        ),
        "student_question_paper_pdf_created": (
            student_pdf_path.exists()
            and student_pdf_path.stat().st_size > 0
        ),
        "teacher_mark_scheme_pdf_created": (
            teacher_pdf_path.exists()
            and teacher_pdf_path.stat().st_size > 0
        ),
        "combined_questions_and_answers_pdf_created": (
            combined_pdf_path.exists()
            and combined_pdf_path.stat().st_size > 0
        ),
        "all_pdfs_render_verified": bool(
            all(
                record.get("render_verified")
                for record in [
                    pdf_verification["student_pdf"],
                    pdf_verification["teacher_pdf"],
                    pdf_verification["combined_audit_pdf"],
                ]
            )
        ),
        "near_duplicate_manifest_created": (
            near_duplicate_manifest_path.exists()
        ),
        "release_readiness_report_created": (
            release_readiness_path.exists()
        ),
        "no_removed_near_duplicate_selected": (
            selected_near_duplicates_removed
            == 0
        ),
        "required_reference_coverage_handled": (
            all_required_references_covered
            or (
                "required_official_reference_not_covered"
                in selection_summary.get(
                    "release_blockers",
                    [],
                )
            )
        ),
        "semantic_rescue_release_decision_consistent": (
            semantic_rescue_decision_consistent
        ),
        "phase2_gate_manifest_created": (
            phase2_gate_manifest_path.exists()
        ),
        "phase2_query_evidence_manifest_created": (
            phase2_query_evidence_path.exists()
        ),
        "all_selected_pass_phase2_gate": (
            selected_phase2_gate_passed
        ),
        "all_selected_pass_exact_detected_concept_gate": (
            selected_exact_detected_concept_passed
        ),
        "adaptive_threshold_profile_created": (
            adaptive_threshold_profile_path.exists()
        ),
        "adaptive_threshold_profile_complete": (
            adaptive_profile_complete
        ),
        "legacy_fixed_thresholds_inactive": (
            legacy_fixed_thresholds_inactive
        ),
        "selected_adaptive_thresholds_present": (
            selected_adaptive_thresholds_present
        ),
        "selected_adaptive_thresholds_within_bounds": (
            selected_adaptive_thresholds_within_bounds
        ),
        "all_selected_pass_adaptive_gate_or_documented_rescue": (
            selected_concept_or_rescue_passed
        ),
        "all_selected_pass_quality_gate": (
            selected_quality_gate_passed
        ),
        "selected_questions_have_no_quality_issues": (
            selected_quality_issue_count == 0
        ),
        "primary_coverage_requirement_handled": (
            selection_summary.get(
                "primary_requirement_met",
                False,
            )
            or (
                "minimum_primary_coverage_not_met"
                in selection_summary.get(
                    "release_blockers",
                    [],
                )
            )
        ),
        "supporting_coverage_requirement_handled": (
            selection_summary.get(
                "supporting_requirement_met",
                False,
            )
            or (
                "minimum_supporting_coverage_not_met"
                in selection_summary.get(
                    "release_blockers",
                    [],
                )
            )
        ),
        "distinct_reference_requirement_handled": (
            selection_summary.get(
                "distinct_reference_requirement_met",
                False,
            )
            or (
                "minimum_distinct_reference_coverage_not_met"
                in selection_summary.get(
                    "release_blockers",
                    [],
                )
            )
        ),
        "phase2_full_instruction_segment_scan_active": (
            "probable_truncated_instruction_segment"
            in detect_question_quality_issues(
                (
                    "Complete the requested boxes and. "
                    "Figure 7"
                )
            )
        ),
        "concept_fit_manifest_created": (
            concept_fit_manifest_path.exists()
        ),
        "concept_fit_profile_created": (
            concept_fit_profile_path.exists()
        ),
        "all_selected_pass_direct_concept_fit_or_documented_rescue": bool(
            (
                selected_candidates_df["concept_fit_gate_passed"]
                | selected_candidates_df["concept_fit_rescue_used"]
            ).all()
        ),
        "context_refinement_manifest_created": (
            context_refinement_manifest_path.exists()
        ),
        "question_group_manifest_created": (
            question_group_manifest_path.exists()
        ),
        "all_effective_visual_dependencies_complete": bool(
            final_df.loc[
                selected_visual_mask,
                "visual_dependency_complete",
            ].astype(bool).all()
        ),
        "all_structured_response_layouts_verified": bool(
            final_df.loc[
                final_df[
                    "structured_response_required"
                ].astype(bool),
                "structured_layout_verified",
            ].astype(bool).all()
        ),
        "no_missing_required_dependency_labels": bool(
            final_df[
                "missing_dependency_labels"
            ].map(
                lambda value: len(
                    value
                )
                if isinstance(
                    value,
                    list,
                )
                else 0
            ).sum()
            == 0
        ),
        "student_release_contains_only_dependency_safe_questions": bool(
            final_df.loc[
                final_df[
                    "student_release_eligible"
                ].astype(bool),
                "visual_dependency_complete",
            ].astype(bool).all()
        ),
        "visual_crop_or_safe_fallback_for_every_visual": bool(
            final_df.loc[
                selected_visual_mask,
                "question_region_crop_status",
            ].isin(
                [
                    "cropped_verified_with_dependencies",
                    "multi_page_cropped_verified_with_dependencies",
                    "full_page_fallback",
                ]
            ).all()
        ),
        "all_required_figure_table_dependencies_resolved": bool(
            (
                final_df[
                    "missing_dependency_labels"
                ].map(
                    lambda value: (
                        len(value) == 0
                        if isinstance(
                            value,
                            list,
                        )
                        else True
                    )
                ).all()
            )
            if (
                "missing_dependency_labels"
                in final_df.columns
            )
            else True
        ),
        "phase3_manifest_created": (
            phase3_manifest_created
        ),
        "phase3_json_report_created": (
            phase3_json_created
        ),
        "phase3_cleanup_rows_complete": (
            phase3_cleanup_rows_complete
        ),
        "phase3_raw_guidance_preserved": (
            phase3_raw_guidance_preserved
        ),
        "phase3_block_parser_current": (
            phase3_block_parser_current
        ),
        "phase3_block_metrics_present": (
            phase3_block_metrics_present
        ),
        "phase3_block_validation_passed": (
            phase3_block_validation_passed
        ),
        "phase3_no_ambiguous_lines": (
            phase3_no_ambiguous_lines
        ),
        "visual_render_manifest_created": (
            visual_manifest_path.exists()
        ),
        "all_selected_visuals_rendered": (
            all_required_visuals_rendered
        ),
        "no_single_page_top_level_visual_spill": bool(
            final_df.apply(
                lambda row: (
                    True
                    if (
                        not bool(
                            row.get(
                                "visual_render_required",
                                False,
                            )
                        )
                        or not isinstance(
                            row.get(
                                "assigned_source_pages"
                            ),
                            list,
                        )
                        or len(
                            row.get(
                                "assigned_source_pages"
                            )
                        )
                        != 1
                        or bool(
                            row.get(
                                "multi_page_question",
                                False,
                            )
                        )
                        or not (
                            row.get(
                                "structural_parent_question_number"
                            )
                            is None
                            or (
                                isinstance(
                                    row.get(
                                        "structural_parent_question_number"
                                    ),
                                    float,
                                )
                                and np.isnan(
                                    row.get(
                                        "structural_parent_question_number"
                                    )
                                )
                            )
                        )
                    )
                    else set(
                        row.get(
                            "source_page_numbers"
                        )
                        if isinstance(
                            row.get(
                                "source_page_numbers"
                            ),
                            list,
                        )
                        else []
                    ).issubset(
                        set(
                            row.get(
                                "assigned_source_pages"
                            )
                        )
                    )
                ),
                axis=1,
            ).all()
        ),
        "no_unanchored_dependency_fallback_in_student_pdf": bool(
            ~final_df[
                "dependency_render_status"
            ].astype(str).eq(
                "unanchored_dependency_rejected"
            ).any()
        ),
        "visual_post_render_sanitization_manifest_created": (
            visual_sanitization_manifest_path.exists()
        ),
        "all_rendered_image_files_exist": (
            all_rendered_image_files_exist
        ),
        "all_selected_have_credible_topic_owner": bool(
            selected_candidates_df.get(
                "ownership_credible_match",
                pd.Series(True, index=selected_candidates_df.index),
            ).astype(bool).all()
        ),
        "all_selected_pass_child_direct_evidence": bool(
            selected_candidates_df.get(
                "child_direct_evidence_passed",
                pd.Series(True, index=selected_candidates_df.index),
            ).astype(bool).all()
        ),
        "all_selected_pass_final_relevance_verifier": bool(
            selected_candidates_df.get(
                "final_relevance_verifier_passed",
                pd.Series(True, index=selected_candidates_df.index),
            ).astype(bool).all()
        ),
        "all_selected_pass_final_general_quality_gate": bool(
            selected_candidates_df.get(
                "final_quality_gate_passed",
                pd.Series(True, index=selected_candidates_df.index),
            ).astype(bool).all()
        ),
        "no_selected_no_owner_questions": bool(
            not selected_candidates_df.get(
                "question_ownership_status",
                pd.Series("owner", index=selected_candidates_df.index),
            ).astype(str).eq("no_owner").any()
        ),
        "qdrant_vectors_unchanged_by_phase_1_and_2": (
            qdrant_point_count
            == EXPECTED_QDRANT_POINTS
        ),
    }

    if STORE_RETRIEVAL_LOGS:
        checks["retrieval_run_logged"] = (
            retrieval_run_id is not None
        )


    checks_df = pd.DataFrame(
        [
            {
                "check": name,
                "passed": bool(value),
            }
            for name, value
            in checks.items()
        ]
    )

    display(checks_df)

    notebook_05_complete = all(
        checks.values()
    )

    release_checks = {
        "semantic_rescue_selected": (
            selected_semantic_rescue_count
            > 0
        ),
        "actual_agent1_chunk_evidence_used": (
            actual_agent1_chunk_evidence_used
        ),
        "phase3_review_recommended_count": (
            phase3_selected_review_count
        ),
        "assessment_ready_for_release": (
            assessment_ready_for_release
        ),
    }

    release_checks_df = pd.DataFrame(
        [
            {
                "release_check": name,
                "passed": bool(value),
            }
            for name, value
            in release_checks.items()
        ]
    )

    print(
        f"Notebook 05 technically complete: "
        f"{notebook_05_complete}"
    )

    print(
        "Assessment release status: "
        f"{selection_summary['assessment_release_status']}"
    )

    print(
        "Agent 2 run status: "
        f"{AGENT2_RUN_STATUS}"
    )

    if AGENT2_USER_MESSAGES:
        print_agent2_user_messages(
            "FINAL USER-FRIENDLY NOTICE"
        )

    display(release_checks_df)

    display(
        pd.DataFrame(
            [retrieval_summary]
        )
    )


# Notebook 05 — retrieval pipeline + HITL retrieval-memory status

## Technical completion

A successful run should show:

```text
Agent 1 topics validated                                  True
official references known                                True
Qdrant collection available                              True
MiniLM vector size remains 384                            True

near-duplicate manifest created                          True
no removed near duplicate selected                       True

adaptive threshold profile created                       True
legacy fixed 0.60 / 0.55 thresholds inactive            True
selected adaptive thresholds within safety bounds       True

full-question instructional-segment scan active          True
all selected questions pass text-quality gate            True
all required approved references covered                 True

all visual-question pages rendered                       True

Phase 3 manifest created                                 True
Phase 3 JSON report created                              True
one cleanup row per selected mark scheme                 True
raw mark-scheme guidance preserved                       True
Phase 3 block parser version current                     True
wrapped-line validation test passes                      True
block-level audit metrics present                        True
no ambiguous mark-scheme lines remain                    True

JSON/CSV/Markdown/TXT outputs created                    True
retrieval and cleanup audit records stored               True
Qdrant point count remains 820                           True

Notebook 05 technically complete                         True
```

## Release-state interpretation

```text
ready_for_release
    marks are within tolerance
    no semantic rescue was selected
    actual Agent 1 source chunk text was used
    Phase 3 does not recommend review

evaluation_ready
    technical retrieval is complete
    lesson-summary fallback was used or Phase 3 recommends review

needs_user_decision
    marks are outside tolerance or semantic rescue was selected
```

The current standalone sample normally remains `evaluation_ready` because it does
not yet receive actual source chunk text from Agent 1 Streamlit.

## Phase 3 interpretation

```text
raw marking_guidance             source of truth
legacy structured fields         retained for comparison
Phase 3 structured fields        cleaner interface view
review_recommended               requires human checking
structured_ready                 suitable for evaluation display
```

## Remaining work

```text
Actual Agent 1 source-chunk integration test             Streamlit integration
Adaptive-parameter comparison                            Notebook 06
Human retrieval relevance capture                        Phase 1 complete in Streamlit + PostgreSQL
Contextual feedback embedding + Qdrant memory storage      Phase 2 implemented after Streamlit feedback
Qdrant memory lookup + compatibility diagnostics            HITL Phase 3 implemented
Memory-aware ranking + exact-context rejection suppression   Phase 4 implemented
Phase 3 rule evaluation across more mark schemes         Notebook 06 / review
```

## Final Phase 3 refinement status

```text
Line-by-line classification approach                    tested
Wrapped-line classification issue                       documented
Inline A. / I. / R. marker splitting                    implemented
Logical block construction                              implemented
Guidance continuation merging                           implemented
Marking-point continuation merging                      implemented
Worked-example boundaries                               preserved
Synthetic wrapped-line validation                       added
Raw guidance preservation                               retained
```

After this notebook is verified, the next development step is Agent 1 Streamlit and
Agent 2 retrieval integration.


## Resilient retrieval and PDF completion

```text
Strict exact-reference Qdrant retrieval                  retained
Legacy Qdrant payload compatibility                      implemented
Optional-filter relaxation                               documented and human-gated
PostgreSQL exact-reference MiniLM fallback               implemented
Official-reference constraint                            never removed
Generic zero-candidate crash                              replaced by controlled recovery
Combined question + mark-scheme PDF                      implemented
PDF path written back into assessment package JSON       implemented
```


## Final generalized refinement criteria

```text
target-marks feasibility is calculated before release              True
weak questions are removed by per-topic concept distributions      True
no question IDs or topic-specific exclusions are used              True
larger questions require a bounded extra relevance margin          True
question-count shortfalls are reported instead of weak filling     True
context is segmented and checked against the current question      True
original context remains preserved                                 True
shared source-parent subquestions are grouped for presentation     True
visual regions are cropped from source-document boundaries         True
full-page fallback prevents accidental clipping                    True
student and teacher PDFs are generated separately                  True
combined audit PDF remains available                               True
PDF files are reopened and render-verified                         True
math/punctuation cleanup is display-only                           True
raw question and mark-scheme text remains unchanged                True
```

These thresholds remain provisional until evaluated across multiple official
topics in Notebook 06. The system deliberately prefers a documented shortfall
over silently adding a weak candidate.


## Issue 2 verification criteria

A successful source-page matching run should show:

```text
Every rendered image has source_page_match_status = verified_question_page
Every successful crop has final_crop_anchor_verified = True
No successful student image uses a full_page_fallback
The final crop contains the selected question text
The crop ends before the next unselected question/subquestion
```

A failed question-page match remains visible in the manifest and is not replaced
with an unrelated complete page.


## Issue 3 verification criteria

A successful dependency-resolution run should show:

```text
Questions mentioning Figure/Table/Diagram:
- required_dependency_labels is populated
- every label is resolved
- dependency crop is rendered before the question crop

Questions requiring blank arrays/tables/grids:
- structured_response_required = True
- structured_layout_verified = True
- original source layout is used in the student paper

Student release:
- visual_dependency_complete = True
- student_release_eligible = True
- unresolved dependency questions do not enter the student PDF
```

The dependency search is bounded around the already verified question page and
does not alter retrieval or question selection.


## Issue 3B spatial-binding regression criteria

```text
Every resolved dependency:
- has a standalone label
- has a recorded structure_region
- has a spatial_direction
- records the boundary that ended its region
- contains structure inside that bounded region

A label immediately followed by another question:
- cannot use that question's boxes, lozenges or answer lines
- remains unresolved when its own region is empty

The resolver never extends a region through the next detected boundary
to manufacture the minimum crop height.
```


## Retrieval HITL Phase 3 status

```text
Dedicated Qdrant retrieval-memory lookup                  implemented
Candidate-level recall                                     implemented
Official-reference compatibility guard                     implemented
Primary/supporting role compatibility guard                implemented
Topic semantic compatibility guard                         implemented
Lesson-evidence semantic compatibility guard               implemented
Question exact-ID / semantic compatibility guard           implemented
Top recalled-memory diagnostics exported                   implemented
Memory ranking adjustment                                  0.0 (forced)
Existing final_score                                       unchanged
Existing phase2_rank                                       unchanged
Selection logic                                             unchanged
```

Phase 3 is diagnostic-only. A recalled `relevant` or `not_relevant` memory is visible in the assessment package/UI but cannot boost or penalise a question until Phase 4.


## Retrieval HITL Phase 4

Phase 4 activates bounded memory-aware ranking after the Phase 3 compatibility
guard. The original `final_score` and `phase2_rank` remain unchanged for audit.
Only `memory_adjusted_final_score` is used as the bounded memory-aware ranking
input inside the existing quality-safe selector.

- Relevant compatible memory: confidence-scaled boost, maximum `+0.04`.
- Exact same question + exact same lesson evidence + Not Relevant:
  excluded from final selection for that lesson context only.
- Broader compatible Not Relevant memory: confidence-scaled penalty,
  maximum `-0.04`.
- No compatible memory: `0.00` adjustment.
- Conflicting compatible decisions: fail closed with `0.00` adjustment.
- Suppressed questions remain in the knowledge base and may appear in a
  different lesson where Phase 3 judges the old memory incompatible.
- Phase 2 quality/paper/concept gates are never bypassed.

The student-PDF post-render safety layer also treats a verified selected-question
crop as self-contained when no dependency labels are declared, no dependency is
missing, no separate context is required, and the question text has no explicit
external Figure/Table/diagram-style reference. Genuine external dependencies
continue to fail closed.


## Retrieval HITL Phase 5

```text
End-to-end memory policy evaluation                         implemented
Current-run candidate/selection invariants                  implemented
Synthetic positive/negative/conflict/no-memory regressions  implemented
Phase 3 thresholds snapshotted                              0.50 / 0.90 / 0.75 / 0.78
Phase 4 score bounds snapshotted                            +0.04 / -0.04
Exact-context Not Relevant policy                           hard suppress same lesson only
Different-context memory policy                             no effect unless Phase 3 compatible
Automatic threshold tuning                                  disabled
Phase 5 evaluation JSON + checks CSV                        implemented
Streamlit lock-status display                               implemented
```

Phase 5 adds no new ranking heuristic. A run is marked `locked` only when all
critical automated invariants and deterministic policy regressions pass. Threshold
changes after this point require an explicit code/version change and a repeat of the
positive- and negative-control validation protocol.
